In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Medium_accounts.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/LI-Small_Trans.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Large_accounts.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Large_Trans.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Medium_Trans.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/LI-Medium_accounts.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Patterns.txt
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Medium_Patterns.txt
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_accounts.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-la

In [2]:
# ============================================================
# CELL 1 — ORIGINAL HI-SMALL DISTRIBUTION DNA AUDIT
#
# SOURCE DATA ONLY:
#   HI-Small_Trans.csv
#   HI-Small_accounts.csv
#   HI-Small_Patterns.txt
#
# IMPORTANT:
#   - Does NOT use augmented data
#   - Does NOT modify any source files
#   - This becomes the baseline for the NEW generator
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import json
import re
import gc
import os

# ============================================================
# 0. LOCATE ORIGINAL IBM HI-SMALL FILES
# ============================================================

IBM_ROOT = Path(
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml"
)

if not IBM_ROOT.exists():
    raise FileNotFoundError(
        f"IBM dataset root not found:\n{IBM_ROOT}"
    )

TX_PATH = IBM_ROOT / "HI-Small_Trans.csv"
ACCOUNTS_PATH = IBM_ROOT / "HI-Small_accounts.csv"
PATTERNS_PATH = IBM_ROOT / "HI-Small_Patterns.txt"

for p in [TX_PATH, ACCOUNTS_PATH, PATTERNS_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{p}"
        )

print("=" * 110)
print("ORIGINAL HI-SMALL DISTRIBUTION DNA AUDIT")
print("=" * 110)

print("Transactions :", TX_PATH)
print("Accounts     :", ACCOUNTS_PATH)
print("Patterns     :", PATTERNS_PATH)
print()

# ============================================================
# 1. BASIC FILE INFORMATION
# ============================================================

print("=" * 110)
print("1. FILE INFORMATION")
print("=" * 110)

print(
    f"Transactions size : "
    f"{TX_PATH.stat().st_size / (1024**3):.3f} GB"
)

print(
    f"Accounts size     : "
    f"{ACCOUNTS_PATH.stat().st_size / (1024**2):.3f} MB"
)

print(
    f"Patterns size     : "
    f"{PATTERNS_PATH.stat().st_size / (1024**2):.3f} MB"
)

# ============================================================
# 2. LOAD HEADER
# ============================================================

header = pd.read_csv(
    TX_PATH,
    nrows=0
)

print()
print("Transaction columns:")
for i, c in enumerate(header.columns, 1):
    print(f"  {i:2d}. {c}")

# ============================================================
# 3. CONFIGURATION
# ============================================================

CHUNK_SIZE = 250_000

TX_COLS = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering"
]

# Reservoir sample size per group for approximate quantiles.
# This avoids storing all 5M amounts in memory.
RESERVOIR_SIZE = 250_000

rng = np.random.default_rng(20260919)

# ============================================================
# 4. COUNTERS / CONTAINERS
# ============================================================

total_rows = 0
legitimate_rows = 0
laundering_rows = 0

timestamp_min = None
timestamp_max = None

# categorical distributions
payment_format_counts = Counter()
receiving_currency_counts = Counter()
payment_currency_counts = Counter()

# time distributions
hour_counts = Counter()
weekday_counts = Counter()
date_counts = Counter()

# bank distributions
from_bank_counts = Counter()
to_bank_counts = Counter()
bank_pair_counts = Counter()

# account distributions
source_account_counts = Counter()
destination_account_counts = Counter()

# account-by-bank combinations
source_node_counts = Counter()
destination_node_counts = Counter()

# account string structure
account_length_counts = Counter()
bank_length_counts = Counter()

account_character_classes = Counter()

# amount diagnostics
amount_received_min = np.inf
amount_received_max = -np.inf
amount_paid_min = np.inf
amount_paid_max = -np.inf

amount_received_nan = 0
amount_paid_nan = 0

amount_ratio_equal = 0
amount_ratio_not_equal = 0

# exact duplicates
duplicate_rows_seen = 0

# ============================================================
# 5. RESERVOIR SAMPLING FUNCTION
# ============================================================

reservoirs = {
    "REAL_LEGITIMATE_received": [],
    "REAL_LAUNDERING_received": [],
    "REAL_LEGITIMATE_paid": [],
    "REAL_LAUNDERING_paid": [],
}

reservoir_seen = {
    k: 0
    for k in reservoirs
}


def reservoir_update(
    reservoir,
    seen_count,
    values,
    max_size=RESERVOIR_SIZE
):
    """
    Reservoir sampling.
    Returns updated reservoir + total seen count.
    """

    values = np.asarray(
        values,
        dtype=np.float64
    )

    values = values[
        np.isfinite(values)
    ]

    for value in values:

        seen_count += 1

        if len(reservoir) < max_size:
            reservoir.append(float(value))

        else:
            j = rng.integers(
                0,
                seen_count
            )

            if j < max_size:
                reservoir[j] = float(value)

    return reservoir, seen_count


# ============================================================
# 6. CHUNKED TRANSACTION AUDIT
# ============================================================

print()
print("=" * 110)
print("2. SCANNING ORIGINAL TRANSACTIONS")
print("=" * 110)

for chunk_no, chunk in enumerate(
    pd.read_csv(
        TX_PATH,
        usecols=TX_COLS,
        dtype=str,
        keep_default_na=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    total_rows += len(chunk)

    # --------------------------------------------------------
    # Normalize flags
    # --------------------------------------------------------

    laundering_mask = (
        chunk["Is Laundering"]
        .astype(str)
        .str.strip()
        == "1"
    )

    legitimate_mask = ~laundering_mask

    n_laundering = int(
        laundering_mask.sum()
    )

    n_legitimate = int(
        legitimate_mask.sum()
    )

    laundering_rows += n_laundering
    legitimate_rows += n_legitimate

    # --------------------------------------------------------
    # Exact-ish duplicate count within chunks
    # Full duplicate validation will be done separately if
    # needed; this captures duplicates occurring inside chunks.
    # --------------------------------------------------------

    duplicate_rows_seen += int(
        chunk.duplicated().sum()
    )

    # --------------------------------------------------------
    # Amounts
    # --------------------------------------------------------

    received = pd.to_numeric(
        chunk["Amount Received"],
        errors="coerce"
    )

    paid = pd.to_numeric(
        chunk["Amount Paid"],
        errors="coerce"
    )

    amount_received_nan += int(
        received.isna().sum()
    )

    amount_paid_nan += int(
        paid.isna().sum()
    )

    valid_received = received[
        received.notna()
    ]

    valid_paid = paid[
        paid.notna()
    ]

    if len(valid_received):

        amount_received_min = min(
            amount_received_min,
            float(valid_received.min())
        )

        amount_received_max = max(
            amount_received_max,
            float(valid_received.max())
        )

    if len(valid_paid):

        amount_paid_min = min(
            amount_paid_min,
            float(valid_paid.min())
        )

        amount_paid_max = max(
            amount_paid_max,
            float(valid_paid.max())
        )

    # Reservoirs
    r, seen = reservoir_update(
        reservoirs["REAL_LEGITIMATE_received"],
        reservoir_seen["REAL_LEGITIMATE_received"],
        received.loc[legitimate_mask].values
    )

    reservoirs["REAL_LEGITIMATE_received"] = r
    reservoir_seen["REAL_LEGITIMATE_received"] = seen

    r, seen = reservoir_update(
        reservoirs["REAL_LAUNDERING_received"],
        reservoir_seen["REAL_LAUNDERING_received"],
        received.loc[laundering_mask].values
    )

    reservoirs["REAL_LAUNDERING_received"] = r
    reservoir_seen["REAL_LAUNDERING_received"] = seen

    r, seen = reservoir_update(
        reservoirs["REAL_LEGITIMATE_paid"],
        reservoir_seen["REAL_LEGITIMATE_paid"],
        paid.loc[legitimate_mask].values
    )

    reservoirs["REAL_LEGITIMATE_paid"] = r
    reservoir_seen["REAL_LEGITIMATE_paid"] = seen

    r, seen = reservoir_update(
        reservoirs["REAL_LAUNDERING_paid"],
        reservoir_seen["REAL_LAUNDERING_paid"],
        paid.loc[laundering_mask].values
    )

    reservoirs["REAL_LAUNDERING_paid"] = r
    reservoir_seen["REAL_LAUNDERING_paid"] = seen

    # Amount relationship
    both_valid = (
        received.notna()
        & paid.notna()
    )

    equal_mask = (
        np.isclose(
            received.loc[both_valid].values,
            paid.loc[both_valid].values,
            rtol=0,
            atol=1e-12
        )
    )

    amount_ratio_equal += int(
        equal_mask.sum()
    )

    amount_ratio_not_equal += int(
        (~equal_mask).sum()
    )

    # --------------------------------------------------------
    # Timestamps
    # --------------------------------------------------------

    ts = pd.to_datetime(
        chunk["Timestamp"],
        errors="coerce"
    )

    valid_ts = ts.notna()

    if valid_ts.any():

        local_min = ts.loc[valid_ts].min()
        local_max = ts.loc[valid_ts].max()

        timestamp_min = (
            local_min
            if timestamp_min is None
            else min(timestamp_min, local_min)
        )

        timestamp_max = (
            local_max
            if timestamp_max is None
            else max(timestamp_max, local_max)
        )

        hour_counts.update(
            ts.loc[valid_ts].dt.hour.tolist()
        )

        weekday_counts.update(
            ts.loc[valid_ts].dt.day_name().tolist()
        )

        date_counts.update(
            ts.loc[valid_ts]
            .dt.strftime("%Y-%m-%d")
            .tolist()
        )

    # --------------------------------------------------------
    # Payment / currency
    # --------------------------------------------------------

    payment_format_counts.update(
        chunk["Payment Format"].tolist()
    )

    receiving_currency_counts.update(
        chunk["Receiving Currency"].tolist()
    )

    payment_currency_counts.update(
        chunk["Payment Currency"].tolist()
    )

    # --------------------------------------------------------
    # Banks
    # --------------------------------------------------------

    from_bank_counts.update(
        chunk["From Bank"].tolist()
    )

    to_bank_counts.update(
        chunk["To Bank"].tolist()
    )

    bank_pair_counts.update(
        zip(
            chunk["From Bank"].tolist(),
            chunk["To Bank"].tolist()
        )
    )

    # --------------------------------------------------------
    # Accounts
    # --------------------------------------------------------

    source_account_counts.update(
        chunk["Account"].tolist()
    )

    destination_account_counts.update(
        chunk["Account.1"].tolist()
    )

    source_node_counts.update(
        zip(
            chunk["From Bank"].tolist(),
            chunk["Account"].tolist()
        )
    )

    destination_node_counts.update(
        zip(
            chunk["To Bank"].tolist(),
            chunk["Account.1"].tolist()
        )
    )

    # --------------------------------------------------------
    # Account / bank ID structure
    # --------------------------------------------------------

    account_lengths = pd.concat([
        chunk["Account"].str.len(),
        chunk["Account.1"].str.len()
    ])

    account_length_counts.update(
        account_lengths.tolist()
    )

    bank_lengths = pd.concat([
        chunk["From Bank"].str.len(),
        chunk["To Bank"].str.len()
    ])

    bank_length_counts.update(
        bank_lengths.tolist()
    )

    # Character classes for accounts
    combined_accounts = pd.concat([
        chunk["Account"],
        chunk["Account.1"]
    ])

    for account in combined_accounts:

        account = str(account)

        if account.isdigit():
            account_character_classes["DIGITS_ONLY"] += 1

        elif account.isalnum() and account.isupper():
            account_character_classes["ALPHANUMERIC_UPPER"] += 1

        elif account.isalnum():
            account_character_classes["ALPHANUMERIC_MIXED"] += 1

        else:
            account_character_classes["OTHER"] += 1

    if chunk_no % 10 == 0:
        print(
            f"Processed {total_rows:,} rows..."
        )

# ============================================================
# 7. BASIC DATASET RESULTS
# ============================================================

print()
print("=" * 110)
print("3. BASIC DATASET RESULTS")
print("=" * 110)

print(
    f"Total rows        : {total_rows:,}"
)

print(
    f"Legitimate rows   : {legitimate_rows:,}"
)

print(
    f"Laundering rows   : {laundering_rows:,}"
)

print(
    f"Laundering rate   : "
    f"{laundering_rows / total_rows * 100:.8f}%"
)

print(
    f"Timestamp minimum : {timestamp_min}"
)

print(
    f"Timestamp maximum : {timestamp_max}"
)

print(
    f"Duplicate rows detected within chunks : "
    f"{duplicate_rows_seen:,}"
)

# ============================================================
# 8. UNIQUE ENTITY COUNTS
# ============================================================

print()
print("=" * 110)
print("4. UNIQUE BANK / ACCOUNT COUNTS")
print("=" * 110)

print(
    f"Unique From Banks          : "
    f"{len(from_bank_counts):,}"
)

print(
    f"Unique To Banks            : "
    f"{len(to_bank_counts):,}"
)

print(
    f"Unique source accounts     : "
    f"{len(source_account_counts):,}"
)

print(
    f"Unique destination accounts: "
    f"{len(destination_account_counts):,}"
)

print(
    f"Unique source bank+account nodes: "
    f"{len(source_node_counts):,}"
)

print(
    f"Unique destination bank+account nodes: "
    f"{len(destination_node_counts):,}"
)

print(
    f"Unique directed bank pairs : "
    f"{len(bank_pair_counts):,}"
)

# ============================================================
# 9. BANK DISTRIBUTIONS
# ============================================================

print()
print("=" * 110)
print("5. TOP SOURCE BANKS")
print("=" * 110)

top_from = (
    pd.Series(
        from_bank_counts,
        dtype="int64"
    )
    .sort_values(
        ascending=False
    )
)

top_from_df = pd.DataFrame({
    "Transactions": top_from,
    "Share_pct": (
        top_from
        / total_rows
        * 100
    )
})

print(
    top_from_df.head(50).to_string()
)

print()
print("=" * 110)
print("6. TOP DESTINATION BANKS")
print("=" * 110)

top_to = (
    pd.Series(
        to_bank_counts,
        dtype="int64"
    )
    .sort_values(
        ascending=False
    )
)

top_to_df = pd.DataFrame({
    "Transactions": top_to,
    "Share_pct": (
        top_to
        / total_rows
        * 100
    )
})

print(
    top_to_df.head(50).to_string()
)

# ============================================================
# 10. TOP BANK PAIRS
# ============================================================

print()
print("=" * 110)
print("7. TOP BANK-TO-BANK PAIRS")
print("=" * 110)

top_pairs = (
    pd.Series(
        bank_pair_counts,
        dtype="int64"
    )
    .sort_values(
        ascending=False
    )
)

top_pairs_df = pd.DataFrame({
    "Transactions": top_pairs,
    "Share_pct": (
        top_pairs
        / total_rows
        * 100
    )
})

print(
    top_pairs_df.head(100).to_string()
)

# ============================================================
# 11. AMOUNT DISTRIBUTIONS
# ============================================================

print()
print("=" * 110)
print("8. AMOUNT DISTRIBUTIONS")
print("=" * 110)

amount_summary = {}

for key, values in reservoirs.items():

    arr = np.asarray(
        values,
        dtype=np.float64
    )

    if len(arr) == 0:
        continue

    qs = np.percentile(
        arr,
        [
            0,
            1,
            5,
            10,
            25,
            50,
            75,
            90,
            95,
            99,
            99.9,
            100
        ]
    )

    amount_summary[key] = {
        "count_seen": int(
            reservoir_seen[key]
        ),
        "sample_size": int(
            len(arr)
        ),
        "min": float(qs[0]),
        "p1": float(qs[1]),
        "p5": float(qs[2]),
        "p10": float(qs[3]),
        "p25": float(qs[4]),
        "median": float(qs[5]),
        "p75": float(qs[6]),
        "p90": float(qs[7]),
        "p95": float(qs[8]),
        "p99": float(qs[9]),
        "p99_9": float(qs[10]),
        "max": float(qs[11]),
    }

    print()
    print(key)

    for k, v in amount_summary[key].items():

        if k in [
            "count_seen",
            "sample_size"
        ]:
            print(
                f"  {k:<12}: {v:,}"
            )
        else:
            print(
                f"  {k:<12}: {v:,.6f}"
            )

print()
print(
    f"Global Amount Received min/max : "
    f"{amount_received_min:,.6f} / "
    f"{amount_received_max:,.6f}"
)

print(
    f"Global Amount Paid min/max     : "
    f"{amount_paid_min:,.6f} / "
    f"{amount_paid_max:,.6f}"
)

print(
    f"Received NaN count             : "
    f"{amount_received_nan:,}"
)

print(
    f"Paid NaN count                 : "
    f"{amount_paid_nan:,}"
)

print(
    f"Paid == Received               : "
    f"{amount_ratio_equal:,}"
)

print(
    f"Paid != Received               : "
    f"{amount_ratio_not_equal:,}"
)

# ============================================================
# 12. PAYMENT FORMAT
# ============================================================

print()
print("=" * 110)
print("9. PAYMENT FORMAT DISTRIBUTION")
print("=" * 110)

payment_df = pd.DataFrame({
    "Count": pd.Series(
        payment_format_counts,
        dtype="int64"
    )
})

payment_df["Share_pct"] = (
    payment_df["Count"]
    / total_rows
    * 100
)

payment_df = payment_df.sort_values(
    "Count",
    ascending=False
)

print(
    payment_df.to_string()
)

# ============================================================
# 13. CURRENCIES
# ============================================================

print()
print("=" * 110)
print("10. RECEIVING CURRENCY DISTRIBUTION")
print("=" * 110)

receiving_currency_df = pd.DataFrame({
    "Count": pd.Series(
        receiving_currency_counts,
        dtype="int64"
    )
})

receiving_currency_df["Share_pct"] = (
    receiving_currency_df["Count"]
    / total_rows
    * 100
)

receiving_currency_df = (
    receiving_currency_df
    .sort_values(
        "Count",
        ascending=False
    )
)

print(
    receiving_currency_df.to_string()
)

print()
print("=" * 110)
print("11. PAYMENT CURRENCY DISTRIBUTION")
print("=" * 110)

payment_currency_df = pd.DataFrame({
    "Count": pd.Series(
        payment_currency_counts,
        dtype="int64"
    )
})

payment_currency_df["Share_pct"] = (
    payment_currency_df["Count"]
    / total_rows
    * 100
)

payment_currency_df = (
    payment_currency_df
    .sort_values(
        "Count",
        ascending=False
    )
)

print(
    payment_currency_df.to_string()
)

# ============================================================
# 14. TIME DISTRIBUTION
# ============================================================

print()
print("=" * 110)
print("12. HOUR-OF-DAY DISTRIBUTION")
print("=" * 110)

hour_df = pd.DataFrame({
    "Count": pd.Series(
        hour_counts,
        dtype="int64"
    )
})

hour_df.index.name = "Hour"

hour_df["Share_pct"] = (
    hour_df["Count"]
    / total_rows
    * 100
)

hour_df = hour_df.sort_index()

print(
    hour_df.to_string()
)

print()
print("=" * 110)
print("13. WEEKDAY DISTRIBUTION")
print("=" * 110)

weekday_df = pd.DataFrame({
    "Count": pd.Series(
        weekday_counts,
        dtype="int64"
    )
})

weekday_df["Share_pct"] = (
    weekday_df["Count"]
    / total_rows
    * 100
)

# chronological weekday order
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_df = weekday_df.reindex(
    [
        d for d in weekday_order
        if d in weekday_df.index
    ]
)

print(
    weekday_df.to_string()
)

print()
print("=" * 110)
print("14. DAILY TRANSACTION COUNTS")
print("=" * 110)

date_df = pd.DataFrame({
    "Count": pd.Series(
        date_counts,
        dtype="int64"
    )
})

date_df["Share_pct"] = (
    date_df["Count"]
    / total_rows
    * 100
)

date_df.index.name = "Date"

date_df = date_df.sort_index()

print(
    date_df.to_string()
)

# ============================================================
# 15. ID STRUCTURE
# ============================================================

print()
print("=" * 110)
print("15. ACCOUNT ID LENGTH DISTRIBUTION")
print("=" * 110)

account_length_df = pd.DataFrame({
    "Count": pd.Series(
        account_length_counts,
        dtype="int64"
    )
})

account_length_df["Share_pct"] = (
    account_length_df["Count"]
    / account_length_df["Count"].sum()
    * 100
)

account_length_df.index.name = "Length"

print(
    account_length_df.sort_index().to_string()
)

print()
print("=" * 110)
print("16. BANK ID LENGTH DISTRIBUTION")
print("=" * 110)

bank_length_df = pd.DataFrame({
    "Count": pd.Series(
        bank_length_counts,
        dtype="int64"
    )
})

bank_length_df["Share_pct"] = (
    bank_length_df["Count"]
    / bank_length_df["Count"].sum()
    * 100
)

bank_length_df.index.name = "Length"

print(
    bank_length_df.sort_index().to_string()
)

print()
print("=" * 110)
print("17. ACCOUNT CHARACTER CLASSES")
print("=" * 110)

account_class_df = pd.DataFrame({
    "Count": pd.Series(
        account_character_classes,
        dtype="int64"
    )
})

account_class_df["Share_pct"] = (
    account_class_df["Count"]
    / account_class_df["Count"].sum()
    * 100
)

print(
    account_class_df.to_string()
)

# ============================================================
# 16. TOP ACCOUNT USAGE
# ============================================================

print()
print("=" * 110)
print("18. TOP SOURCE ACCOUNTS")
print("=" * 110)

top_source_accounts = pd.DataFrame({
    "Transactions": pd.Series(
        source_account_counts,
        dtype="int64"
    )
}).sort_values(
    "Transactions",
    ascending=False
)

top_source_accounts["Share_pct"] = (
    top_source_accounts["Transactions"]
    / total_rows
    * 100
)

print(
    top_source_accounts.head(50).to_string()
)

print()
print("=" * 110)
print("19. TOP DESTINATION ACCOUNTS")
print("=" * 110)

top_destination_accounts = pd.DataFrame({
    "Transactions": pd.Series(
        destination_account_counts,
        dtype="int64"
    )
}).sort_values(
    "Transactions",
    ascending=False
)

top_destination_accounts["Share_pct"] = (
    top_destination_accounts["Transactions"]
    / total_rows
    * 100
)

print(
    top_destination_accounts.head(50).to_string()
)

# ============================================================
# 17. ORIGINAL ACCOUNTS.CSV AUDIT
# ============================================================

print()
print("=" * 110)
print("20. ORIGINAL ACCOUNTS.CSV AUDIT")
print("=" * 110)

accounts = pd.read_csv(
    ACCOUNTS_PATH,
    dtype=str,
    keep_default_na=False
)

print(
    "Shape:",
    accounts.shape
)

print(
    "Columns:",
    list(accounts.columns)
)

for col in accounts.columns:

    unique_count = (
        accounts[col]
        .nunique()
    )

    print(
        f"  {col:<20}: "
        f"{unique_count:,} unique values"
    )

print()
print(
    "Duplicate account rows:",
    f"{accounts.duplicated().sum():,}"
)

# Canonical node count
if {
    "Bank ID",
    "Account Number"
}.issubset(accounts.columns):

    canonical_nodes = (
        accounts["Bank ID"].astype(str)
        + "\x1f"
        + accounts["Account Number"].astype(str)
    )

    print(
        "Unique Bank ID + Account Number:",
        f"{canonical_nodes.nunique():,}"
    )

# ============================================================
# 18. ORIGINAL PATTERNS.TXT AUDIT
# ============================================================

print()
print("=" * 110)
print("21. ORIGINAL PATTERNS.TXT AUDIT")
print("=" * 110)

with open(
    PATTERNS_PATH,
    "r",
    encoding="utf-8"
) as f:

    pattern_lines = [
        line.rstrip("\n")
        for line in f
    ]

print(
    "Total pattern lines:",
    f"{len(pattern_lines):,}"
)

# Exact BEGIN/END parser
episodes = []

current = None

begin_re = re.compile(
    r"BEGIN:\s*(.+)"
)

end_re = re.compile(
    r"END"
)

for line_no, line in enumerate(
    pattern_lines,
    start=1
):

    stripped = line.strip()

    m = begin_re.match(
        stripped
    )

    if m:

        if current is not None:
            raise ValueError(
                f"Nested BEGIN at line {line_no}"
            )

        current = {
            "typology": m.group(1).strip().upper(),
            "begin_line": line_no,
            "transactions": []
        }

        continue

    if end_re.fullmatch(
        stripped
    ):

        if current is None:
            raise ValueError(
                f"END without BEGIN at line {line_no}"
            )

        current["end_line"] = line_no

        episodes.append(
            current
        )

        current = None

        continue

    if current is not None:

        if stripped:
            current["transactions"].append(
                stripped
            )

if current is not None:
    raise ValueError(
        "Patterns file ended before END."
    )

pattern_episode_counts = Counter()

pattern_tx_counts = Counter()

episode_sizes = []

for ep in episodes:

    typology = ep["typology"]
    n = len(
        ep["transactions"]
    )

    pattern_episode_counts[
        typology
    ] += 1

    pattern_tx_counts[
        typology
    ] += n

    episode_sizes.append(
        n
    )

print(
    "Pattern episodes:",
    f"{len(episodes):,}"
)

print(
    "Pattern transaction rows:",
    f"{sum(episode_sizes):,}"
)

print()
print("Typology episode counts:")

for typology, count in sorted(
    pattern_episode_counts.items()
):

    tx_count = pattern_tx_counts[
        typology
    ]

    print(
        f"  {typology:<18} "
        f"episodes={count:>5,} | "
        f"transactions={tx_count:>6,} | "
        f"avg={tx_count / count:.2f}"
    )

if episode_sizes:

    print()
    print("Episode size statistics:")
    print(
        f"  min    : {min(episode_sizes)}"
    )
    print(
        f"  median : {np.median(episode_sizes):.2f}"
    )
    print(
        f"  mean   : {np.mean(episode_sizes):.2f}"
    )
    print(
        f"  p90    : {np.percentile(episode_sizes, 90):.2f}"
    )
    print(
        f"  p99    : {np.percentile(episode_sizes, 99):.2f}"
    )
    print(
        f"  max    : {max(episode_sizes)}"
    )

# ============================================================
# 19. SAVE BASELINE ARTIFACTS
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/AML_GNN_ONLY"
) / "data"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Bank distributions
top_from_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_from_bank_distribution.csv"
)

top_to_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_to_bank_distribution.csv"
)

top_pairs_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_bank_pair_distribution.csv"
)

top_source_accounts.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_source_account_distribution.csv"
)

top_destination_accounts.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_destination_account_distribution.csv"
)

payment_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_payment_format_distribution.csv"
)

receiving_currency_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_receiving_currency_distribution.csv"
)

payment_currency_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_payment_currency_distribution.csv"
)

hour_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_hour_distribution.csv"
)

weekday_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_weekday_distribution.csv"
)

date_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_daily_distribution.csv"
)

account_length_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_account_length_distribution.csv"
)

bank_length_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_bank_length_distribution.csv"
)

account_class_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_original_account_character_classes.csv"
)

# ============================================================
# 20. MASTER JSON
# ============================================================

baseline = {
    "source": {
        "transactions": str(TX_PATH),
        "accounts": str(ACCOUNTS_PATH),
        "patterns": str(PATTERNS_PATH)
    },

    "transactions": {
        "total_rows": int(total_rows),
        "legitimate_rows": int(legitimate_rows),
        "laundering_rows": int(laundering_rows),
        "laundering_rate_pct": float(
            laundering_rows / total_rows * 100
        ),
        "timestamp_min": str(timestamp_min),
        "timestamp_max": str(timestamp_max),
        "duplicate_rows_detected_within_chunks": int(
            duplicate_rows_seen
        )
    },

    "unique_counts": {
        "unique_from_banks": int(
            len(from_bank_counts)
        ),
        "unique_to_banks": int(
            len(to_bank_counts)
        ),
        "unique_source_accounts": int(
            len(source_account_counts)
        ),
        "unique_destination_accounts": int(
            len(destination_account_counts)
        ),
        "unique_source_nodes": int(
            len(source_node_counts)
        ),
        "unique_destination_nodes": int(
            len(destination_node_counts)
        ),
        "unique_bank_pairs": int(
            len(bank_pair_counts)
        )
    },

    "amounts": amount_summary,

    "amount_global": {
        "received_min": float(
            amount_received_min
        ),
        "received_max": float(
            amount_received_max
        ),
        "paid_min": float(
            amount_paid_min
        ),
        "paid_max": float(
            amount_paid_max
        ),
        "received_nan": int(
            amount_received_nan
        ),
        "paid_nan": int(
            amount_paid_nan
        ),
        "equal_received_paid": int(
            amount_ratio_equal
        ),
        "different_received_paid": int(
            amount_ratio_not_equal
        )
    },

    "payment_formats": {
        str(k): int(v)
        for k, v in payment_format_counts.items()
    },

    "receiving_currencies": {
        str(k): int(v)
        for k, v in receiving_currency_counts.items()
    },

    "payment_currencies": {
        str(k): int(v)
        for k, v in payment_currency_counts.items()
    },

    "hours": {
        str(k): int(v)
        for k, v in hour_counts.items()
    },

    "weekdays": {
        str(k): int(v)
        for k, v in weekday_counts.items()
    },

    "daily_counts": {
        str(k): int(v)
        for k, v in date_counts.items()
    },

    "account_id_lengths": {
        str(k): int(v)
        for k, v in account_length_counts.items()
    },

    "bank_id_lengths": {
        str(k): int(v)
        for k, v in bank_length_counts.items()
    },

    "account_character_classes": {
        str(k): int(v)
        for k, v in account_character_classes.items()
    },

    "patterns": {
        "episodes": int(len(episodes)),
        "transactions": int(sum(episode_sizes)),
        "episode_counts": {
            str(k): int(v)
            for k, v in pattern_episode_counts.items()
        },
        "transaction_counts": {
            str(k): int(v)
            for k, v in pattern_tx_counts.items()
        },
        "episode_size": {
            "min": int(min(episode_sizes)),
            "median": float(np.median(episode_sizes)),
            "mean": float(np.mean(episode_sizes)),
            "p90": float(
                np.percentile(
                    episode_sizes,
                    90
                )
            ),
            "p99": float(
                np.percentile(
                    episode_sizes,
                    99
                )
            ),
            "max": int(max(episode_sizes))
        }
    }
}

baseline_path = (
    OUTPUT_DIR /
    "HI-Small_original_distribution_dna.json"
)

with open(
    baseline_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        baseline,
        f,
        indent=2
    )

# ============================================================
# 21. FINAL
# ============================================================

print()
print("=" * 110)
print("✅ ORIGINAL HI-SMALL DISTRIBUTION DNA AUDIT COMPLETE")
print("=" * 110)

print(
    "Master baseline:",
    baseline_path
)

print()
print(
    "IMPORTANT:"
)

print(
    "The existing 10M synthetic CSV was NOT used."
)

print(
    "No source dataset was modified."
)

print(
    "This baseline will become the reference distribution "
    "for the NEW pattern-preserving generator."
)

print("=" * 110)

ORIGINAL HI-SMALL DISTRIBUTION DNA AUDIT
Transactions : /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv
Accounts     : /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_accounts.csv
Patterns     : /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Patterns.txt

1. FILE INFORMATION
Transactions size : 0.443 GB
Accounts size     : 32.476 MB
Patterns size     : 0.309 MB

Transaction columns:
   1. Timestamp
   2. From Bank
   3. Account
   4. To Bank
   5. Account.1
   6. Amount Received
   7. Receiving Currency
   8. Amount Paid
   9. Payment Currency
  10. Payment Format
  11. Is Laundering

2. SCANNING ORIGINAL TRANSACTIONS
Processed 2,500,000 rows...
Processed 5,000,000 rows...

3. BASIC DATASET RESULTS
Total rows        : 5,078,345
Legitimate rows   : 5,073,168
Laundering rows   : 5,177
Laundering rate   : 0.10194266%
Timestamp minimum : 2022-09-01 00:

ValueError: min() iterable argument is empty

In [3]:
# ============================================================
# CELL 2 — INSPECT EXACT HI-SMALL PATTERNS.TXT FORMAT
#
# Purpose:
#   We need to understand the exact BEGIN / END syntax before
#   building the pattern-aware synthetic generator.
#
# NO files are modified.
# ============================================================

from pathlib import Path
import re

PATTERNS_PATH = Path(
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Patterns.txt"
)

if not PATTERNS_PATH.exists():
    raise FileNotFoundError(PATTERNS_PATH)

print("=" * 100)
print("HI-SMALL PATTERNS.TXT FORMAT INSPECTION")
print("=" * 100)

# ------------------------------------------------------------
# 1. READ RAW LINES
# ------------------------------------------------------------

with open(
    PATTERNS_PATH,
    "r",
    encoding="utf-8"
) as f:
    lines = f.readlines()

print("Total lines:", len(lines))

# ------------------------------------------------------------
# 2. PRINT FIRST 100 NON-EMPTY LINES
# ------------------------------------------------------------

print()
print("=" * 100)
print("FIRST 100 NON-EMPTY LINES")
print("=" * 100)

non_empty_counter = 0

for line_no, raw in enumerate(lines, start=1):

    stripped = raw.rstrip("\n\r")

    if not stripped.strip():
        continue

    print(
        f"{line_no:5d}: {repr(stripped)}"
    )

    non_empty_counter += 1

    if non_empty_counter >= 100:
        break

# ------------------------------------------------------------
# 3. FIND ALL LINES CONTAINING BEGIN / END
# ------------------------------------------------------------

print()
print("=" * 100)
print("BEGIN / END MARKER INSPECTION")
print("=" * 100)

begin_matches = []
end_matches = []

for line_no, raw in enumerate(lines, start=1):

    stripped = raw.strip()

    if re.search(
        r"\bBEGIN\b",
        stripped,
        flags=re.IGNORECASE
    ):
        begin_matches.append(
            (line_no, stripped)
        )

    if re.search(
        r"\bEND\b",
        stripped,
        flags=re.IGNORECASE
    ):
        end_matches.append(
            (line_no, stripped)
        )

print(
    "Lines containing BEGIN:",
    len(begin_matches)
)

print(
    "Lines containing END:",
    len(end_matches)
)

print()
print("First 30 BEGIN candidates:")

for line_no, text in begin_matches[:30]:
    print(
        f"{line_no:5d}: {repr(text)}"
    )

print()
print("First 30 END candidates:")

for line_no, text in end_matches[:30]:
    print(
        f"{line_no:5d}: {repr(text)}"
    )

# ------------------------------------------------------------
# 4. SEARCH FOR TYPOLOGY NAMES
# ------------------------------------------------------------

typologies = [
    "FAN-OUT",
    "FAN-IN",
    "CYCLE",
    "BIPARTITE",
    "STACK",
    "RANDOM",
    "SCATTER-GATHER",
    "GATHER-SCATTER",
]

print()
print("=" * 100)
print("TYPOLOGY MARKER INSPECTION")
print("=" * 100)

for typology in typologies:

    occurrences = []

    pattern = re.compile(
        re.escape(typology),
        flags=re.IGNORECASE
    )

    for line_no, raw in enumerate(
        lines,
        start=1
    ):

        if pattern.search(raw):
            occurrences.append(
                (
                    line_no,
                    raw.rstrip("\n\r")
                )
            )

    print()
    print(
        f"{typology:<18}: "
        f"{len(occurrences):,} matching lines"
    )

    for line_no, text in occurrences[:5]:
        print(
            f"    {line_no:5d}: {repr(text)}"
        )

# ------------------------------------------------------------
# 5. SHOW REPRESENTATIVE SECTIONS AROUND TYPOLOGY MARKERS
# ------------------------------------------------------------

print()
print("=" * 100)
print("REPRESENTATIVE PATTERN SECTIONS")
print("=" * 100)

shown_sections = 0
used_lines = set()

for line_no, raw in enumerate(
    lines,
    start=1
):

    upper = raw.upper()

    found_typology = None

    for typology in typologies:
        if typology in upper:
            found_typology = typology
            break

    if found_typology is None:
        continue

    # Avoid dumping overlapping sections
    if any(
        abs(line_no - x) <= 10
        for x in used_lines
    ):
        continue

    start = max(
        1,
        line_no - 3
    )

    end = min(
        len(lines),
        line_no + 12
    )

    print()
    print(
        f"--- around {found_typology} at line {line_no} ---"
    )

    for i in range(
        start,
        end + 1
    ):

        print(
            f"{i:5d}: "
            f"{repr(lines[i-1].rstrip(chr(10)+chr(13)))}"
        )

        used_lines.add(i)

    shown_sections += 1

    if shown_sections >= 8:
        break

# ------------------------------------------------------------
# 6. LOOK FOR OTHER COMMON STRUCTURAL MARKERS
# ------------------------------------------------------------

print()
print("=" * 100)
print("OTHER STRUCTURAL MARKERS")
print("=" * 100)

marker_candidates = [
    "START",
    "BEGIN",
    "END",
    "TYPE",
    "PATTERN",
    "TRANSACTION",
    "Timestamp",
    "From Bank",
    "To Bank",
    "Amount",
]

for marker in marker_candidates:

    count = sum(
        1
        for raw in lines
        if marker.lower() in raw.lower()
    )

    print(
        f"{marker:<15}: {count:,} lines"
    )

print()
print("=" * 100)
print("✅ PATTERNS FORMAT INSPECTION COMPLETE")
print("=" * 100)

print(
    "Nothing was modified."
)

print(
    "We will use the exact format revealed here "
    "for the generator/parser."
)

print("=" * 100)

HI-SMALL PATTERNS.TXT FORMAT INSPECTION
Total lines: 4319

FIRST 100 NON-EMPTY LINES
    1: 'BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out'
    2: '2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1'
    3: '2022/09/01 04:33,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1'
    4: '2022/09/01 09:14,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1'
    5: '2022/09/01 09:56,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1'
    6: '2022/09/01 11:28,021174,800737690,001244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1'
    7: '2022/09/01 13:13,021174,800737690,00513,80078E200,6990.87,US Dollar,6990.87,US Dollar,ACH,1'
    8: '2022/09/01 14:11,021174,800737690,020,80066B990,12536.92,Euro,12536.92,Euro,ACH,1'
    9: '2022/09/02 15:40,021174,800737690,00410,8002CC310,3511.82,Euro,3511.82,Euro,ACH,1'
   10: '2022/09/02 21:23,021174,800737690,01292,8004030A0,16135.09,US Dollar,16135.09,

In [4]:
# ============================================================
# CELL 2 — ORIGINAL HI-SMALL CONDITIONAL / BEHAVIORAL DNA
#
# This cell answers:
#
#   "How does a normal account/bank/entity behave?"
#
# We will later use these empirical distributions to generate:
#   - NEW realistic account IDs
#   - NEW realistic bank assignments
#   - realistic transaction frequency
#   - realistic account reuse
#   - realistic entity/account relationships
#   - realistic timing
#
# SOURCE:
#   Original HI-Small only
#
# NO synthetic data is generated here.
# NO source file is modified.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import json
import re
import gc

# ============================================================
# 0. PATHS
# ============================================================

IBM_ROOT = Path(
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml"
)

TX_PATH = IBM_ROOT / "HI-Small_Trans.csv"
ACCOUNTS_PATH = IBM_ROOT / "HI-Small_accounts.csv"

if not TX_PATH.exists():
    raise FileNotFoundError(TX_PATH)

if not ACCOUNTS_PATH.exists():
    raise FileNotFoundError(ACCOUNTS_PATH)

print("=" * 110)
print("ORIGINAL HI-SMALL CONDITIONAL / BEHAVIORAL DNA")
print("=" * 110)
print()
print("Transactions:", TX_PATH)
print("Accounts    :", ACCOUNTS_PATH)

# ============================================================
# 1. LOAD ORIGINAL ACCOUNTS
# ============================================================

accounts = pd.read_csv(
    ACCOUNTS_PATH,
    dtype=str,
    keep_default_na=False
)

accounts["Bank ID"] = (
    accounts["Bank ID"]
    .astype(str)
    .str.strip()
)

accounts["Account Number"] = (
    accounts["Account Number"]
    .astype(str)
    .str.strip()
)

accounts["Entity ID"] = (
    accounts["Entity ID"]
    .astype(str)
    .str.strip()
)

accounts["Bank Name"] = (
    accounts["Bank Name"]
    .astype(str)
    .str.strip()
)

accounts["Entity Name"] = (
    accounts["Entity Name"]
    .astype(str)
    .str.strip()
)

print()
print("=" * 110)
print("1. ORIGINAL ACCOUNT / ENTITY STRUCTURE")
print("=" * 110)

print(
    "Account rows:",
    f"{len(accounts):,}"
)

print(
    "Unique banks:",
    f"{accounts['Bank ID'].nunique():,}"
)

print(
    "Unique entities:",
    f"{accounts['Entity ID'].nunique():,}"
)

# ============================================================
# 2. ACCOUNTS PER ENTITY
# ============================================================

accounts_per_entity = (
    accounts
    .groupby("Entity ID")
    .size()
)

print()
print("Accounts per entity:")

for stat_name, value in {
    "min": accounts_per_entity.min(),
    "p25": accounts_per_entity.quantile(.25),
    "median": accounts_per_entity.median(),
    "mean": accounts_per_entity.mean(),
    "p75": accounts_per_entity.quantile(.75),
    "p90": accounts_per_entity.quantile(.90),
    "p95": accounts_per_entity.quantile(.95),
    "p99": accounts_per_entity.quantile(.99),
    "max": accounts_per_entity.max(),
}.items():

    print(
        f"  {stat_name:<8}: {value:.3f}"
    )

print()
print("Entity account-count frequency:")
print(
    accounts_per_entity
    .value_counts()
    .sort_index()
    .head(30)
    .to_string()
)

# ============================================================
# 3. ENTITIES WITH MULTIPLE BANKS
# ============================================================

banks_per_entity = (
    accounts
    .groupby("Entity ID")["Bank ID"]
    .nunique()
)

multi_bank_entities = (
    banks_per_entity >= 2
)

print()
print(
    "Entities with >=2 bank relationships:",
    f"{multi_bank_entities.sum():,}"
)

print(
    "Percentage of entities with >=2 banks:",
    f"{multi_bank_entities.mean()*100:.4f}%"
)

print(
    "Maximum banks associated with one entity:",
    f"{banks_per_entity.max():,}"
)

# ============================================================
# 4. ACCOUNTS PER BANK
# ============================================================

accounts_per_bank = (
    accounts
    .groupby("Bank ID")
    .size()
    .sort_values(
        ascending=False
    )
)

print()
print("=" * 110)
print("2. ACCOUNTS PER BANK")
print("=" * 110)

for stat_name, value in {
    "min": accounts_per_bank.min(),
    "p25": accounts_per_bank.quantile(.25),
    "median": accounts_per_bank.median(),
    "mean": accounts_per_bank.mean(),
    "p75": accounts_per_bank.quantile(.75),
    "p90": accounts_per_bank.quantile(.90),
    "p95": accounts_per_bank.quantile(.95),
    "p99": accounts_per_bank.quantile(.99),
    "max": accounts_per_bank.max(),
}.items():

    print(
        f"  {stat_name:<8}: {value:.3f}"
    )

print()
print("Top 30 banks by number of accounts:")

bank_account_df = pd.DataFrame({
    "Account_Count": accounts_per_bank
})

bank_account_df["Share_of_Accounts_pct"] = (
    bank_account_df["Account_Count"]
    / len(accounts)
    * 100
)

print(
    bank_account_df
    .head(30)
    .to_string()
)

# ============================================================
# 5. BANK NAME RELATIONSHIP
# ============================================================

bank_name_per_id = (
    accounts
    .groupby("Bank ID")["Bank Name"]
    .nunique()
)

print()
print("=" * 110)
print("3. BANK ID → BANK NAME STRUCTURE")
print("=" * 110)

print(
    "Bank IDs with exactly one Bank Name:",
    f"{(bank_name_per_id == 1).sum():,}"
)

print(
    "Bank IDs with multiple Bank Names:",
    f"{(bank_name_per_id > 1).sum():,}"
)

# ============================================================
# 6. ACCOUNT ID CHARACTER / LENGTH STRUCTURE
# ============================================================

account_lengths = (
    accounts["Account Number"]
    .str.len()
)

print()
print("=" * 110)
print("4. ORIGINAL ACCOUNT ID STRUCTURE")
print("=" * 110)

print(
    "Unique account ID values:",
    f"{accounts['Account Number'].nunique():,}"
)

print()
print("Lengths:")
print(
    account_lengths
    .value_counts()
    .sort_index()
    .to_string()
)

digit_only_accounts = (
    accounts["Account Number"]
    .str.fullmatch(r"\d+")
)

uppercase_alnum_accounts = (
    accounts["Account Number"]
    .str.fullmatch(r"[A-Z0-9]+")
)

print()
print(
    "Digits only:",
    f"{digit_only_accounts.sum():,}",
    f"({digit_only_accounts.mean()*100:.4f}%)"
)

print(
    "Uppercase alphanumeric:",
    f"{uppercase_alnum_accounts.sum():,}",
    f"({uppercase_alnum_accounts.mean()*100:.4f}%)"
)

print()
print("Sample real account IDs:")

for value in (
    accounts["Account Number"]
    .drop_duplicates()
    .sample(
        min(30, accounts["Account Number"].nunique()),
        random_state=20260919
    )
    .tolist()
):
    print(" ", value)

# ============================================================
# 7. ENTITY ID / ENTITY NAME STRUCTURE
# ============================================================

print()
print("=" * 110)
print("5. ENTITY ID STRUCTURE")
print("=" * 110)

print(
    "Unique Entity IDs:",
    f"{accounts['Entity ID'].nunique():,}"
)

print(
    "Unique Entity Names:",
    f"{accounts['Entity Name'].nunique():,}"
)

print(
    "Entity ID == Entity Name unique-count:",
    accounts["Entity ID"].nunique()
    == accounts["Entity Name"].nunique()
)

# ============================================================
# 8. TRANSACTION ACTIVITY COUNTERS
# ============================================================

print()
print("=" * 110)
print("6. SCANNING ORIGINAL TRANSACTION ACTIVITY")
print("=" * 110)

CHUNK_SIZE = 250_000

usecols = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Is Laundering"
]

source_account_counts = Counter()
destination_account_counts = Counter()

source_bank_counts = Counter()
destination_bank_counts = Counter()

source_node_counts = Counter()
destination_node_counts = Counter()

laundering_source_account_counts = Counter()
laundering_destination_account_counts = Counter()

laundering_source_bank_counts = Counter()
laundering_destination_bank_counts = Counter()

legitimate_source_account_counts = Counter()
legitimate_destination_account_counts = Counter()

legitimate_source_bank_counts = Counter()
legitimate_destination_bank_counts = Counter()

# For active account inter-arrival samples
#
# We keep only timestamps for a bounded sample of accounts so
# memory remains safe.
MAX_ACTIVE_ACCOUNTS = 50_000

active_account_timestamps = defaultdict(list)

rng = np.random.default_rng(20260919)

total_rows = 0
legitimate_rows = 0
laundering_rows = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        TX_PATH,
        usecols=usecols,
        dtype=str,
        keep_default_na=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    total_rows += len(chunk)

    laundering_mask = (
        chunk["Is Laundering"]
        .astype(str)
        .str.strip()
        == "1"
    )

    legitimate_mask = ~laundering_mask

    legitimate_rows += int(
        legitimate_mask.sum()
    )

    laundering_rows += int(
        laundering_mask.sum()
    )

    # --------------------------------------------------------
    # SOURCE ACCOUNT
    # --------------------------------------------------------

    source_account_counts.update(
        chunk["Account"].tolist()
    )

    destination_account_counts.update(
        chunk["Account.1"].tolist()
    )

    # --------------------------------------------------------
    # SOURCE / DESTINATION BANK
    # --------------------------------------------------------

    source_bank_counts.update(
        chunk["From Bank"].tolist()
    )

    destination_bank_counts.update(
        chunk["To Bank"].tolist()
    )

    # --------------------------------------------------------
    # BANK + ACCOUNT NODE
    # --------------------------------------------------------

    source_node_counts.update(
        zip(
            chunk["From Bank"].tolist(),
            chunk["Account"].tolist()
        )
    )

    destination_node_counts.update(
        zip(
            chunk["To Bank"].tolist(),
            chunk["Account.1"].tolist()
        )
    )

    # --------------------------------------------------------
    # LAUNDERING
    # --------------------------------------------------------

    laundering = chunk.loc[
        laundering_mask
    ]

    legitimate = chunk.loc[
        legitimate_mask
    ]

    laundering_source_account_counts.update(
        laundering["Account"].tolist()
    )

    laundering_destination_account_counts.update(
        laundering["Account.1"].tolist()
    )

    laundering_source_bank_counts.update(
        laundering["From Bank"].tolist()
    )

    laundering_destination_bank_counts.update(
        laundering["To Bank"].tolist()
    )

    # --------------------------------------------------------
    # LEGITIMATE
    # --------------------------------------------------------

    legitimate_source_account_counts.update(
        legitimate["Account"].tolist()
    )

    legitimate_destination_account_counts.update(
        legitimate["Account.1"].tolist()
    )

    legitimate_source_bank_counts.update(
        legitimate["From Bank"].tolist()
    )

    legitimate_destination_bank_counts.update(
        legitimate["To Bank"].tolist()
    )

    # --------------------------------------------------------
    # ACCOUNT TIMESTAMP SAMPLE
    # --------------------------------------------------------

    # Only a bounded set of accounts is tracked.
    #
    # This gives us a representative inter-arrival sample
    # without storing timestamps for every account.
    if len(active_account_timestamps) < MAX_ACTIVE_ACCOUNTS:

        ts = pd.to_datetime(
            chunk["Timestamp"],
            errors="coerce"
        )

        for account, timestamp in zip(
            chunk["Account"],
            ts
        ):

            if pd.isna(timestamp):
                continue

            if account not in active_account_timestamps:

                if len(
                    active_account_timestamps
                ) < MAX_ACTIVE_ACCOUNTS:

                    active_account_timestamps[
                        account
                    ] = [
                        timestamp
                    ]

            else:

                # Store at most 100 timestamps per account.
                if len(
                    active_account_timestamps[account]
                ) < 100:

                    active_account_timestamps[
                        account
                    ].append(
                        timestamp
                    )

    if chunk_no % 10 == 0:
        print(
            f"Processed {total_rows:,} rows..."
        )

# ============================================================
# 9. BASIC ACTIVITY RESULTS
# ============================================================

print()
print("=" * 110)
print("7. TRANSACTION ACTIVITY RESULTS")
print("=" * 110)

print(
    "Total rows:",
    f"{total_rows:,}"
)

print(
    "Legitimate:",
    f"{legitimate_rows:,}"
)

print(
    "Laundering:",
    f"{laundering_rows:,}"
)

# ============================================================
# 10. ACCOUNT TRANSACTION COUNT DISTRIBUTIONS
# ============================================================

def summarize_counter(
    counter,
    name
):

    s = pd.Series(
        counter,
        dtype="int64"
    )

    if len(s) == 0:
        return None

    summary = {
        "count": int(len(s)),
        "min": float(s.min()),
        "p25": float(s.quantile(.25)),
        "median": float(s.median()),
        "mean": float(s.mean()),
        "p75": float(s.quantile(.75)),
        "p90": float(s.quantile(.90)),
        "p95": float(s.quantile(.95)),
        "p99": float(s.quantile(.99)),
        "max": float(s.max())
    }

    print()
    print(name)

    for k, v in summary.items():

        if k == "count":
            print(
                f"  {k:<8}: {int(v):,}"
            )
        else:
            print(
                f"  {k:<8}: {v:.3f}"
            )

    return summary


print()
print("=" * 110)
print("8. SOURCE ACCOUNT ACTIVITY")
print("=" * 110)

source_account_summary = summarize_counter(
    source_account_counts,
    "ALL SOURCE ACCOUNTS"
)

legit_source_account_summary = summarize_counter(
    legitimate_source_account_counts,
    "LEGITIMATE SOURCE ACCOUNTS"
)

launder_source_account_summary = summarize_counter(
    laundering_source_account_counts,
    "LAUNDERING SOURCE ACCOUNTS"
)

print()
print("=" * 110)
print("9. DESTINATION ACCOUNT ACTIVITY")
print("=" * 110)

destination_account_summary = summarize_counter(
    destination_account_counts,
    "ALL DESTINATION ACCOUNTS"
)

legit_destination_account_summary = summarize_counter(
    legitimate_destination_account_counts,
    "LEGITIMATE DESTINATION ACCOUNTS"
)

launder_destination_account_summary = summarize_counter(
    laundering_destination_account_counts,
    "LAUNDERING DESTINATION ACCOUNTS"
)

# ============================================================
# 11. BANK ACTIVITY DISTRIBUTIONS
# ============================================================

print()
print("=" * 110)
print("10. SOURCE BANK ACTIVITY")
print("=" * 110)

source_bank_summary = summarize_counter(
    source_bank_counts,
    "ALL SOURCE BANKS"
)

legit_source_bank_summary = summarize_counter(
    legitimate_source_bank_counts,
    "LEGITIMATE SOURCE BANKS"
)

launder_source_bank_summary = summarize_counter(
    laundering_source_bank_counts,
    "LAUNDERING SOURCE BANKS"
)

print()
print("=" * 110)
print("11. DESTINATION BANK ACTIVITY")
print("=" * 110)

destination_bank_summary = summarize_counter(
    destination_bank_counts,
    "ALL DESTINATION BANKS"
)

legit_destination_bank_summary = summarize_counter(
    legitimate_destination_bank_counts,
    "LEGITIMATE DESTINATION BANKS"
)

launder_destination_bank_summary = summarize_counter(
    laundering_destination_bank_counts,
    "LAUNDERING DESTINATION BANKS"
)

# ============================================================
# 12. TOP ACCOUNT ACTIVITY
# ============================================================

print()
print("=" * 110)
print("12. TOP SOURCE ACCOUNTS BY TRANSACTION COUNT")
print("=" * 110)

top_source_accounts = pd.DataFrame({
    "Transactions": pd.Series(
        source_account_counts,
        dtype="int64"
    )
}).sort_values(
    "Transactions",
    ascending=False
)

top_source_accounts["Share_pct"] = (
    top_source_accounts["Transactions"]
    / total_rows
    * 100
)

print(
    top_source_accounts.head(50).to_string()
)

print()
print("=" * 110)
print("13. TOP SOURCE ACCOUNTS — LEGITIMATE ONLY")
print("=" * 110)

top_legit_source_accounts = pd.DataFrame({
    "Transactions": pd.Series(
        legitimate_source_account_counts,
        dtype="int64"
    )
}).sort_values(
    "Transactions",
    ascending=False
)

top_legit_source_accounts["Share_pct"] = (
    top_legit_source_accounts["Transactions"]
    / legitimate_rows
    * 100
)

print(
    top_legit_source_accounts.head(50).to_string()
)

print()
print("=" * 110)
print("14. TOP SOURCE ACCOUNTS — LAUNDERING")
print("=" * 110)

top_launder_source_accounts = pd.DataFrame({
    "Transactions": pd.Series(
        laundering_source_account_counts,
        dtype="int64"
    )
}).sort_values(
    "Transactions",
    ascending=False
)

top_launder_source_accounts["Share_pct"] = (
    top_launder_source_accounts["Transactions"]
    / laundering_rows
    * 100
)

print(
    top_launder_source_accounts.head(50).to_string()
)

# ============================================================
# 13. INTER-ARRIVAL TIME DISTRIBUTION
# ============================================================

print()
print("=" * 110)
print("15. ACCOUNT INTER-ARRIVAL TIME")
print("=" * 110)

interarrival_minutes = []

for account, timestamps in (
    active_account_timestamps.items()
):

    if len(timestamps) < 2:
        continue

    timestamps = sorted(
        timestamps
    )

    diffs = pd.Series(
        timestamps
    ).diff()

    mins = (
        diffs
        .dropna()
        .dt.total_seconds()
        / 60.0
    )

    interarrival_minutes.extend(
        mins[
            np.isfinite(mins)
        ].tolist()
    )

if interarrival_minutes:

    interarrival = np.asarray(
        interarrival_minutes,
        dtype=np.float64
    )

    print(
        "Sampled inter-arrivals:",
        f"{len(interarrival):,}"
    )

    for q in [
        1,
        5,
        10,
        25,
        50,
        75,
        90,
        95,
        99,
        99.9
    ]:

        print(
            f"  p{q:<4}: "
            f"{np.percentile(interarrival, q):,.4f} minutes"
        )

    print(
        "  max  :",
        f"{np.max(interarrival):,.4f} minutes"
    )

else:

    print(
        "⚠️ No inter-arrival samples available."
    )

# ============================================================
# 14. ACTIVITY BUCKETS
# ============================================================

print()
print("=" * 110)
print("16. ACCOUNT ACTIVITY BUCKETS")
print("=" * 110)

activity = pd.Series(
    source_account_counts,
    dtype="int64"
)

activity_buckets = pd.cut(
    activity,
    bins=[
        0,
        1,
        2,
        5,
        10,
        25,
        50,
        100,
        250,
        500,
        1000,
        np.inf
    ],
    right=True
)

bucket_summary = (
    activity_buckets
    .value_counts()
    .sort_index()
)

bucket_df = pd.DataFrame({
    "Accounts": bucket_summary
})

bucket_df["Share_pct"] = (
    bucket_df["Accounts"]
    / len(activity)
    * 100
)

print(
    bucket_df.to_string()
)

# ============================================================
# 15. BANK ACTIVITY BUCKETS
# ============================================================

print()
print("=" * 110)
print("17. BANK ACTIVITY BUCKETS")
print("=" * 110)

bank_activity = pd.Series(
    source_bank_counts,
    dtype="int64"
)

bank_activity_buckets = pd.cut(
    bank_activity,
    bins=[
        0,
        100,
        500,
        1000,
        5000,
        10000,
        25000,
        50000,
        100000,
        250000,
        np.inf
    ],
    right=True
)

bank_bucket_df = pd.DataFrame({
    "Banks": (
        bank_activity_buckets
        .value_counts()
        .sort_index()
    )
})

bank_bucket_df["Share_pct"] = (
    bank_bucket_df["Banks"]
    / len(bank_activity)
    * 100
)

print(
    bank_bucket_df.to_string()
)

# ============================================================
# 16. SAVE BEHAVIORAL DNA
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/AML_GNN_ONLY/data"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save top-account activity tables
top_source_accounts.to_csv(
    OUTPUT_DIR /
    "HI-Small_behavior_top_source_accounts.csv"
)

top_legit_source_accounts.to_csv(
    OUTPUT_DIR /
    "HI-Small_behavior_top_legitimate_source_accounts.csv"
)

top_launder_source_accounts.to_csv(
    OUTPUT_DIR /
    "HI-Small_behavior_top_laundering_source_accounts.csv"
)

bank_account_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_behavior_accounts_per_bank.csv"
)

bucket_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_behavior_account_activity_buckets.csv"
)

bank_bucket_df.to_csv(
    OUTPUT_DIR /
    "HI-Small_behavior_bank_activity_buckets.csv"
)

# ============================================================
# 17. MASTER JSON
# ============================================================

behavioral_dna = {
    "source_transactions": str(TX_PATH),
    "source_accounts": str(ACCOUNTS_PATH),

    "transaction_counts": {
        "total": int(total_rows),
        "legitimate": int(legitimate_rows),
        "laundering": int(laundering_rows)
    },

    "accounts": {
        "rows": int(len(accounts)),
        "unique_accounts": int(
            accounts["Account Number"].nunique()
        ),
        "unique_banks": int(
            accounts["Bank ID"].nunique()
        ),
        "unique_entities": int(
            accounts["Entity ID"].nunique()
        )
    },

    "entity_structure": {
        "accounts_per_entity": {
            "min": float(accounts_per_entity.min()),
            "p25": float(accounts_per_entity.quantile(.25)),
            "median": float(accounts_per_entity.median()),
            "mean": float(accounts_per_entity.mean()),
            "p75": float(accounts_per_entity.quantile(.75)),
            "p90": float(accounts_per_entity.quantile(.90)),
            "p95": float(accounts_per_entity.quantile(.95)),
            "p99": float(accounts_per_entity.quantile(.99)),
            "max": float(accounts_per_entity.max())
        },
        "entities_with_multiple_banks": int(
            multi_bank_entities.sum()
        ),
        "percentage_entities_with_multiple_banks": float(
            multi_bank_entities.mean() * 100
        ),
        "max_banks_per_entity": int(
            banks_per_entity.max()
        )
    },

    "accounts_per_bank": {
        "min": float(accounts_per_bank.min()),
        "p25": float(accounts_per_bank.quantile(.25)),
        "median": float(accounts_per_bank.median()),
        "mean": float(accounts_per_bank.mean()),
        "p75": float(accounts_per_bank.quantile(.75)),
        "p90": float(accounts_per_bank.quantile(.90)),
        "p95": float(accounts_per_bank.quantile(.95)),
        "p99": float(accounts_per_bank.quantile(.99)),
        "max": float(accounts_per_bank.max())
    },

    "account_id_structure": {
        "length_distribution": {
            str(k): int(v)
            for k, v in (
                account_lengths
                .value_counts()
                .sort_index()
                .items()
            )
        },
        "digits_only": int(
            digit_only_accounts.sum()
        ),
        "uppercase_alphanumeric": int(
            uppercase_alnum_accounts.sum()
        )
    },

    "source_account_activity": {
        "all": source_account_summary,
        "legitimate": legit_source_account_summary,
        "laundering": launder_source_account_summary
    },

    "destination_account_activity": {
        "all": destination_account_summary,
        "legitimate": legit_destination_account_summary,
        "laundering": launder_destination_account_summary
    },

    "source_bank_activity": {
        "all": source_bank_summary,
        "legitimate": legit_source_bank_summary,
        "laundering": launder_source_bank_summary
    },

    "destination_bank_activity": {
        "all": destination_bank_summary,
        "legitimate": legit_destination_bank_summary,
        "laundering": launder_destination_bank_summary
    },

    "interarrival_minutes": {
        "sample_size": int(
            len(interarrival_minutes)
        ),
        "p1": float(np.percentile(interarrival, 1))
        if interarrival_minutes else None,
        "p5": float(np.percentile(interarrival, 5))
        if interarrival_minutes else None,
        "p10": float(np.percentile(interarrival, 10))
        if interarrival_minutes else None,
        "p25": float(np.percentile(interarrival, 25))
        if interarrival_minutes else None,
        "median": float(np.percentile(interarrival, 50))
        if interarrival_minutes else None,
        "p75": float(np.percentile(interarrival, 75))
        if interarrival_minutes else None,
        "p90": float(np.percentile(interarrival, 90))
        if interarrival_minutes else None,
        "p95": float(np.percentile(interarrival, 95))
        if interarrival_minutes else None,
        "p99": float(np.percentile(interarrival, 99))
        if interarrival_minutes else None,
        "p99_9": float(np.percentile(interarrival, 99.9))
        if interarrival_minutes else None,
        "max": float(np.max(interarrival))
        if interarrival_minutes else None
    }
}

DNA_PATH = (
    OUTPUT_DIR /
    "HI-Small_original_behavioral_dna.json"
)

with open(
    DNA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        behavioral_dna,
        f,
        indent=2
    )

# ============================================================
# 18. CLEANUP
# ============================================================

del accounts
del source_account_counts
del destination_account_counts
del source_bank_counts
del destination_bank_counts
del source_node_counts
del destination_node_counts
del legitimate_source_account_counts
del legitimate_destination_account_counts
del laundering_source_account_counts
del laundering_destination_account_counts
del active_account_timestamps

gc.collect()

# ============================================================
# FINAL
# ============================================================

print()
print("=" * 110)
print("✅ CONDITIONAL / BEHAVIORAL DNA AUDIT COMPLETE")
print("=" * 110)

print(
    "Master behavioral DNA:",
    DNA_PATH
)

print()
print(
    "This is now the empirical reference for the NEW generator."
)

print(
    "No synthetic transactions were generated."
)

print(
    "No original source files were modified."
)

print("=" * 110)

ORIGINAL HI-SMALL CONDITIONAL / BEHAVIORAL DNA

Transactions: /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv
Accounts    : /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_accounts.csv

1. ORIGINAL ACCOUNT / ENTITY STRUCTURE
Account rows: 518,581
Unique banks: 30,470
Unique entities: 166,207

Accounts per entity:
  min     : 1.000
  p25     : 1.000
  median  : 1.000
  mean    : 3.120
  p75     : 3.000
  p90     : 6.000
  p95     : 7.000
  p99     : 14.000
  max     : 7820.000

Entity account-count frequency:
1     102689
2      15732
3      11443
4       9840
5       8573
6       6173
7       4271
8       2524
9       1420
10       802
11       438
12       299
13       192
14       165
15       112
16       111
17        98
18        78
19        59
20        70
21        61
22        77
23        46
24        60
25        40
26        43
27        40
28        38
29        26
30        18


In [6]:
# ============================================================
# CELL 3 — ORIGINAL HI-SMALL PATTERN-WISE DNA AUDIT
# CORRECTED VERSION
#
# Source:
#   HI-Small_Patterns.txt
#
# Purpose:
#   Extract the exact structural + behavioral characteristics
#   of every original IBM laundering typology.
#
# NO files are modified.
# NO synthetic transactions are generated.
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import json
import re

# ============================================================
# 1. PATH
# ============================================================

PATTERNS_PATH = Path(
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Patterns.txt"
)

if not PATTERNS_PATH.exists():
    raise FileNotFoundError(PATTERNS_PATH)

print("=" * 110)
print("ORIGINAL HI-SMALL PATTERN-WISE DNA AUDIT")
print("=" * 110)
print("Patterns file:")
print(PATTERNS_PATH)

# ============================================================
# 2. EXACT PATTERN PARSER
# ============================================================

BEGIN_RE = re.compile(
    r"^BEGIN LAUNDERING ATTEMPT - (.+?)\s*$",
    re.IGNORECASE
)

END_RE = re.compile(
    r"^END LAUNDERING ATTEMPT - (.+?)\s*$",
    re.IGNORECASE
)

KNOWN_TYPOLOGIES = {
    "FAN-OUT",
    "FAN-IN",
    "CYCLE",
    "BIPARTITE",
    "STACK",
    "RANDOM",
    "SCATTER-GATHER",
    "GATHER-SCATTER",
}

with open(
    PATTERNS_PATH,
    "r",
    encoding="utf-8"
) as f:
    raw_lines = [
        line.rstrip("\r\n")
        for line in f
    ]

episodes = []
current = None

for line_number, raw_line in enumerate(
    raw_lines,
    start=1
):

    line = raw_line.strip()

    if not line:
        continue

    # --------------------------------------------------------
    # BEGIN
    # --------------------------------------------------------

    begin_match = BEGIN_RE.match(line)

    if begin_match:

        if current is not None:
            raise ValueError(
                f"Nested BEGIN at line {line_number}"
            )

        header = begin_match.group(1).strip()

        # Example:
        # FAN-OUT: Max 16-degree Fan-Out
        #
        # We only keep FAN-OUT.
        typology = (
            header
            .split(":", 1)[0]
            .strip()
            .upper()
        )

        if typology not in KNOWN_TYPOLOGIES:
            raise ValueError(
                f"Unknown typology {typology!r} "
                f"at line {line_number}"
            )

        current = {
            "episode_id": len(episodes) + 1,
            "typology": typology,
            "header": line,
            "begin_line": line_number,
            "transaction_lines": [],
        }

        continue

    # --------------------------------------------------------
    # END
    # --------------------------------------------------------

    end_match = END_RE.match(line)

    if end_match:

        if current is None:
            raise ValueError(
                f"END without BEGIN at line {line_number}"
            )

        end_typology = (
            end_match.group(1)
            .strip()
            .upper()
        )

        if end_typology != current["typology"]:
            raise ValueError(
                f"BEGIN/END mismatch at line {line_number}: "
                f"{current['typology']} vs {end_typology}"
            )

        current["end_line"] = line_number

        episodes.append(current)

        current = None

        continue

    # --------------------------------------------------------
    # TRANSACTION LINE
    # --------------------------------------------------------

    if current is None:
        raise ValueError(
            f"Unexpected line outside episode "
            f"at line {line_number}: {line}"
        )

    current["transaction_lines"].append(line)

# ------------------------------------------------------------
# FINAL PARSER CHECK
# ------------------------------------------------------------

if current is not None:
    raise ValueError(
        "Patterns file ended before final END."
    )

print()
print(
    "Parsed episodes:",
    f"{len(episodes):,}"
)

if len(episodes) != 370:
    raise ValueError(
        f"Expected 370 episodes, got {len(episodes)}"
    )

# ============================================================
# 3. PARSE TRANSACTION ROWS
# ============================================================

COLUMNS = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]

transaction_rows = []

for episode in episodes:

    for order, line in enumerate(
        episode["transaction_lines"],
        start=1
    ):

        parts = line.split(",")

        if len(parts) != 11:
            raise ValueError(
                f"Episode {episode['episode_id']} "
                f"contains a row with {len(parts)} fields:\n"
                f"{line}"
            )

        row = {}

        for column, value in zip(
            COLUMNS,
            parts
        ):
            row[column] = value

        row["episode_id"] = (
            episode["episode_id"]
        )

        row["typology"] = (
            episode["typology"]
        )

        row["episode_order"] = order

        transaction_rows.append(row)

patterns_df = pd.DataFrame(
    transaction_rows
)

# ============================================================
# 4. TYPE CONVERSION
# ============================================================

patterns_df["Timestamp"] = pd.to_datetime(
    patterns_df["Timestamp"],
    format="%Y/%m/%d %H:%M",
    errors="raise"
)

patterns_df["Amount Received"] = pd.to_numeric(
    patterns_df["Amount Received"],
    errors="raise"
)

patterns_df["Amount Paid"] = pd.to_numeric(
    patterns_df["Amount Paid"],
    errors="raise"
)

patterns_df["Is Laundering"] = pd.to_numeric(
    patterns_df["Is Laundering"],
    errors="raise"
).astype(int)

# Canonical graph nodes
patterns_df["src_node"] = (
    patterns_df["From Bank"].astype(str)
    + "\x1f"
    + patterns_df["Account"].astype(str)
)

patterns_df["dst_node"] = (
    patterns_df["To Bank"].astype(str)
    + "\x1f"
    + patterns_df["Account.1"].astype(str)
)

print(
    "Pattern transactions:",
    f"{len(patterns_df):,}"
)

if len(patterns_df) != 3209:
    raise ValueError(
        f"Expected 3209 pattern transactions, "
        f"got {len(patterns_df)}"
    )

if not patterns_df["Is Laundering"].eq(1).all():
    raise ValueError(
        "Patterns file contains non-laundering transaction rows."
    )

# ============================================================
# 5. GRAPH METRICS FUNCTION
# ============================================================

def calculate_graph_metrics(group):

    edges = set(
        zip(
            group["src_node"].tolist(),
            group["dst_node"].tolist()
        )
    )

    nodes = set(
        group["src_node"].tolist()
    )

    nodes.update(
        group["dst_node"].tolist()
    )

    indegree = Counter()
    outdegree = Counter()

    for src, dst in edges:

        outdegree[src] += 1
        indegree[dst] += 1

    degree_pairs = []

    for node in nodes:

        incoming = indegree.get(
            node,
            0
        )

        outgoing = outdegree.get(
            node,
            0
        )

        degree_pairs.append(
            (
                int(incoming),
                int(outgoing)
            )
        )

    degree_pairs.sort()

    intermediate_nodes = 0
    source_only_nodes = 0
    destination_only_nodes = 0
    self_loops = 0

    for node in nodes:

        incoming = indegree.get(
            node,
            0
        )

        outgoing = outdegree.get(
            node,
            0
        )

        if incoming > 0 and outgoing > 0:
            intermediate_nodes += 1

        elif incoming == 0 and outgoing > 0:
            source_only_nodes += 1

        elif incoming > 0 and outgoing == 0:
            destination_only_nodes += 1

    for src, dst in edges:

        if src == dst:
            self_loops += 1

    result = {
        "transaction_count": int(
            len(group)
        ),
        "unique_edge_count": int(
            len(edges)
        ),
        "unique_node_count": int(
            len(nodes)
        ),
        "unique_source_accounts": int(
            group["Account"].nunique()
        ),
        "unique_destination_accounts": int(
            group["Account.1"].nunique()
        ),
        "unique_source_banks": int(
            group["From Bank"].nunique()
        ),
        "unique_destination_banks": int(
            group["To Bank"].nunique()
        ),
        "unique_bank_pairs": int(
            group[
                ["From Bank", "To Bank"]
            ]
            .drop_duplicates()
            .shape[0]
        ),
        "max_in_degree": int(
            max(indegree.values())
            if indegree
            else 0
        ),
        "max_out_degree": int(
            max(outdegree.values())
            if outdegree
            else 0
        ),
        "intermediate_nodes": int(
            intermediate_nodes
        ),
        "source_only_nodes": int(
            source_only_nodes
        ),
        "destination_only_nodes": int(
            destination_only_nodes
        ),
        "self_loops": int(
            self_loops
        ),
        "degree_signature": str(
            degree_pairs
        ),
    }

    return result


# ============================================================
# 6. EPISODE-LEVEL DNA
# ============================================================

episode_records = []

for episode_id, group in patterns_df.groupby(
    "episode_id",
    sort=True
):

    group = group.sort_values(
        "episode_order"
    )

    typology = (
        group["typology"]
        .iloc[0]
    )

    graph = calculate_graph_metrics(
        group
    )

    # --------------------------------------------------------
    # Chronological timing
    # --------------------------------------------------------

    chronological = (
        group["Timestamp"]
        .sort_values()
        .reset_index(drop=True)
    )

    total_span = (
        chronological.iloc[-1]
        - chronological.iloc[0]
    )

    total_span_minutes = (
        total_span.total_seconds()
        / 60.0
    )

    if len(chronological) > 1:

        gaps = (
            chronological
            .diff()
            .dropna()
            .dt.total_seconds()
            / 60.0
        )

        gap_min = float(
            gaps.min()
        )

        gap_median = float(
            gaps.median()
        )

        gap_max = float(
            gaps.max()
        )

    else:

        gap_min = 0.0
        gap_median = 0.0
        gap_max = 0.0

    # --------------------------------------------------------
    # Amounts
    # --------------------------------------------------------

    amounts = (
        group["Amount Received"]
        .to_numpy(
            dtype=float
        )
    )

    # --------------------------------------------------------
    # Categorical distributions
    # --------------------------------------------------------

    payment_formats = (
        group["Payment Format"]
        .value_counts()
        .to_dict()
    )

    receiving_currencies = (
        group["Receiving Currency"]
        .value_counts()
        .to_dict()
    )

    payment_currencies = (
        group["Payment Currency"]
        .value_counts()
        .to_dict()
    )

    # --------------------------------------------------------
    # Final record
    # --------------------------------------------------------

    record = {
        "episode_id": int(
            episode_id
        ),
        "typology": typology,

        "transaction_count": graph[
            "transaction_count"
        ],

        "unique_edge_count": graph[
            "unique_edge_count"
        ],

        "unique_node_count": graph[
            "unique_node_count"
        ],

        "unique_source_accounts": graph[
            "unique_source_accounts"
        ],

        "unique_destination_accounts": graph[
            "unique_destination_accounts"
        ],

        "unique_source_banks": graph[
            "unique_source_banks"
        ],

        "unique_destination_banks": graph[
            "unique_destination_banks"
        ],

        "unique_bank_pairs": graph[
            "unique_bank_pairs"
        ],

        "max_in_degree": graph[
            "max_in_degree"
        ],

        "max_out_degree": graph[
            "max_out_degree"
        ],

        "intermediate_nodes": graph[
            "intermediate_nodes"
        ],

        "source_only_nodes": graph[
            "source_only_nodes"
        ],

        "destination_only_nodes": graph[
            "destination_only_nodes"
        ],

        "self_loops": graph[
            "self_loops"
        ],

        "degree_signature": graph[
            "degree_signature"
        ],

        "time_span_minutes": float(
            total_span_minutes
        ),

        "chronological_gap_min": float(
            gap_min
        ),

        "chronological_gap_median": float(
            gap_median
        ),

        "chronological_gap_max": float(
            gap_max
        ),

        "amount_min": float(
            amounts.min()
        ),

        "amount_p25": float(
            np.percentile(
                amounts,
                25
            )
        ),

        "amount_median": float(
            np.median(amounts)
        ),

        "amount_p75": float(
            np.percentile(
                amounts,
                75
            )
        ),

        "amount_max": float(
            amounts.max()
        ),

        "payment_formats": json.dumps(
            payment_formats
        ),

        "receiving_currencies": json.dumps(
            receiving_currencies
        ),

        "payment_currencies": json.dumps(
            payment_currencies
        ),
    }

    episode_records.append(
        record
    )

episode_df = pd.DataFrame(
    episode_records
)

# ============================================================
# 7. VALIDATION
# ============================================================

print()
print("=" * 110)
print("1. PATTERN VALIDATION")
print("=" * 110)

print(
    "Episodes:",
    f"{len(episode_df):,}"
)

print(
    "Transactions:",
    f"{len(patterns_df):,}"
)

print(
    "Typologies:",
    episode_df["typology"].nunique()
)

print(
    "All labels = 1:",
    patterns_df["Is Laundering"].eq(1).all()
)

print()
print("Episode count by typology:")

print(
    episode_df[
        "typology"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print()
print("Transaction count by typology:")

print(
    patterns_df[
        "typology"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 8. TYPOLOGY-WISE DNA
# ============================================================

METRICS = [
    "transaction_count",
    "unique_node_count",
    "unique_source_banks",
    "unique_destination_banks",
    "unique_bank_pairs",
    "max_in_degree",
    "max_out_degree",
    "intermediate_nodes",
    "source_only_nodes",
    "destination_only_nodes",
    "self_loops",
    "time_span_minutes",
    "chronological_gap_median",
    "amount_median",
]

typology_records = []

print()
print("=" * 110)
print("2. TYPOLOGY-WISE DNA")
print("=" * 110)

for typology in sorted(
    episode_df["typology"].unique()
):

    group = episode_df[
        episode_df["typology"] == typology
    ]

    print()
    print("-" * 110)
    print(
        f"{typology} "
        f"({len(group)} episodes)"
    )

    record = {
        "typology": typology,
        "episodes": int(
            len(group)
        ),
    }

    for metric in METRICS:

        values = group[
            metric
        ]

        median = float(
            values.median()
        )

        mean = float(
            values.mean()
        )

        p10 = float(
            values.quantile(
                0.10
            )
        )

        p90 = float(
            values.quantile(
                0.90
            )
        )

        print(
            f"  {metric:<30} "
            f"median={median:>10.3f} "
            f"mean={mean:>10.3f} "
            f"p10={p10:>10.3f} "
            f"p90={p90:>10.3f}"
        )

        record[
            metric + "_median"
        ] = median

        record[
            metric + "_mean"
        ] = mean

        record[
            metric + "_p10"
        ] = p10

        record[
            metric + "_p90"
        ] = p90

    typology_records.append(
        record
    )

typology_summary_df = pd.DataFrame(
    typology_records
)

# ============================================================
# 9. AMOUNT DNA BY TYPOLOGY
# ============================================================

print()
print("=" * 110)
print("3. AMOUNT DNA BY TYPOLOGY")
print("=" * 110)

for typology in sorted(
    patterns_df["typology"].unique()
):

    values = patterns_df.loc[
        patterns_df["typology"] == typology,
        "Amount Received"
    ].to_numpy(
        dtype=float
    )

    print()
    print(typology)

    for percentile in [
        1, 5, 10, 25, 50,
        75, 90, 95, 99
    ]:

        value = np.percentile(
            values,
            percentile
        )

        print(
            f"  p{percentile:<2}: "
            f"{value:,.6f}"
        )

# ============================================================
# 10. PAYMENT FORMAT BY TYPOLOGY
# ============================================================

print()
print("=" * 110)
print("4. PAYMENT FORMAT BY TYPOLOGY")
print("=" * 110)

for typology in sorted(
    patterns_df["typology"].unique()
):

    group = patterns_df[
        patterns_df["typology"] == typology
    ]

    counts = (
        group[
            "Payment Format"
        ]
        .value_counts()
    )

    total = counts.sum()

    output = pd.DataFrame({
        "Count": counts,
        "Share_pct": (
            counts / total * 100.0
        )
    })

    print()
    print(typology)
    print(
        output.to_string()
    )

# ============================================================
# 11. RECEIVING CURRENCY BY TYPOLOGY
# ============================================================

print()
print("=" * 110)
print("5. RECEIVING CURRENCY BY TYPOLOGY")
print("=" * 110)

for typology in sorted(
    patterns_df["typology"].unique()
):

    group = patterns_df[
        patterns_df["typology"] == typology
    ]

    counts = (
        group[
            "Receiving Currency"
        ]
        .value_counts()
    )

    total = counts.sum()

    output = pd.DataFrame({
        "Count": counts,
        "Share_pct": (
            counts / total * 100.0
        )
    })

    print()
    print(typology)
    print(
        output.to_string()
    )

# ============================================================
# 12. TIME DNA
# ============================================================

print()
print("=" * 110)
print("6. TIMING DNA BY TYPOLOGY")
print("=" * 110)

for typology in sorted(
    episode_df["typology"].unique()
):

    group = episode_df[
        episode_df["typology"] == typology
    ]

    print()
    print(typology)

    print(
        "  Episode span median:",
        f"{group['time_span_minutes'].median():.3f} min"
    )

    print(
        "  Gap median:",
        f"{group['chronological_gap_median'].median():.3f} min"
    )

    print(
        "  Gap p90:",
        f"{group['chronological_gap_median'].quantile(.90):.3f} min"
    )

    print(
        "  Maximum episode span:",
        f"{group['time_span_minutes'].max():.3f} min"
    )

# ============================================================
# 13. NODE ROLE DNA
# ============================================================

print()
print("=" * 110)
print("7. NODE ROLE DNA")
print("=" * 110)

role_output = []

for typology in sorted(
    episode_df["typology"].unique()
):

    group = episode_df[
        episode_df["typology"] == typology
    ]

    record = {
        "typology": typology,
        "episodes": int(
            len(group)
        ),
        "median_max_in_degree": float(
            group["max_in_degree"].median()
        ),
        "median_max_out_degree": float(
            group["max_out_degree"].median()
        ),
        "median_intermediate_nodes": float(
            group["intermediate_nodes"].median()
        ),
        "median_source_only_nodes": float(
            group["source_only_nodes"].median()
        ),
        "median_destination_only_nodes": float(
            group["destination_only_nodes"].median()
        ),
        "median_self_loops": float(
            group["self_loops"].median()
        ),
    }

    role_output.append(
        record
    )

role_df = pd.DataFrame(
    role_output
)

print(
    role_df.round(3).to_string(
        index=False
    )
)

# ============================================================
# 14. EPISODE SIZE MATRIX
# ============================================================

print()
print("=" * 110)
print("8. EPISODE SIZE DISTRIBUTION")
print("=" * 110)

size_matrix = (
    episode_df
    .groupby(
        [
            "typology",
            "transaction_count"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print(
    size_matrix.to_string()
)

# ============================================================
# 15. GRAPH SIGNATURE FREQUENCY
# ============================================================

print()
print("=" * 110)
print("9. GRAPH SIGNATURE FREQUENCY")
print("=" * 110)

signature_counts = (
    episode_df
    .groupby(
        [
            "typology",
            "degree_signature"
        ]
    )
    .size()
    .reset_index(
        name="episodes"
    )
    .sort_values(
        "episodes",
        ascending=False
    )
)

print(
    signature_counts
    .head(100)
    .to_string(
        index=False
    )
)

# ============================================================
# 16. SAVE OUTPUTS
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/AML_GNN_ONLY/data"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

episode_csv = (
    OUTPUT_DIR /
    "HI-Small_original_pattern_dna_episodes.csv"
)

typology_csv = (
    OUTPUT_DIR /
    "HI-Small_original_pattern_dna_typologies.csv"
)

roles_csv = (
    OUTPUT_DIR /
    "HI-Small_original_pattern_dna_node_roles.csv"
)

sizes_csv = (
    OUTPUT_DIR /
    "HI-Small_original_pattern_dna_episode_sizes.csv"
)

signatures_csv = (
    OUTPUT_DIR /
    "HI-Small_original_pattern_dna_signatures.csv"
)

episode_df.to_csv(
    episode_csv,
    index=False
)

typology_summary_df.to_csv(
    typology_csv,
    index=False
)

role_df.to_csv(
    roles_csv,
    index=False
)

size_matrix.to_csv(
    sizes_csv
)

signature_counts.to_csv(
    signatures_csv,
    index=False
)

# ============================================================
# 17. MASTER JSON
# ============================================================

master = {}

master["source_patterns"] = str(
    PATTERNS_PATH
)

master["episode_count"] = int(
    len(episode_df)
)

master["transaction_count"] = int(
    len(patterns_df)
)

master["typologies"] = {}

for typology in sorted(
    episode_df["typology"].unique()
):

    episode_count = int(
        (
            episode_df["typology"]
            == typology
        ).sum()
    )

    transaction_count = int(
        (
            patterns_df["typology"]
            == typology
        ).sum()
    )

    master["typologies"][typology] = {
        "episodes": episode_count,
        "transactions": transaction_count,
    }

master["episode_size"] = {
    "min": int(
        episode_df[
            "transaction_count"
        ].min()
    ),
    "median": float(
        episode_df[
            "transaction_count"
        ].median()
    ),
    "mean": float(
        episode_df[
            "transaction_count"
        ].mean()
    ),
    "p90": float(
        episode_df[
            "transaction_count"
        ].quantile(.90)
    ),
    "p99": float(
        episode_df[
            "transaction_count"
        ].quantile(.99)
    ),
    "max": int(
        episode_df[
            "transaction_count"
        ].max()
    ),
}

# Store role information
master["node_role_metrics"] = []

for row in role_df.to_dict(
    orient="records"
):

    cleaned = {}

    for key, value in row.items():

        if isinstance(
            value,
            (np.integer, int)
        ):
            cleaned[key] = int(value)

        elif isinstance(
            value,
            (np.floating, float)
        ):
            cleaned[key] = float(value)

        else:
            cleaned[key] = value

    master[
        "node_role_metrics"
    ].append(
        cleaned
    )

master_path = (
    OUTPUT_DIR /
    "HI-Small_original_pattern_dna.json"
)

with open(
    master_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        master,
        f,
        indent=2
    )

# ============================================================
# 18. FINAL
# ============================================================

print()
print("=" * 110)
print("✅ PATTERN-WISE DNA AUDIT COMPLETE")
print("=" * 110)

print(
    "Episode DNA CSV:",
    episode_csv
)

print(
    "Typology DNA CSV:",
    typology_csv
)

print(
    "Node-role CSV:",
    roles_csv
)

print(
    "Episode-size CSV:",
    sizes_csv
)

print(
    "Signature CSV:",
    signatures_csv
)

print(
    "Master JSON:",
    master_path
)

print()
print("No transactions generated.")
print("No source files modified.")
print("=" * 110)

ORIGINAL HI-SMALL PATTERN-WISE DNA AUDIT
Patterns file:
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Patterns.txt

Parsed episodes: 370
Pattern transactions: 3,209

1. PATTERN VALIDATION
Episodes: 370
Transactions: 3,209
Typologies: 8
All labels = 1: True

Episode count by typology:
typology
BIPARTITE         49
CYCLE             54
FAN-IN            40
FAN-OUT           48
GATHER-SCATTER    51
RANDOM            41
SCATTER-GATHER    44
STACK             43

Transaction count by typology:
typology
BIPARTITE         263
CYCLE             287
FAN-IN            318
FAN-OUT           342
GATHER-SCATTER    716
RANDOM            191
SCATTER-GATHER    626
STACK             466

2. TYPOLOGY-WISE DNA

--------------------------------------------------------------------------------------------------------------
BIPARTITE (49 episodes)
  transaction_count              median=     4.000 mean=     5.367 p10=     1.000 p90=    12.200
  unique_node_count  

In [7]:
# ============================================================
# CELL 4 — ACCOUNT ID / BANK ID GENERATION DNA
#
# PURPOSE:
#   Learn exactly how IBM HI-Small identifiers look.
#
# ACCOUNT:
#   - length
#   - character distribution by position
#   - digit vs letter probability by position
#   - common prefixes
#   - common suffixes
#
# BANK:
#   - length distribution
#   - character distribution
#   - examples
#
# This will be used later to generate NEW realistic IDs.
#
# NO SOURCE FILES MODIFIED.
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import json
import string
import re

# ============================================================
# 1. PATH
# ============================================================

ACCOUNTS_PATH = Path(
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_accounts.csv"
)

TX_PATH = Path(
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Trans.csv"
)

if not ACCOUNTS_PATH.exists():
    raise FileNotFoundError(ACCOUNTS_PATH)

if not TX_PATH.exists():
    raise FileNotFoundError(TX_PATH)

# ============================================================
# 2. LOAD ACCOUNTS
# ============================================================

accounts = pd.read_csv(
    ACCOUNTS_PATH,
    dtype=str,
    keep_default_na=False
)

accounts["Account Number"] = (
    accounts["Account Number"]
    .astype(str)
    .str.strip()
)

accounts["Bank ID"] = (
    accounts["Bank ID"]
    .astype(str)
    .str.strip()
)

print("=" * 110)
print("ACCOUNT ID / BANK ID GENERATION DNA")
print("=" * 110)

print(
    "Account rows:",
    f"{len(accounts):,}"
)

# ============================================================
# 3. ACCOUNT ID VALIDATION
# ============================================================

account_series = (
    accounts["Account Number"]
    .drop_duplicates()
)

lengths = account_series.str.len()

print()
print("=" * 110)
print("1. ACCOUNT ID VALIDATION")
print("=" * 110)

print(
    "Unique account IDs:",
    f"{len(account_series):,}"
)

print(
    "Length distribution:"
)

print(
    lengths.value_counts()
    .sort_index()
    .to_string()
)

invalid_format = account_series[
    ~account_series.str.fullmatch(
        r"[A-Z0-9]{9}"
    )
]

print()
print(
    "Invalid 9-character uppercase-alphanumeric IDs:",
    f"{len(invalid_format):,}"
)

if len(invalid_format):
    print(
        invalid_format.head(20).to_string(
            index=False
        )
    )
else:
    print(
        "✅ Every unique account ID matches [A-Z0-9]{9}."
    )

# ============================================================
# 4. CHARACTER DISTRIBUTION BY POSITION
# ============================================================

print()
print("=" * 110)
print("2. CHARACTER DISTRIBUTION BY ACCOUNT POSITION")
print("=" * 110)

position_distributions = {}

for position in range(9):

    chars = (
        account_series
        .str[position]
    )

    counts = (
        chars.value_counts()
        .sort_values(
            ascending=False
        )
    )

    total = counts.sum()

    output = pd.DataFrame({
        "Count": counts,
        "Share_pct": (
            counts / total * 100.0
        )
    })

    position_distributions[
        str(position + 1)
    ] = {
        str(k): int(v)
        for k, v in counts.items()
    }

    print()
    print(
        f"POSITION {position + 1}"
    )

    print(
        output.head(20).to_string()
    )

# ============================================================
# 5. DIGIT / LETTER PROBABILITY BY POSITION
# ============================================================

print()
print("=" * 110)
print("3. DIGIT VS LETTER PROBABILITY BY POSITION")
print("=" * 110)

position_type_rows = []

for position in range(9):

    chars = (
        account_series
        .str[position]
    )

    is_digit = chars.str.isdigit()

    digit_count = int(
        is_digit.sum()
    )

    letter_count = int(
        (~is_digit).sum()
    )

    total = (
        digit_count
        + letter_count
    )

    row = {
        "Position": position + 1,
        "Digit_Count": digit_count,
        "Digit_Pct": (
            digit_count / total * 100.0
        ),
        "Letter_Count": letter_count,
        "Letter_Pct": (
            letter_count / total * 100.0
        )
    }

    position_type_rows.append(
        row
    )

position_type_df = pd.DataFrame(
    position_type_rows
)

print(
    position_type_df.round(4)
    .to_string(index=False)
)

# ============================================================
# 6. FIRST CHARACTER
# ============================================================

print()
print("=" * 110)
print("4. ACCOUNT PREFIX DISTRIBUTION")
print("=" * 110)

for prefix_length in [
    1,
    2,
    3,
    4
]:

    prefixes = (
        account_series
        .str[:prefix_length]
        .value_counts()
        .head(30)
    )

    print()
    print(
        f"TOP {prefix_length}-CHARACTER PREFIXES"
    )

    print(
        prefixes.to_string()
    )

# ============================================================
# 7. SUFFIX DISTRIBUTION
# ============================================================

print()
print("=" * 110)
print("5. ACCOUNT SUFFIX DISTRIBUTION")
print("=" * 110)

for suffix_length in [
    1,
    2,
    3
]:

    suffixes = (
        account_series
        .str[-suffix_length:]
        .value_counts()
        .head(30)
    )

    print()
    print(
        f"TOP {suffix_length}-CHARACTER SUFFIXES"
    )

    print(
        suffixes.to_string()
    )

# ============================================================
# 8. ACCOUNT ID EXAMPLES
# ============================================================

print()
print("=" * 110)
print("6. REAL ACCOUNT ID EXAMPLES")
print("=" * 110)

sample_accounts = (
    account_series
    .sample(
        min(
            100,
            len(account_series)
        ),
        random_state=20260919
    )
)

for account in sample_accounts.tolist():
    print(
        account
    )

# ============================================================
# 9. BANK ID STRUCTURE
# ============================================================

print()
print("=" * 110)
print("7. BANK ID STRUCTURE")
print("=" * 110)

bank_series = (
    accounts["Bank ID"]
    .drop_duplicates()
)

bank_lengths = (
    bank_series
    .str.len()
)

print(
    "Unique bank IDs:",
    f"{len(bank_series):,}"
)

print()
print(
    "Bank ID length distribution:"
)

print(
    bank_lengths
    .value_counts()
    .sort_index()
    .to_string()
)

invalid_bank_ids = bank_series[
    ~bank_series.str.fullmatch(
        r"\d+"
    )
]

print()
print(
    "Non-numeric bank IDs:",
    f"{len(invalid_bank_ids):,}"
)

# ============================================================
# 10. BANK ID CHARACTER DISTRIBUTION BY POSITION
# ============================================================

print()
print("=" * 110)
print("8. BANK ID CHARACTER DISTRIBUTION")
print("=" * 110)

bank_position_distributions = {}

max_bank_length = int(
    bank_lengths.max()
)

for position in range(
    max_bank_length
):

    chars = (
        bank_series
        .str[position]
    )

    chars = chars[
        chars.notna()
    ]

    counts = (
        chars.value_counts()
        .sort_values(
            ascending=False
        )
    )

    if counts.empty:
        continue

    bank_position_distributions[
        str(position + 1)
    ] = {
        str(k): int(v)
        for k, v in counts.items()
    }

    print()
    print(
        f"BANK POSITION {position + 1}"
    )

    print(
        counts.head(15)
        .to_string()
    )

# ============================================================
# 11. BANK EXAMPLES
# ============================================================

print()
print("=" * 110)
print("9. REAL BANK ID EXAMPLES")
print("=" * 110)

for bank in (
    bank_series
    .sample(
        min(
            100,
            len(bank_series)
        ),
        random_state=20260919
    )
    .tolist()
):

    print(
        bank
    )

# ============================================================
# 12. BANK ID → ACCOUNT COUNT
# ============================================================

print()
print("=" * 110)
print("10. BANK → ACCOUNT RELATIONSHIP")
print("=" * 110)

accounts_per_bank = (
    accounts
    .groupby(
        "Bank ID"
    )
    .size()
    .sort_values(
        ascending=False
    )
)

print(
    "Median accounts/bank:",
    f"{accounts_per_bank.median():.3f}"
)

print(
    "Mean accounts/bank:",
    f"{accounts_per_bank.mean():.3f}"
)

print(
    "P90 accounts/bank:",
    f"{accounts_per_bank.quantile(.90):.3f}"
)

print(
    "P95 accounts/bank:",
    f"{accounts_per_bank.quantile(.95):.3f}"
)

print(
    "P99 accounts/bank:",
    f"{accounts_per_bank.quantile(.99):.3f}"
)

print(
    "Max accounts/bank:",
    f"{accounts_per_bank.max():,}"
)

# ============================================================
# 13. BANK-SPECIFIC ACCOUNT ID CHARACTERISTICS
# ============================================================

print()
print("=" * 110)
print("11. BANK-SPECIFIC ACCOUNT PREFIX CHECK")
print("=" * 110)

# We don't want to assume that account ID prefixes encode
# banks. Check statistically.

bank_prefix_records = []

for bank_id, group in (
    accounts
    .groupby("Bank ID")
):

    accs = (
        group["Account Number"]
        .astype(str)
    )

    if len(accs) < 5:
        continue

    prefix3_unique = (
        accs
        .str[:3]
        .nunique()
    )

    prefix3_top_share = (
        accs
        .str[:3]
        .value_counts()
        .iloc[0]
        / len(accs)
    )

    bank_prefix_records.append({
        "Bank ID": bank_id,
        "Account_Count": len(accs),
        "Unique_3char_Prefixes": int(
            prefix3_unique
        ),
        "Top_3char_Prefix_Share_pct": (
            prefix3_top_share * 100.0
        )
    })

bank_prefix_df = pd.DataFrame(
    bank_prefix_records
)

if not bank_prefix_df.empty:

    print(
        bank_prefix_df[
            "Top_3char_Prefix_Share_pct"
        ]
        .describe()
        .round(4)
        .to_string()
    )

    print()
    print(
        "Banks with strongest 3-char account-prefix "
        "concentration:"
    )

    print(
        bank_prefix_df
        .sort_values(
            "Top_3char_Prefix_Share_pct",
            ascending=False
        )
        .head(30)
        .to_string(index=False)
    )

# ============================================================
# 14. SAVE ID DNA
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/AML_GNN_ONLY/data"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

position_type_path = (
    OUTPUT_DIR /
    "HI-Small_account_id_position_types.csv"
)

position_type_df.to_csv(
    position_type_path,
    index=False
)

accounts_per_bank_path = (
    OUTPUT_DIR /
    "HI-Small_accounts_per_bank.csv"
)

accounts_per_bank.to_csv(
    accounts_per_bank_path,
    header=["Account_Count"]
)

if not bank_prefix_df.empty:

    bank_prefix_path = (
        OUTPUT_DIR /
        "HI-Small_bank_account_prefix_behavior.csv"
    )

    bank_prefix_df.to_csv(
        bank_prefix_path,
        index=False
    )

# ============================================================
# 15. MASTER JSON
# ============================================================

id_dna = {
    "account_format": {
        "length": 9,
        "regex": "[A-Z0-9]{9}",
        "unique_account_ids": int(
            len(account_series)
        ),
        "position_character_distribution":
            position_distributions,
        "position_digit_letter":
            position_type_df.to_dict(
                orient="records"
            )
    },

    "bank_format": {
        "unique_bank_ids": int(
            len(bank_series)
        ),
        "length_distribution": {
            str(k): int(v)
            for k, v in (
                bank_lengths
                .value_counts()
                .sort_index()
                .items()
            )
        },
        "all_numeric": bool(
            len(invalid_bank_ids) == 0
        ),
        "position_distribution":
            bank_position_distributions
    },

    "bank_account_relationship": {
        "median_accounts_per_bank": float(
            accounts_per_bank.median()
        ),
        "mean_accounts_per_bank": float(
            accounts_per_bank.mean()
        ),
        "p90_accounts_per_bank": float(
            accounts_per_bank.quantile(.90)
        ),
        "p95_accounts_per_bank": float(
            accounts_per_bank.quantile(.95)
        ),
        "p99_accounts_per_bank": float(
            accounts_per_bank.quantile(.99)
        ),
        "max_accounts_per_bank": int(
            accounts_per_bank.max()
        )
    }
}

id_dna_path = (
    OUTPUT_DIR /
    "HI-Small_original_identifier_dna.json"
)

with open(
    id_dna_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        id_dna,
        f,
        indent=2
    )

# ============================================================
# 16. FINAL
# ============================================================

print()
print("=" * 110)
print("✅ IDENTIFIER DNA AUDIT COMPLETE")
print("=" * 110)

print(
    "Identifier DNA:",
    id_dna_path
)

print(
    "Position model:",
    position_type_path
)

print(
    "Accounts/bank:",
    accounts_per_bank_path
)

if not bank_prefix_df.empty:
    print(
        "Bank-prefix behavior:",
        bank_prefix_path
    )

print()
print(
    "NO NEW ACCOUNTS WERE CREATED."
)

print(
    "NO TRANSACTIONS WERE CREATED."
)

print(
    "NO ORIGINAL FILE WAS MODIFIED."
)

print("=" * 110)

ACCOUNT ID / BANK ID GENERATION DNA
Account rows: 518,581

1. ACCOUNT ID VALIDATION
Unique account IDs: 518,573
Length distribution:
Account Number
9    518573

Invalid 9-character uppercase-alphanumeric IDs: 0
✅ Every unique account ID matches [A-Z0-9]{9}.

2. CHARACTER DISTRIBUTION BY ACCOUNT POSITION

POSITION 1
                 Count  Share_pct
Account Number                   
8               518558  99.997107
1                   15   0.002893

POSITION 2
                 Count  Share_pct
Account Number                   
0               403414  77.793098
1               115159  22.206902

POSITION 3
                Count  Share_pct
Account Number                  
3               50670   9.771045
0               50396   9.718207
1               50142   9.669227
2               49749   9.593442
4               40812   7.870059
5               25444   4.906542
6               25268   4.872602
7               25255   4.870095
A               25224   4.864117
E               25191   

In [8]:
# ============================================================
# CELL 5 — REAL TRANSACTION BEHAVIOR SAMPLER / DNA
# ============================================================
# Purpose:
#   Learn the empirical conditional distributions from ORIGINAL
#   HI-Small so the future synthetic laundering generator
#   behaves like the real dataset.
#
# IMPORTANT:
#   - Reads ORIGINAL HI-Small only
#   - Creates NO synthetic transactions
#   - Modifies NO source files
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
WORK_DIR  = "/kaggle/working/AML_GNN_ONLY/data"

TRANS_PATH = os.path.join(DATA_ROOT, "HI-Small_Trans.csv")
OUT_JSON   = os.path.join(WORK_DIR, "HI-Small_transaction_behavior_sampler.json")

os.makedirs(WORK_DIR, exist_ok=True)

print("=" * 100)
print("CELL 5 — REAL TRANSACTION BEHAVIOR SAMPLER")
print("=" * 100)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
usecols = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering"
]

df = pd.read_csv(
    TRANS_PATH,
    usecols=usecols
)

df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

print("\nLoaded rows:", len(df))
print("Columns:", list(df.columns))

# ------------------------------------------------------------
# BASIC CLEANING FOR DNA ESTIMATION ONLY
# ------------------------------------------------------------
df["From Bank"] = df["From Bank"].astype(str)
df["To Bank"] = df["To Bank"].astype(str)
df["Account"] = df["Account"].astype(str)
df["Account.1"] = df["Account.1"].astype(str)

df["Amount Received"] = pd.to_numeric(
    df["Amount Received"], errors="coerce"
)

df["Amount Paid"] = pd.to_numeric(
    df["Amount Paid"], errors="coerce"
)

df["Is Laundering"] = pd.to_numeric(
    df["Is Laundering"], errors="coerce"
).fillna(0).astype(int)

df["Receiving Currency"] = df["Receiving Currency"].astype(str)
df["Payment Currency"] = df["Payment Currency"].astype(str)
df["Payment Format"] = df["Payment Format"].astype(str)

df = df.dropna(subset=[
    "Timestamp",
    "Amount Received",
    "Amount Paid"
]).copy()

# ------------------------------------------------------------
# TIME FEATURES
# ------------------------------------------------------------
df["hour"] = df["Timestamp"].dt.hour
df["minute"] = df["Timestamp"].dt.minute
df["weekday"] = df["Timestamp"].dt.dayofweek
df["date"] = df["Timestamp"].dt.date

# Minute-of-day is useful for realistic timestamp generation
df["minute_of_day"] = (
    df["Timestamp"].dt.hour * 60
    + df["Timestamp"].dt.minute
)

# ------------------------------------------------------------
# AMOUNT FEATURES
# ------------------------------------------------------------
# Log transform is much more useful for this heavy-tailed
# transaction distribution.
df["log_amount_received"] = np.log1p(
    np.clip(df["Amount Received"].values, 0, None)
)

df["log_amount_paid"] = np.log1p(
    np.clip(df["Amount Paid"].values, 0, None)
)

# ------------------------------------------------------------
# HELPER
# ------------------------------------------------------------
def value_counts_pct(series):
    vc = series.value_counts(dropna=False)
    total = vc.sum()

    return {
        str(k): {
            "count": int(v),
            "share_pct": float(v / total * 100)
        }
        for k, v in vc.items()
    }

def quantiles(series):
    s = pd.to_numeric(series, errors="coerce").dropna()

    q = s.quantile(
        [0.001, 0.01, 0.05, 0.10, 0.25,
         0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )

    return {
        str(k): float(v)
        for k, v in q.items()
    }

# ============================================================
# 1. GLOBAL TIME DNA
# ============================================================
print("\n" + "=" * 80)
print("1. GLOBAL TIME DNA")
print("=" * 80)

time_dna = {
    "hour_distribution": value_counts_pct(df["hour"]),
    "weekday_distribution": value_counts_pct(df["weekday"]),
    "minute_of_day_distribution": value_counts_pct(df["minute_of_day"])
}

print("\nTop transaction hours:")
print(
    df["hour"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 2. AMOUNT DNA
# ============================================================
print("\n" + "=" * 80)
print("2. AMOUNT DNA")
print("=" * 80)

amount_dna = {
    "received": {
        "quantiles": quantiles(df["Amount Received"]),
        "log_quantiles": quantiles(df["log_amount_received"])
    },
    "paid": {
        "quantiles": quantiles(df["Amount Paid"]),
        "log_quantiles": quantiles(df["log_amount_paid"])
    }
}

print("\nAmount Received quantiles:")
print(pd.Series(amount_dna["received"]["quantiles"]))

# ============================================================
# 3. CURRENCY DNA
# ============================================================
print("\n" + "=" * 80)
print("3. CURRENCY DNA")
print("=" * 80)

currency_dna = {
    "receiving_currency": value_counts_pct(df["Receiving Currency"]),
    "payment_currency": value_counts_pct(df["Payment Currency"])
}

print("\nReceiving Currency:")
print(
    df["Receiving Currency"]
    .value_counts(normalize=True)
    .mul(100)
    .round(4)
    .to_string()
)

print("\nPayment Currency:")
print(
    df["Payment Currency"]
    .value_counts(normalize=True)
    .mul(100)
    .round(4)
    .to_string()
)

# ============================================================
# 4. PAYMENT FORMAT DNA
# ============================================================
print("\n" + "=" * 80)
print("4. PAYMENT FORMAT DNA")
print("=" * 80)

payment_format_dna = value_counts_pct(df["Payment Format"])

print(
    df["Payment Format"]
    .value_counts(normalize=True)
    .mul(100)
    .round(4)
    .to_string()
)

# ============================================================
# 5. CONDITIONAL PAYMENT FORMAT -> CURRENCY
# ============================================================
print("\n" + "=" * 80)
print("5. CONDITIONAL PAYMENT/CURRENCY DNA")
print("=" * 80)

format_currency = (
    df.groupby(
        ["Payment Format", "Payment Currency"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

format_currency["share_pct"] = (
    format_currency["count"]
    /
    format_currency.groupby("Payment Format")["count"]
        .transform("sum")
    * 100
)

format_currency_map = {}

for fmt, g in format_currency.groupby("Payment Format"):
    format_currency_map[str(fmt)] = {
        str(row["Payment Currency"]): {
            "count": int(row["count"]),
            "share_pct": float(row["share_pct"])
        }
        for _, row in g.iterrows()
    }

# ------------------------------------------------------------
# Receiving currency conditioned on payment currency
# ------------------------------------------------------------
recv_pay = (
    df.groupby(
        ["Payment Currency", "Receiving Currency"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

recv_pay["share_pct"] = (
    recv_pay["count"]
    /
    recv_pay.groupby("Payment Currency")["count"]
        .transform("sum")
    * 100
)

recv_pay_map = {}

for pay, g in recv_pay.groupby("Payment Currency"):
    recv_pay_map[str(pay)] = {
        str(row["Receiving Currency"]): {
            "count": int(row["count"]),
            "share_pct": float(row["share_pct"])
        }
        for _, row in g.iterrows()
    }

# ============================================================
# 6. PAYMENT FORMAT -> LAUNDERING RATE
# ============================================================
print("\n" + "=" * 80)
print("6. LAUNDERING RATE BY PAYMENT FORMAT")
print("=" * 80)

fmt_laundering = (
    df.groupby("Payment Format")["Is Laundering"]
    .agg(["count", "sum", "mean"])
)

fmt_laundering["rate_pct"] = (
    fmt_laundering["mean"] * 100
)

print(fmt_laundering.to_string())

# ============================================================
# 7. CURRENCY -> LAUNDERING RATE
# ============================================================
print("\n" + "=" * 80)
print("7. LAUNDERING RATE BY CURRENCY")
print("=" * 80)

currency_laundering = (
    df.groupby("Payment Currency")["Is Laundering"]
    .agg(["count", "sum", "mean"])
)

currency_laundering["rate_pct"] = (
    currency_laundering["mean"] * 100
)

print(currency_laundering.to_string())

# ============================================================
# 8. BANK RELATIONSHIP DNA
# ============================================================
print("\n" + "=" * 80)
print("8. BANK RELATIONSHIP DNA")
print("=" * 80)

# Global From-Bank frequency
from_bank_counts = (
    df["From Bank"]
    .value_counts()
)

to_bank_counts = (
    df["To Bank"]
    .value_counts()
)

bank_pair_counts = (
    df.groupby(
        ["From Bank", "To Bank"],
        sort=False
    )
    .size()
    .reset_index(name="count")
)

# Store top empirical pairs only.
# We do NOT need all 283k pairs in the JSON.
top_pairs = (
    bank_pair_counts
    .sort_values("count", ascending=False)
    .head(5000)
)

top_bank_pairs = [
    {
        "from_bank": str(row["From Bank"]),
        "to_bank": str(row["To Bank"]),
        "count": int(row["count"])
    }
    for _, row in top_pairs.iterrows()
]

bank_dna = {
    "unique_from_banks": int(df["From Bank"].nunique()),
    "unique_to_banks": int(df["To Bank"].nunique()),
    "unique_bank_pairs": int(
        bank_pair_counts.shape[0]
    ),
    "from_bank_top": {
        str(k): int(v)
        for k, v in from_bank_counts.head(1000).items()
    },
    "to_bank_top": {
        str(k): int(v)
        for k, v in to_bank_counts.head(1000).items()
    },
    "top_5000_bank_pairs": top_bank_pairs
}

print("Unique From Banks:", bank_dna["unique_from_banks"])
print("Unique To Banks:", bank_dna["unique_to_banks"])
print("Unique Bank Pairs:", bank_dna["unique_bank_pairs"])

# ============================================================
# 9. TRANSACTION AMOUNT BY PAYMENT FORMAT
# ============================================================
print("\n" + "=" * 80)
print("9. AMOUNT DNA BY PAYMENT FORMAT")
print("=" * 80)

amount_by_format = {}

for fmt, g in df.groupby("Payment Format"):
    amount_by_format[str(fmt)] = {
        "count": int(len(g)),
        "received_quantiles": quantiles(g["Amount Received"]),
        "paid_quantiles": quantiles(g["Amount Paid"])
    }

# ============================================================
# 10. AMOUNT DNA BY CURRENCY
# ============================================================
print("\n" + "=" * 80)
print("10. AMOUNT DNA BY PAYMENT CURRENCY")
print("=" * 80)

amount_by_currency = {}

for cur, g in df.groupby("Payment Currency"):
    amount_by_currency[str(cur)] = {
        "count": int(len(g)),
        "received_quantiles": quantiles(g["Amount Received"]),
        "paid_quantiles": quantiles(g["Amount Paid"])
    }

# ============================================================
# 11. LEGITIMATE-ONLY DNA
# ============================================================
print("\n" + "=" * 80)
print("11. LEGITIMATE-ONLY BEHAVIOR DNA")
print("=" * 80)

legit = df[df["Is Laundering"] == 0].copy()

legit_dna = {
    "rows": int(len(legit)),
    "amount_received_quantiles": quantiles(
        legit["Amount Received"]
    ),
    "amount_paid_quantiles": quantiles(
        legit["Amount Paid"]
    ),
    "hour_distribution": value_counts_pct(
        legit["hour"]
    ),
    "payment_format_distribution": value_counts_pct(
        legit["Payment Format"]
    ),
    "receiving_currency_distribution": value_counts_pct(
        legit["Receiving Currency"]
    ),
    "payment_currency_distribution": value_counts_pct(
        legit["Payment Currency"]
    )
}

# ============================================================
# 12. ORIGINAL LAUNDERING-ONLY DNA
# ============================================================
print("\n" + "=" * 80)
print("12. ORIGINAL LAUNDERING-ONLY DNA")
print("=" * 80)

laund = df[df["Is Laundering"] == 1].copy()

laund_dna = {
    "rows": int(len(laund)),
    "amount_received_quantiles": quantiles(
        laund["Amount Received"]
    ),
    "amount_paid_quantiles": quantiles(
        laund["Amount Paid"]
    ),
    "hour_distribution": value_counts_pct(
        laund["hour"]
    ),
    "payment_format_distribution": value_counts_pct(
        laund["Payment Format"]
    ),
    "receiving_currency_distribution": value_counts_pct(
        laund["Receiving Currency"]
    ),
    "payment_currency_distribution": value_counts_pct(
        laund["Payment Currency"]
    )
}

print("Original laundering rows:", len(laund))

# ============================================================
# 13. FINAL DNA MANIFEST
# ============================================================
dna = {
    "dataset": "IBM HI-Small",
    "source": TRANS_PATH,
    "rows_loaded": int(len(df)),
    "legitimate_rows": int((df["Is Laundering"] == 0).sum()),
    "laundering_rows": int((df["Is Laundering"] == 1).sum()),

    "time": time_dna,

    "amount": amount_dna,

    "currency": currency_dna,

    "payment_format": payment_format_dna,

    "payment_format_to_currency": format_currency_map,

    "payment_currency_to_receiving_currency": recv_pay_map,

    "payment_format_laundering_rate": {
        str(idx): {
            "count": int(row["count"]),
            "laundering_count": int(row["sum"]),
            "rate_pct": float(row["rate_pct"])
        }
        for idx, row in fmt_laundering.iterrows()
    },

    "payment_currency_laundering_rate": {
        str(idx): {
            "count": int(row["count"]),
            "laundering_count": int(row["sum"]),
            "rate_pct": float(row["rate_pct"])
        }
        for idx, row in currency_laundering.iterrows()
    },

    "bank": bank_dna,

    "amount_by_payment_format": amount_by_format,

    "amount_by_payment_currency": amount_by_currency,

    "legitimate_dna": legit_dna,

    "original_laundering_dna": laund_dna
}

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
with open(OUT_JSON, "w") as f:
    json.dump(dna, f, indent=2)

print("\n" + "=" * 100)
print("✅ CELL 5 COMPLETE")
print("=" * 100)
print("Saved:", OUT_JSON)
print("Rows used:", len(df))
print("No synthetic transactions created.")
print("No original file modified.")
print("=" * 100)

CELL 5 — REAL TRANSACTION BEHAVIOR SAMPLER

Loaded rows: 5078345
Columns: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

1. GLOBAL TIME DNA

Top transaction hours:
hour
0     634726
1     193728
2     193143
3     193249
4     193166
5     193900
6     194456
7     193336
8     192921
9     192952
10    193229
11    193500
12    192972
13    192521
14    192787
15    194871
16    193407
17    193154
18    193321
19    192824
20    192809
21    192192
22    191836
23    193345

2. AMOUNT DNA

Amount Received quantiles:
0.001    2.463440e-04
0.01     1.680776e-02
0.05     6.850000e+00
0.1      2.501000e+01
0.25     1.833700e+02
0.5      1.411010e+03
0.75     1.234627e+04
0.9      1.407368e+05
0.95     6.499065e+05
0.99     1.475123e+07
0.999    5.226685e+08
dtype: float64

3. CURRENCY DNA

Receiving Currency:
Receiving Currency
US Dollar            37.0070
Euro  

In [11]:
# ============================================================
# CELL 6 — EXACT IBM LAUNDERING PATTERN TEMPLATES
# ULTRA-ROBUST IBM BLOCK PARSER
# ============================================================

import os
import re
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

PATTERN_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_Patterns.txt"
)

OUT_JSON = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_templates.json"
)

OUT_SUMMARY = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_template_summary.csv"
)

os.makedirs(WORK_DIR, exist_ok=True)

print("=" * 100)
print("CELL 6 — EXACT IBM LAUNDERING PATTERN TEMPLATES")
print("ULTRA-ROBUST IBM BLOCK PARSER")
print("=" * 100)

# ============================================================
# 1. READ FILE
# ============================================================
with open(
    PATTERN_PATH,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:
    raw_lines = f.readlines()

print("\nTotal lines:", len(raw_lines))

# Normalize only for parsing.
# Original transaction content is retained separately.
lines = []

for i, raw in enumerate(raw_lines, start=1):

    normalized = (
        raw
        .replace("\ufeff", "")
        .replace("\xa0", " ")
        .replace("\r", "")
        .replace("\n", "")
        .strip()
    )

    lines.append(
        {
            "line_no": i,
            "raw": raw.rstrip("\r\n"),
            "text": normalized
        }
    )

# ============================================================
# 2. HELPERS
# ============================================================
def is_id_line(text):
    """
    Detect:
        id="xxxxxx"
    even if spacing is unusual.
    """
    return (
        text.lower().startswith("id")
        and "=" in text
        and '"' in text
    )


def extract_id(text):

    m = re.search(
        r'id\s*=\s*"([^"]+)"',
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1)

    return None


def is_begin_line(text):

    upper = text.upper()

    return (
        "BEGIN" in upper
        and "LAUNDERING" in upper
        and "ATTEMPT" in upper
    )


def is_end_line(text):

    upper = text.upper()

    return (
        "END" in upper
        and "LAUNDERING" in upper
        and "ATTEMPT" in upper
    )


def extract_typology(text):

    """
    Handles formats such as:

    BEGIN LAUNDERING ATTEMPT - FAN-OUT: ...
    BEGIN LAUNDERING ATTEMPT-FAN-OUT: ...
    BEGIN LAUNDERING ATTEMPT - FAN-OUT
    """

    upper = text.upper()

    marker = "LAUNDERING ATTEMPT"

    if marker not in upper:
        return None

    tail = text[
        upper.index(marker) + len(marker):
    ].strip()

    # Remove leading separators
    tail = tail.lstrip(" :-=\t")

    # Anything before ':' is the typology
    typology = tail.split(":", 1)[0].strip()

    # Remove accidental trailing separators
    typology = typology.strip(" :-=")

    if not typology:
        return None

    return typology.upper()


def parse_transaction(text):

    """
    IBM pattern rows contain exactly 11 comma-separated fields.
    """

    parts = [
        x.strip()
        for x in text.split(",")
    ]

    if len(parts) != 11:
        return None

    timestamp = parts[0]

    # Timestamp validation
    ts = pd.to_datetime(
        timestamp,
        format="%Y/%m/%d %H:%M",
        errors="coerce"
    )

    if pd.isna(ts):
        return None

    try:

        amount_received = float(
            parts[5]
        )

        amount_paid = float(
            parts[7]
        )

        label = int(
            parts[10]
        )

    except Exception:

        return None

    return {

        "timestamp": timestamp,

        "from_bank": str(
            parts[1]
        ),

        "from_account": str(
            parts[2]
        ),

        "to_bank": str(
            parts[3]
        ),

        "to_account": str(
            parts[4]
        ),

        "amount_received": amount_received,

        "receiving_currency": str(
            parts[6]
        ),

        "amount_paid": amount_paid,

        "payment_currency": str(
            parts[8]
        ),

        "payment_format": str(
            parts[9]
        ),

        "label": label
    }


# ============================================================
# 3. PARSE EPISODE BLOCKS
# ============================================================
#
# KEY CHANGE:
#
# We no longer require the BEGIN line to match an exact regex.
#
# We identify the BEGIN/END markers based on their textual
# content and parse everything between them.
# ============================================================

episodes = []

current = None
pending_id = None

for row in lines:

    line_no = row["line_no"]
    text = row["text"]

    if not text:
        continue

    # --------------------------------------------------------
    # ID
    # --------------------------------------------------------
    if is_id_line(text):

        extracted = extract_id(text)

        if extracted is not None:
            pending_id = extracted

        continue

    # --------------------------------------------------------
    # BEGIN
    # --------------------------------------------------------
    if is_begin_line(text):

        if current is not None:

            raise RuntimeError(
                f"\nNested BEGIN detected.\n"
                f"Line: {line_no}\n"
                f"Content: {text}\n"
                f"Current episode: "
                f"{current['typology']}"
            )

        typology = extract_typology(text)

        if typology is None:

            raise RuntimeError(
                f"\nCould not extract typology "
                f"from BEGIN line.\n"
                f"Line: {line_no}\n"
                f"Content: {text}"
            )

        current = {

            "episode_index": len(
                episodes
            ),

            "episode_id": pending_id,

            "typology": typology,

            "transactions": [],

            "source_begin_line": line_no
        }

        pending_id = None

        continue

    # --------------------------------------------------------
    # END
    # --------------------------------------------------------
    if is_end_line(text):

        if current is None:

            raise RuntimeError(
                f"\nEND marker encountered "
                f"without BEGIN.\n"
                f"Line: {line_no}\n"
                f"Content: {text}"
            )

        end_typology = extract_typology(text)

        if (
            end_typology is not None
            and end_typology != current["typology"]
        ):

            raise RuntimeError(
                f"\nTypology mismatch.\n"
                f"BEGIN: {current['typology']}\n"
                f"END: {end_typology}\n"
                f"Line: {line_no}"
            )

        if len(
            current["transactions"]
        ) == 0:

            raise RuntimeError(
                f"\nEmpty episode.\n"
                f"Episode: "
                f"{current['episode_index']}\n"
                f"Typology: "
                f"{current['typology']}"
            )

        current[
            "source_end_line"
        ] = line_no

        episodes.append(
            current
        )

        current = None

        continue

    # --------------------------------------------------------
    # TRANSACTION
    # --------------------------------------------------------
    tx = parse_transaction(text)

    if tx is not None:

        if current is None:

            raise RuntimeError(
                f"\nTransaction found outside "
                f"laundering episode.\n"
                f"Line: {line_no}\n"
                f"Content: {text}"
            )

        tx[
            "source_order"
        ] = len(
            current["transactions"]
        )

        tx[
            "source_line"
        ] = line_no

        current[
            "transactions"
        ].append(tx)

        continue

# ============================================================
# 4. FINAL STATE VALIDATION
# ============================================================
if current is not None:

    raise RuntimeError(
        "\nFile ended while an episode "
        "was still open.\n"
        f"Episode: "
        f"{current['episode_index']}\n"
        f"Typology: "
        f"{current['typology']}"
    )

print("\nParsed episodes:", len(episodes))

total_transactions = sum(
    len(ep["transactions"])
    for ep in episodes
)

print(
    "Parsed pattern transactions:",
    total_transactions
)

# ============================================================
# 5. EXPECTED IBM VALUES
# ============================================================
print("\n" + "=" * 80)
print("EXPECTED VS ACTUAL")
print("=" * 80)

print(
    "Expected episodes:       370"
)

print(
    "Actual episodes:        ",
    len(episodes)
)

print(
    "Expected transactions:   3209"
)

print(
    "Actual transactions:    ",
    total_transactions
)

# ============================================================
# 6. TYPOLOGY COUNTS
# ============================================================
typology_episode_counts = {}

typology_transaction_counts = {}

for ep in episodes:

    typ = ep["typology"]

    typology_episode_counts[typ] = (
        typology_episode_counts.get(
            typ,
            0
        ) + 1
    )

    typology_transaction_counts[typ] = (
        typology_transaction_counts.get(
            typ,
            0
        )
        +
        len(ep["transactions"])
    )

print("\nEpisode counts:")

for typ in sorted(
    typology_episode_counts
):

    print(
        f"{typ:20s}",
        typology_episode_counts[typ]
    )

print("\nTransaction counts:")

for typ in sorted(
    typology_transaction_counts
):

    print(
        f"{typ:20s}",
        typology_transaction_counts[typ]
    )

# ============================================================
# 7. BUILD GRAPH TEMPLATES
# ============================================================
templates = []
summary_rows = []

for ep in episodes:

    txs = ep["transactions"]

    # --------------------------------------------------------
    # Local node mapping
    # --------------------------------------------------------
    node_map = {}
    nodes = []

    def get_node(
        bank,
        account
    ):

        key = (
            str(bank),
            str(account)
        )

        if key not in node_map:

            node_map[key] = len(
                nodes
            )

            nodes.append(
                {
                    "node_id": len(nodes),
                    "original_bank": str(bank),
                    "original_account": str(account)
                }
            )

        return node_map[key]

    # --------------------------------------------------------
    # Timestamps
    # --------------------------------------------------------
    timestamps = pd.to_datetime(
        [
            tx["timestamp"]
            for tx in txs
        ],
        format="%Y/%m/%d %H:%M",
        errors="coerce"
    )

    if timestamps.isna().any():

        raise RuntimeError(
            f"Timestamp failure in "
            f"episode {ep['episode_index']}"
        )

    min_ts = timestamps.min()
    max_ts = timestamps.max()

    span_minutes = (
        max_ts - min_ts
    ).total_seconds() / 60.0

    # --------------------------------------------------------
    # Amount reference
    # --------------------------------------------------------
    amounts = np.array(
        [
            tx["amount_received"]
            for tx in txs
        ],
        dtype=float
    )

    median_amount = float(
        np.median(amounts)
    )

    if median_amount <= 0:
        median_amount = 1.0

    # --------------------------------------------------------
    # Transactions
    # --------------------------------------------------------
    template_transactions = []

    for order, (
        tx,
        ts
    ) in enumerate(
        zip(txs, timestamps)
    ):

        source_node = get_node(
            tx["from_bank"],
            tx["from_account"]
        )

        destination_node = get_node(
            tx["to_bank"],
            tx["to_account"]
        )

        relative_minutes = (
            ts - min_ts
        ).total_seconds() / 60.0

        amount_ratio = (
            tx["amount_received"]
            /
            median_amount
        )

        template_transactions.append(
            {

                "source_order": int(
                    order
                ),

                "source_node": int(
                    source_node
                ),

                "destination_node": int(
                    destination_node
                ),

                "relative_minutes": float(
                    relative_minutes
                ),

                "amount_received": float(
                    tx["amount_received"]
                ),

                "amount_paid": float(
                    tx["amount_paid"]
                ),

                "amount_ratio_to_episode_median": float(
                    amount_ratio
                ),

                "receiving_currency": (
                    tx["receiving_currency"]
                ),

                "payment_currency": (
                    tx["payment_currency"]
                ),

                "payment_format": (
                    tx["payment_format"]
                ),

                "label": int(
                    tx["label"]
                )
            }
        )

    # ========================================================
    # GRAPH STATISTICS
    # ========================================================
    in_degree = {
        i: 0
        for i in range(len(nodes))
    }

    out_degree = {
        i: 0
        for i in range(len(nodes))
    }

    adjacency = {
        i: []
        for i in range(len(nodes))
    }

    edges = []

    for tx in template_transactions:

        s = tx["source_node"]
        d = tx["destination_node"]

        out_degree[s] += 1
        in_degree[d] += 1

        adjacency[s].append(
            d
        )

        edges.append(
            {
                "source_node": int(s),
                "destination_node": int(d)
            }
        )

    # --------------------------------------------------------
    # Cycle detection
    # --------------------------------------------------------
    def detect_cycle(
        adjacency,
        n
    ):

        WHITE = 0
        GREY = 1
        BLACK = 2

        state = {
            i: WHITE
            for i in range(n)
        }

        def dfs(u):

            state[u] = GREY

            for v in adjacency[u]:

                if state[v] == GREY:
                    return True

                if state[v] == WHITE:

                    if dfs(v):
                        return True

            state[u] = BLACK

            return False

        for i in range(n):

            if state[i] == WHITE:

                if dfs(i):
                    return True

        return False

    cycle_present = detect_cycle(
        adjacency,
        len(nodes)
    )

    # --------------------------------------------------------
    # Bank counts
    # --------------------------------------------------------
    source_banks = set(
        tx["from_bank"]
        for tx in txs
    )

    destination_banks = set(
        tx["to_bank"]
        for tx in txs
    )

    # ========================================================
    # TEMPLATE
    # ========================================================
    template_id = (
        ep["episode_id"]
        if ep["episode_id"]
        else
        f"{ep['typology']}_{ep['episode_index']:04d}"
    )

    template = {

        "template_id": str(
            template_id
        ),

        "episode_index": int(
            ep["episode_index"]
        ),

        "source_episode_id": (
            str(ep["episode_id"])
            if ep["episode_id"]
            else None
        ),

        "typology": ep[
            "typology"
        ],

        "transaction_count": int(
            len(template_transactions)
        ),

        "node_count": int(
            len(nodes)
        ),

        "source_bank_count": int(
            len(source_banks)
        ),

        "destination_bank_count": int(
            len(destination_banks)
        ),

        "max_in_degree": int(
            max(in_degree.values())
        ),

        "max_out_degree": int(
            max(out_degree.values())
        ),

        "cycle_present": bool(
            cycle_present
        ),

        "episode_span_minutes": float(
            span_minutes
        ),

        "median_amount": float(
            median_amount
        ),

        "nodes": [
            {
                "node_id": int(
                    node["node_id"]
                )
            }
            for node in nodes
        ],

        "edges": edges,

        "transactions": template_transactions
    }

    templates.append(
        template
    )

    summary_rows.append(
        {

            "template_id": str(
                template_id
            ),

            "episode_index": int(
                ep["episode_index"]
            ),

            "source_episode_id": (
                str(ep["episode_id"])
                if ep["episode_id"]
                else None
            ),

            "typology": ep[
                "typology"
            ],

            "transaction_count": len(
                template_transactions
            ),

            "node_count": len(
                nodes
            ),

            "source_bank_count": len(
                source_banks
            ),

            "destination_bank_count": len(
                destination_banks
            ),

            "max_in_degree": max(
                in_degree.values()
            ),

            "max_out_degree": max(
                out_degree.values()
            ),

            "cycle_present": cycle_present,

            "episode_span_minutes": span_minutes,

            "median_amount": median_amount
        }
    )

# ============================================================
# 8. SUMMARY DATAFRAME
# ============================================================
summary_df = pd.DataFrame(
    summary_rows
)

# ============================================================
# 9. IBM PATTERN SUMMARY
# ============================================================
print("\n" + "=" * 100)
print("PATTERN TEMPLATE SUMMARY")
print("=" * 100)

typology_summary = (
    summary_df
    .groupby("typology")
    .agg(
        episodes=(
            "template_id",
            "count"
        ),

        transactions=(
            "transaction_count",
            "sum"
        ),

        median_transactions=(
            "transaction_count",
            "median"
        ),

        median_nodes=(
            "node_count",
            "median"
        ),

        median_source_banks=(
            "source_bank_count",
            "median"
        ),

        median_destination_banks=(
            "destination_bank_count",
            "median"
        ),

        median_span_minutes=(
            "episode_span_minutes",
            "median"
        ),

        median_amount=(
            "median_amount",
            "median"
        )
    )
    .sort_index()
)

print(
    typology_summary.to_string()
)

# ============================================================
# 10. EPISODE ID VALIDATION
# ============================================================
episode_ids = [
    ep["episode_id"]
    for ep in episodes
    if ep["episode_id"] is not None
]

print("\n" + "=" * 100)
print("EPISODE ID VALIDATION")
print("=" * 100)

print(
    "Episodes with IDs:",
    len(episode_ids)
)

print(
    "Unique IDs:",
    len(set(episode_ids))
)

# ============================================================
# 11. LABEL VALIDATION
# ============================================================
labels = []

for ep in episodes:

    for tx in ep["transactions"]:

        labels.append(
            tx["label"]
        )

label_counts = (
    pd.Series(labels)
    .value_counts()
    .sort_index()
)

print("\nPattern transaction labels:")
print(
    label_counts.to_string()
)

# ============================================================
# 12. SAVE SUMMARY
# ============================================================
summary_df.to_csv(
    OUT_SUMMARY,
    index=False
)

# ============================================================
# 13. SAVE TEMPLATES
# ============================================================
manifest = {

    "dataset": "IBM HI-Small",

    "source_file": PATTERN_PATH,

    "template_count": int(
        len(templates)
    ),

    "total_template_transactions": int(
        total_transactions
    ),

    "typologies": sorted(
        typology_episode_counts.keys()
    ),

    "templates": templates
}

with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ============================================================
# 14. FINAL VALIDATION
# ============================================================
print("\n" + "=" * 100)
print("FINAL VALIDATION")
print("=" * 100)

print(
    "Episodes:",
    len(templates)
)

print(
    "Transactions:",
    total_transactions
)

print(
    "Typologies:",
    sorted(
        typology_episode_counts.keys()
    )
)

print(
    "Episodes with IBM IDs:",
    len(episode_ids)
)

print(
    "Unique IBM IDs:",
    len(set(episode_ids))
)

print(
    "Episodes containing cycles:",
    int(
        summary_df[
            "cycle_present"
        ].sum()
    )
)

print(
    "Zero-transaction templates:",
    int(
        (
            summary_df[
                "transaction_count"
            ] == 0
        ).sum()
    )
)

print("\nSaved:")
print(
    OUT_JSON
)

print(
    OUT_SUMMARY
)

print("\n" + "=" * 100)
print("✅ CELL 6 COMPLETE")
print("=" * 100)
print("NO SYNTHETIC TRANSACTIONS CREATED.")
print("NO ORIGINAL FILE MODIFIED.")
print("=" * 100)

CELL 6 — EXACT IBM LAUNDERING PATTERN TEMPLATES
ULTRA-ROBUST IBM BLOCK PARSER

Total lines: 4319

Parsed episodes: 370
Parsed pattern transactions: 3209

EXPECTED VS ACTUAL
Expected episodes:       370
Actual episodes:         370
Expected transactions:   3209
Actual transactions:     3209

Episode counts:
BIPARTITE            49
CYCLE                54
FAN-IN               40
FAN-OUT              48
GATHER-SCATTER       51
RANDOM               41
SCATTER-GATHER       44
STACK                43

Transaction counts:
BIPARTITE            263
CYCLE                287
FAN-IN               318
FAN-OUT              342
GATHER-SCATTER       716
RANDOM               191
SCATTER-GATHER       626
STACK                466

PATTERN TEMPLATE SUMMARY
                episodes  transactions  median_transactions  median_nodes  median_source_banks  median_destination_banks  median_span_minutes  median_amount
typology                                                                                        

In [12]:
# ============================================================
# CELL 7 — REAL BANK CONNECTIVITY + ACCOUNT REGISTRY
# ============================================================
# Purpose:
#   Build generator-ready registries from ORIGINAL HI-Small.
#
# Creates:
#   1. Real bank registry
#   2. Legitimate bank-pair registry
#   3. Top legitimate neighbors per bank
#   4. Entity/account schema DNA
#
# IMPORTANT:
#   - ORIGINAL DATA ONLY
#   - NO synthetic transactions
#   - NO synthetic accounts
#   - NO original files modified
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

TRANS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_Trans.csv"
)

ACCOUNTS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_accounts.csv"
)

BANK_REGISTRY_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_real_bank_registry.csv"
)

BANK_PAIR_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_pairs.csv"
)

BANK_NEIGHBOR_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_neighbors.json"
)

ENTITY_DNA_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_entity_identifier_dna.json"
)

print("=" * 100)
print("CELL 7 — REAL BANK CONNECTIVITY + ACCOUNT REGISTRY")
print("=" * 100)

# ============================================================
# 1. LOAD ACCOUNTS TABLE
# ============================================================
print("\n" + "=" * 80)
print("1. ACCOUNTS TABLE")
print("=" * 80)

accounts = pd.read_csv(
    ACCOUNTS_PATH,
    dtype=str
)

print("Rows:", len(accounts))
print("Columns:")
for c in accounts.columns:
    print("  ", c)

# ------------------------------------------------------------
# Required fields
# ------------------------------------------------------------
required_columns = [
    "Bank Name",
    "Bank ID",
    "Account Number",
    "Entity ID",
    "Entity Name"
]

missing = [
    c for c in required_columns
    if c not in accounts.columns
]

if missing:
    raise RuntimeError(
        f"Missing required account columns: {missing}"
    )

# ============================================================
# 2. BANK ID -> BANK NAME VALIDATION
# ============================================================
print("\n" + "=" * 80)
print("2. BANK ID / BANK NAME VALIDATION")
print("=" * 80)

bank_name_conflicts = (
    accounts
    .groupby("Bank ID")["Bank Name"]
    .nunique()
)

conflicting_banks = (
    bank_name_conflicts[
        bank_name_conflicts > 1
    ]
)

print(
    "Unique Bank IDs:",
    accounts["Bank ID"].nunique()
)

print(
    "Unique Bank Names:",
    accounts["Bank Name"].nunique()
)

print(
    "Bank IDs with multiple names:",
    len(conflicting_banks)
)

if len(conflicting_banks) > 0:
    raise RuntimeError(
        "Bank ID -> Bank Name conflicts detected."
    )

# One canonical bank name
bank_name_map = (
    accounts[
        ["Bank ID", "Bank Name"]
    ]
    .drop_duplicates("Bank ID")
)

# ============================================================
# 3. ENTITY STRUCTURE DNA
# ============================================================
print("\n" + "=" * 80)
print("3. ENTITY STRUCTURE DNA")
print("=" * 80)

entity_accounts = (
    accounts
    .groupby("Entity ID")
    .size()
)

entity_banks = (
    accounts
    .groupby("Entity ID")["Bank ID"]
    .nunique()
)

print(
    "Unique entities:",
    accounts["Entity ID"].nunique()
)

print(
    "Entities with >=2 accounts:",
    int((entity_accounts >= 2).sum())
)

print(
    "Entities with >=2 banks:",
    int((entity_banks >= 2).sum())
)

print(
    "Maximum accounts/entity:",
    int(entity_accounts.max())
)

print(
    "Maximum banks/entity:",
    int(entity_banks.max())
)

# ------------------------------------------------------------
# Entity ID string DNA
# ------------------------------------------------------------
entity_id_lengths = (
    accounts["Entity ID"]
    .dropna()
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

entity_name_lengths = (
    accounts["Entity Name"]
    .dropna()
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

entity_id_digit_pct = (
    accounts["Entity ID"]
    .astype(str)
    .str.fullmatch(r"\d+")
    .mean()
    * 100
)

entity_id_alnum_pct = (
    accounts["Entity ID"]
    .astype(str)
    .str.fullmatch(r"[A-Za-z0-9]+")
    .mean()
    * 100
)

entity_dna = {
    "rows": int(len(accounts)),

    "unique_entities": int(
        accounts["Entity ID"].nunique()
    ),

    "unique_entity_names": int(
        accounts["Entity Name"].nunique()
    ),

    "entity_accounts": {
        "min": int(entity_accounts.min()),
        "median": float(entity_accounts.median()),
        "mean": float(entity_accounts.mean()),
        "p90": float(entity_accounts.quantile(0.90)),
        "p95": float(entity_accounts.quantile(0.95)),
        "p99": float(entity_accounts.quantile(0.99)),
        "max": int(entity_accounts.max())
    },

    "entity_banks": {
        "min": int(entity_banks.min()),
        "median": float(entity_banks.median()),
        "mean": float(entity_banks.mean()),
        "p90": float(entity_banks.quantile(0.90)),
        "p95": float(entity_banks.quantile(0.95)),
        "p99": float(entity_banks.quantile(0.99)),
        "max": int(entity_banks.max())
    },

    "entities_with_multiple_accounts": int(
        (entity_accounts >= 2).sum()
    ),

    "entities_with_multiple_banks": int(
        (entity_banks >= 2).sum()
    ),

    "entity_id_length_distribution": {
        str(k): int(v)
        for k, v in entity_id_lengths.items()
    },

    "entity_name_length_distribution": {
        str(k): int(v)
        for k, v in entity_name_lengths.items()
    },

    "entity_id_digits_only_pct": float(
        entity_id_digit_pct
    ),

    "entity_id_alphanumeric_pct": float(
        entity_id_alnum_pct
    )
}

print(
    "\nEntity ID length distribution:"
)

print(
    entity_id_lengths.to_string()
)

print(
    "\nEntity ID digits-only %:",
    round(entity_id_digit_pct, 4)
)

print(
    "Entity ID alphanumeric %:",
    round(entity_id_alnum_pct, 4)
)

# ============================================================
# 4. BANK ACCOUNT COUNTS
# ============================================================
print("\n" + "=" * 80)
print("4. ACCOUNTS PER BANK")
print("=" * 80)

accounts_per_bank = (
    accounts
    .groupby("Bank ID")
    .size()
    .rename("account_count")
)

print(
    "Median:",
    accounts_per_bank.median()
)

print(
    "Mean:",
    accounts_per_bank.mean()
)

print(
    "P90:",
    accounts_per_bank.quantile(0.90)
)

print(
    "P95:",
    accounts_per_bank.quantile(0.95)
)

print(
    "P99:",
    accounts_per_bank.quantile(0.99)
)

print(
    "Max:",
    accounts_per_bank.max()
)

# ============================================================
# 5. SCAN ORIGINAL TRANSACTIONS FOR BANK CONNECTIVITY
# ============================================================
print("\n" + "=" * 80)
print("5. SCANNING ORIGINAL TRANSACTIONS")
print("=" * 80)

tx_usecols = [
    "From Bank",
    "To Bank",
    "Is Laundering"
]

# ------------------------------------------------------------
# Aggregation containers
# ------------------------------------------------------------
from_counts = {}
to_counts = {}

legit_from_counts = {}
legit_to_counts = {}

pair_total = {}
pair_legit = {}

# ------------------------------------------------------------
# Chunked read
# ------------------------------------------------------------
chunk_size = 500_000

chunk_number = 0
total_rows_seen = 0

for chunk in pd.read_csv(
    TRANS_PATH,
    usecols=tx_usecols,
    chunksize=chunk_size,
    dtype={
        "From Bank": str,
        "To Bank": str
    }
):

    chunk_number += 1
    total_rows_seen += len(chunk)

    chunk["Is Laundering"] = (
        pd.to_numeric(
            chunk["Is Laundering"],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
    )

    # --------------------------------------------------------
    # All transaction counts
    # --------------------------------------------------------
    fc = chunk["From Bank"].value_counts()

    for bank, count in fc.items():
        from_counts[bank] = (
            from_counts.get(bank, 0)
            + int(count)
        )

    tc = chunk["To Bank"].value_counts()

    for bank, count in tc.items():
        to_counts[bank] = (
            to_counts.get(bank, 0)
            + int(count)
        )

    # --------------------------------------------------------
    # Legitimate only
    # --------------------------------------------------------
    legit_chunk = chunk[
        chunk["Is Laundering"] == 0
    ]

    lfc = legit_chunk[
        "From Bank"
    ].value_counts()

    for bank, count in lfc.items():
        legit_from_counts[bank] = (
            legit_from_counts.get(bank, 0)
            + int(count)
        )

    ltc = legit_chunk[
        "To Bank"
    ].value_counts()

    for bank, count in ltc.items():
        legit_to_counts[bank] = (
            legit_to_counts.get(bank, 0)
            + int(count)
        )

    # --------------------------------------------------------
    # Bank pairs
    # --------------------------------------------------------
    all_pairs = (
        chunk
        .groupby(
            ["From Bank", "To Bank"]
        )
        .size()
    )

    for (
        from_bank,
        to_bank
    ), count in all_pairs.items():

        key = (
            str(from_bank),
            str(to_bank)
        )

        pair_total[key] = (
            pair_total.get(key, 0)
            + int(count)
        )

    legit_pairs = (
        legit_chunk
        .groupby(
            ["From Bank", "To Bank"]
        )
        .size()
    )

    for (
        from_bank,
        to_bank
    ), count in legit_pairs.items():

        key = (
            str(from_bank),
            str(to_bank)
        )

        pair_legit[key] = (
            pair_legit.get(key, 0)
            + int(count)
        )

    print(
        f"  Chunk {chunk_number} "
        f"| rows processed = {total_rows_seen:,}"
    )

print(
    "\nTotal transaction rows scanned:",
    f"{total_rows_seen:,}"
)

# ============================================================
# 6. BUILD BANK REGISTRY
# ============================================================
print("\n" + "=" * 80)
print("6. BUILDING BANK REGISTRY")
print("=" * 80)

all_bank_ids = sorted(
    set(
        accounts["Bank ID"].astype(str)
    )
    |
    set(from_counts.keys())
    |
    set(to_counts.keys())
)

bank_registry_rows = []

for bank_id in all_bank_ids:

    bank_name = bank_name_map.loc[
        bank_name_map["Bank ID"] == bank_id,
        "Bank Name"
    ]

    if len(bank_name) > 0:
        bank_name = str(
            bank_name.iloc[0]
        )
    else:
        bank_name = "UNKNOWN"

    account_count = int(
        accounts_per_bank.get(
            bank_id,
            0
        )
    )

    source_count = int(
        from_counts.get(
            bank_id,
            0
        )
    )

    destination_count = int(
        to_counts.get(
            bank_id,
            0
        )
    )

    legit_source_count = int(
        legit_from_counts.get(
            bank_id,
            0
        )
    )

    legit_destination_count = int(
        legit_to_counts.get(
            bank_id,
            0
        )
    )

    bank_registry_rows.append(
        {
            "Bank ID": bank_id,
            "Bank Name": bank_name,
            "account_count": account_count,
            "source_transactions": source_count,
            "destination_transactions": destination_count,
            "total_transactions": (
                source_count
                +
                destination_count
            ),
            "legitimate_source_transactions":
                legit_source_count,
            "legitimate_destination_transactions":
                legit_destination_count,
            "legitimate_total_transactions":
                (
                    legit_source_count
                    +
                    legit_destination_count
                )
        }
    )

bank_registry = pd.DataFrame(
    bank_registry_rows
)

bank_registry = (
    bank_registry
    .sort_values(
        "legitimate_total_transactions",
        ascending=False
    )
    .reset_index(drop=True)
)

bank_registry.to_csv(
    BANK_REGISTRY_PATH,
    index=False
)

print(
    "Banks in registry:",
    len(bank_registry)
)

print(
    "\nTop 20 banks by legitimate activity:"
)

print(
    bank_registry.head(20).to_string(
        index=False
    )
)

# ============================================================
# 7. BUILD LEGITIMATE BANK-PAIR REGISTRY
# ============================================================
print("\n" + "=" * 80)
print("7. BUILDING LEGITIMATE BANK-PAIR REGISTRY")
print("=" * 80)

pair_rows = []

for (
    from_bank,
    to_bank
), total_count in pair_total.items():

    legit_count = pair_legit.get(
        (
            from_bank,
            to_bank
        ),
        0
    )

    pair_rows.append(
        {
            "From Bank": from_bank,
            "To Bank": to_bank,
            "total_count": int(
                total_count
            ),
            "legitimate_count": int(
                legit_count
            )
        }
    )

bank_pairs = pd.DataFrame(
    pair_rows
)

bank_pairs = (
    bank_pairs
    .sort_values(
        [
            "legitimate_count",
            "total_count"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

bank_pairs.to_csv(
    BANK_PAIR_PATH,
    index=False
)

print(
    "Unique bank pairs:",
    len(bank_pairs)
)

print(
    "Bank pairs observed in legitimate traffic:",
    int(
        (bank_pairs["legitimate_count"] > 0).sum()
    )
)

# ============================================================
# 8. BUILD TOP LEGITIMATE NEIGHBOR REGISTRY
# ============================================================
print("\n" + "=" * 80)
print("8. TOP LEGITIMATE BANK NEIGHBORS")
print("=" * 80)

neighbors_out = {}

# ------------------------------------------------------------
# Outgoing neighbors
# ------------------------------------------------------------
for bank_id, g in (
    bank_pairs[
        bank_pairs["legitimate_count"] > 0
    ]
    .groupby("From Bank")
):

    g = (
        g
        .sort_values(
            "legitimate_count",
            ascending=False
        )
        .head(50)
    )

    neighbors_out[str(bank_id)] = [
        {
            "to_bank": str(
                row["To Bank"]
            ),
            "count": int(
                row["legitimate_count"]
            )
        }
        for _, row in g.iterrows()
    ]

# ------------------------------------------------------------
# Incoming neighbors
# ------------------------------------------------------------
neighbors_in = {}

for bank_id, g in (
    bank_pairs[
        bank_pairs["legitimate_count"] > 0
    ]
    .groupby("To Bank")
):

    g = (
        g
        .sort_values(
            "legitimate_count",
            ascending=False
        )
        .head(50)
    )

    neighbors_in[str(bank_id)] = [
        {
            "from_bank": str(
                row["From Bank"]
            ),
            "count": int(
                row["legitimate_count"]
            )
        }
        for _, row in g.iterrows()
    ]

neighbor_registry = {
    "outgoing_top50": neighbors_out,
    "incoming_top50": neighbors_in
}

with open(
    BANK_NEIGHBOR_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        neighbor_registry,
        f,
        indent=2
    )

# ============================================================
# 9. BANK DISTRIBUTION STATISTICS
# ============================================================
print("\n" + "=" * 80)
print("9. BANK DISTRIBUTION CHECK")
print("=" * 80)

source_activity = (
    bank_registry[
        "legitimate_source_transactions"
    ]
)

destination_activity = (
    bank_registry[
        "legitimate_destination_transactions"
    ]
)

print(
    "Legitimate source-bank median:",
    source_activity.median()
)

print(
    "Legitimate source-bank mean:",
    source_activity.mean()
)

print(
    "Legitimate source-bank P90:",
    source_activity.quantile(0.90)
)

print(
    "Legitimate source-bank P95:",
    source_activity.quantile(0.95)
)

print(
    "Legitimate source-bank P99:",
    source_activity.quantile(0.99)
)

print(
    "Legitimate destination-bank median:",
    destination_activity.median()
)

print(
    "Legitimate destination-bank mean:",
    destination_activity.mean()
)

print(
    "Legitimate destination-bank P90:",
    destination_activity.quantile(0.90)
)

print(
    "Legitimate destination-bank P95:",
    destination_activity.quantile(0.95)
)

print(
    "Legitimate destination-bank P99:",
    destination_activity.quantile(0.99)
)

# ============================================================
# 10. SAVE ENTITY DNA
# ============================================================
with open(
    ENTITY_DNA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        entity_dna,
        f,
        indent=2
    )

# ============================================================
# 11. FINAL FILE SUMMARY
# ============================================================
print("\n" + "=" * 100)
print("✅ CELL 7 COMPLETE")
print("=" * 100)

print("\nCreated:")
print(
    BANK_REGISTRY_PATH
)
print(
    BANK_PAIR_PATH
)
print(
    BANK_NEIGHBOR_PATH
)
print(
    ENTITY_DNA_PATH
)

print("\nNO SYNTHETIC ACCOUNTS CREATED.")
print("NO SYNTHETIC TRANSACTIONS CREATED.")
print("NO ORIGINAL FILE MODIFIED.")

print("=" * 100)

CELL 7 — REAL BANK CONNECTIVITY + ACCOUNT REGISTRY

1. ACCOUNTS TABLE
Rows: 518581
Columns:
   Bank Name
   Bank ID
   Account Number
   Entity ID
   Entity Name

2. BANK ID / BANK NAME VALIDATION
Unique Bank IDs: 30470
Unique Bank Names: 20053
Bank IDs with multiple names: 0

3. ENTITY STRUCTURE DNA
Unique entities: 166207
Entities with >=2 accounts: 63518
Entities with >=2 banks: 63362
Maximum accounts/entity: 7820
Maximum banks/entity: 1185

Entity ID length distribution:
Entity ID
9    518581

Entity ID digits-only %: 6.5583
Entity ID alphanumeric %: 100.0

4. ACCOUNTS PER BANK
Median: 4.0
Mean: 17.019396127338364
P90: 13.0
P95: 31.0
P99: 372.6200000000026
Max: 3161

5. SCANNING ORIGINAL TRANSACTIONS
  Chunk 1 | rows processed = 500,000
  Chunk 2 | rows processed = 1,000,000
  Chunk 3 | rows processed = 1,500,000
  Chunk 4 | rows processed = 2,000,000
  Chunk 5 | rows processed = 2,500,000
  Chunk 6 | rows processed = 3,000,000
  Chunk 7 | rows processed = 3,500,000
  Chunk 8 | row

In [14]:
# ============================================================
# CELL 8 — BANK ID FORMAT DIAGNOSTIC + CANONICAL MAPPING
# OPTIMIZED VERSION
# ============================================================

import os
import json
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

TRANS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_Trans.csv"
)

ACCOUNTS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_accounts.csv"
)

OUT_MAPPING = os.path.join(
    WORK_DIR,
    "HI-Small_bank_id_canonical_mapping.csv"
)

OUT_JSON = os.path.join(
    WORK_DIR,
    "HI-Small_bank_id_canonical_dna.json"
)

os.makedirs(WORK_DIR, exist_ok=True)

print("=" * 100)
print("CELL 8 — BANK ID FORMAT DIAGNOSTIC + CANONICAL MAPPING")
print("OPTIMIZED VERSION")
print("=" * 100)

# ============================================================
# 1. LOAD UNIQUE IDS FROM TRANSACTION FILE
# ============================================================
print("\n" + "=" * 80)
print("1. LOADING TRANSACTION BANK IDS")
print("=" * 80)

tx_from = pd.read_csv(
    TRANS_PATH,
    usecols=["From Bank"],
    dtype=str
)

tx_to = pd.read_csv(
    TRANS_PATH,
    usecols=["To Bank"],
    dtype=str
)

tx_bank_series = pd.concat(
    [
        tx_from["From Bank"],
        tx_to["To Bank"]
    ],
    ignore_index=True
).dropna().astype(str)

transaction_bank_ids = set(
    tx_bank_series.unique()
)

print(
    "Unique transaction bank IDs:",
    len(transaction_bank_ids)
)

# ============================================================
# 2. LOAD ACCOUNT BANK IDS
# ============================================================
accounts = pd.read_csv(
    ACCOUNTS_PATH,
    usecols=["Bank ID", "Bank Name"],
    dtype=str
)

account_bank_ids = set(
    accounts["Bank ID"]
    .dropna()
    .astype(str)
    .unique()
)

print(
    "Unique accounts.csv bank IDs:",
    len(account_bank_ids)
)

# ============================================================
# 3. EXACT STRING OVERLAP
# ============================================================
print("\n" + "=" * 80)
print("2. EXACT STRING OVERLAP")
print("=" * 80)

exact_overlap = (
    transaction_bank_ids
    &
    account_bank_ids
)

print(
    "Exact overlapping IDs:",
    len(exact_overlap)
)

print(
    "Exact overlap %:",
    round(
        len(exact_overlap)
        /
        len(transaction_bank_ids)
        * 100,
        4
    )
)

# ============================================================
# 4. NORMALIZATION FUNCTION
# ============================================================
def normalize_bank_id(x):

    x = str(x).strip()

    if x == "":
        return None

    try:
        return str(int(x))
    except Exception:
        return x.lstrip("0") or "0"


# ============================================================
# 5. NORMALIZED ID SETS
# ============================================================
print("\n" + "=" * 80)
print("3. NORMALIZED NUMERIC OVERLAP")
print("=" * 80)

tx_norm_map = {
    x: normalize_bank_id(x)
    for x in transaction_bank_ids
}

acc_norm_map = {
    x: normalize_bank_id(x)
    for x in account_bank_ids
}

tx_normalized = set(
    tx_norm_map.values()
)

acc_normalized = set(
    acc_norm_map.values()
)

normalized_overlap = (
    tx_normalized
    &
    acc_normalized
)

print(
    "Unique normalized transaction IDs:",
    len(tx_normalized)
)

print(
    "Unique normalized account IDs:",
    len(acc_normalized)
)

print(
    "Normalized overlapping IDs:",
    len(normalized_overlap)
)

print(
    "Transaction normalized coverage:",
    round(
        len(normalized_overlap)
        /
        len(tx_normalized)
        * 100,
        4
    ),
    "%"
)

# ============================================================
# 6. OPTIMIZED TRANSACTION ID FREQUENCY
# ============================================================
print("\n" + "=" * 80)
print("4. TRANSACTION BANK-ID FREQUENCY")
print("=" * 80)

tx_id_counts = (
    tx_bank_series
    .value_counts()
    .rename("transaction_rows")
    .reset_index()
)

tx_id_counts.columns = [
    "transaction_bank_id",
    "transaction_rows"
]

tx_id_counts["normalized_bank_id"] = (
    tx_id_counts["transaction_bank_id"]
    .map(normalize_bank_id)
)

print(
    "Transaction bank-ID variants:",
    len(tx_id_counts)
)

# ============================================================
# 7. ACCOUNT-ID VARIANT TABLE
# ============================================================
account_id_variants = pd.DataFrame(
    {
        "account_bank_id": list(
            account_bank_ids
        )
    }
)

account_id_variants[
    "normalized_bank_id"
] = (
    account_id_variants[
        "account_bank_id"
    ]
    .map(normalize_bank_id)
)

# ============================================================
# 8. BUILD TRANSACTION-SIDE CANONICAL REPRESENTATION
# ============================================================
print("\n" + "=" * 80)
print("5. BUILDING CANONICAL TRANSACTION REPRESENTATION")
print("=" * 80)

# Sort by normalized ID and transaction frequency.
# The most frequently observed transaction representation
# becomes canonical.
canonical_candidates = (
    tx_id_counts
    .sort_values(
        [
            "normalized_bank_id",
            "transaction_rows"
        ],
        ascending=[
            True,
            False
        ]
    )
)

canonical = (
    canonical_candidates
    .drop_duplicates(
        "normalized_bank_id",
        keep="first"
    )
    .copy()
)

canonical = canonical.rename(
    columns={
        "transaction_bank_id":
            "canonical_transaction_bank_id"
    }
)

# ============================================================
# 9. LIST ALL TRANSACTION-SIDE VARIANTS
# ============================================================
tx_variants = (
    tx_id_counts
    .groupby(
        "normalized_bank_id"
    )["transaction_bank_id"]
    .agg(
        lambda x: "|".join(
            sorted(
                set(x),
                key=lambda z: (
                    len(str(z)),
                    str(z)
                )
            )
        )
    )
    .rename(
        "transaction_id_variants"
    )
    .reset_index()
)

# ============================================================
# 10. LIST ALL ACCOUNT-SIDE VARIANTS
# ============================================================
acc_variants = (
    account_id_variants
    .groupby(
        "normalized_bank_id"
    )["account_bank_id"]
    .agg(
        lambda x: "|".join(
            sorted(
                set(x),
                key=lambda z: (
                    len(str(z)),
                    str(z)
                )
            )
        )
    )
    .rename(
        "account_table_id_variants"
    )
    .reset_index()
)

# ============================================================
# 11. MERGE CANONICAL MAPPING
# ============================================================
mapping = (
    canonical[
        [
            "normalized_bank_id",
            "canonical_transaction_bank_id",
            "transaction_rows"
        ]
    ]
    .merge(
        tx_variants,
        on="normalized_bank_id",
        how="left"
    )
    .merge(
        acc_variants,
        on="normalized_bank_id",
        how="left"
    )
)

mapping[
    "has_account_record"
] = (
    mapping["normalized_bank_id"]
    .isin(
        acc_normalized
    )
    .astype(int)
)

mapping[
    "transaction_variant_count"
] = (
    mapping[
        "transaction_id_variants"
    ]
    .fillna("")
    .apply(
        lambda x:
        0 if x == ""
        else len(x.split("|"))
    )
)

mapping[
    "account_variant_count"
] = (
    mapping[
        "account_table_id_variants"
    ]
    .fillna("")
    .apply(
        lambda x:
        0 if x == ""
        else len(x.split("|"))
    )
)

# ============================================================
# 12. SHOW FORMAT MISMATCHES
# ============================================================
print("\n" + "=" * 80)
print("6. FORMAT-MISMATCH EXAMPLES")
print("=" * 80)

mismatch = mapping[
    mapping[
        "transaction_id_variants"
    ].fillna("")
    !=
    mapping[
        "account_table_id_variants"
    ].fillna("")
].copy()

print(
    "Normalized IDs with different string representations:",
    len(mismatch)
)

print(
    "\nFirst 30 examples:"
)

for _, row in mismatch.head(30).iterrows():

    print(
        f"Normalized={row['normalized_bank_id']} | "
        f"Transaction={row['transaction_id_variants']} | "
        f"Accounts={row['account_table_id_variants']} | "
        f"Canonical={row['canonical_transaction_bank_id']}"
    )

# ============================================================
# 13. BANK NAME CONSISTENCY AFTER NORMALIZATION
# ============================================================
print("\n" + "=" * 80)
print("7. BANK NAME CONSISTENCY")
print("=" * 80)

accounts["_normalized_bank_id"] = (
    accounts["Bank ID"]
    .map(normalize_bank_id)
)

name_counts = (
    accounts
    .groupby(
        "_normalized_bank_id"
    )["Bank Name"]
    .nunique()
)

name_conflicts = (
    name_counts[
        name_counts > 1
    ]
)

print(
    "Normalized bank IDs with multiple names:",
    len(name_conflicts)
)

# ============================================================
# 14. MATCHING COVERAGE
# ============================================================
print("\n" + "=" * 80)
print("8. MATCHING COVERAGE")
print("=" * 80)

matched_count = int(
    mapping[
        "has_account_record"
    ].sum()
)

unmatched_count = (
    len(tx_normalized)
    -
    matched_count
)

print(
    "Transaction normalized banks:",
    len(tx_normalized)
)

print(
    "Matched to accounts.csv:",
    matched_count
)

print(
    "Unmatched:",
    unmatched_count
)

print(
    "Coverage:",
    round(
        matched_count
        /
        len(tx_normalized)
        * 100,
        4
    ),
    "%"
)

# ============================================================
# 15. BANK-ID LENGTH DISTRIBUTIONS
# ============================================================
print("\n" + "=" * 80)
print("9. BANK-ID LENGTH DISTRIBUTIONS")
print("=" * 80)

tx_lengths = (
    pd.Series(
        [
            len(str(x))
            for x in transaction_bank_ids
        ]
    )
    .value_counts()
    .sort_index()
)

acc_lengths = (
    pd.Series(
        [
            len(str(x))
            for x in account_bank_ids
        ]
    )
    .value_counts()
    .sort_index()
)

print(
    "\nTransaction bank-ID lengths:"
)

print(
    tx_lengths.to_string()
)

print(
    "\nAccounts.csv bank-ID lengths:"
)

print(
    acc_lengths.to_string()
)

# ============================================================
# 16. CANONICAL FORMAT CHECK
# ============================================================
print("\n" + "=" * 80)
print("10. CANONICAL REPRESENTATION")
print("=" * 80)

print(
    "Canonical IDs:",
    len(mapping)
)

print(
    "Canonical IDs missing accounts record:",
    int(
        (
            mapping[
                "has_account_record"
            ] == 0
        ).sum()
    )
)

print(
    "\nTop 20 canonical IDs by transaction activity:"
)

display(
    mapping
    .sort_values(
        "transaction_rows",
        ascending=False
    )
    .head(20)
)

# ============================================================
# 17. SAVE CSV
# ============================================================
mapping.to_csv(
    OUT_MAPPING,
    index=False
)

# ============================================================
# 18. SAVE JSON DNA
# ============================================================
canonical_dna = {

    "transaction_unique_bank_ids": int(
        len(transaction_bank_ids)
    ),

    "accounts_unique_bank_ids": int(
        len(account_bank_ids)
    ),

    "exact_overlap": int(
        len(exact_overlap)
    ),

    "normalized_transaction_ids": int(
        len(tx_normalized)
    ),

    "normalized_account_ids": int(
        len(acc_normalized)
    ),

    "normalized_overlap": int(
        len(normalized_overlap)
    ),

    "normalized_coverage_pct": float(
        len(normalized_overlap)
        /
        len(tx_normalized)
        * 100
    ),

    "unmatched_normalized_ids": [
        str(x)
        for x in sorted(
            tx_normalized
            -
            acc_normalized,
            key=lambda z:
                int(z)
                if str(z).isdigit()
                else str(z)
        )
    ],

    "normalized_ids_with_format_mismatch":
        int(len(mismatch)),

    "normalized_bank_name_conflicts":
        int(len(name_conflicts)),

    "transaction_bank_id_length_distribution":
        {
            str(k): int(v)
            for k, v in tx_lengths.items()
        },

    "account_bank_id_length_distribution":
        {
            str(k): int(v)
            for k, v in acc_lengths.items()
        }
}

with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        canonical_dna,
        f,
        indent=2
    )

# ============================================================
# 19. FINAL
# ============================================================
print("\n" + "=" * 100)
print("✅ CELL 8 COMPLETE")
print("=" * 100)

print(
    "\nCanonical bank registry:",
    OUT_MAPPING
)

print(
    "Canonical bank DNA:",
    OUT_JSON
)

print(
    "\nKey result:"
)

print(
    f"  Transaction bank IDs      : "
    f"{len(transaction_bank_ids):,}"
)

print(
    f"  Accounts bank IDs         : "
    f"{len(account_bank_ids):,}"
)

print(
    f"  Normalized bank IDs       : "
    f"{len(tx_normalized):,}"
)

print(
    f"  Normalized overlap        : "
    f"{len(normalized_overlap):,}"
)

print(
    f"  Coverage                  : "
    f"{len(normalized_overlap) / len(tx_normalized) * 100:.2f}%"
)

print("\nNO SYNTHETIC DATA CREATED.")
print("NO ORIGINAL FILE MODIFIED.")

print("=" * 100)

CELL 8 — BANK ID FORMAT DIAGNOSTIC + CANONICAL MAPPING
OPTIMIZED VERSION

1. LOADING TRANSACTION BANK IDS
Unique transaction bank IDs: 30528
Unique accounts.csv bank IDs: 30470

2. EXACT STRING OVERLAP
Exact overlapping IDs: 0
Exact overlap %: 0.0

3. NORMALIZED NUMERIC OVERLAP
Unique normalized transaction IDs: 30470
Unique normalized account IDs: 30470
Normalized overlapping IDs: 30470
Transaction normalized coverage: 100.0 %

4. TRANSACTION BANK-ID FREQUENCY
Transaction bank-ID variants: 30528

5. BUILDING CANONICAL TRANSACTION REPRESENTATION

6. FORMAT-MISMATCH EXAMPLES
Normalized IDs with different string representations: 30470

First 30 examples:
Normalized=1 | Transaction=001 | Accounts=1 | Canonical=001
Normalized=10 | Transaction=010|0010 | Accounts=10 | Canonical=010
Normalized=10057 | Transaction=0010057 | Accounts=10057 | Canonical=0010057
Normalized=10060 | Transaction=0010060 | Accounts=10060 | Canonical=0010060
Normalized=10099 | Transaction=0010099 | Accounts=10099 | Ca

,normalized_bank_id,canonical_transaction_bank_id,transaction_rows,transaction_id_variants,account_table_id_variants,has_account_record,transaction_variant_count,account_variant_count
30434,70,070,452751,070,70,1,1,1
135,12,012,105909,012|0012,12,1,2,1
1,10,010,94717,010|0010,10,1,2,1
0,1,001,92326,001,1,1,1,1
1406,3,003,64040,003,3,1,1,1
721,20,020,63056,020,20,1,1,1
30433,7,007,54115,007,7,1,1,1
526,15,015,52757,015|0015,15,1,2,1
745,211,0211,51027,0211,211,1,1,1
94,116,0116,50472,0116,116,1,1,1



✅ CELL 8 COMPLETE

Canonical bank registry: /kaggle/working/AML_GNN_ONLY/data/HI-Small_bank_id_canonical_mapping.csv
Canonical bank DNA: /kaggle/working/AML_GNN_ONLY/data/HI-Small_bank_id_canonical_dna.json

Key result:
  Transaction bank IDs      : 30,528
  Accounts bank IDs         : 30,470
  Normalized bank IDs       : 30,470
  Normalized overlap        : 30,470
  Coverage                  : 100.00%

NO SYNTHETIC DATA CREATED.
NO ORIGINAL FILE MODIFIED.


In [15]:
# ============================================================
# CELL 9 — FINAL GENERATOR BANK POOL + ACCOUNT-ID SAMPLER
# ============================================================
#
# Uses:
#   - Original HI-Small_accounts.csv
#   - Corrected bank-ID mapping from Cell 8
#   - Bank activity registry generated in Cell 7
#
# Creates:
#   1. Final 30,470-bank generator pool
#   2. Bank sampling weights
#   3. Bank-specific account-prefix distributions
#   4. Account-ID positional character distributions
#   5. Compact generator configuration
#
# IMPORTANT:
#   - NO synthetic accounts
#   - NO synthetic transactions
#   - NO original files modified
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

ACCOUNTS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_accounts.csv"
)

BANK_MAPPING_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_bank_id_canonical_mapping.csv"
)

OLD_BANK_REGISTRY_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_real_bank_registry.csv"
)

OUT_BANK_POOL = os.path.join(
    WORK_DIR,
    "HI-Small_final_generator_bank_pool.csv"
)

OUT_PREFIX_JSON = os.path.join(
    WORK_DIR,
    "HI-Small_bank_account_prefix_sampler.json"
)

OUT_ACCOUNT_JSON = os.path.join(
    WORK_DIR,
    "HI-Small_account_id_generator.json"
)

print("=" * 100)
print("CELL 9 — FINAL GENERATOR BANK POOL + ACCOUNT-ID SAMPLER")
print("=" * 100)

# ============================================================
# 1. LOAD SOURCE TABLES
# ============================================================
print("\n" + "=" * 80)
print("1. LOADING SOURCE TABLES")
print("=" * 80)

accounts = pd.read_csv(
    ACCOUNTS_PATH,
    dtype=str
)

mapping = pd.read_csv(
    BANK_MAPPING_PATH,
    dtype=str
)

old_registry = pd.read_csv(
    OLD_BANK_REGISTRY_PATH,
    dtype=str
)

print(
    "Accounts rows:",
    f"{len(accounts):,}"
)

print(
    "Canonical bank IDs:",
    f"{len(mapping):,}"
)

print(
    "Old registry rows:",
    f"{len(old_registry):,}"
)

# ============================================================
# 2. NORMALIZE BANK IDs
# ============================================================
def normalize_bank_id(x):

    x = str(x).strip()

    if x == "":
        return None

    try:
        return str(int(x))
    except Exception:
        return x.lstrip("0") or "0"


accounts[
    "normalized_bank_id"
] = (
    accounts["Bank ID"]
    .map(normalize_bank_id)
)

old_registry[
    "normalized_bank_id"
] = (
    old_registry["Bank ID"]
    .map(normalize_bank_id)
)

mapping[
    "normalized_bank_id"
] = (
    mapping[
        "normalized_bank_id"
    ].astype(str)
)

# ============================================================
# 3. COLLAPSE OLD REGISTRY TO NORMALIZED BANK LEVEL
# ============================================================
print("\n" + "=" * 80)
print("2. COLLAPSING BANK REGISTRY")
print("=" * 80)

# Convert numeric activity fields
activity_cols = [
    "source_transactions",
    "destination_transactions",
    "legitimate_source_transactions",
    "legitimate_destination_transactions"
]

for col in activity_cols:

    old_registry[col] = pd.to_numeric(
        old_registry[col],
        errors="coerce"
    ).fillna(0)

collapsed_activity = (
    old_registry
    .groupby(
        "normalized_bank_id",
        as_index=False
    )[activity_cols]
    .sum()
)

print(
    "Collapsed normalized banks:",
    len(collapsed_activity)
)

# ============================================================
# 4. BANK ACCOUNT COUNTS
# ============================================================
accounts_per_bank = (
    accounts
    .groupby(
        "normalized_bank_id"
    )
    .size()
    .rename("account_count")
    .reset_index()
)

# ============================================================
# 5. CANONICAL BANK REGISTRY
# ============================================================
print("\n" + "=" * 80)
print("3. BUILDING FINAL 30,470-BANK POOL")
print("=" * 80)

# Canonical transaction-side representation
canonical_bank = (
    mapping[
        [
            "normalized_bank_id",
            "canonical_transaction_bank_id"
        ]
    ]
    .drop_duplicates(
        "normalized_bank_id"
    )
)

# Canonical bank name from accounts.csv
bank_names = (
    accounts[
        [
            "normalized_bank_id",
            "Bank Name"
        ]
    ]
    .drop_duplicates(
        "normalized_bank_id"
    )
)

bank_pool = (
    canonical_bank
    .merge(
        bank_names,
        on="normalized_bank_id",
        how="left"
    )
    .merge(
        collapsed_activity,
        on="normalized_bank_id",
        how="left"
    )
    .merge(
        accounts_per_bank,
        on="normalized_bank_id",
        how="left"
    )
)

# Fill missing activity
for col in activity_cols:
    bank_pool[col] = (
        pd.to_numeric(
            bank_pool[col],
            errors="coerce"
        )
        .fillna(0)
        .astype(np.int64)
    )

bank_pool["account_count"] = (
    pd.to_numeric(
        bank_pool["account_count"],
        errors="coerce"
    )
    .fillna(0)
    .astype(np.int64)
)

# ------------------------------------------------------------
# Legitimate transaction weights
# ------------------------------------------------------------
bank_pool[
    "legit_source_weight"
] = (
    bank_pool[
        "legitimate_source_transactions"
    ]
    .astype(float)
)

bank_pool[
    "legit_destination_weight"
] = (
    bank_pool[
        "legitimate_destination_transactions"
    ]
    .astype(float)
)

# Avoid zero-probability banks in generic sampling.
# A tiny floor is used only for banks with no observed
# legitimate activity.
bank_pool[
    "source_sampling_weight"
] = (
    bank_pool[
        "legit_source_weight"
    ]
    .clip(lower=1.0)
)

bank_pool[
    "destination_sampling_weight"
] = (
    bank_pool[
        "legit_destination_weight"
    ]
    .clip(lower=1.0)
)

# ------------------------------------------------------------
# Normalize weights
# ------------------------------------------------------------
source_total = (
    bank_pool[
        "source_sampling_weight"
    ].sum()
)

destination_total = (
    bank_pool[
        "destination_sampling_weight"
    ].sum()
)

bank_pool[
    "source_probability"
] = (
    bank_pool[
        "source_sampling_weight"
    ]
    /
    source_total
)

bank_pool[
    "destination_probability"
] = (
    bank_pool[
        "destination_sampling_weight"
    ]
    /
    destination_total
)

# ------------------------------------------------------------
# Sort by normalized bank ID
# ------------------------------------------------------------
bank_pool = (
    bank_pool
    .sort_values(
        "normalized_bank_id"
    )
    .reset_index(drop=True)
)

print(
    "FINAL BANK POOL:",
    f"{len(bank_pool):,}",
    "banks"
)

# ============================================================
# 6. VALIDATION
# ============================================================
print("\n" + "=" * 80)
print("4. BANK POOL VALIDATION")
print("=" * 80)

print(
    "Expected normalized banks:",
    30470
)

print(
    "Actual normalized banks:",
    len(bank_pool)
)

print(
    "Unique canonical IDs:",
    bank_pool[
        "canonical_transaction_bank_id"
    ].nunique()
)

print(
    "Banks missing Bank Name:",
    int(
        bank_pool["Bank Name"]
        .isna()
        .sum()
    )
)

print(
    "Banks missing account records:",
    int(
        (
            bank_pool["account_count"] == 0
        ).sum()
    )
)

print(
    "Source probability sum:",
    bank_pool[
        "source_probability"
    ].sum()
)

print(
    "Destination probability sum:",
    bank_pool[
        "destination_probability"
    ].sum()
)

# ============================================================
# 7. TOP BANKS
# ============================================================
print("\n" + "=" * 80)
print("5. TOP BANKS BY LEGITIMATE ACTIVITY")
print("=" * 80)

display(
    bank_pool[
        [
            "normalized_bank_id",
            "canonical_transaction_bank_id",
            "Bank Name",
            "account_count",
            "legitimate_source_transactions",
            "legitimate_destination_transactions",
            "source_probability",
            "destination_probability"
        ]
    ]
    .sort_values(
        "legitimate_source_transactions",
        ascending=False
    )
    .head(20)
)

# ============================================================
# 8. BANK-SPECIFIC ACCOUNT PREFIX SAMPLER
# ============================================================
print("\n" + "=" * 80)
print("6. BANK-SPECIFIC ACCOUNT PREFIX SAMPLER")
print("=" * 80)

# Account IDs are known to be 9-character values.
accounts[
    "account_prefix_3"
] = (
    accounts["Account Number"]
    .astype(str)
    .str[:3]
)

accounts[
    "account_prefix_4"
] = (
    accounts["Account Number"]
    .astype(str)
    .str[:4]
)

# ------------------------------------------------------------
# For each normalized bank:
#   top 3-character account prefixes
#   empirical probability
# ------------------------------------------------------------
prefix_sampler = {}

for bank_id, g in (
    accounts
    .groupby(
        "normalized_bank_id"
    )
):

    prefix_counts = (
        g["account_prefix_3"]
        .value_counts()
    )

    total = prefix_counts.sum()

    prefix_sampler[str(bank_id)] = {
        "account_count": int(len(g)),
        "prefixes": [
            {
                "prefix": str(prefix),
                "count": int(count),
                "probability": float(
                    count / total
                )
            }
            for prefix, count in
            prefix_counts.head(100).items()
        ]
    }

print(
    "Banks with prefix models:",
    len(prefix_sampler)
)

# ============================================================
# 9. GLOBAL ACCOUNT-ID POSITION MODEL
# ============================================================
print("\n" + "=" * 80)
print("7. GLOBAL ACCOUNT-ID POSITION MODEL")
print("=" * 80)

account_values = (
    accounts["Account Number"]
    .astype(str)
)

position_models = {}

for pos in range(9):

    vc = (
        account_values
        .str[pos]
        .value_counts()
    )

    total = vc.sum()

    rows = []

    for char, count in vc.items():

        rows.append(
            {
                "char": str(char),
                "count": int(count),
                "probability": float(
                    count / total
                )
            }
        )

    position_models[str(pos + 1)] = rows

    print(
        f"Position {pos + 1}: "
        f"{len(rows)} possible characters"
    )

# ============================================================
# 10. POSITION DIGIT/LETTER DNA
# ============================================================
position_type_model = {}

for pos in range(9):

    chars = (
        account_values
        .str[pos]
    )

    digit_count = int(
        chars.str.isdigit().sum()
    )

    letter_count = int(
        chars.str.isalpha().sum()
    )

    total = len(chars)

    position_type_model[
        str(pos + 1)
    ] = {
        "digit_probability": float(
            digit_count / total
        ),
        "letter_probability": float(
            letter_count / total
        )
    }

# ============================================================
# 11. ACCOUNT ID STRUCTURAL DNA
# ============================================================
account_generator_config = {

    "format": {
        "length": 9,
        "regex": "^[A-Z0-9]{9}$"
    },

    "fixed_position_behavior": {
        "position_1": {
            "values": {
                str(k): int(v)
                for k, v in
                account_values
                .str[0]
                .value_counts()
                .items()
            }
        },

        "position_2": {
            "values": {
                str(k): int(v)
                for k, v in
                account_values
                .str[1]
                .value_counts()
                .items()
            }
        },

        "position_9": {
            "values": {
                str(k): int(v)
                for k, v in
                account_values
                .str[8]
                .value_counts()
                .items()
            }
        }
    },

    "position_models": position_models,

    "position_digit_letter_model":
        position_type_model,

    "bank_specific_prefix_length": 3,

    "bank_specific_prefix_models_available":
        len(prefix_sampler),

    "original_account_rows":
        int(len(accounts)),

    "original_unique_account_ids":
        int(
            account_values.nunique()
        )
}

# ============================================================
# 12. SAVE BANK POOL
# ============================================================
bank_pool.to_csv(
    OUT_BANK_POOL,
    index=False
)

# ============================================================
# 13. SAVE PREFIX SAMPLER
# ============================================================
with open(
    OUT_PREFIX_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        prefix_sampler,
        f,
        indent=2
    )

# ============================================================
# 14. SAVE ACCOUNT GENERATOR CONFIG
# ============================================================
with open(
    OUT_ACCOUNT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        account_generator_config,
        f,
        indent=2
    )

# ============================================================
# 15. FINAL VALIDATION
# ============================================================
print("\n" + "=" * 100)
print("✅ CELL 9 COMPLETE")
print("=" * 100)

print(
    "\nFinal normalized bank pool:",
    f"{len(bank_pool):,}"
)

print(
    "Expected:",
    "30,470"
)

print(
    "Bank-name conflicts:",
    int(
        bank_pool
        .groupby(
            "normalized_bank_id"
        )["Bank Name"]
        .nunique()
        .gt(1)
        .sum()
    )
)

print(
    "Banks with account records:",
    int(
        (
            bank_pool["account_count"] > 0
        ).sum()
    )
)

print(
    "Bank prefix models:",
    len(prefix_sampler)
)

print(
    "Account ID length:",
    9
)

print(
    "Account ID regex:",
    "^[A-Z0-9]{9}$"
)

print("\nSaved:")
print(
    OUT_BANK_POOL
)
print(
    OUT_PREFIX_JSON
)
print(
    OUT_ACCOUNT_JSON
)

print("\nNO SYNTHETIC ACCOUNTS CREATED.")
print("NO SYNTHETIC TRANSACTIONS CREATED.")
print("NO ORIGINAL FILE MODIFIED.")

print("=" * 100)

CELL 9 — FINAL GENERATOR BANK POOL + ACCOUNT-ID SAMPLER

1. LOADING SOURCE TABLES
Accounts rows: 518,581
Canonical bank IDs: 30,470
Old registry rows: 60,998

2. COLLAPSING BANK REGISTRY
Collapsed normalized banks: 30470

3. BUILDING FINAL 30,470-BANK POOL
FINAL BANK POOL: 30,470 banks

4. BANK POOL VALIDATION
Expected normalized banks: 30470
Actual normalized banks: 30470
Unique canonical IDs: 30470
Banks missing Bank Name: 0
Banks missing account records: 0
Source probability sum: 0.9999999999999999
Destination probability sum: 1.0

5. TOP BANKS BY LEGITIMATE ACTIVITY


,normalized_bank_id,canonical_transaction_bank_id,Bank Name,account_count,legitimate_source_transactions,legitimate_destination_transactions,source_probability,destination_probability
30434,70,070,Oasis Thrift,15,449226,2892,0.088549,0.000568
1,10,010,National Bank of Laramie,3108,81578,42496,0.016080,0.008352
135,12,012,National Bank of the East,3161,79678,41783,0.015706,0.008212
0,1,001,Arbor Savings Bank,2172,62161,30062,0.012253,0.005909
526,15,015,Japan Bank #0,3051,52465,38679,0.010342,0.007602
846,220,00220,National Bank of Pittsburgh,2530,52395,30586,0.010328,0.006012
721,20,020,China Bank #6,1625,40941,21994,0.008070,0.004323
1406,3,003,China Bank #14,1777,38400,25614,0.007569,0.005034
30433,7,007,India Bank #25,1665,31075,23015,0.006125,0.004524
745,211,0211,UK Bank #29,1507,30425,20549,0.005997,0.004039



6. BANK-SPECIFIC ACCOUNT PREFIX SAMPLER
Banks with prefix models: 30470

7. GLOBAL ACCOUNT-ID POSITION MODEL
Position 1: 2 possible characters
Position 2: 2 possible characters
Position 3: 16 possible characters
Position 4: 16 possible characters
Position 5: 16 possible characters
Position 6: 16 possible characters
Position 7: 16 possible characters
Position 8: 16 possible characters
Position 9: 3 possible characters

✅ CELL 9 COMPLETE

Final normalized bank pool: 30,470
Expected: 30,470
Bank-name conflicts: 0
Banks with account records: 30470
Bank prefix models: 30470
Account ID length: 9
Account ID regex: ^[A-Z0-9]{9}$

Saved:
/kaggle/working/AML_GNN_ONLY/data/HI-Small_final_generator_bank_pool.csv
/kaggle/working/AML_GNN_ONLY/data/HI-Small_bank_account_prefix_sampler.json
/kaggle/working/AML_GNN_ONLY/data/HI-Small_account_id_generator.json

NO SYNTHETIC ACCOUNTS CREATED.
NO SYNTHETIC TRANSACTIONS CREATED.
NO ORIGINAL FILE MODIFIED.


In [16]:
# ============================================================
# CELL 10 — ACCOUNT REUSE + ENTITY DNA
# ============================================================
#
# Purpose:
#   Learn how frequently real legitimate accounts are reused,
#   and how Entity IDs / Entity Names are structured.
#
# Reads:
#   ORIGINAL HI-Small only
#
# Creates:
#   HI-Small_account_reuse_dna.json
#   HI-Small_entity_name_dna.json
#
# NO synthetic data created.
# NO original data modified.
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

TRANS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_Trans.csv"
)

ACCOUNTS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_accounts.csv"
)

OUT_ACCOUNT_DNA = os.path.join(
    WORK_DIR,
    "HI-Small_account_reuse_dna.json"
)

OUT_ENTITY_DNA = os.path.join(
    WORK_DIR,
    "HI-Small_entity_name_dna.json"
)

print("=" * 100)
print("CELL 10 — ACCOUNT REUSE + ENTITY DNA")
print("=" * 100)

# ============================================================
# 1. LOAD ACCOUNTS TABLE
# ============================================================
accounts = pd.read_csv(
    ACCOUNTS_PATH,
    dtype=str
)

required = [
    "Bank ID",
    "Account Number",
    "Entity ID",
    "Entity Name"
]

missing = [
    c for c in required
    if c not in accounts.columns
]

if missing:
    raise RuntimeError(
        f"Missing columns: {missing}"
    )

# ------------------------------------------------------------
# Normalize IDs
# ------------------------------------------------------------
def normalize_bank_id(x):

    x = str(x).strip()

    if x == "":
        return None

    try:
        return str(int(x))
    except Exception:
        return x.lstrip("0") or "0"


accounts[
    "normalized_bank_id"
] = (
    accounts["Bank ID"]
    .map(normalize_bank_id)
)

accounts[
    "Account Number"
] = (
    accounts["Account Number"]
    .astype(str)
)

# ============================================================
# 2. ACCOUNT TABLE STRUCTURE
# ============================================================
print("\n" + "=" * 80)
print("1. ACCOUNT TABLE STRUCTURE")
print("=" * 80)

account_rows = len(accounts)

unique_accounts = (
    accounts["Account Number"]
    .nunique()
)

unique_account_bank_pairs = (
    accounts[
        [
            "normalized_bank_id",
            "Account Number"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Account rows:",
    f"{account_rows:,}"
)

print(
    "Unique account numbers:",
    f"{unique_accounts:,}"
)

print(
    "Unique bank-account pairs:",
    f"{unique_account_bank_pairs:,}"
)

print(
    "Duplicate account-table rows:",
    f"{account_rows - unique_account_bank_pairs:,}"
)

# ============================================================
# 3. ENTITY ACCOUNT/BANK REUSE
# ============================================================
print("\n" + "=" * 80)
print("2. ENTITY REUSE")
print("=" * 80)

entity_account_count = (
    accounts
    .groupby("Entity ID")
    .size()
)

entity_bank_count = (
    accounts
    .groupby("Entity ID")[
        "normalized_bank_id"
    ]
    .nunique()
)

print(
    "Entities:",
    f"{accounts['Entity ID'].nunique():,}"
)

print(
    "Median accounts/entity:",
    entity_account_count.median()
)

print(
    "Mean accounts/entity:",
    entity_account_count.mean()
)

print(
    "P90 accounts/entity:",
    entity_account_count.quantile(0.90)
)

print(
    "P95 accounts/entity:",
    entity_account_count.quantile(0.95)
)

print(
    "P99 accounts/entity:",
    entity_account_count.quantile(0.99)
)

print(
    "Max accounts/entity:",
    entity_account_count.max()
)

print(
    "Median banks/entity:",
    entity_bank_count.median()
)

print(
    "Mean banks/entity:",
    entity_bank_count.mean()
)

print(
    "Entities with >=2 banks:",
    int(
        (entity_bank_count >= 2).sum()
    )
)

# ============================================================
# 4. SCAN LEGITIMATE TRANSACTIONS
# ============================================================
print("\n" + "=" * 80)
print("3. ACCOUNT TRANSACTION REUSE")
print("=" * 80)

# We count source and destination activity separately.
#
# Key:
#   account number
#
# This is sufficient for the activity-distribution DNA.
# Bank-account uniqueness is handled by the generator.
# ------------------------------------------------------------

source_counts = {}
destination_counts = {}

legit_source_counts = {}
legit_destination_counts = {}

chunk_size = 500_000
rows_seen = 0
chunk_no = 0

for chunk in pd.read_csv(
    TRANS_PATH,
    usecols=[
        "Account",
        "Account.1",
        "Is Laundering"
    ],
    chunksize=chunk_size,
    dtype=str
):

    chunk_no += 1
    rows_seen += len(chunk)

    labels = pd.to_numeric(
        chunk["Is Laundering"],
        errors="coerce"
    ).fillna(0).astype(int)

    # --------------------------------------------------------
    # All source activity
    # --------------------------------------------------------
    src = (
        chunk["Account"]
        .astype(str)
        .value_counts()
    )

    for account, count in src.items():

        source_counts[account] = (
            source_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    # --------------------------------------------------------
    # All destination activity
    # --------------------------------------------------------
    dst = (
        chunk["Account.1"]
        .astype(str)
        .value_counts()
    )

    for account, count in dst.items():

        destination_counts[account] = (
            destination_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    # --------------------------------------------------------
    # Legitimate only
    # --------------------------------------------------------
    legit = chunk[
        labels == 0
    ]

    legit_src = (
        legit["Account"]
        .astype(str)
        .value_counts()
    )

    for account, count in legit_src.items():

        legit_source_counts[account] = (
            legit_source_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    legit_dst = (
        legit["Account.1"]
        .astype(str)
        .value_counts()
    )

    for account, count in legit_dst.items():

        legit_destination_counts[account] = (
            legit_destination_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    print(
        f"  Chunk {chunk_no} "
        f"| rows processed = {rows_seen:,}"
    )

print(
    "\nRows scanned:",
    f"{rows_seen:,}"
)

# ============================================================
# 5. ACCOUNT ACTIVITY DISTRIBUTIONS
# ============================================================
def activity_summary(
    counts_dict
):

    s = pd.Series(
        counts_dict,
        dtype="int64"
    )

    if len(s) == 0:
        return {}

    return {
        "accounts": int(len(s)),
        "min": int(s.min()),
        "median": float(s.median()),
        "mean": float(s.mean()),
        "p75": float(s.quantile(0.75)),
        "p90": float(s.quantile(0.90)),
        "p95": float(s.quantile(0.95)),
        "p99": float(s.quantile(0.99)),
        "max": int(s.max())
    }


source_all_summary = activity_summary(
    source_counts
)

destination_all_summary = activity_summary(
    destination_counts
)

source_legit_summary = activity_summary(
    legit_source_counts
)

destination_legit_summary = activity_summary(
    legit_destination_counts
)

print("\n" + "=" * 80)
print("4. ACTIVITY SUMMARY")
print("=" * 80)

print("\nAll source accounts:")
print(source_all_summary)

print("\nAll destination accounts:")
print(destination_all_summary)

print("\nLegitimate source accounts:")
print(source_legit_summary)

print("\nLegitimate destination accounts:")
print(destination_legit_summary)

# ============================================================
# 6. LEGITIMATE ACTIVITY BUCKETS
# ============================================================
def make_activity_buckets(
    counts_dict
):

    s = pd.Series(
        counts_dict,
        dtype="int64"
    )

    buckets = pd.cut(
        s,
        bins=[
            0,
            1,
            2,
            5,
            10,
            25,
            50,
            100,
            250,
            500,
            1000,
            np.inf
        ],
        labels=[
            "1",
            "2",
            "3-5",
            "6-10",
            "11-25",
            "26-50",
            "51-100",
            "101-250",
            "251-500",
            "501-1000",
            ">1000"
        ]
    )

    counts = (
        buckets
        .value_counts(
            sort=False
        )
    )

    total = counts.sum()

    return {
        str(bucket): {
            "count": int(count),
            "share_pct": float(
                count / total * 100
            )
        }
        for bucket, count
        in counts.items()
    }

legit_source_buckets = make_activity_buckets(
    legit_source_counts
)

legit_destination_buckets = make_activity_buckets(
    legit_destination_counts
)

print("\n" + "=" * 80)
print("5. LEGITIMATE ACCOUNT ACTIVITY BUCKETS")
print("=" * 80)

print("\nSource:")
for k, v in legit_source_buckets.items():
    print(
        f"{k:10s}"
        f"{v['count']:10,d}"
        f"{v['share_pct']:10.4f}%"
    )

print("\nDestination:")
for k, v in legit_destination_buckets.items():
    print(
        f"{k:10s}"
        f"{v['count']:10,d}"
        f"{v['share_pct']:10.4f}%"
    )

# ============================================================
# 7. ENTITY ID DNA
# ============================================================
print("\n" + "=" * 80)
print("6. ENTITY ID DNA")
print("=" * 80)

entity_ids = (
    accounts["Entity ID"]
    .astype(str)
)

entity_id_lengths = (
    entity_ids
    .str.len()
    .value_counts()
    .sort_index()
)

entity_id_position_models = {}

for pos in range(
    int(entity_ids.str.len().max())
):

    vc = (
        entity_ids
        .str[pos]
        .value_counts()
    )

    total = vc.sum()

    entity_id_position_models[
        str(pos + 1)
    ] = {
        str(char): {
            "count": int(count),
            "probability": float(
                count / total
            )
        }
        for char, count
        in vc.items()
    }

print(
    "Entity ID lengths:"
)

print(
    entity_id_lengths.to_string()
)

# ============================================================
# 8. ENTITY NAME DNA
# ============================================================
print("\n" + "=" * 80)
print("7. ENTITY NAME DNA")
print("=" * 80)

entity_names = (
    accounts["Entity Name"]
    .astype(str)
)

name_lengths = (
    entity_names
    .str.len()
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print(
    name_lengths.to_string()
)

# Character composition
name_digit_pct = (
    entity_names
    .str.contains(
        r"\d",
        regex=True
    )
    .mean()
    * 100
)

name_letter_pct = (
    entity_names
    .str.contains(
        r"[A-Za-z]",
        regex=True
    )
    .mean()
    * 100
)

name_space_pct = (
    entity_names
    .str.contains(
        r"\s",
        regex=True
    )
    .mean()
    * 100
)

print(
    "\nNames containing digits:",
    f"{name_digit_pct:.4f}%"
)

print(
    "Names containing letters:",
    f"{name_letter_pct:.4f}%"
)

print(
    "Names containing spaces:",
    f"{name_space_pct:.4f}%"
)

print("\nExample Entity IDs:")

for x in (
    entity_ids
    .drop_duplicates()
    .head(30)
):
    print(x)

print("\nExample Entity Names:")

for x in (
    entity_names
    .drop_duplicates()
    .head(30)
):
    print(x)

# ============================================================
# 9. ENTITY NAME TOP PATTERNS
# ============================================================
#
# We inspect whether names follow simple templates such as:
#   Entity 123
#   Company 45
#   Individual 987
# etc.
#
# No names are copied into the generator yet.
# ============================================================

name_prefixes = (
    entity_names
    .str.extract(
        r"^([A-Za-z]+)",
        expand=False
    )
    .value_counts()
    .head(30)
)

print(
    "\nTop alphabetic name prefixes:"
)

print(
    name_prefixes.to_string()
)

# ============================================================
# 10. SAVE ACCOUNT REUSE DNA
# ============================================================
account_reuse_dna = {

    "source_activity_all":
        source_all_summary,

    "destination_activity_all":
        destination_all_summary,

    "source_activity_legitimate":
        source_legit_summary,

    "destination_activity_legitimate":
        destination_legit_summary,

    "legitimate_source_activity_buckets":
        legit_source_buckets,

    "legitimate_destination_activity_buckets":
        legit_destination_buckets,

    "unique_source_accounts":
        int(len(source_counts)),

    "unique_destination_accounts":
        int(len(destination_counts)),

    "unique_legitimate_source_accounts":
        int(len(legit_source_counts)),

    "unique_legitimate_destination_accounts":
        int(len(legit_destination_counts))
}

with open(
    OUT_ACCOUNT_DNA,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        account_reuse_dna,
        f,
        indent=2
    )

# ============================================================
# 11. SAVE ENTITY DNA
# ============================================================
entity_name_dna = {

    "unique_entity_ids":
        int(entity_ids.nunique()),

    "unique_entity_names":
        int(entity_names.nunique()),

    "entity_id_length_distribution": {
        str(k): int(v)
        for k, v in
        entity_id_lengths.items()
    },

    "entity_id_position_models":
        entity_id_position_models,

    "entity_account_summary": {
        "min": int(entity_account_count.min()),
        "median": float(entity_account_count.median()),
        "mean": float(entity_account_count.mean()),
        "p90": float(entity_account_count.quantile(0.90)),
        "p95": float(entity_account_count.quantile(0.95)),
        "p99": float(entity_account_count.quantile(0.99)),
        "max": int(entity_account_count.max())
    },

    "entity_bank_summary": {
        "min": int(entity_bank_count.min()),
        "median": float(entity_bank_count.median()),
        "mean": float(entity_bank_count.mean()),
        "p90": float(entity_bank_count.quantile(0.90)),
        "p95": float(entity_bank_count.quantile(0.95)),
        "p99": float(entity_bank_count.quantile(0.99)),
        "max": int(entity_bank_count.max())
    },

    "entity_name_length_summary": {
        str(k): float(v)
        for k, v in name_lengths.items()
    },

    "names_with_digits_pct":
        float(name_digit_pct),

    "names_with_letters_pct":
        float(name_letter_pct),

    "names_with_spaces_pct":
        float(name_space_pct),

    "top_name_prefixes": {
        str(k): int(v)
        for k, v in name_prefixes.items()
    }
}

with open(
    OUT_ENTITY_DNA,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        entity_name_dna,
        f,
        indent=2
    )

# ============================================================
# 12. FINAL
# ============================================================
print("\n" + "=" * 100)
print("✅ CELL 10 COMPLETE")
print("=" * 100)

print(
    "Account reuse DNA:",
    OUT_ACCOUNT_DNA
)

print(
    "Entity DNA:",
    OUT_ENTITY_DNA
)

print("\nNO SYNTHETIC ACCOUNTS CREATED.")
print("NO SYNTHETIC TRANSACTIONS CREATED.")
print("NO ORIGINAL FILE MODIFIED.")

print("=" * 100)

CELL 10 — ACCOUNT REUSE + ENTITY DNA

1. ACCOUNT TABLE STRUCTURE
Account rows: 518,581
Unique account numbers: 518,573
Unique bank-account pairs: 518,581
Duplicate account-table rows: 0

2. ENTITY REUSE
Entities: 166,207
Median accounts/entity: 1.0
Mean accounts/entity: 3.1200912115614865
P90 accounts/entity: 6.0
P95 accounts/entity: 7.0
P99 accounts/entity: 14.0
Max accounts/entity: 7820
Median banks/entity: 1.0
Mean banks/entity: 2.6658624486333307
Entities with >=2 banks: 63362

3. ACCOUNT TRANSACTION REUSE
  Chunk 1 | rows processed = 500,000
  Chunk 2 | rows processed = 1,000,000
  Chunk 3 | rows processed = 1,500,000
  Chunk 4 | rows processed = 2,000,000
  Chunk 5 | rows processed = 2,500,000
  Chunk 6 | rows processed = 3,000,000
  Chunk 7 | rows processed = 3,500,000
  Chunk 8 | rows processed = 4,000,000
  Chunk 9 | rows processed = 4,500,000
  Chunk 10 | rows processed = 5,000,000
  Chunk 11 | rows processed = 5,078,345

Rows scanned: 5,078,345

4. ACTIVITY SUMMARY

All sour

In [17]:
# ============================================================
# CELL 11 — GENERATOR ACCOUNT REGISTRY
# ============================================================
#
# Purpose:
#   Build a generator-ready account registry from ORIGINAL
#   HI-Small only.
#
# This is important because the final synthetic dataset should
# NOT contain millions of one-use synthetic accounts.
#
# We will preferentially REUSE realistic original accounts
# when constructing laundering patterns.
#
# Creates:
#   HI-Small_generator_account_registry.csv
#   HI-Small_generator_account_registry_summary.json
#
# NO synthetic accounts
# NO synthetic transactions
# NO original files modified
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

ACCOUNTS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_accounts.csv"
)

TRANS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_Trans.csv"
)

OUT_REGISTRY = os.path.join(
    WORK_DIR,
    "HI-Small_generator_account_registry.csv"
)

OUT_SUMMARY = os.path.join(
    WORK_DIR,
    "HI-Small_generator_account_registry_summary.json"
)

print("=" * 100)
print("CELL 11 — GENERATOR ACCOUNT REGISTRY")
print("=" * 100)

# ============================================================
# 1. LOAD ACCOUNTS TABLE
# ============================================================
print("\n" + "=" * 80)
print("1. LOADING ORIGINAL ACCOUNT TABLE")
print("=" * 80)

accounts = pd.read_csv(
    ACCOUNTS_PATH,
    dtype=str
)

# ------------------------------------------------------------
# Normalize bank IDs
# ------------------------------------------------------------
def normalize_bank_id(x):

    x = str(x).strip()

    if x == "":
        return None

    try:
        return str(int(x))
    except Exception:
        return x.lstrip("0") or "0"


accounts["normalized_bank_id"] = (
    accounts["Bank ID"]
    .map(normalize_bank_id)
)

accounts["Account Number"] = (
    accounts["Account Number"]
    .astype(str)
)

accounts["Entity ID"] = (
    accounts["Entity ID"]
    .astype(str)
)

accounts["Entity Name"] = (
    accounts["Entity Name"]
    .astype(str)
)

# ============================================================
# 2. REMOVE EXACT ACCOUNT-TABLE DUPLICATES
# ============================================================
#
# There are duplicate ACCOUNT NUMBER VALUES in the original
# data, but they are not necessarily duplicate rows.
#
# We retain unique bank-account combinations.
# ============================================================

registry = (
    accounts[
        [
            "normalized_bank_id",
            "Bank ID",
            "Bank Name",
            "Account Number",
            "Entity ID",
            "Entity Name"
        ]
    ]
    .drop_duplicates(
        [
            "normalized_bank_id",
            "Account Number"
        ]
    )
    .copy()
)

print(
    "Original account rows:",
    f"{len(accounts):,}"
)

print(
    "Unique bank-account registry rows:",
    f"{len(registry):,}"
)

# ============================================================
# 3. SCAN ORIGINAL TRANSACTIONS FOR ACCOUNT ACTIVITY
# ============================================================
print("\n" + "=" * 80)
print("2. SCANNING ORIGINAL TRANSACTION ACTIVITY")
print("=" * 80)

src_counts = {}
dst_counts = {}

legit_src_counts = {}
legit_dst_counts = {}

laund_src_counts = {}
laund_dst_counts = {}

chunk_size = 500_000
rows_seen = 0
chunk_no = 0

for chunk in pd.read_csv(
    TRANS_PATH,
    usecols=[
        "Account",
        "Account.1",
        "Is Laundering"
    ],
    dtype=str,
    chunksize=chunk_size
):

    chunk_no += 1
    rows_seen += len(chunk)

    labels = (
        pd.to_numeric(
            chunk["Is Laundering"],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
    )

    # --------------------------------------------------------
    # All source
    # --------------------------------------------------------
    src = (
        chunk["Account"]
        .astype(str)
        .value_counts()
    )

    for account, count in src.items():

        src_counts[account] = (
            src_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    # --------------------------------------------------------
    # All destination
    # --------------------------------------------------------
    dst = (
        chunk["Account.1"]
        .astype(str)
        .value_counts()
    )

    for account, count in dst.items():

        dst_counts[account] = (
            dst_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    # --------------------------------------------------------
    # Legitimate
    # --------------------------------------------------------
    legit = chunk[
        labels == 0
    ]

    legit_src = (
        legit["Account"]
        .astype(str)
        .value_counts()
    )

    for account, count in legit_src.items():

        legit_src_counts[account] = (
            legit_src_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    legit_dst = (
        legit["Account.1"]
        .astype(str)
        .value_counts()
    )

    for account, count in legit_dst.items():

        legit_dst_counts[account] = (
            legit_dst_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    # --------------------------------------------------------
    # Original laundering
    # --------------------------------------------------------
    laund = chunk[
        labels == 1
    ]

    laund_src = (
        laund["Account"]
        .astype(str)
        .value_counts()
    )

    for account, count in laund_src.items():

        laund_src_counts[account] = (
            laund_src_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    laund_dst = (
        laund["Account.1"]
        .astype(str)
        .value_counts()
    )

    for account, count in laund_dst.items():

        laund_dst_counts[account] = (
            laund_dst_counts.get(
                account,
                0
            )
            +
            int(count)
        )

    print(
        f"  Chunk {chunk_no} "
        f"| rows processed = {rows_seen:,}"
    )

print(
    "\nRows scanned:",
    f"{rows_seen:,}"
)

# ============================================================
# 4. ADD ACTIVITY FEATURES
# ============================================================

registry["source_tx_count"] = (
    registry["Account Number"]
    .map(src_counts)
    .fillna(0)
    .astype(int)
)

registry["destination_tx_count"] = (
    registry["Account Number"]
    .map(dst_counts)
    .fillna(0)
    .astype(int)
)

registry["legitimate_source_tx_count"] = (
    registry["Account Number"]
    .map(legit_src_counts)
    .fillna(0)
    .astype(int)
)

registry["legitimate_destination_tx_count"] = (
    registry["Account Number"]
    .map(legit_dst_counts)
    .fillna(0)
    .astype(int)
)

registry["original_laundering_source_count"] = (
    registry["Account Number"]
    .map(laund_src_counts)
    .fillna(0)
    .astype(int)
)

registry["original_laundering_destination_count"] = (
    registry["Account Number"]
    .map(laund_dst_counts)
    .fillna(0)
    .astype(int)
)

# ============================================================
# 5. ACCOUNT ROLE CLASSIFICATION
# ============================================================
#
# These are generator convenience flags, NOT labels.
#
# source_only
# destination_only
# both
# inactive
# ============================================================

registry["role"] = np.select(
    [
        (
            (registry["legitimate_source_tx_count"] > 0)
            &
            (registry["legitimate_destination_tx_count"] > 0)
        ),

        (
            (registry["legitimate_source_tx_count"] > 0)
            &
            (registry["legitimate_destination_tx_count"] == 0)
        ),

        (
            (registry["legitimate_source_tx_count"] == 0)
            &
            (registry["legitimate_destination_tx_count"] > 0)
        )
    ],
    [
        "both",
        "source_only",
        "destination_only"
    ],
    default="inactive"
)

# ============================================================
# 6. LEGITIMATE ACCOUNT ELIGIBILITY
# ============================================================
#
# Accounts that never appeared in an original laundering
# transaction are our cleanest candidate pool.
#
# This avoids inheriting original laundering labels into
# newly synthesized episodes.
# ============================================================

registry["originally_clean"] = (
    (
        registry[
            "original_laundering_source_count"
        ] == 0
    )
    &
    (
        registry[
            "original_laundering_destination_count"
        ] == 0
    )
)

# Accounts with legitimate activity
registry["has_legitimate_activity"] = (
    (
        registry[
            "legitimate_source_tx_count"
        ]
        +
        registry[
            "legitimate_destination_tx_count"
        ]
    ) > 0
)

# Primary generator pool
registry["generator_eligible"] = (
    registry["originally_clean"]
    &
    registry["has_legitimate_activity"]
)

# ============================================================
# 7. ACCOUNT ACTIVITY TOTAL
# ============================================================
registry["legitimate_total_activity"] = (
    registry[
        "legitimate_source_tx_count"
    ]
    +
    registry[
        "legitimate_destination_tx_count"
    ]
)

# ============================================================
# 8. ACTIVITY BUCKET
# ============================================================
bins = [
    0,
    1,
    2,
    5,
    10,
    25,
    50,
    100,
    250,
    500,
    1000,
    np.inf
]

labels = [
    "1",
    "2",
    "3-5",
    "6-10",
    "11-25",
    "26-50",
    "51-100",
    "101-250",
    "251-500",
    "501-1000",
    ">1000"
]

registry["legitimate_activity_bucket"] = (
    pd.cut(
        registry[
            "legitimate_total_activity"
        ].clip(lower=1),
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    .astype(str)
)

# ============================================================
# 9. ACCOUNT ROLE COUNTS
# ============================================================
print("\n" + "=" * 80)
print("3. ACCOUNT ROLE DISTRIBUTION")
print("=" * 80)

print(
    registry["role"]
    .value_counts()
    .to_string()
)

print("\nGenerator eligible accounts:")
print(
    int(
        registry[
            "generator_eligible"
        ].sum()
    )
)

print("\nOriginally clean accounts:")
print(
    int(
        registry[
            "originally_clean"
        ].sum()
    )
)

# ============================================================
# 10. ACTIVITY DISTRIBUTION
# ============================================================
print("\n" + "=" * 80)
print("4. GENERATOR-ELIGIBLE ACTIVITY DISTRIBUTION")
print("=" * 80)

eligible = registry[
    registry["generator_eligible"]
].copy()

activity = (
    eligible[
        "legitimate_total_activity"
    ]
)

print(
    "Eligible accounts:",
    f"{len(eligible):,}"
)

print(
    "Min:",
    activity.min()
)

print(
    "Median:",
    activity.median()
)

print(
    "Mean:",
    activity.mean()
)

print(
    "P75:",
    activity.quantile(0.75)
)

print(
    "P90:",
    activity.quantile(0.90)
)

print(
    "P95:",
    activity.quantile(0.95)
)

print(
    "P99:",
    activity.quantile(0.99)
)

print(
    "Max:",
    activity.max()
)

# ============================================================
# 11. ROLE × ACTIVITY DISTRIBUTION
# ============================================================
print("\n" + "=" * 80)
print("5. ROLE × ACTIVITY BUCKET")
print("=" * 80)

role_activity = pd.crosstab(
    eligible["role"],
    eligible[
        "legitimate_activity_bucket"
    ],
    normalize="index"
) * 100

print(
    role_activity.round(4).to_string()
)

# ============================================================
# 12. BANK-LEVEL ACCOUNT POOL SIZE
# ============================================================
print("\n" + "=" * 80)
print("6. BANK × ELIGIBLE ACCOUNT COUNTS")
print("=" * 80)

eligible_per_bank = (
    eligible
    .groupby(
        "normalized_bank_id"
    )
    .size()
    .rename(
        "eligible_account_count"
    )
)

print(
    "Banks with eligible accounts:",
    len(eligible_per_bank)
)

print(
    "Median eligible accounts/bank:",
    eligible_per_bank.median()
)

print(
    "Mean eligible accounts/bank:",
    eligible_per_bank.mean()
)

print(
    "P90:",
    eligible_per_bank.quantile(0.90)
)

print(
    "P95:",
    eligible_per_bank.quantile(0.95)
)

print(
    "P99:",
    eligible_per_bank.quantile(0.99)
)

print(
    "Max:",
    eligible_per_bank.max()
)

# ============================================================
# 13. ENTITY STRUCTURE IN ELIGIBLE POOL
# ============================================================
print("\n" + "=" * 80)
print("7. ENTITY STRUCTURE")
print("=" * 80)

eligible_entity_account_counts = (
    eligible
    .groupby("Entity ID")
    .size()
)

eligible_entity_bank_counts = (
    eligible
    .groupby("Entity ID")[
        "normalized_bank_id"
    ]
    .nunique()
)

print(
    "Eligible entities:",
    f"{len(eligible_entity_account_counts):,}"
)

print(
    "Median eligible accounts/entity:",
    eligible_entity_account_counts.median()
)

print(
    "Mean eligible accounts/entity:",
    eligible_entity_account_counts.mean()
)

print(
    "P90:",
    eligible_entity_account_counts.quantile(0.90)
)

print(
    "P95:",
    eligible_entity_account_counts.quantile(0.95)
)

print(
    "P99:",
    eligible_entity_account_counts.quantile(0.99)
)

print(
    "Max:",
    eligible_entity_account_counts.max()
)

print(
    "Eligible entities with >=2 banks:",
    int(
        (
            eligible_entity_bank_counts >= 2
        ).sum()
    )
)

# ============================================================
# 14. SAVE REGISTRY
# ============================================================
registry = registry.sort_values(
    [
        "normalized_bank_id",
        "legitimate_total_activity",
        "Account Number"
    ],
    ascending=[
        True,
        False,
        True
    ]
).reset_index(drop=True)

registry.to_csv(
    OUT_REGISTRY,
    index=False
)

# ============================================================
# 15. SAVE SUMMARY JSON
# ============================================================
summary = {

    "original_account_rows":
        int(len(accounts)),

    "unique_bank_account_rows":
        int(len(registry)),

    "generator_eligible_accounts":
        int(
            registry[
                "generator_eligible"
            ].sum()
        ),

    "originally_clean_accounts":
        int(
            registry[
                "originally_clean"
            ].sum()
        ),

    "accounts_with_legitimate_activity":
        int(
            registry[
                "has_legitimate_activity"
            ].sum()
        ),

    "role_distribution":
        {
            str(k): int(v)
            for k, v in registry[
                "role"
            ].value_counts().items()
        },

    "eligible_activity_summary": {
        "min": int(activity.min()),
        "median": float(activity.median()),
        "mean": float(activity.mean()),
        "p75": float(activity.quantile(0.75)),
        "p90": float(activity.quantile(0.90)),
        "p95": float(activity.quantile(0.95)),
        "p99": float(activity.quantile(0.99)),
        "max": int(activity.max())
    },

    "eligible_entities":
        int(
            len(
                eligible_entity_account_counts
            )
        ),

    "eligible_entities_multiple_banks":
        int(
            (
                eligible_entity_bank_counts >= 2
            ).sum()
        ),

    "banks_with_eligible_accounts":
        int(
            len(eligible_per_bank)
        )
}

with open(
    OUT_SUMMARY,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# ============================================================
# 16. FINAL VALIDATION
# ============================================================
print("\n" + "=" * 100)
print("✅ CELL 11 COMPLETE")
print("=" * 100)

print(
    "Generator account registry:",
    OUT_REGISTRY
)

print(
    "Registry summary:",
    OUT_SUMMARY
)

print(
    "\nTotal registry rows:",
    f"{len(registry):,}"
)

print(
    "Generator-eligible:",
    f"{int(registry['generator_eligible'].sum()):,}"
)

print(
    "Originally clean:",
    f"{int(registry['originally_clean'].sum()):,}"
)

print("\nNO SYNTHETIC ACCOUNTS CREATED.")
print("NO SYNTHETIC TRANSACTIONS CREATED.")
print("NO ORIGINAL FILE MODIFIED.")

print("=" * 100)

CELL 11 — GENERATOR ACCOUNT REGISTRY

1. LOADING ORIGINAL ACCOUNT TABLE
Original account rows: 518,581
Unique bank-account registry rows: 518,581

2. SCANNING ORIGINAL TRANSACTION ACTIVITY
  Chunk 1 | rows processed = 500,000
  Chunk 2 | rows processed = 1,000,000
  Chunk 3 | rows processed = 1,500,000
  Chunk 4 | rows processed = 2,000,000
  Chunk 5 | rows processed = 2,500,000
  Chunk 6 | rows processed = 3,000,000
  Chunk 7 | rows processed = 3,500,000
  Chunk 8 | rows processed = 4,000,000
  Chunk 9 | rows processed = 4,500,000
  Chunk 10 | rows processed = 5,000,000
  Chunk 11 | rows processed = 5,078,345

Rows scanned: 5,078,345

3. ACCOUNT ROLE DISTRIBUTION
role
both                402449
source_only          94430
destination_only     18140
inactive              3562

Generator eligible accounts:
508731

Originally clean accounts:
512224

4. GENERATOR-ELIGIBLE ACTIVITY DISTRIBUTION
Eligible accounts: 508,731
Min: 1
Median: 6.0
Mean: 18.68091781314683
P75: 27.0
P90: 51.0
P95: 70

In [18]:
# ============================================================
# CELL 12 — EXACT SYNTHETIC PATTERN ALLOCATION FEASIBILITY
# ============================================================
#
# TARGET:
#   Legitimate rows already in original HI-Small:
#       5,073,168
#
#   Required synthetic laundering rows:
#       5,067,982
#
#   Final total:
#      10,146,150 rows
#
# NOTE:
#   This is based on ORIGINAL HI-Small counts.
#   The previous synthetic dataset is NOT used.
#
# This cell:
#   1. Loads exact IBM templates
#   2. Calculates typology transaction proportions
#   3. Calculates target synthetic transaction count
#   4. Tests exact feasibility per typology
#   5. Checks GCD / reachable transaction counts
#   6. Produces generation targets
#
# NO SYNTHETIC DATA CREATED.
# ============================================================

import os
import json
import math
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

TEMPLATE_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_templates.json"
)

OUT_PLAN = os.path.join(
    WORK_DIR,
    "HI-Small_synthetic_generation_plan.json"
)

# ------------------------------------------------------------
# ORIGINAL DATA COUNTS
# ------------------------------------------------------------
ORIGINAL_LEGITIMATE = 5_073_168
ORIGINAL_LAUNDERING = 5_177

TARGET_SYNTHETIC_LAUNDERING = (
    5_073_168 - ORIGINAL_LAUNDERING
)

FINAL_TOTAL = (
    ORIGINAL_LEGITIMATE
    +
    ORIGINAL_LAUNDERING
    +
    TARGET_SYNTHETIC_LAUNDERING
)

print("=" * 100)
print("CELL 12 — EXACT SYNTHETIC PATTERN ALLOCATION FEASIBILITY")
print("=" * 100)

print("\nOriginal legitimate rows:",
      f"{ORIGINAL_LEGITIMATE:,}")

print("Original laundering rows:",
      f"{ORIGINAL_LAUNDERING:,}")

print("Synthetic laundering required:",
      f"{TARGET_SYNTHETIC_LAUNDERING:,}")

print("Final balanced total:",
      f"{FINAL_TOTAL:,}")

# ============================================================
# 1. LOAD TEMPLATES
# ============================================================
with open(
    TEMPLATE_PATH,
    "r",
    encoding="utf-8"
) as f:

    manifest = json.load(f)

templates = manifest["templates"]

print("\nTemplates loaded:",
      len(templates))

# ============================================================
# 2. BUILD TEMPLATE DATAFRAME
# ============================================================
template_df = pd.DataFrame(
    [
        {
            "template_id":
                t["template_id"],

            "episode_index":
                t["episode_index"],

            "typology":
                t["typology"],

            "transaction_count":
                t["transaction_count"],

            "node_count":
                t["node_count"],

            "span_minutes":
                t["episode_span_minutes"]
        }
        for t in templates
    ]
)

# ============================================================
# 3. ORIGINAL IBM TYPOLOGY DISTRIBUTION
# ============================================================
print("\n" + "=" * 80)
print("1. ORIGINAL IBM TYPOLOGY DISTRIBUTION")
print("=" * 80)

typology_stats = (
    template_df
    .groupby("typology")
    .agg(
        episodes=(
            "template_id",
            "count"
        ),

        transactions=(
            "transaction_count",
            "sum"
        ),

        mean_transactions=(
            "transaction_count",
            "mean"
        ),

        median_transactions=(
            "transaction_count",
            "median"
        ),

        min_transactions=(
            "transaction_count",
            "min"
        ),

        max_transactions=(
            "transaction_count",
            "max"
        )
    )
    .sort_index()
)

total_pattern_transactions = (
    typology_stats[
        "transactions"
    ].sum()
)

typology_stats[
    "transaction_share"
] = (
    typology_stats[
        "transactions"
    ]
    /
    total_pattern_transactions
)

print(
    typology_stats.to_string()
)

print(
    "\nTotal IBM pattern transactions:",
    total_pattern_transactions
)

# ============================================================
# 4. TARGET TRANSACTIONS BY TYPOLOGY
# ============================================================
print("\n" + "=" * 80)
print("2. TARGET SYNTHETIC TRANSACTIONS BY TYPOLOGY")
print("=" * 80)

# Largest-remainder allocation guarantees that the
# integer typology targets sum EXACTLY to the target.
raw_targets = (
    typology_stats[
        "transaction_share"
    ]
    *
    TARGET_SYNTHETIC_LAUNDERING
)

floor_targets = np.floor(
    raw_targets
).astype(int)

remainder = (
    TARGET_SYNTHETIC_LAUNDERING
    -
    int(floor_targets.sum())
)

fractional_parts = (
    raw_targets
    -
    floor_targets
)

largest_fractional = (
    fractional_parts
    .sort_values(
        ascending=False
    )
)

target_transactions = floor_targets.copy()

for typology in (
    largest_fractional
    .head(remainder)
    .index
):

    target_transactions.loc[
        typology
    ] += 1

typology_stats[
    "target_transactions"
] = target_transactions

print(
    typology_stats[
        [
            "episodes",
            "transactions",
            "transaction_share",
            "target_transactions"
        ]
    ].to_string()
)

print(
    "\nTarget sum:",
    f"{typology_stats['target_transactions'].sum():,}"
)

assert (
    typology_stats[
        "target_transactions"
    ].sum()
    ==
    TARGET_SYNTHETIC_LAUNDERING
)

# ============================================================
# 5. TEMPLATE SIZE FEASIBILITY
# ============================================================
print("\n" + "=" * 80)
print("3. TEMPLATE SIZE FEASIBILITY")
print("=" * 80)

feasibility = {}

for typology, g in (
    template_df
    .groupby("typology")
):

    sizes = sorted(
        set(
            g[
                "transaction_count"
            ].astype(int)
        )
    )

    gcd_all = 0

    for size in sizes:
        gcd_all = math.gcd(
            gcd_all,
            int(size)
        )

    feasibility[
        typology
    ] = {

        "unique_transaction_sizes":
            sizes,

        "min_template_transactions":
            int(min(sizes)),

        "max_template_transactions":
            int(max(sizes)),

        "gcd_of_template_sizes":
            int(gcd_all),

        "template_count":
            int(len(g))
    }

    print(
        f"\n{typology}"
    )

    print(
        "  Template count:",
        len(g)
    )

    print(
        "  Transaction sizes:",
        sizes
    )

    print(
        "  Min:",
        min(sizes)
    )

    print(
        "  Max:",
        max(sizes)
    )

    print(
        "  GCD:",
        gcd_all
    )

# ============================================================
# 6. EXACT REACHABILITY CHECK
# ============================================================
print("\n" + "=" * 80)
print("4. EXACT TARGET REACHABILITY")
print("=" * 80)

def reachable_with_unbounded_templates(
    target,
    sizes
):
    """
    Check whether target can be formed using any number
    of template transaction sizes.

    We only search a bounded tail because target is huge.
    With realistic template sizes and gcd=1, every
    sufficiently large value is reachable.

    Returns:
        True / False
    """

    target = int(target)

    sizes = sorted(
        set(
            int(x)
            for x in sizes
            if int(x) > 0
        )
    )

    if not sizes:
        return False

    # GCD necessary condition
    g = 0

    for size in sizes:
        g = math.gcd(g, size)

    if target % g != 0:
        return False

    # For very large targets, inspect a bounded region
    # near the target.
    max_size = max(sizes)

    # Dynamic programming only needs a finite tail.
    limit = min(
        target,
        max_size * 20_000
    )

    reachable = np.zeros(
        limit + 1,
        dtype=np.bool_
    )

    reachable[0] = True

    for value in range(
        1,
        limit + 1
    ):

        for size in sizes:

            if size > value:
                break

            if reachable[
                value - size
            ]:

                reachable[value] = True
                break

    # If target itself is within calculated region
    if target <= limit:
        return bool(
            reachable[target]
        )

    # Otherwise, test a reachable tail.
    # If gcd=1 and the largest target region contains a
    # consecutive reachable stretch of max_size values,
    # then every sufficiently larger target is reachable.
    tail_start = (
        limit
        -
        max_size
        +
        1
    )

    tail = reachable[
        tail_start:
        limit + 1
    ]

    return bool(
        tail.all()
        and
        g == 1
    )

reachability = {}

for typology in typology_stats.index:

    target = int(
        typology_stats.loc[
            typology,
            "target_transactions"
        ]
    )

    sizes = feasibility[
        typology
    ]["unique_transaction_sizes"]

    possible = (
        reachable_with_unbounded_templates(
            target,
            sizes
        )
    )

    reachability[
        typology
    ] = bool(possible)

    print(
        f"{typology:20s}"
        f" target={target:,}"
        f" | reachable={possible}"
    )

# ============================================================
# 7. ESTIMATED EPISODE COUNTS
# ============================================================
print("\n" + "=" * 80)
print("5. ESTIMATED EPISODE COUNTS")
print("=" * 80)

for typology in typology_stats.index:

    target_tx = int(
        typology_stats.loc[
            typology,
            "target_transactions"
        ]
    )

    mean_tx = float(
        typology_stats.loc[
            typology,
            "mean_transactions"
        ]
    )

    estimated_episodes = int(
        round(
            target_tx
            /
            mean_tx
        )
    )

    print(
        f"{typology:20s}"
        f" target_tx={target_tx:,}"
        f" | mean_tx/episode={mean_tx:.3f}"
        f" | estimated episodes={estimated_episodes:,}"
    )

# ============================================================
# 8. GLOBAL ESTIMATED EPISODE COUNT
# ============================================================
estimated_total_episodes = 0

for typology in typology_stats.index:

    target_tx = int(
        typology_stats.loc[
            typology,
            "target_transactions"
        ]
    )

    mean_tx = float(
        typology_stats.loc[
            typology,
            "mean_transactions"
        ]
    )

    estimated_total_episodes += int(
        round(
            target_tx / mean_tx
        )
    )

print(
    "\nEstimated synthetic laundering episodes:",
    f"{estimated_total_episodes:,}"
)

# ============================================================
# 9. SAVE PLAN
# ============================================================
plan = {

    "source_dataset":
        "IBM HI-Small",

    "original_legitimate_rows":
        ORIGINAL_LEGITIMATE,

    "original_laundering_rows":
        ORIGINAL_LAUNDERING,

    "synthetic_laundering_rows_required":
        TARGET_SYNTHETIC_LAUNDERING,

    "final_total_rows":
        FINAL_TOTAL,

    "original_pattern_episodes":
        int(len(templates)),

    "original_pattern_transactions":
        int(total_pattern_transactions),

    "typology_targets": {

        str(typology): {

            "original_episodes":
                int(
                    typology_stats.loc[
                        typology,
                        "episodes"
                    ]
                ),

            "original_transactions":
                int(
                    typology_stats.loc[
                        typology,
                        "transactions"
                    ]
                ),

            "original_transaction_share":
                float(
                    typology_stats.loc[
                        typology,
                        "transaction_share"
                    ]
                ),

            "target_synthetic_transactions":
                int(
                    typology_stats.loc[
                        typology,
                        "target_transactions"
                    ]
                ),

            "template_sizes":
                feasibility[
                    typology
                ][
                    "unique_transaction_sizes"
                ],

            "template_size_gcd":
                feasibility[
                    typology
                ][
                    "gcd_of_template_sizes"
                ],

            "exactly_reachable":
                bool(
                    reachability[
                        typology
                    ]
                )
        }

        for typology
        in typology_stats.index
    }
}

with open(
    OUT_PLAN,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        plan,
        f,
        indent=2
    )

# ============================================================
# 10. FINAL
# ============================================================
print("\n" + "=" * 100)
print("✅ CELL 12 COMPLETE")
print("=" * 100)

print(
    "Synthetic laundering target:",
    f"{TARGET_SYNTHETIC_LAUNDERING:,}"
)

print(
    "Final balanced dataset:",
    f"{FINAL_TOTAL:,}"
)

print(
    "All typology targets sum exactly:",
    int(
        typology_stats[
            "target_transactions"
        ].sum()
    )
    ==
    TARGET_SYNTHETIC_LAUNDERING
)

print(
    "All typologies exactly reachable:",
    all(
        reachability.values()
    )
)

print(
    "\nSaved plan:"
)

print(
    OUT_PLAN
)

print(
    "\nNO SYNTHETIC ACCOUNTS CREATED."
)

print(
    "NO SYNTHETIC TRANSACTIONS CREATED."
)

print(
    "NO ORIGINAL FILE MODIFIED."
)

print("=" * 100)

CELL 12 — EXACT SYNTHETIC PATTERN ALLOCATION FEASIBILITY

Original legitimate rows: 5,073,168
Original laundering rows: 5,177
Synthetic laundering required: 5,067,991
Final balanced total: 10,146,336

Templates loaded: 370

1. ORIGINAL IBM TYPOLOGY DISTRIBUTION
                episodes  transactions  mean_transactions  median_transactions  min_transactions  max_transactions  transaction_share
typology                                                                                                                             
BIPARTITE             49           263           5.367347                  4.0                 1                15           0.081957
CYCLE                 54           287           5.314815                  4.0                 2                12           0.089436
FAN-IN                40           318           7.950000                  8.0                 1                16           0.099096
FAN-OUT               48           342           7.125000           

In [19]:
# ============================================================
# CELL 13 — PARITY-CORRECTED EXACT GENERATION PLAN
# ============================================================
#
# Problem discovered in CELL 12:
#
#   SCATTER-GATHER template sizes are all EVEN
#   STACK template sizes are all EVEN
#
# Therefore their target transaction counts MUST be even.
#
# Current targets:
#   SCATTER-GATHER = 988,645  -> impossible (odd)
#   STACK           = 735,956  -> already possible (even)
#
# Wait:
#   STACK is already even.
#   Therefore only SCATTER-GATHER requires correction.
#
# We will:
#   +1 SCATTER-GATHER
#   -1 from a typology whose templates can produce any large
#      integer count.
#
# The adjustment is selected to minimize proportional distortion.
#
# NO SYNTHETIC DATA CREATED.
# ============================================================

import os
import json
import math
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

TEMPLATE_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_templates.json"
)

OLD_PLAN_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_synthetic_generation_plan.json"
)

OUT_PLAN = os.path.join(
    WORK_DIR,
    "HI-Small_synthetic_generation_plan_v2.json"
)

# ------------------------------------------------------------
# TARGET
# ------------------------------------------------------------
TARGET_SYNTHETIC = 5_067_991

# ============================================================
# 1. LOAD DATA
# ============================================================
with open(
    TEMPLATE_PATH,
    "r",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

templates = manifest["templates"]

template_df = pd.DataFrame(
    [
        {
            "template_id": t["template_id"],
            "typology": t["typology"],
            "transaction_count": int(
                t["transaction_count"]
            )
        }
        for t in templates
    ]
)

# ============================================================
# 2. ORIGINAL IBM TRANSACTION SHARES
# ============================================================
typology_original = (
    template_df
    .groupby("typology")[
        "transaction_count"
    ]
    .sum()
    .sort_index()
)

total_original = (
    typology_original.sum()
)

shares = (
    typology_original
    /
    total_original
)

# ============================================================
# 3. START WITH ORIGINAL CELL-12 TARGETS
# ============================================================
raw_targets = (
    shares
    *
    TARGET_SYNTHETIC
)

targets = np.floor(
    raw_targets
).astype(int)

remaining = (
    TARGET_SYNTHETIC
    -
    int(targets.sum())
)

fractional = (
    raw_targets
    -
    targets
)

for typology in (
    fractional
    .sort_values(
        ascending=False
    )
    .head(remaining)
    .index
):

    targets.loc[
        typology
    ] += 1

targets = targets.astype(int)

# ============================================================
# 4. TEMPLATE-SIZE FEASIBILITY
# ============================================================
size_info = {}

for typology, g in (
    template_df
    .groupby("typology")
):

    sizes = sorted(
        g[
            "transaction_count"
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    gcd_value = 0

    for s in sizes:
        gcd_value = math.gcd(
            gcd_value,
            int(s)
        )

    size_info[
        typology
    ] = {
        "sizes": sizes,
        "gcd": gcd_value,
        "min": min(sizes),
        "max": max(sizes)
    }

# ============================================================
# 5. IDENTIFY PARITY-CONSTRAINED TYPOLOGIES
# ============================================================
print("=" * 100)
print("CELL 13 — PARITY-CORRECTED EXACT GENERATION PLAN")
print("=" * 100)

print("\nInitial targets:")
print(
    targets.to_string()
)

parity_constrained = []

for typology in targets.index:

    sizes = size_info[
        typology
    ]["sizes"]

    # If all template sizes are even, target must be even.
    all_even = all(
        int(s) % 2 == 0
        for s in sizes
    )

    if all_even:

        parity_constrained.append(
            typology
        )

        print(
            f"\n{typology}:"
        )

        print(
            "  Template sizes:",
            sizes
        )

        print(
            "  GCD:",
            size_info[
                typology
            ]["gcd"]
        )

        print(
            "  Target:",
            int(
                targets.loc[
                    typology
                ]
            )
        )

        print(
            "  Target parity:",
            int(
                targets.loc[
                    typology
                ]
            ) % 2
        )

# ============================================================
# 6. FIND INFEASIBLE TARGETS
# ============================================================
infeasible = []

for typology in targets.index:

    target = int(
        targets.loc[
            typology
        ]
    )

    gcd_value = size_info[
        typology
    ]["gcd"]

    if target % gcd_value != 0:

        infeasible.append(
            typology
        )

print("\n" + "=" * 80)
print("1. INFEASIBLE TARGETS")
print("=" * 80)

for typology in infeasible:

    print(
        f"{typology:20s}"
        f" target={targets.loc[typology]:,}"
        f" gcd={size_info[typology]['gcd']}"
    )

# ============================================================
# 7. FIND BEST SOURCE TYPOLOGY FOR REDISTRIBUTION
# ============================================================
#
# We need:
#
#   target[SCATTER-GATHER] += 1
#
# and
#
#   target[source] -= 1
#
# so the total stays unchanged.
#
# Source must NOT be parity-constrained.
# We choose the largest target among eligible types because
# removing one transaction creates the smallest proportional
# disturbance.
# ============================================================

if len(infeasible) == 0:

    print(
        "\n✅ No feasibility correction required."
    )

else:

    source_candidates = []

    for typology in targets.index:

        if typology in infeasible:
            continue

        gcd_value = size_info[
            typology
        ]["gcd"]

        # gcd=1 means no parity restriction at the
        # transaction-count level.
        if gcd_value == 1:

            source_candidates.append(
                typology
            )

    source_candidates = sorted(
        source_candidates,
        key=lambda x:
            int(
                targets.loc[x]
            ),
        reverse=True
    )

    if len(source_candidates) == 0:

        raise RuntimeError(
            "No suitable typology available "
            "for target redistribution."
        )

    print("\nEligible source typologies:")

    for typology in source_candidates:

        print(
            f"  {typology:20s}"
            f" target={targets.loc[typology]:,}"
        )

    # --------------------------------------------------------
    # Determine correction.
    # --------------------------------------------------------
    #
    # In this dataset SCATTER-GATHER has an odd target and
    # requires +1.
    #
    correction_targets = targets.copy()

    for typology in infeasible:

        current = int(
            correction_targets.loc[
                typology
            ]
        )

        # Find nearest higher target satisfying gcd.
        new_target = current

        while (
            new_target % size_info[
                typology
            ]["gcd"] != 0
        ):
            new_target += 1

        delta = (
            new_target
            -
            current
        )

        correction_targets.loc[
            typology
        ] = new_target

        # Remove the same amount from the largest
        # eligible source.
        remaining_correction = delta

        for source in source_candidates:

            available = int(
                correction_targets.loc[
                    source
                ]
            )

            if available <= 0:
                continue

            remove = min(
                remaining_correction,
                available
            )

            correction_targets.loc[
                source
            ] -= remove

            remaining_correction -= remove

            if remaining_correction == 0:
                break

        if remaining_correction != 0:

            raise RuntimeError(
                "Could not redistribute "
                f"{delta} transactions for {typology}."
            )

    targets = correction_targets.astype(int)

# ============================================================
# 8. FINAL TARGET CHECK
# ============================================================
print("\n" + "=" * 80)
print("2. CORRECTED TARGETS")
print("=" * 80)

comparison = pd.DataFrame(
    {
        "original_target": (
            pd.Series(
                raw_targets
            )
            .round()
            .astype(int)
        ),

        "initial_target": (
            pd.Series(
                np.floor(
                    raw_targets
                ).astype(int)
            )
        ),

        "corrected_target": targets
    }
)

# Add actual proportional share
comparison[
    "corrected_share"
] = (
    comparison[
        "corrected_target"
    ]
    /
    TARGET_SYNTHETIC
)

comparison[
    "original_share"
] = shares

comparison[
    "delta"
] = (
    comparison[
        "corrected_target"
    ]
    -
    comparison[
        "initial_target"
    ]
)

print(
    comparison[
        [
            "initial_target",
            "corrected_target",
            "delta",
            "original_share",
            "corrected_share"
        ]
    ].to_string()
)

print(
    "\nCorrected target sum:",
    f"{targets.sum():,}"
)

assert (
    int(targets.sum())
    ==
    TARGET_SYNTHETIC
)

# ============================================================
# 9. EXACT GCD FEASIBILITY CHECK
# ============================================================
print("\n" + "=" * 80)
print("3. GCD FEASIBILITY")
print("=" * 80)

all_feasible = True

for typology in targets.index:

    target = int(
        targets.loc[
            typology
        ]
    )

    gcd_value = size_info[
        typology
    ]["gcd"]

    feasible = (
        target % gcd_value == 0
    )

    print(
        f"{typology:20s}"
        f" target={target:,}"
        f" | gcd={gcd_value}"
        f" | feasible_by_gcd={feasible}"
    )

    if not feasible:
        all_feasible = False

if not all_feasible:

    raise RuntimeError(
        "At least one corrected target "
        "still violates its template-size GCD."
    )

# ============================================================
# 10. ACTUAL TEMPLATE COUNTS NEEDED
# ============================================================
#
# Number of episodes depends on which templates are sampled.
#
# We estimate using original mean transaction count.
# ============================================================

print("\n" + "=" * 80)
print("4. ESTIMATED EPISODE COUNTS")
print("=" * 80)

episode_estimates = {}

for typology in targets.index:

    g = template_df[
        template_df["typology"]
        == typology
    ]

    mean_tx = g[
        "transaction_count"
    ].mean()

    estimate = int(
        round(
            int(
                targets.loc[
                    typology
                ]
            )
            /
            mean_tx
        )
    )

    episode_estimates[
        typology
    ] = estimate

    print(
        f"{typology:20s}"
        f" transactions={targets.loc[typology]:,}"
        f" | mean tx/template={mean_tx:.4f}"
        f" | estimated episodes={estimate:,}"
    )

# ============================================================
# 11. SAVE PLAN
# ============================================================
plan = {

    "dataset":
        "IBM HI-Small",

    "original_legitimate_rows":
        5_073_168,

    "original_laundering_rows":
        5_177,

    "synthetic_laundering_required":
        TARGET_SYNTHETIC,

    "final_balanced_total":
        5_073_168 + 5_177 + TARGET_SYNTHETIC,

    "template_count":
        len(templates),

    "template_transaction_count":
        int(total_original),

    "allocation_method":
        "IBM transaction-share allocation with minimal GCD correction",

    "typology_targets": {

        str(typology): {

            "original_ibm_transactions":
                int(
                    typology_original.loc[
                        typology
                    ]
                ),

            "original_ibm_share":
                float(
                    shares.loc[
                        typology
                    ]
                ),

            "initial_target":
                int(
                    comparison.loc[
                        typology,
                        "initial_target"
                    ]
                ),

            "corrected_target":
                int(
                    targets.loc[
                        typology
                    ]
                ),

            "target_delta":
                int(
                    comparison.loc[
                        typology,
                        "delta"
                    ]
                ),

            "template_sizes":
                size_info[
                    typology
                ]["sizes"],

            "template_gcd":
                int(
                    size_info[
                        typology
                    ]["gcd"]
                ),

            "estimated_episode_count":
                int(
                    episode_estimates[
                        typology
                    ]
                )
        }

        for typology in targets.index
    }
}

with open(
    OUT_PLAN,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        plan,
        f,
        indent=2
    )

# ============================================================
# 12. FINAL
# ============================================================
print("\n" + "=" * 100)
print("✅ CELL 13 COMPLETE")
print("=" * 100)

print(
    "Synthetic laundering target:",
    f"{TARGET_SYNTHETIC:,}"
)

print(
    "Corrected target sum:",
    f"{targets.sum():,}"
)

print(
    "GCD-feasible:",
    all_feasible
)

print(
    "Number of target adjustments:",
    int(
        (
            comparison["delta"]
            != 0
        ).sum()
    )
)

print(
    "Total transaction adjustment:",
    int(
        comparison["delta"].abs().sum()
    )
)

print(
    "\nSaved:"
)

print(
    OUT_PLAN
)

print(
    "\nNO SYNTHETIC DATA CREATED."
)

print(
    "NO ORIGINAL FILE MODIFIED."
)

print("=" * 100)

CELL 13 — PARITY-CORRECTED EXACT GENERATION PLAN

Initial targets:
typology
BIPARTITE          415357
CYCLE              453261
FAN-IN             502219
FAN-OUT            540123
GATHER-SCATTER    1130783
RANDOM             301647
SCATTER-GATHER     988645
STACK              735956

SCATTER-GATHER:
  Template sizes: [2, 4, 6, 12, 14, 16, 18, 22, 24, 26, 28, 30, 32]
  GCD: 2
  Target: 988645
  Target parity: 1

STACK:
  Template sizes: [2, 4, 6, 8, 10, 12, 14, 16, 20, 22, 24, 26, 28, 30]
  GCD: 2
  Target: 735956
  Target parity: 0

1. INFEASIBLE TARGETS
SCATTER-GATHER       target=988,645 gcd=2

Eligible source typologies:
  GATHER-SCATTER       target=1,130,783
  FAN-OUT              target=540,123
  FAN-IN               target=502,219
  CYCLE                target=453,261
  BIPARTITE            target=415,357
  RANDOM               target=301,647

2. CORRECTED TARGETS
                initial_target  corrected_target  delta  original_share  corrected_share
typology                   

In [20]:
# ============================================================
# CELL 14 — EXACT EPISODE-LEVEL TEMPLATE ALLOCATION
# ============================================================
#
# Purpose:
#   Convert exact typology transaction targets into an exact
#   list of synthetic laundering episodes.
#
# Design:
#   - Use ORIGINAL IBM template-size distribution
#   - Preserve typology-specific episode morphology
#   - Sample actual IBM templates with replacement
#   - Exact transaction total per typology
#   - Exact global synthetic transaction total
#
# Creates:
#   HI-Small_synthetic_episode_plan.csv
#   HI-Small_synthetic_episode_plan.json
#
# NO synthetic transactions are generated here.
# NO original files modified.
# ============================================================

import os
import json
import math
import random
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

TEMPLATE_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_templates.json"
)

PLAN_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_synthetic_generation_plan_v2.json"
)

OUT_CSV = os.path.join(
    WORK_DIR,
    "HI-Small_synthetic_episode_plan.csv"
)

OUT_JSON = os.path.join(
    WORK_DIR,
    "HI-Small_synthetic_episode_plan.json"
)

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------
SEED = 20260919

random.seed(SEED)
np.random.seed(SEED)

print("=" * 100)
print("CELL 14 — EXACT EPISODE-LEVEL TEMPLATE ALLOCATION")
print("=" * 100)

# ============================================================
# 1. LOAD IBM TEMPLATES
# ============================================================
with open(
    TEMPLATE_PATH,
    "r",
    encoding="utf-8"
) as f:
    template_manifest = json.load(f)

templates = template_manifest[
    "templates"
]

template_df = pd.DataFrame(
    [
        {
            "template_id":
                str(t["template_id"]),

            "episode_index":
                int(t["episode_index"]),

            "typology":
                str(t["typology"]),

            "transaction_count":
                int(t["transaction_count"])
        }
        for t in templates
    ]
)

print(
    "IBM templates loaded:",
    len(template_df)
)

# ============================================================
# 2. LOAD EXACT TYPOLOGY TARGETS
# ============================================================
with open(
    PLAN_PATH,
    "r",
    encoding="utf-8"
) as f:
    generation_plan = json.load(f)

typology_targets = {
    str(k): int(
        v["corrected_target"]
    )
    for k, v in generation_plan[
        "typology_targets"
    ].items()
}

TARGET_TOTAL = int(
    generation_plan[
        "synthetic_laundering_required"
    ]
)

print(
    "\nSynthetic target:",
    f"{TARGET_TOTAL:,}"
)

# ============================================================
# 3. SMALL EXACT INTEGER COMPOSITION SOLVER
# ============================================================
def exact_size_composition(
    target,
    sizes,
    approximate_counts
):
    """
    Find non-negative counts for transaction sizes such that:

        sum(count[size] * size) == target

    Starting point:
        approximate_counts

    Since approximate_counts are floors of expected counts,
    the residual is generally small.

    Dynamic programming is performed only on the residual.
    """

    sizes = sorted(
        set(
            int(s)
            for s in sizes
            if int(s) > 0
        )
    )

    current_total = sum(
        int(approximate_counts.get(s, 0))
        * int(s)
        for s in sizes
    )

    residual = int(
        target - current_total
    )

    if residual < 0:
        raise RuntimeError(
            "Approximate composition exceeded target."
        )

    if residual == 0:
        return {
            int(s): int(
                approximate_counts.get(
                    s,
                    0
                )
            )
            for s in sizes
        }

    # --------------------------------------------------------
    # Unbounded DP over residual
    # --------------------------------------------------------
    reachable = np.full(
        residual + 1,
        -1,
        dtype=np.int32
    )

    reachable[0] = 0

    parent_size = np.full(
        residual + 1,
        -1,
        dtype=np.int32
    )

    for amount in range(
        1,
        residual + 1
    ):

        for size in sizes:

            if size > amount:
                break

            prev = (
                amount - size
            )

            if reachable[prev] != -1:

                reachable[amount] = (
                    reachable[prev] + 1
                )

                parent_size[
                    amount
                ] = size

                break

    if reachable[residual] == -1:

        # This should only happen if the residual violates
        # the GCD constraint.
        gcd_value = 0

        for s in sizes:
            gcd_value = math.gcd(
                gcd_value,
                int(s)
            )

        raise RuntimeError(
            f"Residual {residual} not reachable. "
            f"Sizes={sizes}, GCD={gcd_value}"
        )

    # --------------------------------------------------------
    # Recover additions
    # --------------------------------------------------------
    additions = {
        int(s): 0
        for s in sizes
    }

    remaining = residual

    while remaining > 0:

        size = int(
            parent_size[remaining]
        )

        if size <= 0:
            raise RuntimeError(
                "Invalid DP parent encountered."
            )

        additions[size] += 1

        remaining -= size

    # --------------------------------------------------------
    # Final composition
    # --------------------------------------------------------
    final_counts = {}

    for size in sizes:

        final_counts[size] = (
            int(
                approximate_counts.get(
                    size,
                    0
                )
            )
            +
            int(
                additions[size]
            )
        )

    # Final validation
    final_total = sum(
        size * count
        for size, count
        in final_counts.items()
    )

    if final_total != target:

        raise RuntimeError(
            f"Composition failed: "
            f"expected={target}, "
            f"actual={final_total}"
        )

    return final_counts

# ============================================================
# 4. CREATE EPISODE PLAN
# ============================================================
episode_plan_rows = []

summary_rows = []

global_episode_counter = 0

for typology in sorted(
    typology_targets.keys()
):

    target_transactions = int(
        typology_targets[
            typology
        ]
    )

    typ_templates = (
        template_df[
            template_df["typology"]
            == typology
        ]
        .copy()
    )

    if typ_templates.empty:
        raise RuntimeError(
            f"No templates found for typology "
            f"{typology}"
        )

    # --------------------------------------------------------
    # Original episode-size distribution
    # --------------------------------------------------------
    size_episode_counts = (
        typ_templates[
            "transaction_count"
        ]
        .value_counts()
        .sort_index()
    )

    total_original_episodes = (
        size_episode_counts.sum()
    )

    size_probs = (
        size_episode_counts
        /
        total_original_episodes
    )

    # --------------------------------------------------------
    # Expected episode count
    # --------------------------------------------------------
    mean_transactions = (
        typ_templates[
            "transaction_count"
        ].mean()
    )

    expected_episode_count = (
        target_transactions
        /
        mean_transactions
    )

    # --------------------------------------------------------
    # Approximate number of episodes
    # --------------------------------------------------------
    episode_count_base = int(
        math.floor(
            expected_episode_count
        )
    )

    # Expected counts per template size
    approximate_counts = {}

    for size, probability in (
        size_probs.items()
    ):

        approximate_counts[
            int(size)
        ] = int(
            math.floor(
                episode_count_base
                *
                float(probability)
            )
        )

    # --------------------------------------------------------
    # If all rounded-down counts somehow produce too few
    # episodes, that is fine; the DP additions will fill the
    # transaction remainder.
    # --------------------------------------------------------
    sizes = sorted(
        approximate_counts.keys()
    )

    final_size_counts = exact_size_composition(
        target_transactions,
        sizes,
        approximate_counts
    )

    # --------------------------------------------------------
    # Validate exact transaction target
    # --------------------------------------------------------
    calculated_transactions = sum(
        size * count
        for size, count
        in final_size_counts.items()
    )

    if (
        calculated_transactions
        != target_transactions
    ):
        raise RuntimeError(
            f"{typology}: exact target failure."
        )

    # --------------------------------------------------------
    # Build template lookup by size
    # --------------------------------------------------------
    templates_by_size = {}

    for size in sizes:

        candidates = (
            typ_templates[
                typ_templates[
                    "transaction_count"
                ] == size
            ]["template_id"]
            .tolist()
        )

        if not candidates:
            raise RuntimeError(
                f"No templates for size {size} "
                f"in {typology}."
            )

        templates_by_size[
            int(size)
        ] = candidates

    # --------------------------------------------------------
    # Create actual episode rows
    # --------------------------------------------------------
    typology_episode_counter = 0
    typology_transaction_counter = 0

    # Shuffle the size groups so the generated episode order
    # is not grouped unnaturally by template size.
    size_jobs = []

    for size, count in (
        final_size_counts.items()
    ):

        for _ in range(
            int(count)
        ):

            size_jobs.append(
                int(size)
            )

    random.shuffle(
        size_jobs
    )

    # --------------------------------------------------------
    # For every required size, choose an IBM template with
    # uniform probability among original episodes of that size.
    # --------------------------------------------------------
    for size in size_jobs:

        template_id = random.choice(
            templates_by_size[
                int(size)
            ]
        )

        synthetic_episode_id = (
            f"SYN_{typology}_{global_episode_counter:07d}"
        )

        episode_row = {

            "synthetic_episode_id":
                synthetic_episode_id,

            "global_episode_index":
                int(global_episode_counter),

            "typology":
                typology,

            "source_template_id":
                str(template_id),

            "transaction_count":
                int(size)
        }

        episode_plan_rows.append(
            episode_row
        )

        global_episode_counter += 1

        typology_episode_counter += 1

        typology_transaction_counter += (
            int(size)
        )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------
    generated_sizes = (
        pd.Series(
            [
                r["transaction_count"]
                for r in episode_plan_rows[
                    -typology_episode_counter:
                ]
            ]
        )
    )

    summary_rows.append(
        {
            "typology":
                typology,

            "target_transactions":
                target_transactions,

            "generated_transactions":
                typology_transaction_counter,

            "episode_count":
                typology_episode_counter,

            "mean_transactions_per_episode":
                float(
                    generated_sizes.mean()
                ),

            "median_transactions_per_episode":
                float(
                    generated_sizes.median()
                ),

            "min_transactions_per_episode":
                int(
                    generated_sizes.min()
                ),

            "max_transactions_per_episode":
                int(
                    generated_sizes.max()
                ),

            "original_mean_transactions":
                float(mean_transactions),

            "original_episode_count":
                int(
                    total_original_episodes
                )
        }
    )

# ============================================================
# 5. DATAFRAME
# ============================================================
episode_plan = pd.DataFrame(
    episode_plan_rows
)

summary_df = pd.DataFrame(
    summary_rows
)

# ============================================================
# 6. GLOBAL VALIDATION
# ============================================================
print("\n" + "=" * 80)
print("1. TYPOLOGY GENERATION SUMMARY")
print("=" * 80)

print(
    summary_df.to_string(
        index=False
    )
)

global_generated_transactions = int(
    episode_plan[
        "transaction_count"
    ].sum()
)

print("\n" + "=" * 80)
print("2. GLOBAL VALIDATION")
print("=" * 80)

print(
    "Generated episodes:",
    f"{len(episode_plan):,}"
)

print(
    "Generated transactions:",
    f"{global_generated_transactions:,}"
)

print(
    "Required transactions:",
    f"{TARGET_TOTAL:,}"
)

print(
    "Exact transaction match:",
    global_generated_transactions
    ==
    TARGET_TOTAL
)

# ============================================================
# 7. TYPOLOGY EXACTNESS
# ============================================================
typology_check = (
    episode_plan
    .groupby("typology")
    .agg(
        generated_episodes=(
            "synthetic_episode_id",
            "count"
        ),
        generated_transactions=(
            "transaction_count",
            "sum"
        )
    )
    .reset_index()
)

target_df = pd.DataFrame(
    [
        {
            "typology": typology,
            "target_transactions":
                target_transactions
        }
        for typology, target_transactions
        in typology_targets.items()
    ]
)

typology_check = (
    target_df
    .merge(
        typology_check,
        on="typology",
        how="left"
    )
)

typology_check[
    "exact_match"
] = (
    typology_check[
        "target_transactions"
    ]
    ==
    typology_check[
        "generated_transactions"
    ]
)

print(
    "\nTypology exactness:"
)

print(
    typology_check.to_string(
        index=False
    )
)

if not bool(
    typology_check[
        "exact_match"
    ].all()
):

    raise RuntimeError(
        "At least one typology does not "
        "match its exact target."
    )

# ============================================================
# 8. SOURCE TEMPLATE COVERAGE
# ============================================================
print("\n" + "=" * 80)
print("3. SOURCE TEMPLATE COVERAGE")
print("=" * 80)

template_usage = (
    episode_plan[
        "source_template_id"
    ]
    .value_counts()
)

print(
    "IBM templates used:",
    len(template_usage),
    "/",
    len(template_df)
)

print(
    "Template coverage:",
    round(
        len(template_usage)
        /
        len(template_df)
        *
        100,
        4
    ),
    "%"
)

# ============================================================
# 9. TEMPLATE USAGE BY TYPOLOGY
# ============================================================
usage_by_type = []

for typology, g in (
    episode_plan
    .groupby("typology")
):

    source_templates = (
        g[
            "source_template_id"
        ]
        .nunique()
    )

    total_templates = (
        template_df[
            template_df["typology"]
            == typology
        ]
        ["template_id"]
        .nunique()
    )

    usage_by_type.append(
        {
            "typology":
                typology,

            "templates_used":
                int(source_templates),

            "templates_available":
                int(total_templates),

            "coverage_pct":
                float(
                    source_templates
                    /
                    total_templates
                    * 100
                )
        }
    )

usage_by_type_df = pd.DataFrame(
    usage_by_type
)

print(
    usage_by_type_df.to_string(
        index=False
    )
)

# ============================================================
# 10. SAVE CSV
# ============================================================
episode_plan.to_csv(
    OUT_CSV,
    index=False
)

# ============================================================
# 11. SAVE JSON
# ============================================================
plan_manifest = {

    "dataset":
        "IBM HI-Small",

    "seed":
        SEED,

    "synthetic_transaction_target":
        TARGET_TOTAL,

    "generated_transaction_count":
        global_generated_transactions,

    "generated_episode_count":
        int(len(episode_plan)),

    "typology_targets":
        typology_targets,

    "typology_summary":
        summary_df.to_dict(
            orient="records"
        ),

    "template_usage":
        {
            str(k): int(v)
            for k, v
            in template_usage.items()
        },

    "episodes":
        episode_plan.to_dict(
            orient="records"
        )
}

with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        plan_manifest,
        f,
        indent=2
    )

# ============================================================
# 12. FINAL
# ============================================================
print("\n" + "=" * 100)
print("✅ CELL 14 COMPLETE")
print("=" * 100)

print(
    "Episode plan:",
    OUT_CSV
)

print(
    "Episode plan manifest:",
    OUT_JSON
)

print(
    "\nSynthetic episodes planned:",
    f"{len(episode_plan):,}"
)

print(
    "Synthetic transactions planned:",
    f"{global_generated_transactions:,}"
)

print(
    "Exact target achieved:",
    global_generated_transactions
    ==
    TARGET_TOTAL
)

print(
    "\nNO SYNTHETIC TRANSACTIONS CREATED."
)

print(
    "NO SYNTHETIC ACCOUNTS CREATED."
)

print(
    "NO ORIGINAL FILE MODIFIED."
)

print("=" * 100)

CELL 14 — EXACT EPISODE-LEVEL TEMPLATE ALLOCATION
IBM templates loaded: 370

Synthetic target: 5,067,991

1. TYPOLOGY GENERATION SUMMARY
      typology  target_transactions  generated_transactions  episode_count  mean_transactions_per_episode  median_transactions_per_episode  min_transactions_per_episode  max_transactions_per_episode  original_mean_transactions  original_episode_count
     BIPARTITE               415357                  415357          77451                       5.362836                              4.0                             1                            15                    5.367347                      49
         CYCLE               453261                  453261          85293                       5.314164                              4.0                             2                            12                    5.314815                      54
        FAN-IN               502219                  502219          63218                       7.944241     

In [21]:
# ============================================================
# CELL 15 — 5,000-EPISODE SYNTHETIC GENERATOR PILOT
# ============================================================
#
# PURPOSE
# -------
# Validate the complete synthetic-generation mechanism before
# scaling to the full 5,067,991 laundering transactions.
#
# PILOT:
#   5,000 laundering episodes
#
# Uses:
#   - Original IBM pattern templates
#   - Original HI-Small legitimate transaction DNA
#   - Real bank vocabulary
#   - Real clean account ecosystem
#   - Fresh structurally realistic accounts for a small
#     fraction of nodes
#
# Creates:
#   1. Pilot synthetic transactions CSV
#   2. Pilot new accounts CSV
#   3. Pilot Patterns.txt
#   4. Pilot manifest JSON
#
# IMPORTANT
# ---------
# Original HI-Small files are NEVER modified.
# ============================================================

import os
import re
import json
import random
import string
from collections import defaultdict

import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

SEED = 20260919

random.seed(SEED)
np.random.seed(SEED)

PILOT_EPISODES = 5000

# Fraction of template nodes that will receive a brand-new
# synthetic account.
#
# Most nodes use the original clean account ecosystem so that
# "new account" itself does not become the dominant synthetic
# signal.
NEW_ACCOUNT_RATE = 0.05

DATA_ROOT = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
)

WORK_DIR = (
    "/kaggle/working/"
    "AML_GNN_ONLY/data"
)

TRANS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_Trans.csv"
)

ACCOUNTS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_accounts.csv"
)

TEMPLATE_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_templates.json"
)

EPISODE_PLAN_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_synthetic_episode_plan.csv"
)

BANK_POOL_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_final_generator_bank_pool.csv"
)

ACCOUNT_REGISTRY_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_generator_account_registry.csv"
)

ACCOUNT_PREFIX_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_bank_account_prefix_sampler.json"
)

ACCOUNT_GENERATOR_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_account_id_generator.json"
)

ENTITY_DNA_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_entity_name_dna.json"
)

# ------------------------------------------------------------
# PILOT OUTPUTS
# ------------------------------------------------------------

PILOT_TRANS_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_transactions.csv"
)

PILOT_ACCOUNTS_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_new_accounts.csv"
)

PILOT_PATTERNS_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_Patterns.txt"
)

PILOT_MANIFEST_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_manifest.json"
)

print("=" * 100)
print("CELL 15 — 5,000-EPISODE SYNTHETIC GENERATOR PILOT")
print("=" * 100)

# ============================================================
# 1. LOAD IBM TEMPLATES
# ============================================================

print("\n" + "=" * 80)
print("1. LOADING IBM TEMPLATES")
print("=" * 80)

with open(
    TEMPLATE_PATH,
    "r",
    encoding="utf-8"
) as f:

    template_manifest = json.load(f)

templates = template_manifest[
    "templates"
]

template_lookup = {
    str(t["template_id"]): t
    for t in templates
}

print(
    "Templates:",
    len(templates)
)

# ============================================================
# 2. LOAD COMPLETE EPISODE PLAN
# ============================================================

episode_plan = pd.read_csv(
    EPISODE_PLAN_PATH,
    dtype=str
)

episode_plan[
    "transaction_count"
] = pd.to_numeric(
    episode_plan[
        "transaction_count"
    ],
    errors="raise"
)

print(
    "Full planned episodes:",
    f"{len(episode_plan):,}"
)

# ============================================================
# 3. STRATIFIED PILOT EPISODE SAMPLE
# ============================================================
#
# Preserve approximate typology composition in the pilot.
# ============================================================

typology_counts = (
    episode_plan["typology"]
    .value_counts()
    .sort_index()
)

target_pilot_counts = (
    typology_counts
    /
    typology_counts.sum()
    *
    PILOT_EPISODES
)

pilot_counts = np.floor(
    target_pilot_counts
).astype(int)

remainder = (
    PILOT_EPISODES
    -
    int(pilot_counts.sum())
)

fractions = (
    target_pilot_counts
    -
    pilot_counts
)

for typology in (
    fractions
    .sort_values(
        ascending=False
    )
    .head(remainder)
    .index
):

    pilot_counts.loc[
        typology
    ] += 1

pilot_parts = []

for typology, count in (
    pilot_counts.items()
):

    available = episode_plan[
        episode_plan["typology"]
        ==
        typology
    ]

    sampled = available.sample(
        n=int(count),
        random_state=SEED
        +
        sum(
            ord(c)
            for c in typology
        )
    )

    pilot_parts.append(
        sampled
    )

pilot_plan = (
    pd.concat(
        pilot_parts,
        ignore_index=True
    )
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)

# Re-number pilot episode IDs
pilot_plan[
    "pilot_episode_index"
] = np.arange(
    len(pilot_plan)
)

pilot_plan[
    "synthetic_episode_id"
] = [
    f"SYN_{i:07d}"
    for i in range(
        len(pilot_plan)
    )
]

print(
    "Pilot episodes:",
    len(pilot_plan)
)

print(
    "\nPilot typology distribution:"
)

print(
    pilot_plan[
        "typology"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 4. LOAD BANK POOL
# ============================================================

print("\n" + "=" * 80)
print("2. LOADING BANK POOL")
print("=" * 80)

bank_pool = pd.read_csv(
    BANK_POOL_PATH,
    dtype=str
)

numeric_bank_cols = [
    "legitimate_source_transactions",
    "legitimate_destination_transactions",
    "account_count",
    "source_probability",
    "destination_probability"
]

for col in numeric_bank_cols:

    bank_pool[col] = pd.to_numeric(
        bank_pool[col],
        errors="coerce"
    ).fillna(0)

print(
    "Banks:",
    len(bank_pool)
)

# ------------------------------------------------------------
# Bank dictionaries
# ------------------------------------------------------------

bank_ids = (
    bank_pool[
        "normalized_bank_id"
    ]
    .astype(str)
    .tolist()
)

bank_tx_id_map = {
    str(row["normalized_bank_id"]):
        str(row["canonical_transaction_bank_id"])
    for _, row in bank_pool.iterrows()
}

bank_account_id_map = {}

# Original accounts.csv representation
accounts_source_for_bank = pd.read_csv(
    ACCOUNTS_PATH,
    usecols=[
        "Bank ID",
        "Account Number"
    ],
    dtype=str
)

def normalize_bank_id(x):

    x = str(x).strip()

    if x == "":
        return None

    try:
        return str(
            int(x)
        )
    except Exception:
        return x.lstrip("0") or "0"

accounts_source_for_bank[
    "normalized_bank_id"
] = (
    accounts_source_for_bank[
        "Bank ID"
    ].map(normalize_bank_id)
)

for bank_id, g in (
    accounts_source_for_bank
    .groupby("normalized_bank_id")
):

    bank_account_id_map[
        str(bank_id)
    ] = str(
        g["Bank ID"].iloc[0]
    )

# ------------------------------------------------------------
# Weighted bank sampling
# ------------------------------------------------------------

source_bank_weights = (
    bank_pool[
        "source_probability"
    ]
    .to_numpy(
        dtype=float
    )
)

destination_bank_weights = (
    bank_pool[
        "destination_probability"
    ]
    .to_numpy(
        dtype=float
    )
)

# ============================================================
# 5. LOAD CLEAN ACCOUNT REGISTRY
# ============================================================

print("\n" + "=" * 80)
print("3. LOADING CLEAN ACCOUNT REGISTRY")
print("=" * 80)

registry = pd.read_csv(
    ACCOUNT_REGISTRY_PATH,
    dtype=str
)

registry[
    "normalized_bank_id"
] = registry[
    "normalized_bank_id"
].astype(str)

for col in [
    "legitimate_source_tx_count",
    "legitimate_destination_tx_count",
    "legitimate_total_activity"
]:

    registry[col] = pd.to_numeric(
        registry[col],
        errors="coerce"
    ).fillna(0).astype(int)

registry[
    "generator_eligible"
] = (
    registry[
        "generator_eligible"
    ].astype(str).str.lower()
    .eq("true")
)

eligible = registry[
    registry[
        "generator_eligible"
    ]
].copy()

print(
    "Eligible accounts:",
    f"{len(eligible):,}"
)

# ------------------------------------------------------------
# Create role-compatible pools
# ------------------------------------------------------------

def compatible_role(row_role, needed_role):

    if needed_role == "source":
        return row_role in {
            "source_only",
            "both"
        }

    if needed_role == "destination":
        return row_role in {
            "destination_only",
            "both"
        }

    if needed_role == "both":
        return row_role == "both"

    return True


account_pool = defaultdict(list)

for _, row in eligible.iterrows():

    role = str(
        row["role"]
    )

    bank = str(
        row["normalized_bank_id"]
    )

    if compatible_role(
        role,
        "source"
    ):

        account_pool[
            (bank, "source")
        ].append(
            str(row["Account Number"])
        )

    if compatible_role(
        role,
        "destination"
    ):

        account_pool[
            (bank, "destination")
        ].append(
            str(row["Account Number"])
        )

    if compatible_role(
        role,
        "both"
    ):

        account_pool[
            (bank, "both")
        ].append(
            str(row["Account Number"])
        )

# ------------------------------------------------------------
# Shuffle each account pool
# ------------------------------------------------------------

for key in account_pool:

    random.shuffle(
        account_pool[key]
    )

account_pool_pointer = defaultdict(int)

# Existing accounts already present in original dataset
existing_account_keys = set(
    (
        str(row["normalized_bank_id"]),
        str(row["Account Number"])
    )
    for _, row in registry.iterrows()
)

# ============================================================
# 6. ACCOUNT / ENTITY GENERATION DNA
# ============================================================

print("\n" + "=" * 80)
print("4. LOADING ACCOUNT / ENTITY GENERATION DNA")
print("=" * 80)

with open(
    ACCOUNT_PREFIX_PATH,
    "r",
    encoding="utf-8"
) as f:

    bank_prefix_sampler = json.load(f)

with open(
    ACCOUNT_GENERATOR_PATH,
    "r",
    encoding="utf-8"
) as f:

    account_generator_dna = json.load(f)

with open(
    ENTITY_DNA_PATH,
    "r",
    encoding="utf-8"
) as f:

    entity_dna = json.load(f)

position_models = (
    account_generator_dna[
        "position_models"
    ]
)

entity_position_models = (
    entity_dna[
        "entity_id_position_models"
    ]
)

name_prefixes = (
    entity_dna[
        "top_name_prefixes"
    ]
)

# ------------------------------------------------------------
# Convert JSON probabilities into arrays
# ------------------------------------------------------------

account_position_arrays = {}

for pos, values in (
    position_models.items()
):

    chars = [
        item["char"]
        for item in values
    ]

    probs = np.array(
        [
            item["probability"]
            for item in values
        ],
        dtype=float
    )

    probs = (
        probs
        /
        probs.sum()
    )

    account_position_arrays[
        int(pos)
    ] = (
        chars,
        probs
    )

# Entity ID arrays
entity_position_arrays = {}

for pos, values in (
    entity_position_models.items()
):

    chars = list(
        values.keys()
    )

    probs = np.array(
        [
            values[c]["probability"]
            for c in chars
        ],
        dtype=float
    )

    probs = (
        probs
        /
        probs.sum()
    )

    entity_position_arrays[
        int(pos)
    ] = (
        chars,
        probs
    )

# Name prefix probabilities
name_prefix_keys = list(
    name_prefixes.keys()
)

name_prefix_probs = np.array(
    [
        name_prefixes[k]
        for k in name_prefix_keys
    ],
    dtype=float
)

name_prefix_probs /= (
    name_prefix_probs.sum()
)

# ------------------------------------------------------------
# Fresh-account generators
# ------------------------------------------------------------

used_new_accounts = set()

used_new_entity_ids = set()

new_accounts_rows = []

def generate_entity_id():

    for _ in range(10000):

        chars = []

        for pos in range(
            1,
            10
        ):

            char_list, probs = (
                entity_position_arrays[
                    pos
                ]
            )

            chars.append(
                np.random.choice(
                    char_list,
                    p=probs
                )
            )

        value = "".join(chars)

        if value not in (
            used_new_entity_ids
        ):

            used_new_entity_ids.add(
                value
            )

            return value

    raise RuntimeError(
        "Unable to generate unique Entity ID."
    )


def generate_entity_name():

    prefix = np.random.choice(
        name_prefix_keys,
        p=name_prefix_probs
    )

    number = np.random.randint(
        1,
        100000
    )

    return (
        f"{prefix} #{number}"
    )


def generate_new_account(
    normalized_bank_id
):

    normalized_bank_id = str(
        normalized_bank_id
    )

    # Get bank-specific 3-char prefixes.
    prefix_info = (
        bank_prefix_sampler
        .get(
            normalized_bank_id
        )
    )

    for _ in range(10000):

        # ----------------------------------------------------
        # Prefix
        # ----------------------------------------------------
        if (
            prefix_info
            and
            prefix_info["prefixes"]
        ):

            prefixes = [
                x["prefix"]
                for x in
                prefix_info["prefixes"]
            ]

            probs = np.array(
                [
                    x["probability"]
                    for x in
                    prefix_info["prefixes"]
                ],
                dtype=float
            )

            probs /= probs.sum()

            prefix = np.random.choice(
                prefixes,
                p=probs
            )

        else:

            # Fallback: sample first 3 positions
            prefix_chars = []

            for pos in [1, 2, 3]:

                chars, probs = (
                    account_position_arrays[
                        pos
                    ]
                )

                prefix_chars.append(
                    np.random.choice(
                        chars,
                        p=probs
                    )
                )

            prefix = "".join(
                prefix_chars
            )

        # ----------------------------------------------------
        # Remaining positions 4-9
        # ----------------------------------------------------
        chars = list(
            prefix
        )

        for pos in range(
            4,
            10
        ):

            char_list, probs = (
                account_position_arrays[
                    pos
                ]
            )

            chars.append(
                np.random.choice(
                    char_list,
                    p=probs
                )
            )

        account_number = "".join(
            chars
        )

        key = (
            normalized_bank_id,
            account_number
        )

        if (
            key not in
            existing_account_keys
            and
            key not in
            used_new_accounts
        ):

            used_new_accounts.add(
                key
            )

            entity_id = (
                generate_entity_id()
            )

            entity_name = (
                generate_entity_name()
            )

            bank_account_id = (
                bank_account_id_map.get(
                    normalized_bank_id,
                    normalized_bank_id
                )
            )

            new_accounts_rows.append(
                {
                    "Bank Name":
                        str(
                            bank_pool.loc[
                                bank_pool[
                                    "normalized_bank_id"
                                ].astype(str)
                                ==
                                normalized_bank_id,
                                "Bank Name"
                            ].iloc[0]
                        ),

                    "Bank ID":
                        bank_account_id,

                    "Account Number":
                        account_number,

                    "Entity ID":
                        entity_id,

                    "Entity Name":
                        entity_name
                }
            )

            return account_number

    raise RuntimeError(
        "Unable to generate unique account."
    )

# ============================================================
# 7. LOAD LEGITIMATE BEHAVIOR SAMPLE
# ============================================================

print("\n" + "=" * 80)
print("5. BUILDING LEGITIMATE BEHAVIOR SAMPLE")
print("=" * 80)

sample_rows = []

chunk_size = 500_000
chunk_no = 0

for chunk in pd.read_csv(
    TRANS_PATH,
    usecols=[
        "Timestamp",
        "Amount Received",
        "Amount Paid",
        "Payment Currency",
        "Receiving Currency",
        "Payment Format",
        "Is Laundering"
    ],
    chunksize=chunk_size,
    dtype=str
):

    chunk_no += 1

    chunk[
        "Is Laundering"
    ] = pd.to_numeric(
        chunk[
            "Is Laundering"
        ],
        errors="coerce"
    ).fillna(0).astype(int)

    legit_chunk = chunk[
        chunk[
            "Is Laundering"
        ] == 0
    ].copy()

    if len(legit_chunk) == 0:
        continue

    # Take up to 10k random legitimate rows/chunk.
    n = min(
        10000,
        len(legit_chunk)
    )

    sampled = (
        legit_chunk
        .sample(
            n=n,
            random_state=SEED + chunk_no
        )
    )

    sample_rows.append(
        sampled
    )

    print(
        f"  Chunk {chunk_no} "
        f"| sampled legitimate rows = "
        f"{sum(len(x) for x in sample_rows):,}"
    )

behavior_sample = (
    pd.concat(
        sample_rows,
        ignore_index=True
    )
)

behavior_sample[
    "Timestamp"
] = pd.to_datetime(
    behavior_sample[
        "Timestamp"
    ],
    errors="coerce"
)

behavior_sample[
    "Amount Received"
] = pd.to_numeric(
    behavior_sample[
        "Amount Received"
    ],
    errors="coerce"
)

behavior_sample[
    "Amount Paid"
] = pd.to_numeric(
    behavior_sample[
        "Amount Paid"
    ],
    errors="coerce"
)

behavior_sample = (
    behavior_sample
    .dropna(
        subset=[
            "Timestamp",
            "Amount Received",
            "Amount Paid"
        ]
    )
    .copy()
)

print(
    "\nLegitimate behavior sample:",
    f"{len(behavior_sample):,}"
)

# ------------------------------------------------------------
# Global empirical samplers
# ------------------------------------------------------------

time_values = (
    behavior_sample[
        "Timestamp"
    ]
    .tolist()
)

received_amount_values = (
    behavior_sample[
        "Amount Received"
    ]
    .clip(
        lower=1e-6
    )
    .to_numpy(
        dtype=float
    )
)

payment_format_values = (
    behavior_sample[
        "Payment Format"
    ]
    .astype(str)
    .tolist()
)

payment_currency_values = (
    behavior_sample[
        "Payment Currency"
    ]
    .astype(str)
    .tolist()
)

# payment currency -> receiving currency
recv_by_pay = defaultdict(list)

for pay, recv in zip(
    behavior_sample[
        "Payment Currency"
    ].astype(str),
    behavior_sample[
        "Receiving Currency"
    ].astype(str)
):

    recv_by_pay[pay].append(
        recv
    )

# Dataset min/max
dataset_min_ts = (
    behavior_sample[
        "Timestamp"
    ].min()
)

dataset_max_ts = (
    behavior_sample[
        "Timestamp"
    ].max()
)

dataset_max_received = float(
    received_amount_values.max()
)

# ============================================================
# 8. PREPARE NODE ROLE FUNCTION
# ============================================================

def get_template_node_roles(
    template
):

    in_degree = defaultdict(int)
    out_degree = defaultdict(int)

    for edge in template["edges"]:

        s = int(
            edge["source_node"]
        )

        d = int(
            edge["destination_node"]
        )

        out_degree[s] += 1
        in_degree[d] += 1

    roles = {}

    for node in template["nodes"]:

        node_id = int(
            node["node_id"]
        )

        indeg = in_degree[
            node_id
        ]

        outdeg = out_degree[
            node_id
        ]

        if (
            indeg > 0
            and
            outdeg > 0
        ):

            role = "both"

        elif outdeg > 0:

            role = "source"

        elif indeg > 0:

            role = "destination"

        else:

            role = "both"

        roles[node_id] = role

    return roles

# ============================================================
# 9. BANK MAPPING
# ============================================================

def sample_distinct_bank(
    used_banks,
    role
):

    if role == "source":
        probs = source_bank_weights

    elif role == "destination":
        probs = destination_bank_weights

    else:
        probs = (
            source_bank_weights
            +
            destination_bank_weights
        )

        probs = (
            probs
            /
            probs.sum()
        )

    # Try weighted candidates.
    for _ in range(200):

        idx = np.random.choice(
            len(bank_pool),
            p=probs
        )

        bank = str(
            bank_pool.iloc[
                idx
            ][
                "normalized_bank_id"
            ]
        )

        if bank not in used_banks:
            return bank

    # Fallback
    candidates = [
        x for x in bank_ids
        if x not in used_banks
    ]

    if not candidates:

        raise RuntimeError(
            "Ran out of distinct banks."
        )

    return random.choice(
        candidates
    )

def map_template_banks(
    template
):

    # Determine the original bank of every node.
    node_original_bank = {}

    for tx in template[
        "transactions"
    ]:

        s = int(
            tx["source_node"]
        )

        d = int(
            tx["destination_node"]
        )

        node_original_bank.setdefault(
            s,
            str(
                template["nodes"][
                    s
                ].get(
                    "original_bank",
                    "0"
                )
            )
        )

        node_original_bank.setdefault(
            d,
            str(
                template["nodes"][
                    d
                ].get(
                    "original_bank",
                    "0"
                )
            )
        )

    # --------------------------------------------------------
    # Templates created by Cell 6 intentionally stripped
    # original node bank values from "nodes". Therefore derive
    # role-level grouping from transaction rows.
    # --------------------------------------------------------
    original_bank_by_node = {}

    for tx in template[
        "_raw_transactions_for_mapping"
    ]:
        original_bank_by_node[
            int(tx["source_node"])
        ] = str(
            tx["from_bank"]
        )

        original_bank_by_node[
            int(tx["destination_node"])
        ] = str(
            tx["to_bank"]
        )

    # --------------------------------------------------------
    # Determine source/destination presence per bank
    # --------------------------------------------------------
    bank_roles = defaultdict(set)

    for tx in template[
        "_raw_transactions_for_mapping"
    ]:

        bank_roles[
            str(tx["from_bank"])
        ].add(
            "source"
        )

        bank_roles[
            str(tx["to_bank"])
        ].add(
            "destination"
        )

    unique_original_banks = list(
        bank_roles.keys()
    )

    # Sort so role-bearing banks are deterministic.
    unique_original_banks.sort()

    mapping = {}
    used_real_banks = set()

    for original_bank in (
        unique_original_banks
    ):

        roles = bank_roles[
            original_bank
        ]

        if (
            "source" in roles
            and
            "destination" in roles
        ):
            role = "both"

        elif "source" in roles:
            role = "source"

        else:
            role = "destination"

        real_bank = sample_distinct_bank(
            used_real_banks,
            role
        )

        mapping[
            original_bank
        ] = real_bank

        used_real_banks.add(
            real_bank
        )

    # Node -> real bank
    node_bank_mapping = {}

    for node_id, original_bank in (
        original_bank_by_node.items()
    ):

        node_bank_mapping[
            int(node_id)
        ] = mapping[
            str(original_bank)
        ]

    return (
        node_bank_mapping,
        mapping
    )

# ============================================================
# 10. ACCOUNT ASSIGNMENT
# ============================================================

def get_account_from_pool(
    bank,
    role,
    used_accounts_this_episode
):

    # --------------------------------------------------------
    # Candidate role order
    # --------------------------------------------------------
    if role == "both":

        role_order = [
            "both"
        ]

    elif role == "source":

        role_order = [
            "source",
            "both"
        ]

    else:

        role_order = [
            "destination",
            "both"
        ]

    for candidate_role in role_order:

        pool = account_pool.get(
            (
                str(bank),
                candidate_role
            ),
            []
        )

        if not pool:
            continue

        start = (
            account_pool_pointer[
                (
                    str(bank),
                    candidate_role
                )
            ]
        )

        for offset in range(
            len(pool)
        ):

            idx = (
                start
                +
                offset
            ) % len(pool)

            account = pool[idx]

            if account not in (
                used_accounts_this_episode
            ):

                account_pool_pointer[
                    (
                        str(bank),
                        candidate_role
                    )
                ] = (
                    idx + 1
                ) % len(pool)

                return account, False

    # No clean original account available.
    # Create a fresh structurally realistic one.
    account = generate_new_account(
        str(bank)
    )

    return account, True

# ============================================================
# 11. TRANSACTION ID GENERATOR
# ============================================================

used_transaction_ids = set()

alphabet = (
    string.ascii_uppercase
    +
    string.digits
)

def generate_transaction_id():

    for _ in range(1000):

        value = "".join(
            random.choices(
                alphabet,
                k=16
            )
        )

        if value not in (
            used_transaction_ids
        ):

            used_transaction_ids.add(
                value
            )

            return value

    raise RuntimeError(
        "Unable to generate unique transaction ID."
    )

# ============================================================
# 12. OUTPUT BUFFERS
# ============================================================

transaction_rows = []

pattern_file = open(
    PILOT_PATTERNS_PATH,
    "w",
    encoding="utf-8"
)

used_edges = set()

episode_audit_rows = []

global_transaction_counter = 0

# ============================================================
# 13. GENERATE PILOT
# ============================================================

print("\n" + "=" * 80)
print("6. GENERATING PILOT")
print("=" * 80)

for pilot_idx, episode_row in (
    pilot_plan.iterrows()
):

    template_id = str(
        episode_row[
            "source_template_id"
        ]
    )

    template = template_lookup[
        template_id
    ]

    typology = str(
        episode_row[
            "typology"
        ]
    )

    synthetic_episode_id = str(
        episode_row[
            "synthetic_episode_id"
        ]
    )

    # --------------------------------------------------------
    # Build raw transaction structure.
    #
    # Since Cell 6 compressed node metadata, the original
    # bank/account information from the template transactions
    # is reconstructed here.
    # --------------------------------------------------------
    raw_template_transactions = []

    for tx in template[
        "transactions"
    ]:

        raw_template_transactions.append(
            {
                "source_node":
                    int(tx["source_node"]),

                "destination_node":
                    int(tx["destination_node"]),

                "from_bank":
                    str(
                        template[
                            "transactions"
                        ][
                            tx["source_order"]
                        ].get(
                            "from_bank",
                            ""
                        )
                    ),

                "to_bank":
                    str(
                        template[
                            "transactions"
                        ][
                            tx["source_order"]
                        ].get(
                            "to_bank",
                            ""
                        )
                    ),

                "source_order":
                    int(
                        tx["source_order"]
                    ),

                "relative_minutes":
                    float(
                        tx["relative_minutes"]
                    ),

                "amount_received":
                    float(
                        tx["amount_received"]
                    ),

                "amount_paid":
                    float(
                        tx["amount_paid"]
                    ),

                "payment_format":
                    str(
                        tx["payment_format"]
                    ),

                "payment_currency":
                    str(
                        tx["payment_currency"]
                    ),

                "receiving_currency":
                    str(
                        tx["receiving_currency"]
                    )
            }
        )

    # --------------------------------------------------------
    # IMPORTANT:
    # Cell 6 JSON does not retain original bank IDs inside
    # each transaction because the reusable structural template
    # intentionally stripped them.
    #
    # Instead we use the source template node IDs and create
    # a fresh bank assignment based on transaction structure.
    #
    # For the pilot, each unique template node receives a real
    # bank according to its source/destination role.
    # --------------------------------------------------------

    # Node degree
    in_degree = defaultdict(int)
    out_degree = defaultdict(int)

    for tx in template[
        "transactions"
    ]:

        s = int(
            tx["source_node"]
        )

        d = int(
            tx["destination_node"]
        )

        out_degree[s] += 1
        in_degree[d] += 1

    node_roles = {}

    for node in template[
        "nodes"
    ]:

        node_id = int(
            node["node_id"]
        )

        if (
            in_degree[node_id] > 0
            and
            out_degree[node_id] > 0
        ):

            node_roles[
                node_id
            ] = "both"

        elif out_degree[node_id] > 0:

            node_roles[
                node_id
            ] = "source"

        else:

            node_roles[
                node_id
            ] = "destination"

    # --------------------------------------------------------
    # Sample one real bank per template node.
    #
    # Distinct node banks are preserved where possible.
    # --------------------------------------------------------
    node_bank_mapping = {}

    used_banks = set()

    for node_id in sorted(
        node_roles.keys()
    ):

        real_bank = (
            sample_distinct_bank(
                used_banks,
                node_roles[node_id]
            )
        )

        node_bank_mapping[
            node_id
        ] = real_bank

        used_banks.add(
            real_bank
        )

    # --------------------------------------------------------
    # Account assignment
    # --------------------------------------------------------
    node_account_mapping = {}

    used_accounts_episode = set()

    fresh_accounts_episode = 0

    for node_id in sorted(
        node_roles.keys()
    ):

        bank = node_bank_mapping[
            node_id
        ]

        role = node_roles[
            node_id
        ]

        # Mostly reuse real eligible accounts.
        use_new = (
            np.random.random()
            <
            NEW_ACCOUNT_RATE
        )

        if use_new:

            account = (
                generate_new_account(
                    bank
                )
            )

            was_new = True

        else:

            account, was_new = (
                get_account_from_pool(
                    bank,
                    role,
                    used_accounts_episode
                )
            )

        node_account_mapping[
            node_id
        ] = account

        used_accounts_episode.add(
            account
        )

        if was_new:
            fresh_accounts_episode += 1

    # --------------------------------------------------------
    # Episode timing anchor
    # --------------------------------------------------------
    template_span = float(
        template[
            "episode_span_minutes"
        ]
    )

    anchor = random.choice(
        time_values
    )

    latest_allowed = (
        dataset_max_ts
        -
        pd.to_timedelta(
            template_span,
            unit="m"
        )
    )

    if anchor > latest_allowed:
        anchor = latest_allowed

    if anchor < dataset_min_ts:
        anchor = dataset_min_ts

    # --------------------------------------------------------
    # Target episode median amount from legitimate DNA
    # --------------------------------------------------------
    target_median_amount = float(
        np.random.choice(
            received_amount_values
        )
    )

    # Keep amount realistic.
    target_median_amount = max(
        1e-6,
        min(
            target_median_amount,
            dataset_max_received
        )
    )

    template_median = float(
        template[
            "median_amount"
        ]
    )

    if template_median <= 0:
        template_median = 1.0

    scale = (
        target_median_amount
        /
        template_median
    )

    # --------------------------------------------------------
    # Build transaction rows
    # --------------------------------------------------------
    episode_pattern_lines = []

    episode_transaction_count = 0

    episode_edge_set = set()

    episode_amounts = []

    for tx in template[
        "transactions"
    ]:

        source_node = int(
            tx["source_node"]
        )

        destination_node = int(
            tx["destination_node"]
        )

        source_bank_norm = (
            node_bank_mapping[
                source_node
            ]
        )

        dest_bank_norm = (
            node_bank_mapping[
                destination_node
            ]
        )

        source_account = (
            node_account_mapping[
                source_node
            ]
        )

        dest_account = (
            node_account_mapping[
                destination_node
            ]
        )

        # ----------------------------------------------------
        # Edge uniqueness
        # ----------------------------------------------------
        edge = (
            source_bank_norm,
            source_account,
            dest_bank_norm,
            dest_account
        )

        episode_edge_set.add(
            edge
        )

        # ----------------------------------------------------
        # Amount
        # ----------------------------------------------------
        amount_received = max(
            1e-6,
            float(
                tx[
                    "amount_received"
                ]
            )
            *
            scale
        )

        amount_paid_ratio = (
            float(
                tx["amount_paid"]
            )
            /
            max(
                float(
                    tx[
                        "amount_received"
                    ]
                ),
                1e-12
            )
        )

        amount_paid = max(
            1e-6,
            amount_received
            *
            amount_paid_ratio
        )

        # Prevent extreme overflow.
        if amount_received > dataset_max_received:

            local_scale = (
                dataset_max_received
                /
                amount_received
            )

            amount_received *= (
                local_scale
            )

            amount_paid *= (
                local_scale
            )

        # ----------------------------------------------------
        # Time
        # ----------------------------------------------------
        timestamp = (
            anchor
            +
            pd.to_timedelta(
                float(
                    tx[
                        "relative_minutes"
                    ]
                ),
                unit="m"
            )
        )

        # ----------------------------------------------------
        # Payment format from NORMAL HI-Small distribution
        # ----------------------------------------------------
        payment_format = random.choice(
            payment_format_values
        )

        payment_currency = random.choice(
            payment_currency_values
        )

        receiving_candidates = (
            recv_by_pay.get(
                payment_currency
            )
        )

        if (
            receiving_candidates
            and
            len(receiving_candidates) > 0
        ):

            receiving_currency = (
                random.choice(
                    receiving_candidates
                )
            )

        else:

            receiving_currency = (
                payment_currency
            )

        # ----------------------------------------------------
        # Bank representation in transactions
        # ----------------------------------------------------
        source_bank_tx = (
            bank_tx_id_map[
                source_bank_norm
            ]
        )

        dest_bank_tx = (
            bank_tx_id_map[
                dest_bank_norm
            ]
        )

        # ----------------------------------------------------
        # Internal synthetic transaction ID
        # ----------------------------------------------------
        transaction_id = (
            generate_transaction_id()
        )

        # ----------------------------------------------------
        # Transaction CSV row
        # ----------------------------------------------------
        transaction_rows.append(
            {
                "transaction_id":
                    transaction_id,

                "Timestamp":
                    timestamp.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    ),

                "From Bank":
                    source_bank_tx,

                "Account":
                    source_account,

                "To Bank":
                    dest_bank_tx,

                "Account.1":
                    dest_account,

                "Amount Received":
                    float(
                        amount_received
                    ),

                "Receiving Currency":
                    receiving_currency,

                "Amount Paid":
                    float(
                        amount_paid
                    ),

                "Payment Currency":
                    payment_currency,

                "Payment Format":
                    payment_format,

                "Is Laundering":
                    1,

                "is_synthetic":
                    1,

                "synthetic_episode_id":
                    synthetic_episode_id,

                "typology":
                    typology,

                "source_template_id":
                    template_id,

                "template_source_order":
                    int(
                        tx[
                            "source_order"
                        ]
                    )
            }
        )

        # ----------------------------------------------------
        # Exact IBM-style Patterns.txt line
        # ----------------------------------------------------
        episode_pattern_lines.append(
            ",".join(
                [
                    timestamp.strftime(
                        "%Y/%m/%d %H:%M"
                    ),

                    source_bank_tx,

                    source_account,

                    dest_bank_tx,

                    dest_account,

                    f"{amount_received:.6f}",

                    receiving_currency,

                    f"{amount_paid:.6f}",

                    payment_currency,

                    payment_format,

                    "1"
                ]
            )
        )

        episode_transaction_count += 1
        episode_amounts.append(
            amount_received
        )

    # --------------------------------------------------------
    # Patterns.txt episode
    # --------------------------------------------------------
    pattern_file.write(
        f'id="{synthetic_episode_id}"\n'
    )

    pattern_file.write(
        f"BEGIN LAUNDERING ATTEMPT - "
        f"{typology}: "
        f"Synthetic instance based on "
        f"IBM {typology} template\n"
    )

    for line in episode_pattern_lines:

        pattern_file.write(
            line + "\n"
        )

    pattern_file.write(
        f"END LAUNDERING ATTEMPT - "
        f"{typology}\n"
    )

    # --------------------------------------------------------
    # Audit
    # --------------------------------------------------------
    episode_audit_rows.append(
        {
            "synthetic_episode_id":
                synthetic_episode_id,

            "typology":
                typology,

            "source_template_id":
                template_id,

            "transaction_count":
                episode_transaction_count,

            "node_count":
                len(
                    node_account_mapping
                ),

            "bank_count":
                len(
                    set(
                        node_bank_mapping.values()
                    )
                ),

            "fresh_account_count":
                fresh_accounts_episode,

            "unique_edges":
                len(
                    episode_edge_set
                ),

            "duplicate_edges":
                episode_transaction_count
                -
                len(
                    episode_edge_set
                ),

            "episode_span_minutes":
                template_span,

            "generated_median_amount":
                float(
                    np.median(
                        episode_amounts
                    )
                )
        }
    )

    global_transaction_counter += (
        episode_transaction_count
    )

    if (
        (pilot_idx + 1) % 500
        == 0
    ):

        print(
            f"  Episodes generated: "
            f"{pilot_idx + 1:,}"
            f" | Transactions: "
            f"{global_transaction_counter:,}"
            f" | New accounts: "
            f"{len(new_accounts_rows):,}"
        )

pattern_file.close()

# ============================================================
# 14. WRITE TRANSACTIONS
# ============================================================

print("\n" + "=" * 80)
print("7. WRITING PILOT FILES")
print("=" * 80)

pilot_transactions = pd.DataFrame(
    transaction_rows
)

pilot_audit = pd.DataFrame(
    episode_audit_rows
)

pilot_new_accounts = pd.DataFrame(
    new_accounts_rows
)

pilot_transactions.to_csv(
    PILOT_TRANS_PATH,
    index=False
)

pilot_new_accounts.to_csv(
    PILOT_ACCOUNTS_PATH,
    index=False
)

# ============================================================
# 15. GLOBAL AUDIT
# ============================================================

print("\n" + "=" * 80)
print("8. PILOT AUDIT")
print("=" * 80)

print(
    "Episodes:",
    len(pilot_audit)
)

print(
    "Transactions:",
    len(pilot_transactions)
)

print(
    "New accounts:",
    len(pilot_new_accounts)
)

print(
    "Unique transaction IDs:",
    pilot_transactions[
        "transaction_id"
    ].nunique()
)

print(
    "Transaction ID duplicates:",
    (
        len(pilot_transactions)
        -
        pilot_transactions[
            "transaction_id"
        ].nunique()
    )
)

print(
    "\nTypology transaction counts:"
)

print(
    pilot_transactions[
        "typology"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    "\nTypology episode counts:"
)

print(
    pilot_audit[
        "typology"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 16. EDGE VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("9. EDGE VALIDATION")
print("=" * 80)

print(
    "Episodes with duplicate edges:",
    int(
        (
            pilot_audit[
                "duplicate_edges"
            ]
            > 0
        ).sum()
    )
)

print(
    "Total duplicate edges inside episodes:",
    int(
        pilot_audit[
            "duplicate_edges"
        ].sum()
    )
)

global_edges = (
    pilot_transactions[
        [
            "From Bank",
            "Account",
            "To Bank",
            "Account.1"
        ]
    ]
    .astype(str)
)

global_edge_keys = (
    global_edges
    .astype(str)
    .agg(
        "|".join,
        axis=1
    )
)

print(
    "Global unique directed edges:",
    global_edge_keys.nunique()
)

print(
    "Global duplicate directed edges:",
    len(global_edge_keys)
    -
    global_edge_keys.nunique()
)

# ============================================================
# 17. ACCOUNT FORMAT VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("10. ACCOUNT FORMAT VALIDATION")
print("=" * 80)

new_ids = (
    pilot_new_accounts[
        "Account Number"
    ]
    .astype(str)
)

print(
    "New account format violations:",
    int(
        ~new_ids.str.fullmatch(
            r"[A-Z0-9]{9}"
        ).sum()
    )
)

if len(new_ids) > 0:

    print(
        "New account length distribution:"
    )

    print(
        new_ids.str.len()
        .value_counts()
        .sort_index()
        .to_string()
    )

# ============================================================
# 18. BANK VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("11. BANK VALIDATION")
print("=" * 80)

valid_transaction_banks = set(
    bank_pool[
        "canonical_transaction_bank_id"
    ].astype(str)
)

generated_from_banks = set(
    pilot_transactions[
        "From Bank"
    ].astype(str)
)

generated_to_banks = set(
    pilot_transactions[
        "To Bank"
    ].astype(str)
)

unknown_banks = (
    (
        generated_from_banks
        |
        generated_to_banks
    )
    -
    valid_transaction_banks
)

print(
    "Unique generated transaction banks:",
    len(
        generated_from_banks
        |
        generated_to_banks
    )
)

print(
    "Unknown generated banks:",
    len(unknown_banks)
)

# ============================================================
# 19. BEHAVIOR DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("12. SYNTHETIC BEHAVIOR DISTRIBUTION")
print("=" * 80)

print(
    "\nPayment Format:"
)

print(
    pilot_transactions[
        "Payment Format"
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(3)
    .to_string()
)

print(
    "\nPayment Currency:"
)

print(
    pilot_transactions[
        "Payment Currency"
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(3)
    .to_string()
)

amounts = (
    pilot_transactions[
        "Amount Received"
    ]
)

print(
    "\nGenerated Amount Received quantiles:"
)

print(
    amounts.quantile(
        [
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    ).to_string()
)

# ============================================================
# 20. TIMESTAMP DISTRIBUTION
# ============================================================

pilot_transactions[
    "_timestamp"
] = pd.to_datetime(
    pilot_transactions[
        "Timestamp"
    ]
)

print(
    "\nGenerated transaction hours:"
)

print(
    pilot_transactions[
        "_timestamp"
    ]
    .dt.hour
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 21. PATTERN FILE VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("13. PATTERNS.TXT VALIDATION")
print("=" * 80)

with open(
    PILOT_PATTERNS_PATH,
    "r",
    encoding="utf-8"
) as f:

    pattern_lines = [
        line.rstrip("\n")
        for line in f
    ]

begin_lines = [
    x for x in pattern_lines
    if x.startswith(
        "BEGIN LAUNDERING ATTEMPT"
    )
]

end_lines = [
    x for x in pattern_lines
    if x.startswith(
        "END LAUNDERING ATTEMPT"
    )
]

id_lines = [
    x for x in pattern_lines
    if x.startswith(
        'id="'
    )
]

print(
    "Patterns file lines:",
    len(pattern_lines)
)

print(
    "ID lines:",
    len(id_lines)
)

print(
    "BEGIN lines:",
    len(begin_lines)
)

print(
    "END lines:",
    len(end_lines)
)

print(
    "Expected episodes:",
    PILOT_EPISODES
)

print(
    "Episode marker count correct:",
    (
        len(id_lines)
        ==
        PILOT_EPISODES
        and
        len(begin_lines)
        ==
        PILOT_EPISODES
        and
        len(end_lines)
        ==
        PILOT_EPISODES
    )
)

# ============================================================
# 22. SAVE MANIFEST
# ============================================================

manifest = {

    "dataset":
        "IBM HI-Small",

    "pilot_seed":
        SEED,

    "pilot_episodes":
        int(PILOT_EPISODES),

    "generated_transactions":
        int(len(pilot_transactions)),

    "new_accounts_created":
        int(len(pilot_new_accounts)),

    "new_account_rate":
        float(NEW_ACCOUNT_RATE),

    "templates_available":
        int(len(templates)),

    "templates_used":
        int(
            pilot_plan[
                "source_template_id"
            ].nunique()
        ),

    "typology_episode_counts":
        {
            str(k): int(v)
            for k, v in
            pilot_audit[
                "typology"
            ].value_counts().items()
        },

    "typology_transaction_counts":
        {
            str(k): int(v)
            for k, v in
            pilot_transactions[
                "typology"
            ].value_counts().items()
        },

    "duplicate_episode_edges":
        int(
            pilot_audit[
                "duplicate_edges"
            ].sum()
        ),

    "duplicate_global_edges":
        int(
            len(global_edge_keys)
            -
            global_edge_keys.nunique()
        ),

    "unknown_banks":
        int(len(unknown_banks)),

    "duplicate_transaction_ids":
        int(
            len(pilot_transactions)
            -
            pilot_transactions[
                "transaction_id"
            ].nunique()
        ),

    "pattern_id_lines":
        int(len(id_lines)),

    "pattern_begin_lines":
        int(len(begin_lines)),

    "pattern_end_lines":
        int(len(end_lines)),

    "outputs": {
        "transactions":
            PILOT_TRANS_PATH,

        "new_accounts":
            PILOT_ACCOUNTS_PATH,

        "patterns":
            PILOT_PATTERNS_PATH
    }
}

with open(
    PILOT_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ============================================================
# 23. CLEAN HELPER COLUMN
# ============================================================

pilot_transactions.drop(
    columns=["_timestamp"],
    inplace=True
)

pilot_transactions.to_csv(
    PILOT_TRANS_PATH,
    index=False
)

# ============================================================
# 24. FINAL
# ============================================================

print("\n" + "=" * 100)
print("✅ CELL 15 COMPLETE — PILOT GENERATED")
print("=" * 100)

print(
    "\nTransactions:",
    f"{len(pilot_transactions):,}"
)

print(
    "Episodes:",
    f"{len(pilot_audit):,}"
)

print(
    "New accounts:",
    f"{len(pilot_new_accounts):,}"
)

print(
    "Templates used:",
    f"{pilot_plan['source_template_id'].nunique():,}"
)

print(
    "Duplicate transaction IDs:",
    int(
        len(pilot_transactions)
        -
        pilot_transactions[
            "transaction_id"
        ].nunique()
    )
)

print(
    "Unknown banks:",
    len(unknown_banks)
)

print(
    "Duplicate global edges:",
    int(
        len(global_edge_keys)
        -
        global_edge_keys.nunique()
    )
)

print(
    "Patterns ID/BEGIN/END correct:",
    (
        len(id_lines)
        ==
        PILOT_EPISODES
        and
        len(begin_lines)
        ==
        PILOT_EPISODES
        and
        len(end_lines)
        ==
        PILOT_EPISODES
    )
)

print("\nSaved:")
print(
    PILOT_TRANS_PATH
)
print(
    PILOT_ACCOUNTS_PATH
)
print(
    PILOT_PATTERNS_PATH
)
print(
    PILOT_MANIFEST_PATH
)

print("\nORIGINAL HI-SMALL FILES WERE NOT MODIFIED.")

print("=" * 100)

CELL 15 — 5,000-EPISODE SYNTHETIC GENERATOR PILOT

1. LOADING IBM TEMPLATES
Templates: 370
Full planned episodes: 584,706
Pilot episodes: 5000

Pilot typology distribution:
typology
BIPARTITE         662
CYCLE             729
FAN-IN            541
FAN-OUT           649
GATHER-SCATTER    689
RANDOM            554
SCATTER-GATHER    595
STACK             581

2. LOADING BANK POOL
Banks: 30470

3. LOADING CLEAN ACCOUNT REGISTRY
Eligible accounts: 508,731

4. LOADING ACCOUNT / ENTITY GENERATION DNA

5. BUILDING LEGITIMATE BEHAVIOR SAMPLE
  Chunk 1 | sampled legitimate rows = 10,000
  Chunk 2 | sampled legitimate rows = 20,000
  Chunk 3 | sampled legitimate rows = 30,000
  Chunk 4 | sampled legitimate rows = 40,000
  Chunk 5 | sampled legitimate rows = 50,000
  Chunk 6 | sampled legitimate rows = 60,000
  Chunk 7 | sampled legitimate rows = 70,000
  Chunk 8 | sampled legitimate rows = 80,000
  Chunk 9 | sampled legitimate rows = 90,000
  Chunk 10 | sampled legitimate rows = 100,000
  Chunk 1

In [22]:
# ============================================================
# CELL 16 — PILOT BANK RELATIONSHIP AUDIT
# ============================================================
#
# Purpose:
#   Audit whether synthetic From Bank -> To Bank relationships
#   are grounded in legitimate HI-Small connectivity.
#
# IMPORTANT:
#   This cell does NOT generate anything.
# ============================================================

import os
import pandas as pd
import numpy as np

WORK_DIR = "/kaggle/working/AML_GNN_ONLY/data"

PILOT_TRANS_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_transactions.csv"
)

LEGIT_PAIR_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_pairs.csv"
)

BANK_POOL_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_final_generator_bank_pool.csv"
)

print("=" * 100)
print("CELL 16 — PILOT BANK RELATIONSHIP AUDIT")
print("=" * 100)

# ============================================================
# 1. LOAD PILOT
# ============================================================
pilot = pd.read_csv(
    PILOT_TRANS_PATH,
    usecols=[
        "From Bank",
        "To Bank",
        "Account",
        "Account.1",
        "typology",
        "synthetic_episode_id"
    ],
    dtype=str
)

print(
    "\nPilot transactions:",
    f"{len(pilot):,}"
)

# ============================================================
# 2. LOAD LEGITIMATE BANK PAIRS
# ============================================================
legit_pairs = pd.read_csv(
    LEGIT_PAIR_PATH,
    usecols=[
        "From Bank",
        "To Bank",
        "legitimate_count"
    ],
    dtype=str
)

legit_pairs[
    "legitimate_count"
] = pd.to_numeric(
    legit_pairs[
        "legitimate_count"
    ],
    errors="coerce"
).fillna(0).astype(int)

# Normalize representations
def normalize_bank_id(x):

    x = str(x).strip()

    try:
        return str(int(x))
    except Exception:
        return x.lstrip("0") or "0"

pilot[
    "from_norm"
] = pilot[
    "From Bank"
].map(normalize_bank_id)

pilot[
    "to_norm"
] = pilot[
    "To Bank"
].map(normalize_bank_id)

legit_pairs[
    "from_norm"
] = legit_pairs[
    "From Bank"
].map(normalize_bank_id)

legit_pairs[
    "to_norm"
] = legit_pairs[
    "To Bank"
].map(normalize_bank_id)

# ============================================================
# 3. BUILD LEGITIMATE PAIR SET
# ============================================================
legit_pair_set = set(
    zip(
        legit_pairs[
            "from_norm"
        ],
        legit_pairs[
            "to_norm"
        ]
    )
)

print(
    "\nLegitimate bank pairs:",
    f"{len(legit_pair_set):,}"
)

# ============================================================
# 4. AUDIT GENERATED PAIRS
# ============================================================
pilot_pairs = (
    pilot[
        [
            "from_norm",
            "to_norm"
        ]
    ]
    .drop_duplicates()
)

pilot_pairs[
    "legitimate_pair"
] = [
    (
        (src, dst)
        in legit_pair_set
    )
    for src, dst in zip(
        pilot_pairs[
            "from_norm"
        ],
        pilot_pairs[
            "to_norm"
        ]
    )
]

print("\n" + "=" * 80)
print("1. GLOBAL BANK-PAIR REALISM")
print("=" * 80)

print(
    "Unique generated bank pairs:",
    f"{len(pilot_pairs):,}"
)

print(
    "Generated pairs observed in legitimate HI-Small:",
    int(
        pilot_pairs[
            "legitimate_pair"
        ].sum()
    )
)

print(
    "Generated pairs NOT observed legitimately:",
    int(
        (~pilot_pairs["legitimate_pair"]).sum()
    )
)

print(
    "Pair-level legitimate coverage:",
    f"{pilot_pairs['legitimate_pair'].mean() * 100:.4f}%"
)

# Transaction-weighted coverage
transaction_pair_flags = [
    (
        (src, dst)
        in legit_pair_set
    )
    for src, dst in zip(
        pilot["from_norm"],
        pilot["to_norm"]
    )
]

print(
    "Transaction-weighted legitimate coverage:",
    f"{np.mean(transaction_pair_flags) * 100:.4f}%"
)

# ============================================================
# 5. SHOW ARTIFICIAL PAIR EXAMPLES
# ============================================================
print("\n" + "=" * 80)
print("2. ARTIFICIAL BANK-PAIR EXAMPLES")
print("=" * 80)

artificial_pairs = (
    pilot_pairs[
        ~pilot_pairs[
            "legitimate_pair"
        ]
    ]
    .head(50)
)

if len(artificial_pairs) == 0:

    print(
        "✅ No artificial bank pairs found."
    )

else:

    print(
        artificial_pairs[
            [
                "from_norm",
                "to_norm"
            ]
        ]
        .to_string(
            index=False
        )
    )

# ============================================================
# 6. BANK-PAIR COVERAGE BY TYPOLOGY
# ============================================================
print("\n" + "=" * 80)
print("3. BANK-PAIR REALISM BY TYPOLOGY")
print("=" * 80)

typology_rows = []

for typology, g in (
    pilot
    .groupby("typology")
):

    flags = [
        (
            (src, dst)
            in legit_pair_set
        )
        for src, dst in zip(
            g["from_norm"],
            g["to_norm"]
        )
    ]

    unique_g = (
        g[
            [
                "from_norm",
                "to_norm"
            ]
        ]
        .drop_duplicates()
    )

    unique_flags = [
        (
            (src, dst)
            in legit_pair_set
        )
        for src, dst in zip(
            unique_g["from_norm"],
            unique_g["to_norm"]
        )
    ]

    typology_rows.append(
        {
            "typology":
                typology,

            "transactions":
                len(g),

            "unique_bank_pairs":
                len(unique_g),

            "transaction_weighted_legit_pct":
                np.mean(flags) * 100,

            "pair_level_legit_pct":
                np.mean(unique_flags) * 100
        }
    )

typology_audit = pd.DataFrame(
    typology_rows
).sort_values(
    "typology"
)

print(
    typology_audit.to_string(
        index=False
    )
)

# ============================================================
# 7. ACCOUNT/BANK CONSISTENCY
# ============================================================
print("\n" + "=" * 80)
print("4. ACCOUNT / BANK CONSISTENCY")
print("=" * 80)

# New pilot accounts
pilot_accounts = pd.read_csv(
    os.path.join(
        WORK_DIR,
        "HI-Small_SYNTH_PILOT_new_accounts.csv"
    ),
    dtype=str
)

# Each new account should belong to the bank recorded
# in the new account registry.
#
# Build normalized account->bank map.
pilot_accounts[
    "bank_norm"
] = (
    pilot_accounts[
        "Bank ID"
    ]
    .map(normalize_bank_id)
)

account_bank_map = {
    str(row["Account Number"]):
    str(row["bank_norm"])
    for _, row in pilot_accounts.iterrows()
}

source_check = []

for _, row in pilot.iterrows():

    account = str(
        row["Account"]
    )

    if account in account_bank_map:

        source_check.append(
            account_bank_map[account]
            ==
            row["from_norm"]
        )

    else:

        source_check.append(
            True
        )

destination_check = []

for _, row in pilot.iterrows():

    account = str(
        row["Account.1"]
    )

    if account in account_bank_map:

        destination_check.append(
            account_bank_map[account]
            ==
            row["to_norm"]
        )

    else:

        destination_check.append(
            True
        )

print(
    "New-account source-bank mismatches:",
    int(
        (~np.array(source_check)).sum()
    )
)

print(
    "New-account destination-bank mismatches:",
    int(
        (~np.array(destination_check)).sum()
    )
)

# ============================================================
# 8. ACCOUNT FORMAT CORRECTED AUDIT
# ============================================================
print("\n" + "=" * 80)
print("5. ACCOUNT FORMAT AUDIT")
print("=" * 80)

if len(pilot_accounts) > 0:

    valid_mask = (
        pilot_accounts[
            "Account Number"
        ]
        .astype(str)
        .str.fullmatch(
            r"[A-Z0-9]{9}"
        )
    )

    print(
        "New accounts:",
        len(pilot_accounts)
    )

    print(
        "Valid 9-char uppercase-alphanumeric:",
        int(valid_mask.sum())
    )

    print(
        "Actual format violations:",
        int((~valid_mask).sum())
    )

# ============================================================
# 9. GLOBAL NODE / EDGE CHARACTERISTICS
# ============================================================
print("\n" + "=" * 80)
print("6. GLOBAL PILOT CHARACTERISTICS")
print("=" * 80)

print(
    "Unique source accounts:",
    pilot["Account"].nunique()
)

print(
    "Unique destination accounts:",
    pilot["Account.1"].nunique()
)

print(
    "Unique source banks:",
    pilot["from_norm"].nunique()
)

print(
    "Unique destination banks:",
    pilot["to_norm"].nunique()
)

# ============================================================
# 10. FINAL VERDICT
# ============================================================
coverage = (
    np.mean(transaction_pair_flags)
    *
    100
)

print("\n" + "=" * 100)
print("✅ CELL 16 COMPLETE")
print("=" * 100)

print(
    f"Transaction-weighted legitimate bank-pair coverage: "
    f"{coverage:.4f}%"
)

if coverage >= 99:

    print(
        "✅ Bank relationship structure is strongly grounded."
    )

elif coverage >= 90:

    print(
        "⚠️ Bank relationship structure needs improvement."
    )

else:

    print(
        "❌ Bank relationship structure is NOT suitable "
        "for final-scale generation."
    )

print(
    "\nNo synthetic data generated by this cell."
)

print(
    "No original files modified."
)

print("=" * 100)

CELL 16 — PILOT BANK RELATIONSHIP AUDIT

Pilot transactions: 43,310

Legitimate bank pairs: 283,353

1. GLOBAL BANK-PAIR REALISM
Unique generated bank pairs: 39,258
Generated pairs observed in legitimate HI-Small: 8076
Generated pairs NOT observed legitimately: 31182
Pair-level legitimate coverage: 20.5716%
Transaction-weighted legitimate coverage: 23.9806%

2. ARTIFICIAL BANK-PAIR EXAMPLES
from_norm to_norm
     1467   14838
    14838    1068
   232583   36056
    36056   30380
    30380  127977
   224449   21258
        1     210
    23833     410
    23833  232880
     1674    1655
   214621      12
       20  212789
   212789   50016
    50016  142501
   142501      12
       12      14
       14   29317
    29317   11056
   226925  240229
   240229  226925
    21611    3514
     3514     126
      126   16163
    16163    3236
     3236   22736
    22736   23525
    23525     701
    16354   19723
       16   19723
   239692   19723
   122055   19723
     1547   19723
   325787   

In [23]:
# ============================================================
# CELL 17 — LEGITIMATE-CONNECTIVITY BANK MAPPING LIBRARY
# ============================================================
#
# PURPOSE
# -------
# Fix the issue discovered in CELL 16.
#
# For every IBM pattern template:
#
#     template node A -> template node B
#
# must map to:
#
#     real bank X -> real bank Y
#
# where X -> Y EXISTS in legitimate HI-Small traffic.
#
# The graph topology of the IBM laundering template is
# preserved exactly.
#
# We create several valid mappings per template so the final
# generator can randomly select among them.
#
# NO synthetic transactions generated.
# NO synthetic accounts generated.
# ============================================================

import os
import json
import random
from collections import defaultdict, deque

import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

SEED = 20260919

random.seed(SEED)
np.random.seed(SEED)

MAPPINGS_PER_TEMPLATE = 5

MAX_ROOT_CANDIDATES = 300

MAX_BACKTRACK_STEPS = 3000

MAX_ATTEMPTS_PER_MAPPING = 100

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORK_DIR = (
    "/kaggle/working/"
    "AML_GNN_ONLY/data"
)

TEMPLATE_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_templates.json"
)

LEGIT_PAIR_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_pairs.csv"
)

BANK_POOL_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_final_generator_bank_pool.csv"
)

OUT_JSON = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_mapping_library.json"
)

OUT_SUMMARY = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_mapping_summary.csv"
)

print("=" * 100)
print("CELL 17 — LEGITIMATE-CONNECTIVITY BANK MAPPING LIBRARY")
print("=" * 100)

# ============================================================
# 1. LOAD TEMPLATES
# ============================================================

with open(
    TEMPLATE_PATH,
    "r",
    encoding="utf-8"
) as f:

    template_manifest = json.load(f)

templates = template_manifest[
    "templates"
]

print(
    "\nIBM templates:",
    len(templates)
)

# ============================================================
# 2. LOAD LEGITIMATE BANK PAIRS
# ============================================================

print("\n" + "=" * 80)
print("1. LOADING LEGITIMATE BANK GRAPH")
print("=" * 80)

pairs = pd.read_csv(
    LEGIT_PAIR_PATH,
    dtype=str
)

pairs["legitimate_count"] = pd.to_numeric(
    pairs["legitimate_count"],
    errors="coerce"
).fillna(0).astype(int)

# ------------------------------------------------------------
# Normalize IDs
# ------------------------------------------------------------

def normalize_bank_id(x):

    x = str(x).strip()

    if x == "":
        return None

    try:
        return str(int(x))
    except Exception:
        return x.lstrip("0") or "0"

pairs[
    "src_norm"
] = pairs[
    "From Bank"
].map(normalize_bank_id)

pairs[
    "dst_norm"
] = pairs[
    "To Bank"
].map(normalize_bank_id)

pairs = pairs[
    pairs[
        "legitimate_count"
    ] > 0
].copy()

print(
    "Legitimate bank pairs:",
    f"{len(pairs):,}"
)

# ============================================================
# 3. BUILD ADJACENCY STRUCTURES
# ============================================================

print("\n" + "=" * 80)
print("2. BUILDING LEGITIMATE BANK GRAPH")
print("=" * 80)

out_neighbors = defaultdict(dict)
in_neighbors = defaultdict(dict)

for _, row in pairs.iterrows():

    src = str(
        row["src_norm"]
    )

    dst = str(
        row["dst_norm"]
    )

    weight = int(
        row["legitimate_count"]
    )

    out_neighbors[src][dst] = weight

    in_neighbors[dst][src] = weight

# ------------------------------------------------------------
# Sets for fast constraint checks
# ------------------------------------------------------------

out_neighbor_sets = {
    src: set(
        neighbors.keys()
    )
    for src, neighbors
    in out_neighbors.items()
}

in_neighbor_sets = {
    dst: set(
        neighbors.keys()
    )
    for dst, neighbors
    in in_neighbors.items()
}

all_banks = sorted(
    set(
        out_neighbors.keys()
    )
    |
    set(
        in_neighbors.keys()
    )
)

print(
    "Banks in legitimate graph:",
    len(all_banks)
)

print(
    "Banks with outgoing legitimate edges:",
    len(out_neighbors)
)

print(
    "Banks with incoming legitimate edges:",
    len(in_neighbors)
)

# ============================================================
# 4. LOAD BANK ACTIVITY WEIGHTS
# ============================================================

bank_pool = pd.read_csv(
    BANK_POOL_PATH,
    dtype=str
)

for col in [
    "legitimate_source_transactions",
    "legitimate_destination_transactions"
]:

    bank_pool[col] = pd.to_numeric(
        bank_pool[col],
        errors="coerce"
    ).fillna(0)

bank_activity = {}

for _, row in bank_pool.iterrows():

    bank = str(
        row["normalized_bank_id"]
    )

    bank_activity[
        bank
    ] = {

        "source_activity":
            float(
                row[
                    "legitimate_source_transactions"
                ]
            ),

        "destination_activity":
            float(
                row[
                    "legitimate_destination_transactions"
                ]
            )
    }

# ============================================================
# 5. BANK DEGREE DATA
# ============================================================

bank_out_degree = {
    bank: len(
        out_neighbor_sets.get(
            bank,
            set()
        )
    )
    for bank in all_banks
}

bank_in_degree = {
    bank: len(
        in_neighbor_sets.get(
            bank,
            set()
        )
    )
    for bank in all_banks
}

# ============================================================
# 6. TEMPLATE GRAPH HELPERS
# ============================================================

def build_template_graph(
    template
):

    nodes = set()

    adjacency = defaultdict(set)

    reverse = defaultdict(set)

    edge_list = []

    for edge in template[
        "edges"
    ]:

        s = int(
            edge["source_node"]
        )

        d = int(
            edge["destination_node"]
        )

        nodes.add(s)
        nodes.add(d)

        adjacency[s].add(d)

        reverse[d].add(s)

        edge_list.append(
            (
                s,
                d
            )
        )

    # Preserve isolated nodes if any
    for node in template[
        "nodes"
    ]:

        nodes.add(
            int(
                node["node_id"]
            )
        )

    return (
        sorted(nodes),
        adjacency,
        reverse,
        edge_list
    )


def node_degrees(
    node,
    adjacency,
    reverse
):

    return (
        len(
            adjacency.get(
                node,
                set()
            )
        ),
        len(
            reverse.get(
                node,
                set()
            )
        )
    )


def connected_components(
    nodes,
    adjacency,
    reverse
):

    seen = set()
    components = []

    for start in nodes:

        if start in seen:
            continue

        q = deque(
            [start]
        )

        seen.add(start)

        component = []

        while q:

            u = q.popleft()

            component.append(u)

            neighbors = (
                adjacency.get(
                    u,
                    set()
                )
                |
                reverse.get(
                    u,
                    set()
                )
            )

            for v in neighbors:

                if v not in seen:

                    seen.add(v)
                    q.append(v)

        components.append(
            sorted(component)
        )

    return components

# ============================================================
# 7. CANDIDATE BANK FUNCTIONS
# ============================================================

def base_bank_score(
    bank,
    role,
    required_out=0,
    required_in=0
):

    bank = str(bank)

    out_deg = bank_out_degree.get(
        bank,
        0
    )

    in_deg = bank_in_degree.get(
        bank,
        0
    )

    if (
        out_deg < required_out
        or
        in_deg < required_in
    ):
        return 0.0

    activity = bank_activity.get(
        bank,
        {}
    )

    if role == "source":

        score = (
            activity.get(
                "source_activity",
                0.0
            )
            +
            1.0
        )

    elif role == "destination":

        score = (
            activity.get(
                "destination_activity",
                0.0
            )
            +
            1.0
        )

    else:

        score = (
            activity.get(
                "source_activity",
                0.0
            )
            +
            activity.get(
                "destination_activity",
                0.0
            )
            +
            1.0
        )

    # Mild degree boost.
    score *= (
        1.0
        +
        0.02
        *
        (
            out_deg
            +
            in_deg
        )
    )

    return float(
        score
    )


def weighted_sample_one(
    candidates,
    role,
    required_out=0,
    required_in=0
):

    if not candidates:
        return None

    scored = []

    for bank in candidates:

        score = base_bank_score(
            bank,
            role,
            required_out,
            required_in
        )

        if score > 0:

            scored.append(
                (
                    str(bank),
                    score
                )
            )

    if not scored:
        return None

    banks = [
        x[0]
        for x in scored
    ]

    scores = np.array(
        [
            x[1]
            for x in scored
        ],
        dtype=float
    )

    # Compress enormous bank-activity differences.
    scores = np.sqrt(
        scores
    )

    scores /= scores.sum()

    return str(
        np.random.choice(
            banks,
            p=scores
        )
    )

# ============================================================
# 8. NODE ROLE
# ============================================================

def get_node_role(
    node,
    adjacency,
    reverse
):

    out_deg = len(
        adjacency.get(
            node,
            set()
        )
    )

    in_deg = len(
        reverse.get(
            node,
            set()
        )
    )

    if (
        in_deg > 0
        and
        out_deg > 0
    ):
        return "both"

    if out_deg > 0:
        return "source"

    if in_deg > 0:
        return "destination"

    return "both"

# ============================================================
# 9. GENERATE ONE VALID BANK MAPPING
# ============================================================

def generate_mapping(
    template,
    mapping_index
):

    (
        nodes,
        adjacency,
        reverse,
        edge_list
    ) = build_template_graph(
        template
    )

    # --------------------------------------------------------
    # Handle empty graph
    # --------------------------------------------------------
    if not nodes:

        return {}

    # --------------------------------------------------------
    # Components
    # --------------------------------------------------------
    components = connected_components(
        nodes,
        adjacency,
        reverse
    )

    mapping = {}

    used_banks = set()

    # --------------------------------------------------------
    # Process each component independently.
    # --------------------------------------------------------
    for component in components:

        if len(component) == 1:

            node = component[0]

            role = get_node_role(
                node,
                adjacency,
                reverse
            )

            candidates = [
                bank
                for bank in all_banks
                if (
                    base_bank_score(
                        bank,
                        role
                    ) > 0
                )
            ]

            selected = weighted_sample_one(
                candidates,
                role
            )

            if selected is None:
                return None

            mapping[node] = selected
            used_banks.add(selected)

            continue

        # ----------------------------------------------------
        # Node ordering:
        # high degree first.
        # ----------------------------------------------------
        component_order = sorted(
            component,
            key=lambda n: (
                -(
                    len(
                        adjacency.get(
                            n,
                            set()
                        )
                    )
                    +
                    len(
                        reverse.get(
                            n,
                            set()
                        )
                    )
                ),
                -len(
                    adjacency.get(
                        n,
                        set()
                    )
                ),
                n
            )
        )

        # ----------------------------------------------------
        # Root selection
        # ----------------------------------------------------
        root = component_order[0]

        root_role = get_node_role(
            root,
            adjacency,
            reverse
        )

        required_out = len(
            adjacency.get(
                root,
                set()
            )
        )

        required_in = len(
            reverse.get(
                root,
                set()
            )
        )

        # Candidate banks that have enough observed
        # neighbors for the root's branching structure.
        root_candidates = []

        for bank in all_banks:

            if bank_out_degree.get(
                bank,
                0
            ) < required_out:
                continue

            if bank_in_degree.get(
                bank,
                0
            ) < required_in:
                continue

            score = base_bank_score(
                bank,
                root_role,
                required_out,
                required_in
            )

            if score > 0:
                root_candidates.append(
                    (
                        bank,
                        score
                    )
                )

        if not root_candidates:
            return None

        root_candidates = sorted(
            root_candidates,
            key=lambda x: x[1],
            reverse=True
        )

        # Keep a diverse but manageable root candidate set.
        if len(root_candidates) > MAX_ROOT_CANDIDATES:

            # Probabilistic selection later.
            root_candidates = root_candidates[
                :MAX_ROOT_CANDIDATES
            ]

        # Shuffle root order weighted by activity.
        root_banks = [
            x[0]
            for x in root_candidates
        ]

        root_scores = np.array(
            [
                x[1]
                for x in root_candidates
            ],
            dtype=float
        )

        root_scores = np.sqrt(
            root_scores
        )

        root_scores /= (
            root_scores.sum()
        )

        root_order = list(
            np.random.choice(
                root_banks,
                size=min(
                    30,
                    len(root_banks)
                ),
                replace=False,
                p=root_scores
            )
        )

        found_component = False

        # ----------------------------------------------------
        # Try multiple roots
        # ----------------------------------------------------
        for root_bank in root_order:

            local_mapping = {
                root: str(
                    root_bank
                )
            }

            local_used = {
                str(root_bank)
            }

            # ------------------------------------------------
            # Backtracking search
            # ------------------------------------------------
            backtrack_counter = [0]

            def choose_next_node():

                unassigned = [
                    n for n in component
                    if n not in local_mapping
                ]

                if not unassigned:
                    return None

                # MRV:
                # choose node with most constraints first.
                best_node = None
                best_key = None

                for n in unassigned:

                    assigned_neighbors = (
                        (
                            adjacency.get(
                                n,
                                set()
                            )
                            |
                            reverse.get(
                                n,
                                set()
                            )
                        )
                        &
                        set(
                            local_mapping.keys()
                        )
                    )

                    key = (
                        len(
                            assigned_neighbors
                        ),
                        len(
                            adjacency.get(
                                n,
                                set()
                            )
                        )
                        +
                        len(
                            reverse.get(
                                n,
                                set()
                            )
                        )
                    )

                    if (
                        best_key is None
                        or
                        key > best_key
                    ):

                        best_key = key
                        best_node = n

                return best_node

            def candidate_banks(
                node
            ):

                role = get_node_role(
                    node,
                    adjacency,
                    reverse
                )

                candidates = None

                # --------------------------------------------
                # Incoming constraints:
                #
                # assigned predecessor P
                # requires:
                #
                #   P_bank -> candidate_bank
                # --------------------------------------------
                for predecessor in (
                    reverse.get(
                        node,
                        set()
                    )
                ):

                    if predecessor not in (
                        local_mapping
                    ):
                        continue

                    pred_bank = local_mapping[
                        predecessor
                    ]

                    valid = (
                        out_neighbor_sets
                        .get(
                            pred_bank,
                            set()
                        )
                    )

                    if candidates is None:
                        candidates = set(valid)
                    else:
                        candidates &= valid

                    if not candidates:
                        return []

                # --------------------------------------------
                # Outgoing constraints:
                #
                # candidate_bank -> assigned successor
                # --------------------------------------------
                for successor in (
                    adjacency.get(
                        node,
                        set()
                    )
                ):

                    if successor not in (
                        local_mapping
                    ):
                        continue

                    succ_bank = local_mapping[
                        successor
                    ]

                    valid = (
                        in_neighbor_sets
                        .get(
                            succ_bank,
                            set()
                        )
                    )

                    if candidates is None:
                        candidates = set(valid)
                    else:
                        candidates &= valid

                    if not candidates:
                        return []

                # --------------------------------------------
                # If no assigned neighbor constraint exists,
                # use global banks.
                # --------------------------------------------
                if candidates is None:

                    required_out_local = len(
                        adjacency.get(
                            node,
                            set()
                        )
                    )

                    required_in_local = len(
                        reverse.get(
                            node,
                            set()
                        )
                    )

                    candidates = {
                        bank
                        for bank in all_banks
                        if (
                            bank_out_degree.get(
                                bank,
                                0
                            )
                            >=
                            required_out_local
                            and
                            bank_in_degree.get(
                                bank,
                                0
                            )
                            >=
                            required_in_local
                        )
                    }

                # --------------------------------------------
                # Prefer unused banks where possible.
                # But DO NOT force uniqueness if that would
                # make the graph impossible.
                # --------------------------------------------
                unused_candidates = (
                    candidates
                    -
                    local_used
                )

                if unused_candidates:
                    candidates = (
                        unused_candidates
                    )

                # --------------------------------------------
                # Ensure node's role has activity support.
                # --------------------------------------------
                scored = []

                required_out_local = len(
                    adjacency.get(
                        node,
                        set()
                    )
                )

                required_in_local = len(
                    reverse.get(
                        node,
                        set()
                    )
                )

                for bank in candidates:

                    score = base_bank_score(
                        bank,
                        role,
                        required_out_local,
                        required_in_local
                    )

                    if score <= 0:
                        continue

                    # Add local connectivity score.
                    local_score = 1.0

                    for predecessor in (
                        reverse.get(
                            node,
                            set()
                        )
                    ):

                        if predecessor in local_mapping:

                            pred_bank = local_mapping[
                                predecessor
                            ]

                            local_score *= max(
                                1.0,
                                float(
                                    out_neighbors[
                                        pred_bank
                                    ].get(
                                        bank,
                                        0
                                    )
                                )
                            )

                    for successor in (
                        adjacency.get(
                            node,
                            set()
                        )
                    ):

                        if successor in local_mapping:

                            succ_bank = local_mapping[
                                successor
                            ]

                            local_score *= max(
                                1.0,
                                float(
                                    in_neighbors[
                                        succ_bank
                                    ].get(
                                        bank,
                                        0
                                    )
                                )
                            )

                    score *= (
                        np.sqrt(
                            local_score
                        )
                    )

                    scored.append(
                        (
                            str(bank),
                            score
                        )
                    )

                if not scored:
                    return []

                # Weighted random ordering.
                banks = [
                    x[0]
                    for x in scored
                ]

                scores = np.array(
                    [
                        x[1]
                        for x in scored
                    ],
                    dtype=float
                )

                scores = np.sqrt(
                    scores
                )

                scores /= (
                    scores.sum()
                )

                order_count = min(
                    40,
                    len(banks)
                )

                return list(
                    np.random.choice(
                        banks,
                        size=order_count,
                        replace=False,
                        p=scores
                    )
                )

            def search():

                if (
                    len(local_mapping)
                    ==
                    len(component)
                ):
                    return True

                backtrack_counter[0] += 1

                if (
                    backtrack_counter[0]
                    >
                    MAX_BACKTRACK_STEPS
                ):
                    return False

                node = choose_next_node()

                if node is None:
                    return True

                candidates = candidate_banks(
                    node
                )

                if not candidates:
                    return False

                for bank in candidates:

                    # Check every currently satisfied edge
                    # explicitly before committing.
                    valid = True

                    for predecessor in (
                        reverse.get(
                            node,
                            set()
                        )
                    ):

                        if predecessor in local_mapping:

                            pred_bank = local_mapping[
                                predecessor
                            ]

                            if bank not in (
                                out_neighbor_sets
                                .get(
                                    pred_bank,
                                    set()
                                )
                            ):

                                valid = False
                                break

                    if not valid:
                        continue

                    for successor in (
                        adjacency.get(
                            node,
                            set()
                        )
                    ):

                        if successor in local_mapping:

                            succ_bank = local_mapping[
                                successor
                            ]

                            if bank not in (
                                in_neighbor_sets
                                .get(
                                    succ_bank,
                                    set()
                                )
                            ):

                                valid = False
                                break

                    if not valid:
                        continue

                    local_mapping[
                        node
                    ] = bank

                    local_used.add(
                        bank
                    )

                    if search():
                        return True

                    local_used.discard(
                        bank
                    )

                    del local_mapping[
                        node
                    ]

                return False

            if search():

                mapping.update(
                    local_mapping
                )

                used_banks.update(
                    local_mapping.values()
                )

                found_component = True
                break

        if not found_component:
            return None

    # ========================================================
    # FINAL EDGE CHECK
    # ========================================================
    for source_node, dest_node in edge_list:

        source_bank = mapping[
            source_node
        ]

        dest_bank = mapping[
            dest_node
        ]

        if dest_bank not in (
            out_neighbor_sets.get(
                source_bank,
                set()
            )
        ):

            return None

    return mapping

# ============================================================
# 10. BUILD MAPPING LIBRARY
# ============================================================

print("\n" + "=" * 80)
print("3. GENERATING VALID BANK MAPPINGS")
print("=" * 80)

mapping_library = {}
summary_rows = []

total_successful = 0

for template_idx, template in enumerate(
    templates,
    start=1
):

    template_id = str(
        template["template_id"]
    )

    typology = str(
        template["typology"]
    )

    target_mappings = []

    attempted = 0

    while (
        len(target_mappings)
        <
        MAPPINGS_PER_TEMPLATE
        and
        attempted
        <
        (
            MAPPINGS_PER_TEMPLATE
            *
            MAX_ATTEMPTS_PER_MAPPING
        )
    ):

        attempted += 1

        mapping = generate_mapping(
            template,
            len(target_mappings)
        )

        if mapping is None:
            continue

        # ----------------------------------------------------
        # Canonical mapping representation
        # ----------------------------------------------------
        mapping_tuple = tuple(
            (
                int(node),
                str(bank)
            )
            for node, bank
            in sorted(
                mapping.items()
            )
        )

        # Avoid duplicate mappings.
        existing = {
            tuple(
                (
                    int(n),
                    str(b)
                )
                for n, b
                in sorted(
                    m.items()
                )
            )
            for m in target_mappings
        }

        if mapping_tuple in existing:
            continue

        target_mappings.append(
            mapping
        )

    mapping_library[
        template_id
    ] = target_mappings

    success = len(
        target_mappings
    )

    total_successful += success

    summary_rows.append(
        {
            "template_id":
                template_id,

            "typology":
                typology,

            "template_nodes":
                int(
                    template["node_count"]
                ),

            "template_edges":
                int(
                    len(
                        template["edges"]
                    )
                ),

            "mappings_required":
                int(
                    MAPPINGS_PER_TEMPLATE
                ),

            "mappings_generated":
                int(
                    success
                ),

            "complete":
                bool(
                    success
                    >=
                    MAPPINGS_PER_TEMPLATE
                ),

            "unique_banks_example":
                int(
                    len(
                        set(
                            target_mappings[0].values()
                        )
                    )
                )
                if target_mappings
                else 0
        }
    )

    if (
        template_idx % 25
        == 0
    ):

        print(
            f"Templates processed: "
            f"{template_idx:,}/{len(templates):,}"
            f" | "
            f"valid mappings stored: "
            f"{total_successful:,}"
        )

# ============================================================
# 11. SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 80)
print("4. MAPPING LIBRARY SUMMARY")
print("=" * 80)

print(
    summary_df[
        [
            "typology",
            "mappings_generated",
            "complete"
        ]
    ]
    .groupby("typology")
    .agg(
        templates=(
            "mappings_generated",
            "count"
        ),

        min_mappings=(
            "mappings_generated",
            "min"
        ),

        median_mappings=(
            "mappings_generated",
            "median"
        ),

        complete_templates=(
            "complete",
            "sum"
        )
    )
    .to_string()
)

print(
    "\nTemplates with 5 valid mappings:",
    int(
        summary_df[
            "complete"
        ].sum()
    ),
        "/",
    len(summary_df)
)

print(
    "Total valid mappings:",
    total_successful
)

# ============================================================
# 12. CHECK ALL EDGES IN ALL MAPPINGS
# ============================================================

print("\n" + "=" * 80)
print("5. FINAL EDGE VALIDATION")
print("=" * 80)

total_checked_edges = 0
invalid_edges = 0

for template in templates:

    template_id = str(
        template["template_id"]
    )

    mappings = mapping_library[
        template_id
    ]

    for mapping in mappings:

        for edge in template[
            "edges"
        ]:

            source_bank = mapping[
                int(
                    edge["source_node"]
                )
            ]

            dest_bank = mapping[
                int(
                    edge["destination_node"]
                )
            ]

            total_checked_edges += 1

            if dest_bank not in (
                out_neighbor_sets.get(
                    source_bank,
                    set()
                )
            ):

                invalid_edges += 1

print(
    "Edges checked:",
    f"{total_checked_edges:,}"
)

print(
    "Invalid edges:",
    invalid_edges
)

print(
    "Legitimate edge coverage:",
    (
        100.0
        if total_checked_edges == 0
        else
        (
            1
            -
            invalid_edges
            /
            total_checked_edges
        )
        *
        100
    )
)

# ============================================================
# 13. BANK-COUNT DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("6. UNIQUE BANK COUNT OF GENERATED MAPPINGS")
print("=" * 80)

bank_count_rows = []

for template in templates:

    template_id = str(
        template["template_id"]
    )

    typology = str(
        template["typology"]
    )

    for mapping in mapping_library[
        template_id
    ]:

        bank_count_rows.append(
            {
                "template_id":
                    template_id,

                "typology":
                    typology,

                "nodes":
                    len(mapping),

                "unique_banks":
                    len(
                        set(
                            mapping.values()
                        )
                    )
            }
        )

bank_count_df = pd.DataFrame(
    bank_count_rows
)

print(
    bank_count_df
    .groupby("typology")[
        "unique_banks"
    ]
    .describe()
    .round(3)
    .to_string()
)

# ============================================================
# 14. SAVE SUMMARY
# ============================================================

summary_df.to_csv(
    OUT_SUMMARY,
    index=False
)

# ============================================================
# 15. SAVE JSON
# ============================================================

serializable_library = {}

for template_id, mappings in (
    mapping_library.items()
):

    serializable_library[
        str(template_id)
    ] = [

        {
            str(node):
                str(bank)

            for node, bank
            in mapping.items()
        }

        for mapping in mappings
    ]

manifest = {

    "dataset":
        "IBM HI-Small",

    "purpose":
        "Reusable bank-node mappings constrained to "
        "legitimate HI-Small bank connectivity",

    "seed":
        SEED,

    "mappings_per_template_requested":
        MAPPINGS_PER_TEMPLATE,

    "template_count":
        len(templates),

    "templates_with_complete_mapping_library":
        int(
            summary_df[
                "complete"
            ].sum()
        ),

    "total_valid_mappings":
        int(total_successful),

    "legitimate_edge_coverage_pct":
        (
            100.0
            if total_checked_edges == 0
            else
            (
                1
                -
                invalid_edges
                /
                total_checked_edges
            )
            *
            100
        ),

    "mappings":
        serializable_library
}

with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ============================================================
# 16. FINAL
# ============================================================

print("\n" + "=" * 100)
print("✅ CELL 17 COMPLETE")
print("=" * 100)

print(
    "Templates:",
    len(templates)
)

print(
    "Templates with complete mapping sets:",
    int(
        summary_df[
            "complete"
        ].sum()
    )
)

print(
    "Required mappings/template:",
    MAPPINGS_PER_TEMPLATE
)

print(
    "Total valid mappings:",
    total_successful
)

print(
    "Invalid mapped edges:",
    invalid_edges
)

print(
    "Legitimate edge coverage:",
    (
        100.0
        if total_checked_edges == 0
        else
        (
            1
            -
            invalid_edges
            /
            total_checked_edges
        )
        *
        100
    ),
    "%"
)

print("\nSaved:")
print(
    OUT_JSON
)

print(
    OUT_SUMMARY
)

print("\nNO SYNTHETIC TRANSACTIONS CREATED.")
print("NO SYNTHETIC ACCOUNTS CREATED.")
print("NO ORIGINAL FILE MODIFIED.")

print("=" * 100)

CELL 17 — LEGITIMATE-CONNECTIVITY BANK MAPPING LIBRARY

IBM templates: 370

1. LOADING LEGITIMATE BANK GRAPH
Legitimate bank pairs: 281,474

2. BUILDING LEGITIMATE BANK GRAPH
Banks in legitimate graph: 30470
Banks with outgoing legitimate edges: 30470
Banks with incoming legitimate edges: 15811

3. GENERATING VALID BANK MAPPINGS
Templates processed: 25/370 | valid mappings stored: 125
Templates processed: 50/370 | valid mappings stored: 250
Templates processed: 75/370 | valid mappings stored: 375
Templates processed: 100/370 | valid mappings stored: 500
Templates processed: 125/370 | valid mappings stored: 625
Templates processed: 150/370 | valid mappings stored: 750
Templates processed: 175/370 | valid mappings stored: 875
Templates processed: 200/370 | valid mappings stored: 1,000
Templates processed: 225/370 | valid mappings stored: 1,125
Templates processed: 250/370 | valid mappings stored: 1,250
Templates processed: 275/370 | valid mappings stored: 1,375
Templates processed: 300/3

In [24]:
# ============================================================
# CELL 18 — CONNECTIVITY-CONSTRAINED SYNTHETIC PILOT V2
# ============================================================
#
# Rebuild the 5,000-episode pilot using CELL 17 mappings.
#
# CRITICAL DIFFERENCE FROM CELL 15:
#
#   Cell 15:
#       template node -> independently sampled bank
#
#   Cell 18:
#       template node -> bank from CELL 17
#       where EVERY template edge maps to a legitimate
#       HI-Small bank relationship.
#
# ============================================================

import os
import json
import random
import string
from collections import defaultdict

import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

SEED = 20260919

random.seed(SEED)
np.random.seed(SEED)

PILOT_EPISODES = 5000

# New accounts are generated only for a small fraction of
# template nodes.
NEW_ACCOUNT_RATE = 0.05

# ============================================================
# PATHS
# ============================================================

DATA_ROOT = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
)

WORK_DIR = (
    "/kaggle/working/"
    "AML_GNN_ONLY/data"
)

TRANS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_Trans.csv"
)

ACCOUNTS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_accounts.csv"
)

TEMPLATE_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_templates.json"
)

EPISODE_PLAN_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_synthetic_episode_plan.csv"
)

BANK_POOL_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_final_generator_bank_pool.csv"
)

ACCOUNT_REGISTRY_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_generator_account_registry.csv"
)

ACCOUNT_PREFIX_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_bank_account_prefix_sampler.json"
)

ACCOUNT_GENERATOR_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_account_id_generator.json"
)

ENTITY_DNA_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_entity_name_dna.json"
)

MAPPING_LIBRARY_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_mapping_library.json"
)

# ------------------------------------------------------------
# OUTPUTS — V2
# ------------------------------------------------------------

OUT_TRANS = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_V2_transactions.csv"
)

OUT_ACCOUNTS = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_V2_new_accounts.csv"
)

OUT_PATTERNS = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_V2_Patterns.txt"
)

OUT_MANIFEST = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_V2_manifest.json"
)

OUT_AUDIT = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_V2_episode_audit.csv"
)

print("=" * 100)
print("CELL 18 — CONNECTIVITY-CONSTRAINED SYNTHETIC PILOT V2")
print("=" * 100)

# ============================================================
# 1. LOAD TEMPLATES
# ============================================================

print("\n" + "=" * 80)
print("1. LOADING TEMPLATES")
print("=" * 80)

with open(
    TEMPLATE_PATH,
    "r",
    encoding="utf-8"
) as f:

    template_manifest = json.load(f)

templates = template_manifest[
    "templates"
]

template_lookup = {
    str(t["template_id"]): t
    for t in templates
}

print(
    "IBM templates:",
    len(templates)
)

# ============================================================
# 2. LOAD EPISODE PLAN
# ============================================================

episode_plan = pd.read_csv(
    EPISODE_PLAN_PATH,
    dtype=str
)

episode_plan[
    "transaction_count"
] = pd.to_numeric(
    episode_plan[
        "transaction_count"
    ],
    errors="raise"
)

# ------------------------------------------------------------
# Stratified pilot
# ------------------------------------------------------------

type_counts = (
    episode_plan[
        "typology"
    ]
    .value_counts()
    .sort_index()
)

pilot_raw = (
    type_counts
    /
    type_counts.sum()
    *
    PILOT_EPISODES
)

pilot_counts = np.floor(
    pilot_raw
).astype(int)

remaining = (
    PILOT_EPISODES
    -
    int(pilot_counts.sum())
)

fractions = (
    pilot_raw
    -
    pilot_counts
)

for typology in (
    fractions
    .sort_values(
        ascending=False
    )
    .head(remaining)
    .index
):

    pilot_counts.loc[
        typology
    ] += 1

pilot_parts = []

for typology, count in (
    pilot_counts.items()
):

    subset = episode_plan[
        episode_plan[
            "typology"
        ]
        ==
        typology
    ]

    pilot_parts.append(
        subset.sample(
            n=int(count),
            random_state=(
                SEED
                +
                sum(
                    ord(c)
                    for c in typology
                )
            )
        )
    )

pilot_plan = (
    pd.concat(
        pilot_parts,
        ignore_index=True
    )
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)

pilot_plan[
    "pilot_episode_index"
] = np.arange(
    len(pilot_plan)
)

pilot_plan[
    "synthetic_episode_id"
] = [
    f"SYN_{i:07d}"
    for i in range(
        len(pilot_plan)
    )
]

print(
    "Pilot episodes:",
    len(pilot_plan)
)

print(
    "\nPilot typology distribution:"
)

print(
    pilot_plan[
        "typology"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 3. LOAD BANK MAPPING LIBRARY
# ============================================================

print("\n" + "=" * 80)
print("2. LOADING LEGITIMATE BANK MAPPINGS")
print("=" * 80)

with open(
    MAPPING_LIBRARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    mapping_manifest = json.load(f)

mapping_library = (
    mapping_manifest[
        "mappings"
    ]
)

print(
    "Templates with mappings:",
    len(mapping_library)
)

print(
    "Mappings per template:",
    mapping_manifest[
        "mappings_per_template_requested"
    ]
)

print(
    "Validated legitimate edge coverage:",
    mapping_manifest[
        "legitimate_edge_coverage_pct"
    ]
)

# Convert node keys from JSON strings back to ints
normalized_mapping_library = {}

for template_id, maps in (
    mapping_library.items()
):

    normalized_mapping_library[
        str(template_id)
    ] = [

        {
            int(node):
                str(bank)

            for node, bank
            in mapping.items()
        }

        for mapping in maps
    ]

# ============================================================
# 4. LOAD BANK POOL
# ============================================================

print("\n" + "=" * 80)
print("3. LOADING BANK POOL")
print("=" * 80)

bank_pool = pd.read_csv(
    BANK_POOL_PATH,
    dtype=str
)

for col in [
    "legitimate_source_transactions",
    "legitimate_destination_transactions",
    "account_count",
    "source_probability",
    "destination_probability"
]:

    bank_pool[col] = pd.to_numeric(
        bank_pool[col],
        errors="coerce"
    ).fillna(0)

bank_tx_id_map = {
    str(row["normalized_bank_id"]):
        str(row["canonical_transaction_bank_id"])
    for _, row in bank_pool.iterrows()
}

bank_name_map = {
    str(row["normalized_bank_id"]):
        str(row["Bank Name"])
    for _, row in bank_pool.iterrows()
}

print(
    "Banks:",
    len(bank_pool)
)

# ============================================================
# 5. LOAD ACCOUNT REGISTRY
# ============================================================

print("\n" + "=" * 80)
print("4. LOADING ACCOUNT REGISTRY")
print("=" * 80)

registry = pd.read_csv(
    ACCOUNT_REGISTRY_PATH,
    dtype=str
)

registry[
    "normalized_bank_id"
] = registry[
    "normalized_bank_id"
].astype(str)

for col in [
    "legitimate_source_tx_count",
    "legitimate_destination_tx_count",
    "legitimate_total_activity"
]:

    registry[col] = pd.to_numeric(
        registry[col],
        errors="coerce"
    ).fillna(0).astype(int)

registry[
    "generator_eligible"
] = (
    registry[
        "generator_eligible"
    ]
    .astype(str)
    .str.lower()
    .eq("true")
)

eligible = registry[
    registry[
        "generator_eligible"
    ]
].copy()

print(
    "Eligible accounts:",
    f"{len(eligible):,}"
)

# ------------------------------------------------------------
# Role-specific pools
# ------------------------------------------------------------

account_pool = defaultdict(list)

for _, row in eligible.iterrows():

    bank = str(
        row["normalized_bank_id"]
    )

    account = str(
        row["Account Number"]
    )

    role = str(
        row["role"]
    )

    # Source-compatible
    if role in {
        "source_only",
        "both"
    }:

        account_pool[
            (bank, "source")
        ].append(account)

    # Destination-compatible
    if role in {
        "destination_only",
        "both"
    }:

        account_pool[
            (bank, "destination")
        ].append(account)

    # Both-side accounts
    if role == "both":

        account_pool[
            (bank, "both")
        ].append(account)

    # Universal bank pool
    account_pool[
        (bank, "any")
    ].append(account)

for key in account_pool:

    random.shuffle(
        account_pool[key]
    )

pool_pointer = defaultdict(int)

# ============================================================
# 6. LOAD ACCOUNT GENERATION DNA
# ============================================================

print("\n" + "=" * 80)
print("5. LOADING ACCOUNT GENERATION DNA")
print("=" * 80)

with open(
    ACCOUNT_PREFIX_PATH,
    "r",
    encoding="utf-8"
) as f:

    bank_prefix_sampler = json.load(f)

with open(
    ACCOUNT_GENERATOR_PATH,
    "r",
    encoding="utf-8"
) as f:

    account_generator_dna = json.load(f)

with open(
    ENTITY_DNA_PATH,
    "r",
    encoding="utf-8"
) as f:

    entity_dna = json.load(f)

position_models = (
    account_generator_dna[
        "position_models"
    ]
)

entity_position_models = (
    entity_dna[
        "entity_id_position_models"
    ]

)

name_prefixes = (
    entity_dna[
        "top_name_prefixes"
    ]
)

# ------------------------------------------------------------
# Position sampler
# ------------------------------------------------------------

account_position_arrays = {}

for pos, entries in (
    position_models.items()
):

    chars = [
        x["char"]
        for x in entries
    ]

    probs = np.array(
        [
            x["probability"]
            for x in entries
        ],
        dtype=float
    )

    probs /= probs.sum()

    account_position_arrays[
        int(pos)
    ] = (
        chars,
        probs
    )

# ------------------------------------------------------------
# Entity sampler
# ------------------------------------------------------------

entity_position_arrays = {}

for pos, values in (
    entity_position_models.items()
):

    chars = list(
        values.keys()
    )

    probs = np.array(
        [
            values[c]["probability"]
            for c in chars
        ],
        dtype=float
    )

    probs /= probs.sum()

    entity_position_arrays[
        int(pos)
    ] = (
        chars,
        probs
    )

name_keys = list(
    name_prefixes.keys()
)

name_probs = np.array(
    [
        name_prefixes[k]
        for k in name_keys
    ],
    dtype=float
)

name_probs /= name_probs.sum()

# ------------------------------------------------------------
# Existing IDs
# ------------------------------------------------------------

existing_bank_account_keys = set(
    (
        str(row["normalized_bank_id"]),
        str(row["Account Number"])
    )
    for _, row
    in registry.iterrows()
)

used_new_accounts = set()
used_new_entities = set()

new_account_rows = []

# ============================================================
# 7. FRESH ACCOUNT GENERATORS
# ============================================================

def generate_entity_id():

    for _ in range(10000):

        chars = []

        for pos in range(1, 10):

            char_list, probs = (
                entity_position_arrays[
                    pos
                ]
            )

            chars.append(
                np.random.choice(
                    char_list,
                    p=probs
                )
            )

        value = "".join(
            chars
        )

        if value not in (
            used_new_entities
        ):

            used_new_entities.add(
                value
            )

            return value

    raise RuntimeError(
        "Could not generate unique Entity ID."
    )


def generate_entity_name():

    prefix = np.random.choice(
        name_keys,
        p=name_probs
    )

    number = np.random.randint(
        1,
        100000
    )

    return (
        f"{prefix} #{number}"
    )


def generate_new_account(
    bank
):

    bank = str(bank)

    prefix_data = (
        bank_prefix_sampler.get(
            bank
        )
    )

    for _ in range(10000):

        # ----------------------------------------------------
        # Bank-specific 3-character prefix
        # ----------------------------------------------------
        if (
            prefix_data
            and
            prefix_data["prefixes"]
        ):

            prefixes = [
                x["prefix"]
                for x
                in prefix_data["prefixes"]
            ]

            probs = np.array(
                [
                    x["probability"]
                    for x
                    in prefix_data["prefixes"]
                ],
                dtype=float
            )

            probs /= probs.sum()

            prefix = np.random.choice(
                prefixes,
                p=probs
            )

        else:

            prefix = ""

            for pos in [1, 2, 3]:

                chars, probs = (
                    account_position_arrays[
                        pos
                    ]
                )

                prefix += (
                    np.random.choice(
                        chars,
                        p=probs
                    )
                )

        account_chars = list(
            prefix
        )

        for pos in range(
            4,
            10
        ):

            chars, probs = (
                account_position_arrays[
                    pos
                ]
            )

            account_chars.append(
                np.random.choice(
                    chars,
                    p=probs
                )
            )

        account = "".join(
            account_chars
        )

        key = (
            bank,
            account
        )

        if (
            key not in
            existing_bank_account_keys
            and
            key not in
            used_new_accounts
        ):

            used_new_accounts.add(
                key
            )

            entity_id = (
                generate_entity_id()
            )

            entity_name = (
                generate_entity_name()
            )

            new_account_rows.append(
                {
                    "Bank Name":
                        bank_name_map[
                            bank
                        ],

                    "Bank ID":
                        str(
                            bank_pool.loc[
                                bank_pool[
                                    "normalized_bank_id"
                                ].astype(str)
                                ==
                                bank,
                                "canonical_transaction_bank_id"
                            ].iloc[0]
                        ),

                    "Account Number":
                        account,

                    "Entity ID":
                        entity_id,

                    "Entity Name":
                        entity_name,

                    "normalized_bank_id":
                        bank
                }
            )

            return account

    raise RuntimeError(
        f"Unable to generate account for bank {bank}"
    )

# ============================================================
# 8. ACCOUNT ASSIGNMENT
# ============================================================

def get_account(
    bank,
    role,
    used_accounts
):

    bank = str(bank)

    role_order = []

    if role == "source":

        role_order = [
            "source",
            "both",
            "any"
        ]

    elif role == "destination":

        role_order = [
            "destination",
            "both",
            "any"
        ]

    else:

        role_order = [
            "both",
            "any"
        ]

    for pool_role in role_order:

        candidates = account_pool.get(
            (
                bank,
                pool_role
            ),
            []
        )

        if not candidates:
            continue

        start = pool_pointer[
            (
                bank,
                pool_role
            )
        ]

        for offset in range(
            len(candidates)
        ):

            idx = (
                start
                +
                offset
            ) % len(candidates)

            account = candidates[
                idx
            ]

            if account in used_accounts:
                continue

            pool_pointer[
                (
                    bank,
                    pool_role
                )
            ] = (
                idx + 1
            ) % len(candidates)

            return (
                account,
                False
            )

    # No original eligible account available.
    return (
        generate_new_account(bank),
        True
    )

# ============================================================
# 9. BEHAVIOR DNA SAMPLE
# ============================================================

print("\n" + "=" * 80)
print("6. LOADING LEGITIMATE BEHAVIOR DNA SAMPLE")
print("=" * 80)

# We only need a representative sample to reproduce the
# normal transaction-level behavior distribution.

behavior_parts = []

chunk_no = 0
rows_seen = 0

for chunk in pd.read_csv(
    TRANS_PATH,
    usecols=[
        "Timestamp",
        "Amount Received",
        "Amount Paid",
        "Payment Currency",
        "Receiving Currency",
        "Payment Format",
        "Is Laundering"
    ],
    dtype=str,
    chunksize=500_000
):

    chunk_no += 1
    rows_seen += len(chunk)

    chunk[
        "Is Laundering"
    ] = pd.to_numeric(
        chunk[
            "Is Laundering"
        ],
        errors="coerce"
    ).fillna(0).astype(int)

    legit = chunk[
        chunk[
            "Is Laundering"
        ] == 0
    ].copy()

    if len(legit) == 0:
        continue

    sample_n = min(
        10000,
        len(legit)
    )

    behavior_parts.append(
        legit.sample(
            n=sample_n,
            random_state=(
                SEED
                +
                chunk_no
            )
        )
    )

behavior = pd.concat(
    behavior_parts,
    ignore_index=True
)

behavior[
    "Timestamp"
] = pd.to_datetime(
    behavior[
        "Timestamp"
    ],
    errors="coerce"
)

behavior[
    "Amount Received"
] = pd.to_numeric(
    behavior[
        "Amount Received"
    ],
    errors="coerce"
)

behavior[
    "Amount Paid"
] = pd.to_numeric(
    behavior[
        "Amount Paid"
    ],
    errors="coerce"
)

behavior = behavior.dropna(
    subset=[
        "Timestamp",
        "Amount Received",
        "Amount Paid"
    ]
)

print(
    "Behavior sample:",
    f"{len(behavior):,}"
)

# ------------------------------------------------------------
# Sampling arrays
# ------------------------------------------------------------

time_values = (
    behavior[
        "Timestamp"
    ].tolist()
)

amount_values = (
    behavior[
        "Amount Received"
    ]
    .clip(
        lower=1e-6
    )
    .to_numpy(
        dtype=float
    )
)

payment_formats = (
    behavior[
        "Payment Format"
    ]
    .astype(str)
    .tolist()
)

payment_currencies = (
    behavior[
        "Payment Currency"
    ]
    .astype(str)
    .tolist()
)

receiving_by_payment = defaultdict(list)

for pay, recv in zip(
    behavior[
        "Payment Currency"
    ].astype(str),
    behavior[
        "Receiving Currency"
    ].astype(str)
):

    receiving_by_payment[
        pay
    ].append(
        recv
    )

dataset_min_ts = (
    behavior[
        "Timestamp"
    ].min()
)

dataset_max_ts = (
    behavior[
        "Timestamp"
    ].max()
)

dataset_max_amount = float(
    amount_values.max()
)

# ============================================================
# 10. TRANSACTION ID GENERATOR
# ============================================================

used_transaction_ids = set()

tx_alphabet = (
    string.ascii_uppercase
    +
    string.digits
)

def generate_transaction_id():

    for _ in range(1000):

        value = "".join(
            random.choices(
                tx_alphabet,
                k=16
            )
        )

        if value not in (
            used_transaction_ids
        ):

            used_transaction_ids.add(
                value
            )

            return value

    raise RuntimeError(
        "Unable to create unique transaction ID."
    )

# ============================================================
# 11. GENERATE PILOT
# ============================================================

print("\n" + "=" * 80)
print("7. GENERATING CONNECTIVITY-CONSTRAINED PILOT")
print("=" * 80)

transaction_rows = []
episode_audit = []

pattern_file = open(
    OUT_PATTERNS,
    "w",
    encoding="utf-8"
)

total_transactions = 0

for idx, episode in (
    pilot_plan.iterrows()
):

    template_id = str(
        episode[
            "source_template_id"
        ]
    )

    typology = str(
        episode[
            "typology"
        ]
    )

    synthetic_episode_id = str(
        episode[
            "synthetic_episode_id"
        ]
    )

    template = template_lookup[
        template_id
    ]

    valid_mappings = (
        normalized_mapping_library[
            template_id
        ]
    )

    if not valid_mappings:

        raise RuntimeError(
            f"No valid bank mappings for "
            f"template {template_id}"
        )

    # --------------------------------------------------------
    # Choose one validated bank mapping.
    # --------------------------------------------------------
    node_bank_mapping = random.choice(
        valid_mappings
    )

    # --------------------------------------------------------
    # Node degree / role
    # --------------------------------------------------------
    in_degree = defaultdict(int)
    out_degree = defaultdict(int)

    for edge in template[
        "edges"
    ]:

        source = int(
            edge["source_node"]
        )

        destination = int(
            edge["destination_node"]
        )

        out_degree[source] += 1
        in_degree[destination] += 1

    node_roles = {}

    for node in template[
        "nodes"
    ]:

        node_id = int(
            node["node_id"]
        )

        if (
            in_degree[node_id] > 0
            and
            out_degree[node_id] > 0
        ):

            role = "both"

        elif out_degree[node_id] > 0:

            role = "source"

        else:

            role = "destination"

        node_roles[
            node_id
        ] = role

    # --------------------------------------------------------
    # Account mapping
    # --------------------------------------------------------
    node_account_mapping = {}

    used_accounts_episode = set()

    fresh_accounts_episode = 0

    for node_id in sorted(
        node_roles
    ):

        bank = str(
            node_bank_mapping[
                node_id
            ]
        )

        role = node_roles[
            node_id
        ]

        force_new = (
            np.random.random()
            <
            NEW_ACCOUNT_RATE
        )

        if force_new:

            account = (
                generate_new_account(
                    bank
                )
            )

            was_new = True

        else:

            account, was_new = (
                get_account(
                    bank,
                    role,
                    used_accounts_episode
                )
            )

        if account in (
            used_accounts_episode
        ):

            raise RuntimeError(
                "Duplicate account inside episode."
            )

        node_account_mapping[
            node_id
        ] = account

        used_accounts_episode.add(
            account
        )

        if was_new:

            fresh_accounts_episode += 1

    # --------------------------------------------------------
    # Anchor time
    # --------------------------------------------------------
    template_span = float(
        template[
            "episode_span_minutes"
        ]
    )

    anchor = random.choice(
        time_values
    )

    latest_anchor = (
        dataset_max_ts
        -
        pd.to_timedelta(
            template_span,
            unit="m"
        )
    )

    if anchor > latest_anchor:

        anchor = latest_anchor

    if anchor < dataset_min_ts:

        anchor = dataset_min_ts

    # --------------------------------------------------------
    # Episode amount scaling
    #
    # Target median follows REAL legitimate transaction
    # distribution.
    # --------------------------------------------------------
    target_median = float(
        random.choice(
            amount_values
        )
    )

    template_median = max(
        1e-9,
        float(
            template[
                "median_amount"
            ]
        )
    )

    amount_scale = (
        target_median
        /
        template_median
    )

    episode_edges = set()
    episode_amounts = []

    pattern_lines = []

    # --------------------------------------------------------
    # Transactions
    # --------------------------------------------------------
    for tx in template[
        "transactions"
    ]:

        source_node = int(
            tx["source_node"]
        )

        destination_node = int(
            tx["destination_node"]
        )

        source_bank_norm = str(
            node_bank_mapping[
                source_node
            ]
        )

        destination_bank_norm = str(
            node_bank_mapping[
                destination_node
            ]
        )

        source_account = (
            node_account_mapping[
                source_node
            ]
        )

        destination_account = (
            node_account_mapping[
                destination_node
            ]
        )

        # ----------------------------------------------------
        # Validated mapped bank edge
        # ----------------------------------------------------
        edge = (
            source_bank_norm,
            destination_bank_norm
        )

        episode_edges_check = (
            source_bank_norm,
            source_account,
            destination_bank_norm,
            destination_account
        )

        episode_edges_check = (
            source_bank_norm,
            source_account,
            destination_bank_norm,
            destination_account
        )

        episode_edges_check = tuple(
            episode_edges_check
        )

        episode_edges_check

        episode_edges_key = (
            source_bank_norm,
            source_account,
            destination_bank_norm,
            destination_account
        )

        episode_edges_set_key = (
            source_bank_norm,
            source_account,
            destination_bank_norm,
            destination_account
        )

        episode_edges_key = (
            source_bank_norm,
            source_account,
            destination_bank_norm,
            destination_account
        )

        # Use full account-level directed edge for audit.
        episode_edges = episode_edges_set_key

        episode_edges

        episode_edge_set = (
            episode_audit
        )

        # Actually store it in local set:
        # ----------------------------------------------------
        episode_edges_unique = (
            episode_edges_set_key
        )

        # We keep a dedicated local set below.
        # ----------------------------------------------------

        # ----------------------------------------------------
        # Amount
        # ----------------------------------------------------
        template_amount = float(
            tx[
                "amount_received"
            ]
        )

        amount_received = max(
            1e-6,
            template_amount
            *
            amount_scale
        )

        original_received = max(
            1e-12,
            template_amount
        )

        original_paid = float(
            tx[
                "amount_paid"
            ]
        )

        paid_ratio = (
            original_paid
            /
            original_received
        )

        amount_paid = max(
            1e-6,
            amount_received
            *
            paid_ratio
        )

        if (
            amount_received
            >
            dataset_max_amount
        ):

            correction = (
                dataset_max_amount
                /
                amount_received
            )

            amount_received *= correction
            amount_paid *= correction

        # ----------------------------------------------------
        # Timestamp
        # ----------------------------------------------------
        timestamp = (
            anchor
            +
            pd.to_timedelta(
                float(
                    tx[
                        "relative_minutes"
                    ]
                ),
                unit="m"
            )
        )

        # ----------------------------------------------------
        # Real legitimate transaction behavior
        # ----------------------------------------------------
        payment_format = random.choice(
            payment_formats
        )

        payment_currency = random.choice(
            payment_currencies
        )

        recv_candidates = (
            receiving_by_payment.get(
                payment_currency
            )
        )

        if (
            recv_candidates
            and
            len(recv_candidates) > 0
        ):

            receiving_currency = (
                random.choice(
                    recv_candidates
                )
            )

        else:

            receiving_currency = (
                payment_currency
            )

        # ----------------------------------------------------
        # Canonical transaction bank IDs
        # ----------------------------------------------------
        source_bank_tx = (
            bank_tx_id_map[
                source_bank_norm
            ]
        )

        destination_bank_tx = (
            bank_tx_id_map[
                destination_bank_norm
            ]
        )

        # ----------------------------------------------------
        # Transaction ID
        # ----------------------------------------------------
        transaction_id = (
            generate_transaction_id()
        )

        transaction_rows.append(
            {
                "transaction_id":
                    transaction_id,

                "Timestamp":
                    timestamp.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    ),

                "From Bank":
                    source_bank_tx,

                "Account":
                    source_account,

                "To Bank":
                    destination_bank_tx,

                "Account.1":
                    destination_account,

                "Amount Received":
                    float(
                        amount_received
                    ),

                "Receiving Currency":
                    receiving_currency,

                "Amount Paid":
                    float(
                        amount_paid
                    ),

                "Payment Currency":
                    payment_currency,

                "Payment Format":
                    payment_format,

                "Is Laundering":
                    1,

                "is_synthetic":
                    1,

                "synthetic_episode_id":
                    synthetic_episode_id,

                "typology":
                    typology,

                "source_template_id":
                    template_id,

                "template_source_order":
                    int(
                        tx[
                            "source_order"
                        ]
                    )
            }
        )

        # ----------------------------------------------------
        # IBM-style pattern transaction line
        # ----------------------------------------------------
        pattern_lines.append(
            ",".join(
                [
                    timestamp.strftime(
                        "%Y/%m/%d %H:%M"
                    ),

                    source_bank_tx,

                    source_account,

                    destination_bank_tx,

                    destination_account,

                    f"{amount_received:.6f}",

                    receiving_currency,

                    f"{amount_paid:.6f}",

                    payment_currency,

                    payment_format,

                    "1"
                ]
            )
        )

        episode_amounts.append(
            amount_received
        )

    # --------------------------------------------------------
    # Write Patterns.txt block
    # --------------------------------------------------------
    pattern_file.write(
        f'id="{synthetic_episode_id}"\n'
    )

    pattern_file.write(
        f"BEGIN LAUNDERING ATTEMPT - "
        f"{typology}: "
        f"Synthetic instance based on "
        f"IBM {typology} template\n"
    )

    for pattern_line in pattern_lines:

        pattern_file.write(
            pattern_line + "\n"
        )

    pattern_file.write(
        f"END LAUNDERING ATTEMPT - "
        f"{typology}\n"
    )

    # --------------------------------------------------------
    # Episode audit
    # --------------------------------------------------------
    #
    # Reconstruct all account-level edges directly from the
    # generated transaction records for this episode.
    # --------------------------------------------------------
    episode_transaction_rows = transaction_rows[
        -len(
            template[
                "transactions"
            ]
        ):
    ]

    full_edges = {
        (
            row["From Bank"],
            row["Account"],
            row["To Bank"],
            row["Account.1"]
        )
        for row
        in episode_transaction_rows
    }

    episode_audit.append(
        {
            "synthetic_episode_id":
                synthetic_episode_id,

            "typology":
                typology,

            "source_template_id":
                template_id,

            "transaction_count":
                len(
                    episode_transaction_rows
                ),

            "template_node_count":
                int(
                    len(
                        template[
                            "nodes"
                        ]
                    )
                ),

            "generated_unique_banks":
                len(
                    set(
                        node_bank_mapping.values()
                    )
                ),

            "fresh_accounts":
                fresh_accounts_episode,

            "unique_accounts":
                len(
                    node_account_mapping
                ),

            "unique_directed_edges":
                len(
                    full_edges
                ),

            "duplicate_directed_edges":
                len(
                    episode_transaction_rows
                )
                -
                len(
                    full_edges
                ),

            "template_span_minutes":
                template_span,

            "generated_median_amount":
                float(
                    np.median(
                        episode_amounts
                    )
                )
        }
    )

    total_transactions += (
        len(
            template[
                "transactions"
            ]
        )
    )

    if (
        (idx + 1) % 500
        == 0
    ):

        print(
            f"Episodes: {idx + 1:,}"
            f" | Transactions: "
            f"{total_transactions:,}"
            f" | New accounts: "
            f"{len(new_account_rows):,}"
        )

pattern_file.close()

# ============================================================
# 12. DATAFRAMES
# ============================================================

synthetic_df = pd.DataFrame(
    transaction_rows
)

new_accounts_df = pd.DataFrame(
    new_account_rows
)

audit_df = pd.DataFrame(
    episode_audit
)

# ============================================================
# 13. SAVE FILES
# ============================================================

print("\n" + "=" * 80)
print("8. SAVING PILOT V2")
print("=" * 80)

synthetic_df.to_csv(
    OUT_TRANS,
    index=False
)

new_accounts_df.to_csv(
    OUT_ACCOUNTS,
    index=False
)

audit_df.to_csv(
    OUT_AUDIT,
    index=False
)

# ============================================================
# 14. GLOBAL EDGE CHECK
# ============================================================

global_edges = (
    synthetic_df[
        [
            "From Bank",
            "Account",
            "To Bank",
            "Account.1"
        ]
    ]
    .astype(str)
    .agg(
        "|".join,
        axis=1
    )
)

# ============================================================
# 15. LEGITIMATE BANK PAIR SET
# ============================================================

legit_pair_path = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_pairs.csv"
)

legit_pairs = pd.read_csv(
    legit_pair_path,
    dtype=str
)

legit_pairs[
    "src_norm"
] = (
    legit_pairs[
        "From Bank"
    ]
    .map(
        lambda x: str(
            int(str(x))
        )
    )
)

legit_pairs[
    "dst_norm"
] = (
    legit_pairs[
        "To Bank"
    ]
    .map(
        lambda x: str(
            int(str(x))
        )
    )
)

legit_pairs = legit_pairs[
    pd.to_numeric(
        legit_pairs[
            "legitimate_count"
        ],
        errors="coerce"
    ).fillna(0)
    > 0
]

legit_pair_set = set(
    zip(
        legit_pairs[
            "src_norm"
        ],
        legit_pairs[
            "dst_norm"
        ]
    )
)

synthetic_df[
    "from_norm"
] = synthetic_df[
    "From Bank"
].map(
    lambda x: str(
        int(str(x))
    )
)

synthetic_df[
    "to_norm"
] = synthetic_df[
    "To Bank"
].map(
    lambda x: str(
        int(str(x))
    )
)

pair_flags = [
    (
        (src, dst)
        in legit_pair_set
    )
    for src, dst in zip(
        synthetic_df[
            "from_norm"
        ],
        synthetic_df[
            "to_norm"
        ]
    )
]

# ============================================================
# 16. ACCOUNT-BANK VALIDATION
# ============================================================

# Original registry account -> normalized bank
account_bank_map = {
    str(row["Account Number"]):
        str(row["normalized_bank_id"])
    for _, row
    in registry.iterrows()
}

# New account -> normalized bank
for _, row in new_accounts_df.iterrows():

    account_bank_map[
        str(row[
            "Account Number"
        ])
    ] = str(
        row[
            "normalized_bank_id"
        ]
    )

source_consistency = []
destination_consistency = []

for _, row in synthetic_df.iterrows():

    account = str(
        row["Account"]
    )

    account2 = str(
        row["Account.1"]
    )

    source_bank = str(
        row["from_norm"]
    )

    destination_bank = str(
        row["to_norm"]
    )

    if account in account_bank_map:

        source_consistency.append(
            account_bank_map[
                account
            ]
            ==
            source_bank
        )

    else:

        source_consistency.append(
            False
        )

    if account2 in account_bank_map:

        destination_consistency.append(
            account_bank_map[
                account2
            ]
            ==
            destination_bank
        )

    else:

        destination_consistency.append(
            False
        )

# ============================================================
# 17. ACCOUNT FORMAT VALIDATION
# ============================================================

new_account_format_ok = (
    new_accounts_df[
        "Account Number"
    ]
    .astype(str)
    .str.fullmatch(
        r"[A-Z0-9]{9}"
    )
)

# ============================================================
# 18. PATTERN FILE VALIDATION
# ============================================================

with open(
    OUT_PATTERNS,
    "r",
    encoding="utf-8"
) as f:

    pattern_lines = [
        x.rstrip("\n")
        for x in f
    ]

pattern_ids = [
    x for x in pattern_lines
    if x.startswith(
        'id="'
    )
]

pattern_begins = [
    x for x in pattern_lines
    if x.startswith(
        "BEGIN LAUNDERING ATTEMPT"
    )
]

pattern_ends = [
    x for x in pattern_lines
    if x.startswith(
        "END LAUNDERING ATTEMPT"
    )
]

# ============================================================
# 19. TYPOLOGY AUDIT
# ============================================================

typology_transactions = (
    synthetic_df[
        "typology"
    ]
    .value_counts()
    .sort_index()
)

typology_episodes = (
    audit_df[
        "typology"
    ]
    .value_counts()
    .sort_index()
)

# ============================================================
# 20. SAVE MANIFEST
# ============================================================

manifest = {

    "dataset":
        "IBM HI-Small",

    "generator_version":
        "connectivity_constrained_v2",

    "seed":
        SEED,

    "pilot_episodes":
        int(PILOT_EPISODES),

    "generated_transactions":
        int(len(synthetic_df)),

    "new_accounts":
        int(len(new_accounts_df)),

    "new_account_rate":
        float(NEW_ACCOUNT_RATE),

    "ibm_templates_available":
        int(len(templates)),

    "ibm_templates_used":
        int(
            pilot_plan[
                "source_template_id"
            ].nunique()
        ),

    "valid_bank_mappings_available":
        int(
            mapping_manifest[
                "total_valid_mappings"
            ]
        ),

    "typology_episode_counts":
        {
            str(k): int(v)
            for k, v
            in typology_episodes.items()
        },

    "typology_transaction_counts":
        {
            str(k): int(v)
            for k, v
            in typology_transactions.items()
        },

    "transaction_weighted_legitimate_bank_pair_pct":
        float(
            np.mean(
                pair_flags
            )
            *
            100
        ),

    "global_unique_directed_edges":
        int(
            global_edges.nunique()
        ),

    "global_duplicate_directed_edges":
        int(
            len(global_edges)
            -
            global_edges.nunique()
        ),

    "source_account_bank_consistency_pct":
        float(
            np.mean(
                source_consistency
            )
            *
            100
        ),

    "destination_account_bank_consistency_pct":
        float(
            np.mean(
                destination_consistency
            )
            *
            100
        ),

    "new_account_format_violations":
        int(
            (~new_account_format_ok).sum()
        ),

    "pattern_id_lines":
        int(
            len(pattern_ids)
        ),

    "pattern_begin_lines":
        int(
            len(pattern_begins)
        ),

    "pattern_end_lines":
        int(
            len(pattern_ends)
        ),

    "outputs": {
        "transactions":
            OUT_TRANS,

        "new_accounts":
            OUT_ACCOUNTS,

        "patterns":
            OUT_PATTERNS,

        "episode_audit":
            OUT_AUDIT
    }
}

with open(
    OUT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ============================================================
# 21. FINAL VALIDATION OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("✅ CELL 18 COMPLETE — PILOT V2")
print("=" * 100)

print(
    "\nTransactions:",
    f"{len(synthetic_df):,}"
)

print(
    "Episodes:",
    f"{len(audit_df):,}"
)

print(
    "New accounts:",
    f"{len(new_accounts_df):,}"
)

print(
    "Unique transaction IDs:",
    synthetic_df[
        "transaction_id"
    ].nunique()
)

print(
    "Transaction ID duplicates:",
    len(synthetic_df)
    -
    synthetic_df[
        "transaction_id"
    ].nunique()
)

print(
    "\nTransaction-weighted legitimate bank-pair coverage:",
    f"{np.mean(pair_flags) * 100:.4f}%"
)

print(
    "Global duplicate directed edges:",
    int(
        len(global_edges)
        -
        global_edges.nunique()
    )
)

print(
    "Source account-bank consistency:",
    f"{np.mean(source_consistency) * 100:.4f}%"
)

print(
    "Destination account-bank consistency:",
    f"{np.mean(destination_consistency) * 100:.4f}%"
)

print(
    "New-account format violations:",
    int(
        (~new_account_format_ok).sum()
    )
)

print(
    "\nPatterns.txt IDs:",
    len(pattern_ids)
)

print(
    "Patterns.txt BEGIN:",
    len(pattern_begins)
)

print(
    "Patterns.txt END:",
    len(pattern_ends)
)

print(
    "Pattern marker counts correct:",
    (
        len(pattern_ids)
        ==
        PILOT_EPISODES
        and
        len(pattern_begins)
        ==
        PILOT_EPISODES
        and
        len(pattern_ends)
        ==
        PILOT_EPISODES
    )
)

print("\nTypology transactions:")

print(
    typology_transactions.to_string()
)

print("\nSaved:")

print(
    OUT_TRANS
)

print(
    OUT_ACCOUNTS
)

print(
    OUT_PATTERNS
)

print(
    OUT_AUDIT
)

print(
    OUT_MANIFEST
)

print("\nORIGINAL HI-SMALL FILES WERE NOT MODIFIED.")

print("=" * 100)

CELL 18 — CONNECTIVITY-CONSTRAINED SYNTHETIC PILOT V2

1. LOADING TEMPLATES
IBM templates: 370
Pilot episodes: 5000

Pilot typology distribution:
typology
BIPARTITE         662
CYCLE             729
FAN-IN            541
FAN-OUT           649
GATHER-SCATTER    689
RANDOM            554
SCATTER-GATHER    595
STACK             581

2. LOADING LEGITIMATE BANK MAPPINGS
Templates with mappings: 370
Mappings per template: 5
Validated legitimate edge coverage: 100.0

3. LOADING BANK POOL
Banks: 30470

4. LOADING ACCOUNT REGISTRY
Eligible accounts: 508,731

5. LOADING ACCOUNT GENERATION DNA

6. LOADING LEGITIMATE BEHAVIOR DNA SAMPLE
Behavior sample: 110,000

7. GENERATING CONNECTIVITY-CONSTRAINED PILOT
Episodes: 500 | Transactions: 4,167 | New accounts: 369
Episodes: 1,000 | Transactions: 8,626 | New accounts: 781
Episodes: 1,500 | Transactions: 12,878 | New accounts: 1,175
Episodes: 2,000 | Transactions: 17,241 | New accounts: 1,639
Episodes: 2,500 | Transactions: 21,692 | New accounts: 2,039

In [25]:
# ============================================================
# CELL 19 — FULL PILOT INTEGRITY + PATTERN FIDELITY AUDIT
# ============================================================
#
# PURPOSE
# -------
# Validate the complete V2 pilot before scaling to ~5.07M
# synthetic laundering transactions.
#
# Checks:
#   1. Exact episode/template transaction counts
#   2. Exact node/edge morphology
#   3. Exact timestamp-relative structure
#   4. Bank-pair legitimacy
#   5. Correct BANK + ACCOUNT identity mapping
#   6. New-account uniqueness/format
#   7. Entity uniqueness
#   8. Pattern.txt integrity
#   9. Amount integrity
#  10. Currency/payment-format integrity
#
# NO DATA IS GENERATED.
# ============================================================

import os
import json
from collections import defaultdict

import numpy as np
import pandas as pd

# ============================================================
# PATHS
# ============================================================

WORK_DIR = (
    "/kaggle/working/"
    "AML_GNN_ONLY/data"
)

PILOT_TRANS_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_V2_transactions.csv"
)

PILOT_ACCOUNTS_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_V2_new_accounts.csv"
)

PILOT_PATTERNS_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_V2_Patterns.txt"
)

PILOT_AUDIT_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_PILOT_V2_episode_audit.csv"
)

TEMPLATE_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_templates.json"
)

BANK_PAIR_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_pairs.csv"
)

ACCOUNT_REGISTRY_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_generator_account_registry.csv"
)

# ============================================================
# 1. LOAD
# ============================================================

print("=" * 100)
print("CELL 19 — FULL PILOT INTEGRITY + PATTERN FIDELITY AUDIT")
print("=" * 100)

pilot = pd.read_csv(
    PILOT_TRANS_PATH,
    dtype=str
)

new_accounts = pd.read_csv(
    PILOT_ACCOUNTS_PATH,
    dtype=str
)

episode_audit = pd.read_csv(
    PILOT_AUDIT_PATH,
    dtype=str
)

with open(
    TEMPLATE_PATH,
    "r",
    encoding="utf-8"
) as f:
    template_manifest = json.load(f)

templates = template_manifest[
    "templates"
]

template_lookup = {
    str(t["template_id"]): t
    for t in templates
}

print(
    "\nPilot transactions:",
    f"{len(pilot):,}"
)

print(
    "Pilot episodes:",
    f"{pilot['synthetic_episode_id'].nunique():,}"
)

print(
    "New accounts:",
    f"{len(new_accounts):,}"
)

print(
    "Templates:",
    len(templates)
)

# ============================================================
# 2. BASIC INTEGRITY
# ============================================================

print("\n" + "=" * 80)
print("1. BASIC TRANSACTION INTEGRITY")
print("=" * 80)

print(
    "Null values:",
    int(pilot.isna().sum().sum())
)

numeric_amount_received = pd.to_numeric(
    pilot["Amount Received"],
    errors="coerce"
)

numeric_amount_paid = pd.to_numeric(
    pilot["Amount Paid"],
    errors="coerce"
)

print(
    "Invalid Amount Received:",
    int(
        numeric_amount_received.isna().sum()
    )
)

print(
    "Invalid Amount Paid:",
    int(
        numeric_amount_paid.isna().sum()
    )
)

print(
    "Non-positive Amount Received:",
    int(
        (numeric_amount_received <= 0).sum()
    )
)

print(
    "Non-positive Amount Paid:",
    int(
        (numeric_amount_paid <= 0).sum()
    )
)

print(
    "Non-laundering rows:",
    int(
        (
            pd.to_numeric(
                pilot["Is Laundering"],
                errors="coerce"
            )
            != 1
        ).sum()
    )
)

# ============================================================
# 3. TRANSACTION ID INTEGRITY
# ============================================================

print("\n" + "=" * 80)
print("2. TRANSACTION ID INTEGRITY")
print("=" * 80)

unique_tx_ids = (
    pilot["transaction_id"]
    .nunique()
)

print(
    "Unique transaction IDs:",
    unique_tx_ids
)

print(
    "Duplicate transaction IDs:",
    len(pilot)
    -
    unique_tx_ids
)

# ============================================================
# 4. LOAD LEGITIMATE BANK PAIRS
# ============================================================

legit_pairs = pd.read_csv(
    BANK_PAIR_PATH,
    dtype=str
)

def norm_bank(x):

    x = str(x).strip()

    try:
        return str(int(x))
    except Exception:
        return x.lstrip("0") or "0"

legit_pairs[
    "src_norm"
] = legit_pairs[
    "From Bank"
].map(norm_bank)

legit_pairs[
    "dst_norm"
] = legit_pairs[
    "To Bank"
].map(norm_bank)

legit_pairs[
    "legitimate_count"
] = pd.to_numeric(
    legit_pairs[
        "legitimate_count"
    ],
    errors="coerce"
).fillna(0).astype(int)

legit_pair_set = set(
    zip(
        legit_pairs[
            legit_pairs[
                "legitimate_count"
            ] > 0
        ]["src_norm"],

        legit_pairs[
            legit_pairs[
                "legitimate_count"
            ] > 0
        ]["dst_norm"]
    )
)

pilot[
    "from_norm"
] = pilot[
    "From Bank"
].map(norm_bank)

pilot[
    "to_norm"
] = pilot[
    "To Bank"
].map(norm_bank)

# ============================================================
# 5. BANK-PAIR INTEGRITY
# ============================================================

print("\n" + "=" * 80)
print("3. LEGITIMATE BANK-PAIR INTEGRITY")
print("=" * 80)

pair_flags = [
    (
        (src, dst)
        in legit_pair_set
    )
    for src, dst in zip(
        pilot["from_norm"],
        pilot["to_norm"]
    )
]

print(
    "Transaction rows checked:",
    f"{len(pair_flags):,}"
)

print(
    "Legitimate bank-pair rows:",
    sum(pair_flags)
)

print(
    "Invalid bank-pair rows:",
    len(pair_flags) - sum(pair_flags)
)

print(
    "Bank-pair legitimate coverage:",
    f"{np.mean(pair_flags) * 100:.6f}%"
)

# ============================================================
# 6. ACCOUNT REGISTRY
# ============================================================

print("\n" + "=" * 80)
print("4. BANK + ACCOUNT INTEGRITY")
print("=" * 80)

registry = pd.read_csv(
    ACCOUNT_REGISTRY_PATH,
    dtype=str
)

registry[
    "normalized_bank_id"
] = registry[
    "normalized_bank_id"
].map(norm_bank)

# Exact key:
#     normalized_bank_id + Account Number
original_account_keys = set(
    (
        str(row["normalized_bank_id"]),
        str(row["Account Number"])
    )
    for _, row in registry.iterrows()
)

# Add new accounts
new_accounts[
    "normalized_bank_id"
] = new_accounts[
    "normalized_bank_id"
].map(norm_bank)

new_account_keys = set(
    (
        str(row["normalized_bank_id"]),
        str(row["Account Number"])
    )
    for _, row
    in new_accounts.iterrows()
)

all_valid_account_keys = (
    original_account_keys
    |
    new_account_keys
)

# ------------------------------------------------------------
# Source account checks
# ------------------------------------------------------------

source_keys = [
    (
        str(row["from_norm"]),
        str(row["Account"])
    )
    for _, row
    in pilot.iterrows()
]

destination_keys = [
    (
        str(row["to_norm"]),
        str(row["Account.1"])
    )
    for _, row
    in pilot.iterrows()
]

source_valid = [
    x in all_valid_account_keys
    for x in source_keys
]

destination_valid = [
    x in all_valid_account_keys
    for x in destination_keys
]

print(
    "Source bank+account valid rows:",
    sum(source_valid)
)

print(
    "Source bank+account invalid rows:",
    len(source_valid) - sum(source_valid)
)

print(
    "Source bank+account coverage:",
    f"{np.mean(source_valid) * 100:.6f}%"
)

print(
    "\nDestination bank+account valid rows:",
    sum(destination_valid)
)

print(
    "Destination bank+account invalid rows:",
    len(destination_valid) - sum(destination_valid)
)

print(
    "Destination bank+account coverage:",
    f"{np.mean(destination_valid) * 100:.6f}%"
)

# ============================================================
# 7. INVESTIGATE RAW ACCOUNT-NUMBER COLLISIONS
# ============================================================

print("\n" + "=" * 80)
print("5. GLOBAL ACCOUNT-NUMBER COLLISION CHECK")
print("=" * 80)

account_to_banks = defaultdict(set)

for bank, account in original_account_keys:

    account_to_banks[
        account
    ].add(bank)

collision_accounts = {
    account: banks
    for account, banks
    in account_to_banks.items()
    if len(banks) > 1
}

print(
    "Account numbers used by >1 bank in original data:",
    len(collision_accounts)
)

if collision_accounts:

    print(
        "\nExamples:"
    )

    for account, banks in list(
        collision_accounts.items()
    )[:20]:

        print(
            account,
            "->",
            sorted(banks)
        )

# ============================================================
# 8. NEW ACCOUNT FORMAT
# ============================================================

print("\n" + "=" * 80)
print("6. NEW ACCOUNT INTEGRITY")
print("=" * 80)

new_account_series = (
    new_accounts[
        "Account Number"
    ].astype(str)
)

valid_format = (
    new_account_series
    .str.fullmatch(
        r"[A-Z0-9]{9}"
    )
)

print(
    "New accounts:",
    len(new_accounts)
)

print(
    "Format-valid:",
    int(valid_format.sum())
)

print(
    "Format violations:",
    int((~valid_format).sum())
)

# Unique within bank
new_bank_account_keys = [
    (
        str(row["normalized_bank_id"]),
        str(row["Account Number"])
    )
    for _, row
    in new_accounts.iterrows()
]

print(
    "Unique bank+account new IDs:",
    len(set(new_bank_account_keys))
)

print(
    "Duplicate new bank+account IDs:",
    len(new_bank_account_keys)
    -
    len(set(new_bank_account_keys))
)

# Global collision with originals
new_vs_original = (
    set(new_bank_account_keys)
    &
    original_account_keys
)

print(
    "New accounts colliding with original bank+account:",
    len(new_vs_original)
)

# ============================================================
# 9. NEW ACCOUNT ENTITY INTEGRITY
# ============================================================

print("\n" + "=" * 80)
print("7. NEW ENTITY INTEGRITY")
print("=" * 80)

entity_ids = (
    new_accounts[
        "Entity ID"
    ].astype(str)
)

entity_names = (
    new_accounts[
        "Entity Name"
    ].astype(str)
)

print(
    "New Entity IDs:",
    entity_ids.nunique()
)

print(
    "Duplicate Entity IDs:",
    len(entity_ids)
    -
    entity_ids.nunique()
)

print(
    "Valid 9-char Entity IDs:",
    int(
        entity_ids
        .str.fullmatch(
            r"[A-Z0-9]{9}"
        )
        .sum()
    )
)

print(
    "Entity name examples:"
)

print(
    entity_names.head(10).to_string(
        index=False
    )
)

# ============================================================
# 10. LOAD EPISODE AUDIT
# ============================================================

episode_audit[
    "transaction_count"
] = pd.to_numeric(
    episode_audit[
        "transaction_count"
    ],
    errors="coerce"
).astype(int)

# ============================================================
# 11. EPISODE ↔ TEMPLATE FIDELITY
# ============================================================

print("\n" + "=" * 80)
print("8. EPISODE / TEMPLATE FIDELITY")
print("=" * 80)

template_count_mismatch = 0
template_node_mismatch = 0
template_edge_mismatch = 0
template_degree_mismatch = 0
timestamp_mismatch = 0

# Make generated transactions accessible by episode
grouped = {
    str(ep_id): g.reset_index(drop=True)
    for ep_id, g
    in pilot.groupby(
        "synthetic_episode_id",
        sort=False
    )
}

for ep_id, g in (
    pilot.groupby(
        "synthetic_episode_id",
        sort=False
    )
):

    ep_id = str(ep_id)

    # Template ID
    template_id = str(
        g["source_template_id"]
        .iloc[0]
    )

    template = template_lookup[
        template_id
    ]

    generated_n = len(g)

    expected_n = int(
        template[
            "transaction_count"
        ]
    )

    if generated_n != expected_n:

        template_count_mismatch += 1

    # --------------------------------------------------------
    # Generated node graph
    # --------------------------------------------------------
    account_nodes = set(
        g[
            "Account"
        ].astype(str)
    ) | set(
        g[
            "Account.1"
        ].astype(str)
    )

    expected_nodes = int(
        template[
            "node_count"
        ]
    )

    if len(account_nodes) != expected_nodes:

        template_node_mismatch += 1

    # --------------------------------------------------------
    # Generated edges
    # --------------------------------------------------------
    generated_edges = {
        (
            str(row["Account"]),
            str(row["Account.1"])
        )
        for _, row
        in g.iterrows()
    }

    expected_edges = len(
        template[
            "edges"
        ]
    )

    if len(generated_edges) != expected_edges:

        template_edge_mismatch += 1

    # --------------------------------------------------------
    # Degree sequence
    # --------------------------------------------------------
    generated_in = defaultdict(int)
    generated_out = defaultdict(int)

    for _, row in g.iterrows():

        s = str(
            row["Account"]
        )

        d = str(
            row["Account.1"]
        )

        generated_out[s] += 1
        generated_in[d] += 1

    generated_degree_signature = sorted(
        [
            (
                generated_in[n],
                generated_out[n]
            )
            for n in account_nodes
        ]
    )

    # Template degree signature
    expected_in = defaultdict(int)
    expected_out = defaultdict(int)

    for edge in template[
        "edges"
    ]:

        s = int(
            edge["source_node"]
        )

        d = int(
            edge["destination_node"]
        )

        expected_out[s] += 1
        expected_in[d] += 1

    template_nodes = [
        int(n["node_id"])
        for n in template[
            "nodes"
        ]
    ]

    expected_degree_signature = sorted(
        [
            (
                expected_in[n],
                expected_out[n]
            )
            for n in template_nodes
        ]
    )

    if (
        generated_degree_signature
        !=
        expected_degree_signature
    ):

        template_degree_mismatch += 1

    # --------------------------------------------------------
    # Timestamp fidelity
    # --------------------------------------------------------
    generated_ts = pd.to_datetime(
        g["Timestamp"],
        errors="coerce"
    )

    if generated_ts.isna().any():

        timestamp_mismatch += 1

        continue

    anchor = generated_ts.min()

    generated_relative = (
        (
            generated_ts
            -
            anchor
        )
        .dt.total_seconds()
        /
        60.0
    )

    expected_relative = np.array(
        [
            float(
                tx[
                    "relative_minutes"
                ]
            )
            for tx in sorted(
                template[
                    "transactions"
                ],
                key=lambda x:
                    int(
                        x["source_order"]
                    )
            )
        ]
    )

    # Generated row order is source order
    generated_relative_array = (
        generated_relative
        .to_numpy()
    )

    if len(
        generated_relative_array
    ) != len(
        expected_relative
    ):

        timestamp_mismatch += 1

    else:

        max_error = np.max(
            np.abs(
                generated_relative_array
                -
                expected_relative
            )
        )

        if max_error > 1e-9:

            timestamp_mismatch += 1

print(
    "Episode transaction-count mismatches:",
    template_count_mismatch
)

print(
    "Episode node-count mismatches:",
    template_node_mismatch
)

print(
    "Episode edge-count mismatches:",
    template_edge_mismatch
)

print(
    "Episode degree-signature mismatches:",
    template_degree_mismatch
)

print(
    "Episode timestamp mismatches:",
    timestamp_mismatch
)

# ============================================================
# 12. DUPLICATE DIRECTED EDGES
# ============================================================

print("\n" + "=" * 80)
print("9. DIRECTED EDGE UNIQUENESS")
print("=" * 80)

edge_strings = (
    pilot[
        [
            "from_norm",
            "Account",
            "to_norm",
            "Account.1"
        ]
    ]
    .astype(str)
    .agg(
        "|".join,
        axis=1
    )
)

print(
    "Total edges:",
    len(edge_strings)
)

print(
    "Unique edges:",
    edge_strings.nunique()
)

print(
    "Duplicate edges:",
    len(edge_strings)
    -
    edge_strings.nunique()
)

# ============================================================
# 13. AMOUNT INTEGRITY
# ============================================================

print("\n" + "=" * 80)
print("10. AMOUNT INTEGRITY")
print("=" * 80)

print(
    "Amount Received quantiles:"
)

print(
    numeric_amount_received
    .quantile(
        [
            0.001,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
    .to_string()
)

print(
    "\nAmount Paid quantiles:"
)

print(
    numeric_amount_paid
    .quantile(
        [
            0.001,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
    .to_string()
)

print(
    "\nAmount Received > 1e12:",
    int(
        (numeric_amount_received > 1e12).sum()
    )
)

print(
    "Amount Received < 1e-6:",
    int(
        (numeric_amount_received < 1e-6).sum()
    )
)

# ============================================================
# 14. PAYMENT BEHAVIOR
# ============================================================

print("\n" + "=" * 80)
print("11. PAYMENT BEHAVIOR")
print("=" * 80)

print(
    "\nPayment Format:"
)

print(
    pilot[
        "Payment Format"
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(4)
    .to_string()
)

print(
    "\nPayment Currency:"
)

print(
    pilot[
        "Payment Currency"
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(4)
    .to_string()
)

print(
    "\nReceiving Currency:"
)

print(
    pilot[
        "Receiving Currency"
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(4)
    .to_string()
)

# ============================================================
# 15. PATTERNS.TXT INTEGRITY
# ============================================================

print("\n" + "=" * 80)
print("12. PATTERNS.TXT INTEGRITY")
print("=" * 80)

with open(
    PILOT_PATTERNS_PATH,
    "r",
    encoding="utf-8"
) as f:

    pattern_lines = [
        x.rstrip("\n")
        for x in f
    ]

id_lines = [
    x for x in pattern_lines
    if x.startswith('id="')
]

begin_lines = [
    x for x in pattern_lines
    if x.startswith(
        "BEGIN LAUNDERING ATTEMPT"
    )
]

end_lines = [
    x for x in pattern_lines
    if x.startswith(
        "END LAUNDERING ATTEMPT"
    )
]

pattern_tx_lines = []

for x in pattern_lines:

    parts = x.split(",")

    if len(parts) == 11:

        parsed_date = pd.to_datetime(
            parts[0],
            format="%Y/%m/%d %H:%M",
            errors="coerce"
        )

        if not pd.isna(parsed_date):

            pattern_tx_lines.append(
                x
            )

print(
    "Pattern IDs:",
    len(id_lines)
)

print(
    "BEGIN markers:",
    len(begin_lines)
)

print(
    "END markers:",
    len(end_lines)
)

print(
    "Pattern transaction rows:",
    len(pattern_tx_lines)
)

print(
    "Expected pattern transaction rows:",
    len(pilot)
)

# ============================================================
# 16. PATTERN TRANSACTION LABELS
# ============================================================

pattern_labels = []

for line in pattern_tx_lines:

    parts = [
        x.strip()
        for x in line.split(",")
    ]

    try:

        pattern_labels.append(
            int(parts[10])
        )

    except Exception:

        pass

print(
    "Pattern labels other than 1:",
    sum(
        x != 1
        for x in pattern_labels
    )
)

# ============================================================
# 17. FINAL DECISION GATES
# ============================================================

print("\n" + "=" * 100)
print("13. FINAL PILOT SCALE-UP GATES")
print("=" * 100)

gates = {

    "transaction_count_nonzero":
        len(pilot) > 0,

    "unique_transaction_ids":
        (
            pilot["transaction_id"].nunique()
            ==
            len(pilot)
        ),

    "bank_pair_coverage_100":
        (
            np.mean(pair_flags)
            >=
            0.999999
        ),

    "source_bank_account_coverage_100":
        (
            np.mean(source_valid)
            >=
            0.999999
        ),

    "destination_bank_account_coverage_100":
        (
            np.mean(destination_valid)
            >=
            0.999999
        ),

    "new_account_format_100":
        (
            int((~valid_format).sum())
            ==
            0
        ),

    "new_account_bank_collision_0":
        (
            len(new_vs_original)
            ==
            0
        ),

    "new_entity_id_unique":
        (
            entity_ids.nunique()
            ==
            len(entity_ids)
        ),

    "episode_transaction_counts_exact":
        (
            template_count_mismatch
            ==
            0
        ),

    "episode_node_counts_exact":
        (
            template_node_mismatch
            ==
            0
        ),

    "episode_edge_counts_exact":
        (
            template_edge_mismatch
            ==
            0
        ),

    "episode_degree_signatures_exact":
        (
            template_degree_mismatch
            ==
            0
        ),

    "episode_timestamps_exact":
        (
            timestamp_mismatch
            ==
            0
        ),

    "duplicate_directed_edges_0":
        (
            len(edge_strings)
            ==
            edge_strings.nunique()
        ),

    "patterns_ids_exact":
        (
            len(id_lines)
            ==
            pilot["synthetic_episode_id"].nunique()
        ),

    "patterns_begin_exact":
        (
            len(begin_lines)
            ==
            pilot["synthetic_episode_id"].nunique()
        ),

    "patterns_end_exact":
        (
            len(end_lines)
            ==
            pilot["synthetic_episode_id"].nunique()
        ),

    "patterns_transaction_count_exact":
        (
            len(pattern_tx_lines)
            ==
            len(pilot)
        ),

    "patterns_all_labeled_1":
        (
            all(
                x == 1
                for x in pattern_labels
            )
        )
}

for name, passed in gates.items():

    print(
        f"{'✅' if passed else '❌'} "
        f"{name}: {passed}"
    )

all_passed = all(
    gates.values()
)

# ============================================================
# 18. FINAL RESULT
# ============================================================

print("\n" + "=" * 100)

if all_passed:

    print(
        "✅ PILOT PASSED ALL SCALE-UP INTEGRITY GATES"
    )

else:

    print(
        "❌ PILOT FAILED ONE OR MORE SCALE-UP GATES"
    )

print("=" * 100)

print(
    "\nNO SYNTHETIC DATA GENERATED BY THIS CELL."
)

print(
    "NO ORIGINAL FILES MODIFIED."
)

print("=" * 100)

CELL 19 — FULL PILOT INTEGRITY + PATTERN FIDELITY AUDIT

Pilot transactions: 43,310
Pilot episodes: 5,000
New accounts: 4,081
Templates: 370

1. BASIC TRANSACTION INTEGRITY
Null values: 0
Invalid Amount Received: 0
Invalid Amount Paid: 0
Non-positive Amount Received: 0
Non-positive Amount Paid: 0
Non-laundering rows: 0

2. TRANSACTION ID INTEGRITY
Unique transaction IDs: 43310
Duplicate transaction IDs: 0

3. LEGITIMATE BANK-PAIR INTEGRITY
Transaction rows checked: 43,310
Legitimate bank-pair rows: 43310
Invalid bank-pair rows: 0
Bank-pair legitimate coverage: 100.000000%

4. BANK + ACCOUNT INTEGRITY
Source bank+account valid rows: 43310
Source bank+account invalid rows: 0
Source bank+account coverage: 100.000000%

Destination bank+account valid rows: 43310
Destination bank+account invalid rows: 0
Destination bank+account coverage: 100.000000%

5. GLOBAL ACCOUNT-NUMBER COLLISION CHECK
Account numbers used by >1 bank in original data: 8

Examples:
8135B8200 -> ['221731', '27444']
8135B8

/tmp/ipykernel_58/3382689211.py:799: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  generated_ts = pd.to_datetime(
/tmp/ipykernel_58/3382689211.py:799: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  generated_ts = pd.to_datetime(


Episode transaction-count mismatches: 0
Episode node-count mismatches: 0
Episode edge-count mismatches: 0
Episode degree-signature mismatches: 0
Episode timestamp mismatches: 0

9. DIRECTED EDGE UNIQUENESS
Total edges: 43310
Unique edges: 43310
Duplicate edges: 0

10. AMOUNT INTEGRITY
Amount Received quantiles:
0.001    7.053857e-05
0.010    8.546211e-03
0.050    2.670620e+00
0.250    1.431546e+02
0.500    1.363441e+03
0.750    1.544714e+04
0.900    2.319521e+05
0.950    1.246118e+06
0.990    3.388815e+07
0.999    3.704223e+09

Amount Paid quantiles:
0.001    7.053857e-05
0.010    8.546211e-03
0.050    2.670620e+00
0.250    1.431546e+02
0.500    1.363441e+03
0.750    1.544714e+04
0.900    2.319521e+05
0.950    1.246118e+06
0.990    3.388815e+07
0.999    3.704223e+09

Amount Received > 1e12: 0
Amount Received < 1e-6: 0

11. PAYMENT BEHAVIOR

Payment Format:
Payment Format
Cheque          37.6380
Credit Card     25.5992
ACH             11.9487
Cash             9.6998
Reinvestment     8.5

In [26]:
# ============================================================
# CELL 20 — FULL-SCALE STREAMING SYNTHETIC GENERATOR
# ============================================================
#
# TARGET:
#   5,067,991 synthetic laundering transactions
#   584,706 synthetic laundering episodes
#
# IMPORTANT:
#   - Original HI-Small files remain untouched.
#   - Writes incrementally to disk.
#   - Does NOT hold 5M transaction dictionaries in RAM.
#
# OUTPUT:
#   HI-Small_SYNTH_FULL_transactions.csv
#   HI-Small_SYNTH_FULL_new_accounts.csv
#   HI-Small_SYNTH_FULL_Patterns.txt
#   HI-Small_SYNTH_FULL_episode_audit.csv
#   HI-Small_SYNTH_FULL_manifest.json
#
# ============================================================

import os
import json
import random
import string
from collections import defaultdict

import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

SEED = 20260919

random.seed(SEED)
np.random.seed(SEED)

NEW_ACCOUNT_RATE = 0.05

FLUSH_EVERY_EPISODES = 500

# ============================================================
# PATHS
# ============================================================

DATA_ROOT = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
)

WORK_DIR = (
    "/kaggle/working/"
    "AML_GNN_ONLY/data"
)

TRANS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_Trans.csv"
)

ACCOUNTS_PATH = os.path.join(
    DATA_ROOT,
    "HI-Small_accounts.csv"
)

TEMPLATE_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_pattern_templates.json"
)

EPISODE_PLAN_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_synthetic_episode_plan.csv"
)

BANK_POOL_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_final_generator_bank_pool.csv"
)

ACCOUNT_REGISTRY_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_generator_account_registry.csv"
)

ACCOUNT_PREFIX_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_bank_account_prefix_sampler.json"
)

ACCOUNT_GENERATOR_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_account_id_generator.json"
)

ENTITY_DNA_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_entity_name_dna.json"
)

MAPPING_LIBRARY_PATH = os.path.join(
    WORK_DIR,
    "HI-Small_legitimate_bank_mapping_library.json"
)

# ------------------------------------------------------------
# FINAL OUTPUTS
# ------------------------------------------------------------

OUT_TRANS = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_FULL_transactions.csv"
)

OUT_ACCOUNTS = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_FULL_new_accounts.csv"
)

OUT_PATTERNS = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_FULL_Patterns.txt"
)

OUT_AUDIT = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_FULL_episode_audit.csv"
)

OUT_MANIFEST = os.path.join(
    WORK_DIR,
    "HI-Small_SYNTH_FULL_manifest.json"
)

# ============================================================
# 1. LOAD STATIC DATA
# ============================================================

print("=" * 100)
print("CELL 20 — FULL-SCALE STREAMING SYNTHETIC GENERATOR")
print("=" * 100)

# ------------------------------------------------------------
# Templates
# ------------------------------------------------------------

with open(
    TEMPLATE_PATH,
    "r",
    encoding="utf-8"
) as f:

    template_manifest = json.load(f)

templates = template_manifest[
    "templates"
]

template_lookup = {
    str(t["template_id"]): t
    for t in templates
}

print(
    "\nIBM templates:",
    len(templates)
)

# ------------------------------------------------------------
# Episode plan
# ------------------------------------------------------------

episode_plan = pd.read_csv(
    EPISODE_PLAN_PATH,
    dtype=str
)

episode_plan[
    "transaction_count"
] = pd.to_numeric(
    episode_plan[
        "transaction_count"
    ],
    errors="raise"
).astype(int)

print(
    "Planned episodes:",
    f"{len(episode_plan):,}"
)

planned_target = int(
    episode_plan[
        "transaction_count"
    ].sum()
)

print(
    "Planned synthetic transactions:",
    f"{planned_target:,}"
)

EXPECTED_TARGET = 5_067_991

if planned_target != EXPECTED_TARGET:

    raise RuntimeError(
        f"Episode plan target mismatch: "
        f"{planned_target:,} != {EXPECTED_TARGET:,}"
    )

# ============================================================
# 2. LOAD BANK MAPPING LIBRARY
# ============================================================

print("\n" + "=" * 80)
print("1. LOADING CONNECTIVITY MAPPINGS")
print("=" * 80)

with open(
    MAPPING_LIBRARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    mapping_manifest = json.load(f)

mapping_library = {}

for template_id, mappings in (
    mapping_manifest[
        "mappings"
    ].items()
):

    mapping_library[
        str(template_id)
    ] = [

        {
            int(node):
                str(bank)

            for node, bank
            in mapping.items()
        }

        for mapping in mappings
    ]

print(
    "Templates with mapping libraries:",
    len(mapping_library)
)

# ============================================================
# 3. BANK POOL
# ============================================================

print("\n" + "=" * 80)
print("2. LOADING BANK POOL")
print("=" * 80)

bank_pool = pd.read_csv(
    BANK_POOL_PATH,
    dtype=str
)

bank_pool[
    "normalized_bank_id"
] = bank_pool[
    "normalized_bank_id"
].astype(str)

bank_tx_id_map = {
    str(row["normalized_bank_id"]):
        str(row["canonical_transaction_bank_id"])
    for _, row in bank_pool.iterrows()
}

bank_name_map = {
    str(row["normalized_bank_id"]):
        str(row["Bank Name"])
    for _, row in bank_pool.iterrows()
}

print(
    "Banks:",
    len(bank_pool)
)

# ============================================================
# 4. ACCOUNT REGISTRY
# ============================================================

print("\n" + "=" * 80)
print("3. LOADING ACCOUNT REGISTRY")
print("=" * 80)

registry = pd.read_csv(
    ACCOUNT_REGISTRY_PATH,
    dtype=str
)

registry[
    "normalized_bank_id"
] = registry[
    "normalized_bank_id"
].astype(str)

for col in [
    "legitimate_source_tx_count",
    "legitimate_destination_tx_count",
    "legitimate_total_activity"
]:

    registry[col] = pd.to_numeric(
        registry[col],
        errors="coerce"
    ).fillna(0).astype(int)

registry[
    "generator_eligible"
] = (
    registry[
        "generator_eligible"
    ]
    .astype(str)
    .str.lower()
    .eq("true")
)

eligible = registry[
    registry[
        "generator_eligible"
    ]
].copy()

print(
    "Eligible accounts:",
    f"{len(eligible):,}"
)

# ------------------------------------------------------------
# Account pools
# ------------------------------------------------------------

account_pool = defaultdict(list)

for _, row in eligible.iterrows():

    bank = str(
        row["normalized_bank_id"]
    )

    account = str(
        row["Account Number"]
    )

    role = str(
        row["role"]
    )

    if role in {
        "source_only",
        "both"
    }:

        account_pool[
            (bank, "source")
        ].append(account)

    if role in {
        "destination_only",
        "both"
    }:

        account_pool[
            (bank, "destination")
        ].append(account)

    if role == "both":

        account_pool[
            (bank, "both")
        ].append(account)

    account_pool[
        (bank, "any")
    ].append(account)

for key in account_pool:

    random.shuffle(
        account_pool[key]
    )

pool_pointer = defaultdict(int)

# ============================================================
# 5. ACCOUNT DNA
# ============================================================

print("\n" + "=" * 80)
print("4. LOADING ACCOUNT GENERATION DNA")
print("=" * 80)

with open(
    ACCOUNT_PREFIX_PATH,
    "r",
    encoding="utf-8"
) as f:

    bank_prefix_sampler = json.load(f)

with open(
    ACCOUNT_GENERATOR_PATH,
    "r",
    encoding="utf-8"
) as f:

    account_generator_dna = json.load(f)

with open(
    ENTITY_DNA_PATH,
    "r",
    encoding="utf-8"
) as f:

    entity_dna = json.load(f)

position_models = (
    account_generator_dna[
        "position_models"
    ]
)

entity_position_models = (
    entity_dna[
        "entity_id_position_models"
    ]

)

name_prefixes = (
    entity_dna[
        "top_name_prefixes"
    ]
)

account_position_arrays = {}

for pos, entries in (
    position_models.items()
):

    chars = [
        x["char"]
        for x in entries
    ]

    probs = np.array(
        [
            x["probability"]
            for x in entries
        ],
        dtype=float
    )

    probs /= probs.sum()

    account_position_arrays[
        int(pos)
    ] = (
        chars,
        probs
    )

entity_position_arrays = {}

for pos, values in (
    entity_position_models.items()
):

    chars = list(
        values.keys()
    )

    probs = np.array(
        [
            values[c]["probability"]
            for c in chars
        ],
        dtype=float
    )

    probs /= probs.sum()

    entity_position_arrays[
        int(pos)
    ] = (
        chars,
        probs
    )

name_keys = list(
    name_prefixes.keys()
)

name_probs = np.array(
    [
        name_prefixes[k]
        for k in name_keys
    ],
    dtype=float
)

name_probs /= name_probs.sum()

# ------------------------------------------------------------
# Existing bank+account combinations
# ------------------------------------------------------------

existing_account_keys = set(
    (
        str(row["normalized_bank_id"]),
        str(row["Account Number"])
    )
    for _, row in registry.iterrows()
)

used_new_accounts = set()
used_new_entities = set()

# ============================================================
# 6. BEHAVIOR DNA SAMPLE
# ============================================================

print("\n" + "=" * 80)
print("5. LOADING LEGITIMATE TRANSACTION BEHAVIOR")
print("=" * 80)

behavior_parts = []

chunk_number = 0

for chunk in pd.read_csv(
    TRANS_PATH,
    usecols=[
        "Timestamp",
        "Amount Received",
        "Amount Paid",
        "Payment Currency",
        "Receiving Currency",
        "Payment Format",
        "Is Laundering"
    ],
    dtype=str,
    chunksize=500_000
):

    chunk_number += 1

    labels = pd.to_numeric(
        chunk[
            "Is Laundering"
        ],
        errors="coerce"
    ).fillna(0).astype(int)

    legit = chunk[
        labels == 0
    ].copy()

    if len(legit) == 0:
        continue

    n = min(
        10_000,
        len(legit)
    )

    behavior_parts.append(
        legit.sample(
            n=n,
            random_state=(
                SEED
                +
                chunk_number
            )
        )
    )

behavior = pd.concat(
    behavior_parts,
    ignore_index=True
)

behavior[
    "Timestamp"
] = pd.to_datetime(
    behavior[
        "Timestamp"
    ],
    errors="coerce"
)

behavior[
    "Amount Received"
] = pd.to_numeric(
    behavior[
        "Amount Received"
    ],
    errors="coerce"
)

behavior[
    "Amount Paid"
] = pd.to_numeric(
    behavior[
        "Amount Paid"
    ],
    errors="coerce"
)

behavior = behavior.dropna(
    subset=[
        "Timestamp",
        "Amount Received",
        "Amount Paid"
    ]
).reset_index(
    drop=True
)

print(
    "Behavior sample:",
    f"{len(behavior):,}"
)

time_values = (
    behavior[
        "Timestamp"
    ].tolist()
)

amount_values = (
    behavior[
        "Amount Received"
    ]
    .clip(
        lower=1e-6
    )
    .to_numpy(
        dtype=float
    )
)

payment_formats = (
    behavior[
        "Payment Format"
    ]
    .astype(str)
    .tolist()
)

payment_currencies = (
    behavior[
        "Payment Currency"
    ]
    .astype(str)
    .tolist()
)

receiving_by_payment = defaultdict(list)

for pay, recv in zip(
    behavior[
        "Payment Currency"
    ].astype(str),
    behavior[
        "Receiving Currency"
    ].astype(str)
):

    receiving_by_payment[
        pay
    ].append(
        recv
    )

dataset_min_ts = (
    behavior["Timestamp"].min()
)

dataset_max_ts = (
    behavior["Timestamp"].max()
)

dataset_max_amount = float(
    amount_values.max()
)

# ============================================================
# 7. GENERATORS
# ============================================================

def generate_entity_id():

    for _ in range(10000):

        chars = []

        for pos in range(
            1,
            10
        ):

            char_list, probs = (
                entity_position_arrays[
                    pos
                ]
            )

            chars.append(
                np.random.choice(
                    char_list,
                    p=probs
                )
            )

        value = "".join(chars)

        if value not in (
            used_new_entities
        ):

            used_new_entities.add(
                value
            )

            return value

    raise RuntimeError(
        "Could not generate Entity ID."
    )


def generate_entity_name():

    prefix = np.random.choice(
        name_keys,
        p=name_probs
    )

    number = np.random.randint(
        1,
        100000
    )

    return (
        f"{prefix} #{number}"
    )


def generate_new_account(
    bank
):

    bank = str(bank)

    prefix_data = (
        bank_prefix_sampler.get(
            bank
        )
    )

    for _ in range(10000):

        if (
            prefix_data
            and
            prefix_data["prefixes"]
        ):

            prefixes = [
                x["prefix"]
                for x
                in prefix_data[
                    "prefixes"
                ]
            ]

            probs = np.array(
                [
                    x["probability"]
                    for x
                    in prefix_data[
                        "prefixes"
                    ]
                ],
                dtype=float
            )

            probs /= probs.sum()

            prefix = np.random.choice(
                prefixes,
                p=probs
            )

        else:

            prefix = ""

            for pos in [
                1,
                2,
                3
            ]:

                chars, probs = (
                    account_position_arrays[
                        pos
                    ]
                )

                prefix += (
                    np.random.choice(
                        chars,
                        p=probs
                    )
                )

        chars = list(
            prefix
        )

        for pos in range(
            4,
            10
        ):

            char_list, probs = (
                account_position_arrays[
                    pos
                ]
            )

            chars.append(
                np.random.choice(
                    char_list,
                    p=probs
                )
            )

        account = "".join(
            chars
        )

        key = (
            bank,
            account
        )

        if (
            key not in
            existing_account_keys
            and
            key not in
            used_new_accounts
        ):

            used_new_accounts.add(
                key
            )

            new_accounts_file.writerow(
                [
                    bank_name_map[
                        bank
                    ],

                    bank_tx_id_map[
                        bank
                    ],

                    account,

                    generate_entity_id(),

                    generate_entity_name()
                ]
            )

            return account

    raise RuntimeError(
        f"Unable to generate account for bank {bank}"
    )


def get_account(
    bank,
    role,
    used_accounts
):

    bank = str(bank)

    if role == "source":

        role_order = [
            "source",
            "both",
            "any"
        ]

    elif role == "destination":

        role_order = [
            "destination",
            "both",
            "any"
        ]

    else:

        role_order = [
            "both",
            "any"
        ]

    for pool_role in role_order:

        pool = account_pool.get(
            (
                bank,
                pool_role
            ),
            []
        )

        if not pool:
            continue

        start = pool_pointer[
            (
                bank,
                pool_role
            )
        ]

        for offset in range(
            len(pool)
        ):

            idx = (
                start
                +
                offset
            ) % len(pool)

            account = pool[
                idx
            ]

            if account in used_accounts:
                continue

            pool_pointer[
                (
                    bank,
                    pool_role
                )
            ] = (
                idx + 1
            ) % len(pool)

            return (
                account,
                False
            )

    return (
        generate_new_account(bank),
        True
    )


def generate_transaction_id():

    for _ in range(1000):

        value = "".join(
            random.choices(
                string.ascii_uppercase
                +
                string.digits,
                k=16
            )
        )

        if value not in (
            used_transaction_ids
        ):

            used_transaction_ids.add(
                value
            )

            return value

    raise RuntimeError(
        "Unable to generate transaction ID."
    )

# ============================================================
# 8. PREPARE OUTPUT FILES
# ============================================================

print("\n" + "=" * 80)
print("6. OPENING STREAM OUTPUTS")
print("=" * 80)

# ------------------------------------------------------------
# Transaction CSV
# ------------------------------------------------------------

transaction_columns = [
    "transaction_id",
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
    "is_synthetic",
    "synthetic_episode_id",
    "typology",
    "source_template_id",
    "template_source_order"
]

trans_file = open(
    OUT_TRANS,
    "w",
    encoding="utf-8",
    buffering=1024 * 1024
)

transactions_csv = (
    __import__("csv").writer(
        trans_file
    )
)

transactions_csv.writerow(
    transaction_columns
)

# ------------------------------------------------------------
# New accounts CSV
# ------------------------------------------------------------

accounts_file = open(
    OUT_ACCOUNTS,
    "w",
    encoding="utf-8",
    buffering=1024 * 1024
)

new_accounts_file = (
    __import__("csv").writer(
        accounts_file
    )
)

new_accounts_file.writerow(
    [
        "Bank Name",
        "Bank ID",
        "Account Number",
        "Entity ID",
        "Entity Name"
    ]
)

# ------------------------------------------------------------
# Patterns
# ------------------------------------------------------------

patterns_file = open(
    OUT_PATTERNS,
    "w",
    encoding="utf-8",
    buffering=1024 * 1024
)

# ============================================================
# 9. GENERATION STATE
# ============================================================

used_transaction_ids = set()

total_transactions = 0
total_episodes = 0

typology_transactions = defaultdict(int)
typology_episodes = defaultdict(int)

new_accounts_total = 0

audit_buffer = []

# ============================================================
# 10. GENERATE ALL EPISODES
# ============================================================

print("\n" + "=" * 80)
print("7. FULL-SCALE GENERATION")
print("=" * 80)

for idx, episode in (
    episode_plan.iterrows()
):

    template_id = str(
        episode[
            "source_template_id"
        ]
    )

    typology = str(
        episode[
            "typology"
        ]
    )

    synthetic_episode_id = str(
        episode[
            "synthetic_episode_id"
        ]
    )

    template = template_lookup[
        template_id
    ]

    mappings = mapping_library[
        template_id
    ]

    if not mappings:
        raise RuntimeError(
            f"No mapping for {template_id}"
        )

    node_bank_mapping = random.choice(
        mappings
    )

    # --------------------------------------------------------
    # Node degree
    # --------------------------------------------------------
    in_degree = defaultdict(int)
    out_degree = defaultdict(int)

    for edge in template[
        "edges"
    ]:

        source = int(
            edge["source_node"]
        )

        destination = int(
            edge["destination_node"]
        )

        out_degree[
            source
        ] += 1

        in_degree[
            destination
        ] += 1

    node_roles = {}

    for node in template[
        "nodes"
    ]:

        node_id = int(
            node["node_id"]
        )

        if (
            in_degree[node_id] > 0
            and
            out_degree[node_id] > 0
        ):

            role = "both"

        elif out_degree[node_id] > 0:

            role = "source"

        else:

            role = "destination"

        node_roles[
            node_id
        ] = role

    # --------------------------------------------------------
    # Account mapping
    # --------------------------------------------------------
    node_account_mapping = {}

    used_accounts_episode = set()

    episode_new_accounts = 0

    for node_id in sorted(
        node_roles
    ):

        bank = str(
            node_bank_mapping[
                node_id
            ]
        )

        role = node_roles[
            node_id
        ]

        force_new = (
            np.random.random()
            <
            NEW_ACCOUNT_RATE
        )

        if force_new:

            account = (
                generate_new_account(
                    bank
                )
            )

            was_new = True

        else:

            account, was_new = (
                get_account(
                    bank,
                    role,
                    used_accounts_episode
                )
            )

        node_account_mapping[
            node_id
        ] = account

        used_accounts_episode.add(
            account
        )

        if was_new:
            episode_new_accounts += 1

    new_accounts_total += (
        episode_new_accounts
    )

    # --------------------------------------------------------
    # Time anchor
    # --------------------------------------------------------
    template_span = float(
        template[
            "episode_span_minutes"
        ]
    )

    anchor = random.choice(
        time_values
    )

    latest_anchor = (
        dataset_max_ts
        -
        pd.to_timedelta(
            template_span,
            unit="m"
        )
    )

    if anchor > latest_anchor:
        anchor = latest_anchor

    if anchor < dataset_min_ts:
        anchor = dataset_min_ts

    # --------------------------------------------------------
    # Amount scaling
    # --------------------------------------------------------
    target_median = float(
        random.choice(
            amount_values
        )
    )

    template_median = max(
        1e-9,
        float(
            template[
                "median_amount"
            ]
        )
    )

    amount_scale = (
        target_median
        /
        template_median
    )

    episode_amounts = []

    episode_edges = set()

    # --------------------------------------------------------
    # Patterns block begin
    # --------------------------------------------------------
    patterns_file.write(
        f'id="{synthetic_episode_id}"\n'
    )

    patterns_file.write(
        f"BEGIN LAUNDERING ATTEMPT - "
        f"{typology}: "
        f"Synthetic instance based on "
        f"IBM {typology} template\n"
    )

    # --------------------------------------------------------
    # Generate transactions in exact template order
    # --------------------------------------------------------
    for tx in template[
        "transactions"
    ]:

        source_node = int(
            tx["source_node"]
        )

        destination_node = int(
            tx["destination_node"]
        )

        source_bank_norm = str(
            node_bank_mapping[
                source_node
            ]
        )

        destination_bank_norm = str(
            node_bank_mapping[
                destination_node
            ]
        )

        source_account = (
            node_account_mapping[
                source_node
            ]
        )

        destination_account = (
            node_account_mapping[
                destination_node
            ]
        )

        # ----------------------------------------------------
        # Amount
        # ----------------------------------------------------
        template_amount = float(
            tx[
                "amount_received"
            ]
        )

        amount_received = max(
            1e-6,
            template_amount
            *
            amount_scale
        )

        original_received = max(
            1e-12,
            template_amount
        )

        original_paid = float(
            tx[
                "amount_paid"
            ]
        )

        paid_ratio = (
            original_paid
            /
            original_received
        )

        amount_paid = max(
            1e-6,
            amount_received
            *
            paid_ratio
        )

        if amount_received > dataset_max_amount:

            correction = (
                dataset_max_amount
                /
                amount_received
            )

            amount_received *= correction
            amount_paid *= correction

        # ----------------------------------------------------
        # Timestamp
        # ----------------------------------------------------
        timestamp = (
            anchor
            +
            pd.to_timedelta(
                float(
                    tx[
                        "relative_minutes"
                    ]
                ),
                unit="m"
            )
        )

        # ----------------------------------------------------
        # Behavior attributes
        # ----------------------------------------------------
        payment_format = random.choice(
            payment_formats
        )

        payment_currency = random.choice(
            payment_currencies
        )

        recv_candidates = (
            receiving_by_payment.get(
                payment_currency
            )
        )

        if recv_candidates:

            receiving_currency = (
                random.choice(
                    recv_candidates
                )
            )

        else:

            receiving_currency = (
                payment_currency
            )

        # ----------------------------------------------------
        # Canonical transaction bank IDs
        # ----------------------------------------------------
        source_bank_tx = (
            bank_tx_id_map[
                source_bank_norm
            ]
        )

        destination_bank_tx = (
            bank_tx_id_map[
                destination_bank_norm
            ]
        )

        # ----------------------------------------------------
        # Full directed edge
        # ----------------------------------------------------
        edge = (
            source_bank_tx,
            source_account,
            destination_bank_tx,
            destination_account
        )

        episode_edges.add(
            edge
        )

        # ----------------------------------------------------
        # Transaction ID
        # ----------------------------------------------------
        transaction_id = (
            generate_transaction_id()
        )

        # ----------------------------------------------------
        # Write transaction
        # ----------------------------------------------------
        transactions_csv.writerow(
            [
                transaction_id,

                timestamp.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),

                source_bank_tx,

                source_account,

                destination_bank_tx,

                destination_account,

                f"{amount_received:.6f}",

                receiving_currency,

                f"{amount_paid:.6f}",

                payment_currency,

                payment_format,

                1,

                1,

                synthetic_episode_id,

                typology,

                template_id,

                int(
                    tx[
                        "source_order"
                    ]
                )
            ]
        )

        # ----------------------------------------------------
        # Write Patterns transaction
        # ----------------------------------------------------
        patterns_file.write(
            ",".join(
                [
                    timestamp.strftime(
                        "%Y/%m/%d %H:%M"
                    ),

                    source_bank_tx,

                    source_account,

                    destination_bank_tx,

                    destination_account,

                    f"{amount_received:.6f}",

                    receiving_currency,

                    f"{amount_paid:.6f}",

                    payment_currency,

                    payment_format,

                    "1"
                ]
            )
            +
            "\n"
        )

        episode_amounts.append(
            amount_received
        )

        total_transactions += 1
        typology_transactions[
            typology
        ] += 1

    # --------------------------------------------------------
    # Patterns end
    # --------------------------------------------------------
    patterns_file.write(
        f"END LAUNDERING ATTEMPT - "
        f"{typology}\n"
    )

    # --------------------------------------------------------
    # Episode counters
    # --------------------------------------------------------
    total_episodes += 1

    typology_episodes[
        typology
    ] += 1

    audit_buffer.append(
        {
            "synthetic_episode_id":
                synthetic_episode_id,

            "typology":
                typology,

            "source_template_id":
                template_id,

            "transaction_count":
                len(
                    template[
                        "transactions"
                    ]
                ),

            "template_nodes":
                len(
                    template[
                        "nodes"
                    ]
                ),

            "generated_unique_accounts":
                len(
                    node_account_mapping
                ),

            "generated_unique_banks":
                len(
                    set(
                        node_bank_mapping.values()
                    )
                ),

            "new_accounts":
                episode_new_accounts,

            "unique_directed_edges":
                len(
                    episode_edges
                ),

            "duplicate_directed_edges":
                len(
                    template[
                        "transactions"
                    ]
                )
                -
                len(
                    episode_edges
                ),

            "template_span_minutes":
                template_span,

            "generated_median_amount":
                float(
                    np.median(
                        episode_amounts
                    )
                )
        }
    )

    # --------------------------------------------------------
    # Flush audit periodically
    # --------------------------------------------------------
    if (
        total_episodes
        %
        FLUSH_EVERY_EPISODES
        ==
        0
    ):

        print(
            f"Episodes: "
            f"{total_episodes:,}/"
            f"{len(episode_plan):,}"
            f" | Transactions: "
            f"{total_transactions:,}"
            f" | New accounts: "
            f"{new_accounts_total:,}"
        )

# ============================================================
# 11. CLOSE FILES
# ============================================================

trans_file.close()
accounts_file.close()
patterns_file.close()

# ============================================================
# 12. FINAL EXACTNESS
# ============================================================

print("\n" + "=" * 80)
print("8. FINAL GENERATION COUNTS")
print("=" * 80)

print(
    "Episodes generated:",
    f"{total_episodes:,}"
)

print(
    "Episodes planned:",
    f"{len(episode_plan):,}"
)

print(
    "Transactions generated:",
    f"{total_transactions:,}"
)

print(
    "Transactions planned:",
    f"{planned_target:,}"
)

print(
    "New accounts generated:",
    f"{new_accounts_total:,}"
)

if (
    total_episodes
    !=
    len(episode_plan)
):

    raise RuntimeError(
        "Episode count mismatch."
    )

if (
    total_transactions
    !=
    planned_target
):

    raise RuntimeError(
        "Transaction count mismatch."
    )

# ============================================================
# 13. AUDIT DATAFRAME
# ============================================================

audit_df = pd.DataFrame(
    audit_buffer
)

audit_df.to_csv(
    OUT_AUDIT,
    index=False
)

# ============================================================
# 14. PATTERN FILE COUNTS
# ============================================================

print("\n" + "=" * 80)
print("9. PATTERNS FILE VALIDATION")
print("=" * 80)

with open(
    OUT_PATTERNS,
    "r",
    encoding="utf-8"
) as f:

    pattern_lines = [
        x.rstrip("\n")
        for x in f
    ]

pattern_ids = [
    x for x in pattern_lines
    if x.startswith(
        'id="'
    )
]

pattern_begins = [
    x for x in pattern_lines
    if x.startswith(
        "BEGIN LAUNDERING ATTEMPT"
    )
]

pattern_ends = [
    x for x in pattern_lines
    if x.startswith(
        "END LAUNDERING ATTEMPT"
    )
]

pattern_transaction_count = 0

for line in pattern_lines:

    parts = line.split(",")

    if len(parts) != 11:
        continue

    parsed = pd.to_datetime(
        parts[0],
        format="%Y/%m/%d %H:%M",
        errors="coerce"
    )

    if not pd.isna(parsed):
        pattern_transaction_count += 1

print(
    "Pattern IDs:",
    len(pattern_ids)
)

print(
    "BEGIN markers:",
    len(pattern_begins)
)

print(
    "END markers:",
    len(pattern_ends)
)

print(
    "Pattern transaction rows:",
    pattern_transaction_count
)

# ============================================================
# 15. BUILD MANIFEST
# ============================================================

manifest = {

    "dataset":
        "IBM HI-Small",

    "generator_version":
        "streaming_connectivity_constrained_v3",

    "seed":
        SEED,

    "target_synthetic_transactions":
        EXPECTED_TARGET,

    "generated_synthetic_transactions":
        int(total_transactions),

    "target_episodes":
        int(len(episode_plan)),

    "generated_episodes":
        int(total_episodes),

    "new_accounts_generated":
        int(new_accounts_total),

    "new_account_rate":
        float(NEW_ACCOUNT_RATE),

    "templates_available":
        int(len(templates)),

    "templates_used":
        int(
            episode_plan[
                "source_template_id"
            ].nunique()
        ),

    "validated_legitimate_bank_mappings":
        int(
            mapping_manifest[
                "total_valid_mappings"
            ]
        ),

    "typology_transactions":
        {
            str(k): int(v)
            for k, v
            in typology_transactions.items()
        },

    "typology_episodes":
        {
            str(k): int(v)
            for k, v
            in typology_episodes.items()
        },

    "pattern_file_ids":
        int(len(pattern_ids)),

    "pattern_file_begin_markers":
        int(len(pattern_begins)),

    "pattern_file_end_markers":
        int(len(pattern_ends)),

    "pattern_file_transaction_rows":
        int(pattern_transaction_count),

    "outputs": {

        "transactions":
            OUT_TRANS,

        "new_accounts":
            OUT_ACCOUNTS,

        "patterns":
            OUT_PATTERNS,

        "episode_audit":
            OUT_AUDIT
    }
}

with open(
    OUT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ============================================================
# 16. FINAL
# ============================================================

print("\n" + "=" * 100)
print("✅ CELL 20 COMPLETE — FULL SYNTHETIC GENERATION")
print("=" * 100)

print(
    "\nSynthetic laundering transactions:",
    f"{total_transactions:,}"
)

print(
    "Required:",
    f"{EXPECTED_TARGET:,}"
)

print(
    "Exact transaction target:",
    total_transactions
    ==
    EXPECTED_TARGET
)

print(
    "Synthetic laundering episodes:",
    f"{total_episodes:,}"
)

print(
    "New accounts:",
    f"{new_accounts_total:,}"
)

print(
    "Templates used:",
    f"{episode_plan['source_template_id'].nunique():,}"
)

print(
    "\nPatterns IDs:",
    len(pattern_ids)
)

print(
    "Patterns BEGIN:",
    len(pattern_begins)
)

print(
    "Patterns END:",
    len(pattern_ends)
)

print(
    "Patterns transaction rows:",
    pattern_transaction_count
)

print(
    "Patterns exact:",
    (
        len(pattern_ids)
        ==
        total_episodes
        and
        len(pattern_begins)
        ==
        total_episodes
        and
        len(pattern_ends)
        ==
        total_episodes
        and
        pattern_transaction_count
        ==
        total_transactions
    )
)

print("\nSaved:")
print(
    OUT_TRANS
)
print(
    OUT_ACCOUNTS
)
print(
    OUT_PATTERNS
)
print(
    OUT_AUDIT
)
print(
    OUT_MANIFEST
)

print(
    "\nORIGINAL HI-SMALL FILES WERE NOT MODIFIED."
)

print("=" * 100)

CELL 20 — FULL-SCALE STREAMING SYNTHETIC GENERATOR

IBM templates: 370
Planned episodes: 584,706
Planned synthetic transactions: 5,067,991

1. LOADING CONNECTIVITY MAPPINGS
Templates with mapping libraries: 370

2. LOADING BANK POOL
Banks: 30470

3. LOADING ACCOUNT REGISTRY
Eligible accounts: 508,731

4. LOADING ACCOUNT GENERATION DNA

5. LOADING LEGITIMATE TRANSACTION BEHAVIOR
Behavior sample: 110,000

6. OPENING STREAM OUTPUTS

7. FULL-SCALE GENERATION
Episodes: 500/584,706 | Transactions: 2,611 | New accounts: 397
Episodes: 1,000/584,706 | Transactions: 5,356 | New accounts: 803
Episodes: 1,500/584,706 | Transactions: 8,227 | New accounts: 1,203
Episodes: 2,000/584,706 | Transactions: 11,081 | New accounts: 1,653
Episodes: 2,500/584,706 | Transactions: 13,595 | New accounts: 2,008
Episodes: 3,000/584,706 | Transactions: 16,279 | New accounts: 2,400
Episodes: 3,500/584,706 | Transactions: 19,015 | New accounts: 2,794
Episodes: 4,000/584,706 | Transactions: 21,628 | New accounts: 3,17

KeyboardInterrupt: 

In [31]:
# ==================================================================================================
# CELL 21 — STREAMING FINAL SYNTHETIC DATA AUDIT
# FULL REPLACEMENT — AUTO-DISCOVERS ORIGINAL ACCOUNTS FILE
#
# IMPORTANT:
#   - DO NOT regenerate anything
#   - Uses the already-generated full synthetic dataset
#   - Automatically searches Kaggle input/workspace for HI-Small_accounts.csv
#   - Creates the missing manifest
# ==================================================================================================

import os
import csv
import json
import re
import math
import glob
import subprocess
from collections import Counter
from datetime import datetime


# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

BASE = "/kaggle/working/AML_GNN_ONLY/data"

TRANSACTIONS_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_transactions.csv"
)

NEW_ACCOUNTS_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_new_accounts.csv"
)

PATTERNS_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_Patterns.txt"
)

EPISODE_AUDIT_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_episode_audit.csv"
)

MANIFEST_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_manifest.json"
)

LEGIT_PAIRS_FILE = os.path.join(
    BASE,
    "HI-Small_legitimate_bank_pairs.csv"
)

EXPECTED_TX = 5_067_991
EXPECTED_EPISODES = 584_706

EXPECTED_TYPOLOGY_TX = {
    "BIPARTITE":        415_357,
    "CYCLE":            453_261,
    "FAN-IN":           502_219,
    "FAN-OUT":          540_123,
    "GATHER-SCATTER": 1_130_782,
    "RANDOM":           301_647,
    "SCATTER-GATHER":   988_646,
    "STACK":            735_956,
}


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def canonical_bank(x):
    if x is None:
        return None

    s = str(x).strip()

    if not s:
        return None

    if s.isdigit():
        return str(int(s))

    return s.upper()


def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None


def safe_int(x):
    try:
        return int(x)
    except Exception:
        return None


def find_column(header, candidates):

    if header is None:
        return None

    lower_map = {
        str(h).strip().lower(): h
        for h in header
    }

    # Exact normalized match
    for c in candidates:

        key = c.strip().lower()

        if key in lower_map:
            return lower_map[key]

    # Substring fallback
    for h in header:

        h_low = str(h).strip().lower()

        for c in candidates:

            if c.strip().lower() in h_low:
                return h

    return None


def require_file(path, label):

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Missing required {label}: {path}"
        )


# --------------------------------------------------------------------------------------------------
# ORIGINAL HI-SMALL ACCOUNTS FILE AUTO-DISCOVERY
# --------------------------------------------------------------------------------------------------

def discover_original_accounts():

    candidates = []

    search_roots = [
        "/kaggle/input",
        "/kaggle/working",
    ]

    for root in search_roots:

        if not os.path.exists(root):
            continue

        pattern = os.path.join(
            root,
            "**",
            "HI-Small_accounts.csv"
        )

        try:

            matches = glob.glob(
                pattern,
                recursive=True
            )

            candidates.extend(matches)

        except Exception:
            pass


    # Remove duplicates
    candidates = sorted(
        set(
            os.path.abspath(x)
            for x in candidates
            if os.path.isfile(x)
        )
    )


    # Prefer exact file in BASE if ever present
    base_exact = os.path.join(
        BASE,
        "HI-Small_accounts.csv"
    )

    if os.path.isfile(base_exact):
        return base_exact, candidates


    if len(candidates) == 0:
        return None, candidates


    # Prefer files under Kaggle input
    input_candidates = [
        x for x in candidates
        if x.startswith("/kaggle/input/")
    ]

    if len(input_candidates) == 1:
        return input_candidates[0], candidates

    if len(input_candidates) > 1:

        # Exact filename already guaranteed.
        # Pick the smallest path depth / deterministic first.
        input_candidates = sorted(
            input_candidates,
            key=lambda x: (
                x.count(os.sep),
                x
            )
        )

        return input_candidates[0], candidates


    return candidates[0], candidates


# --------------------------------------------------------------------------------------------------
# START
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("CELL 21 — STREAMING FINAL SYNTHETIC DATA AUDIT")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. OUTPUT FILE CHECK
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. OUTPUT FILE CHECK")
print("=" * 80)

required_outputs = {
    "transactions": TRANSACTIONS_FILE,
    "new_accounts": NEW_ACCOUNTS_FILE,
    "patterns": PATTERNS_FILE,
    "episode_audit": EPISODE_AUDIT_FILE,
    "legitimate_pairs": LEGIT_PAIRS_FILE,
}

for label, path in required_outputs.items():

    exists = os.path.exists(path)

    if exists:

        print(
            f"{label:18s} | exists=True  | "
            f"size={file_size_mb(path):,.2f} MB | "
            f"{os.path.basename(path)}"
        )

    else:

        print(
            f"{label:18s} | exists=False | "
            f"{os.path.basename(path)}"
        )


for label, path in required_outputs.items():

    require_file(
        path,
        label
    )


print(
    f"manifest           | exists={os.path.exists(MANIFEST_FILE)} | "
    f"{os.path.basename(MANIFEST_FILE)}"
)


# --------------------------------------------------------------------------------------------------
# 2. TRANSACTION CSV STRUCTURE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. TRANSACTION CSV STRUCTURE")
print("=" * 80)


with open(
    TRANSACTIONS_FILE,
    "r",
    newline="",
    encoding="utf-8"
) as f:

    reader = csv.reader(f)
    tx_header = next(reader)


print("Columns:")

for i, col in enumerate(tx_header):

    print(
        f"  [{i}] {col}"
    )


tx_id_col = find_column(
    tx_header,
    [
        "transaction_id",
        "Transaction ID",
        "transaction id"
    ]
)

timestamp_col = find_column(
    tx_header,
    ["Timestamp"]
)

from_bank_col = find_column(
    tx_header,
    ["From Bank", "from_bank"]
)

account_col = find_column(
    tx_header,
    ["Account"]
)

to_bank_col = find_column(
    tx_header,
    ["To Bank", "to_bank"]
)

account1_col = find_column(
    tx_header,
    [
        "Account.1",
        "Account1",
        "To Account",
        "to_account"
    ]
)

amount_received_col = find_column(
    tx_header,
    [
        "Amount Received",
        "amount_received"
    ]
)

amount_paid_col = find_column(
    tx_header,
    [
        "Amount Paid",
        "amount_paid"
    ]
)

recv_currency_col = find_column(
    tx_header,
    [
        "Receiving Currency",
        "receiving_currency"
    ]
)

pay_currency_col = find_column(
    tx_header,
    [
        "Payment Currency",
        "payment_currency"
    ]
)

payment_format_col = find_column(
    tx_header,
    [
        "Payment Format",
        "payment_format"
    ]
)

label_col = find_column(
    tx_header,
    [
        "Is Laundering",
        "is_laundering"
    ]
)

synthetic_col = find_column(
    tx_header,
    [
        "is_synthetic",
        "Is Synthetic"
    ]
)

synthetic_episode_col = find_column(
    tx_header,
    [
        "synthetic_episode_id",
        "Synthetic Episode ID"
    ]
)

typology_col = find_column(
    tx_header,
    [
        "typology",
        "Typology"
    ]
)

template_col = find_column(
    tx_header,
    [
        "source_template_id",
        "Source Template ID"
    ]
)

template_order_col = find_column(
    tx_header,
    [
        "template_source_order",
        "Template Source Order"
    ]
)


required_columns = {
    "transaction_id": tx_id_col,
    "Timestamp": timestamp_col,
    "From Bank": from_bank_col,
    "Account": account_col,
    "To Bank": to_bank_col,
    "Account.1": account1_col,
    "Amount Received": amount_received_col,
    "Amount Paid": amount_paid_col,
    "Receiving Currency": recv_currency_col,
    "Payment Currency": pay_currency_col,
    "Payment Format": payment_format_col,
    "Is Laundering": label_col,
}


missing_cols = [
    name
    for name, value in required_columns.items()
    if value is None
]


if missing_cols:

    raise RuntimeError(
        "Missing required transaction columns: "
        + ", ".join(missing_cols)
    )


print("\n✅ All required transaction columns found.")


# --------------------------------------------------------------------------------------------------
# 3. FULL TRANSACTION CSV STREAMING AUDIT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. FULL TRANSACTION CSV STREAMING AUDIT")
print("=" * 80)


header_idx = {
    name: i
    for i, name in enumerate(tx_header)
}


idx_id = header_idx[tx_id_col]
idx_ts = header_idx[timestamp_col]
idx_fb = header_idx[from_bank_col]
idx_acc = header_idx[account_col]
idx_tb = header_idx[to_bank_col]
idx_acc1 = header_idx[account1_col]
idx_ar = header_idx[amount_received_col]
idx_ap = header_idx[amount_paid_col]
idx_rc = header_idx[recv_currency_col]
idx_pc = header_idx[pay_currency_col]
idx_pf = header_idx[payment_format_col]
idx_label = header_idx[label_col]

idx_synth = (
    header_idx[synthetic_col]
    if synthetic_col
    else None
)

idx_episode = (
    header_idx[synthetic_episode_col]
    if synthetic_episode_col
    else None
)

idx_typology = (
    header_idx[typology_col]
    if typology_col
    else None
)

idx_template = (
    header_idx[template_col]
    if template_col
    else None
)


tx_count = 0
malformed_rows = 0
blank_ids = 0
invalid_labels = 0
invalid_synthetic_flags = 0
invalid_episode_ids = 0
blank_typology = 0
blank_template_ids = 0

min_amount_received = math.inf
max_amount_received = -math.inf

min_amount_paid = math.inf
max_amount_paid = -math.inf

min_timestamp = None
max_timestamp = None

format_counts = Counter()
payment_currency_counts = Counter()
receiving_currency_counts = Counter()
typology_counts = Counter()
template_counts = Counter()

unique_episode_ids = set()

source_banks = set()
destination_banks = set()
bank_pairs = set()


id_temp = os.path.join(
    BASE,
    "_HI-Small_SYNTH_FULL_tx_ids.tmp"
)

id_sorted = os.path.join(
    BASE,
    "_HI-Small_SYNTH_FULL_tx_ids_sorted.tmp"
)


with open(
    TRANSACTIONS_FILE,
    "r",
    newline="",
    encoding="utf-8"
) as f, open(
    id_temp,
    "w",
    encoding="utf-8"
) as id_out:

    reader = csv.reader(f)

    next(reader)

    for row in reader:

        tx_count += 1


        if len(row) != len(tx_header):

            malformed_rows += 1
            continue


        tx_id = row[idx_id].strip()

        if not tx_id:

            blank_ids += 1

        else:

            id_out.write(
                tx_id + "\n"
            )


        ts = row[idx_ts].strip()

        if ts:

            if (
                min_timestamp is None
                or ts < min_timestamp
            ):
                min_timestamp = ts

            if (
                max_timestamp is None
                or ts > max_timestamp
            ):
                max_timestamp = ts


        ar = safe_float(
            row[idx_ar]
        )

        ap = safe_float(
            row[idx_ap]
        )


        if (
            ar is None
            or not math.isfinite(ar)
            or ar <= 0
        ):

            invalid_labels += 1

        else:

            min_amount_received = min(
                min_amount_received,
                ar
            )

            max_amount_received = max(
                max_amount_received,
                ar
            )


        if (
            ap is None
            or not math.isfinite(ap)
            or ap <= 0
        ):

            invalid_labels += 1

        else:

            min_amount_paid = min(
                min_amount_paid,
                ap
            )

            max_amount_paid = max(
                max_amount_paid,
                ap
            )


        label = row[idx_label].strip()

        if label not in {
            "1",
            "1.0"
        }:

            invalid_labels += 1


        if idx_synth is not None:

            syn = row[idx_synth].strip()

            if syn not in {
                "1",
                "1.0",
                "True",
                "true"
            }:

                invalid_synthetic_flags += 1


        if idx_episode is not None:

            episode_id = row[idx_episode].strip()

            if not episode_id:

                invalid_episode_ids += 1

            else:

                unique_episode_ids.add(
                    episode_id
                )


        if idx_typology is not None:

            typ = row[idx_typology].strip().upper()

            if not typ:

                blank_typology += 1

            else:

                typology_counts[
                    typ
                ] += 1


        if idx_template is not None:

            template_id = row[
                idx_template
            ].strip()

            if not template_id:

                blank_template_ids += 1

            else:

                template_counts[
                    template_id
                ] += 1


        format_counts[
            row[idx_pf].strip()
        ] += 1

        payment_currency_counts[
            row[idx_pc].strip()
        ] += 1

        receiving_currency_counts[
            row[idx_rc].strip()
        ] += 1


        src_bank = canonical_bank(
            row[idx_fb]
        )

        dst_bank = canonical_bank(
            row[idx_tb]
        )


        source_banks.add(
            src_bank
        )

        destination_banks.add(
            dst_bank
        )

        bank_pairs.add(
            (
                src_bank,
                dst_bank
            )
        )


print(
    f"Transaction rows                : {tx_count:,}"
)

print(
    f"Expected                         : {EXPECTED_TX:,}"
)

print(
    f"Malformed rows                  : {malformed_rows:,}"
)

print(
    f"Blank transaction IDs           : {blank_ids:,}"
)

print(
    f"Invalid laundering labels       : {invalid_labels:,}"
)

print(
    f"Invalid synthetic flags         : {invalid_synthetic_flags:,}"
)

print(
    f"Invalid episode IDs             : {invalid_episode_ids:,}"
)

print(
    f"Blank typology values           : {blank_typology:,}"
)

print(
    f"Blank source-template IDs       : {blank_template_ids:,}"
)

print(
    f"Unique synthetic episode IDs    : "
    f"{len(unique_episode_ids):,}"
)

print(
    f"Expected synthetic episodes     : "
    f"{EXPECTED_EPISODES:,}"
)

print(
    f"Unique source templates         : "
    f"{len(template_counts):,}"
)

print(
    f"Min Timestamp                   : "
    f"{min_timestamp}"
)

print(
    f"Max Timestamp                   : "
    f"{max_timestamp}"
)

print(
    f"Min Amount Received             : "
    f"{min_amount_received}"
)

print(
    f"Max Amount Received             : "
    f"{max_amount_received}"
)

print(
    f"Min Amount Paid                 : "
    f"{min_amount_paid}"
)

print(
    f"Max Amount Paid                 : "
    f"{max_amount_paid}"
)

print(
    f"Unique source banks             : "
    f"{len(source_banks):,}"
)

print(
    f"Unique destination banks        : "
    f"{len(destination_banks):,}"
)

print(
    f"Unique generated bank pairs     : "
    f"{len(bank_pairs):,}"
)


print("\nPayment Format:")

for k, v in format_counts.most_common():

    print(
        f"  {k:20s} "
        f"{v:12,} "
        f"({100*v/tx_count:8.4f}%)"
    )


print("\nPayment Currency:")

for k, v in payment_currency_counts.most_common():

    print(
        f"  {k:20s} "
        f"{v:12,} "
        f"({100*v/tx_count:8.4f}%)"
    )


print("\nReceiving Currency:")

for k, v in receiving_currency_counts.most_common():

    print(
        f"  {k:20s} "
        f"{v:12,} "
        f"({100*v/tx_count:8.4f}%)"
    )


print("\nTypology counts in transaction CSV:")

for typ in sorted(
    typology_counts
):

    expected = EXPECTED_TYPOLOGY_TX.get(
        typ,
        "N/A"
    )

    print(
        f"  {typ:18s} "
        f"{typology_counts[typ]:10,} "
        f"| expected={expected}"
    )


# --------------------------------------------------------------------------------------------------
# 4. GLOBAL TRANSACTION ID UNIQUENESS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. GLOBAL TRANSACTION ID UNIQUENESS")
print("=" * 80)


print(
    "Running external unique sort on transaction IDs..."
)


result = subprocess.run(
    [
        "sort",
        "-u",
        id_temp,
        "-o",
        id_sorted
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)


if result.returncode != 0:

    raise RuntimeError(
        "External transaction-ID sort failed:\n"
        + result.stderr
    )


with open(
    id_sorted,
    "r",
    encoding="utf-8"
) as f:

    unique_tx_ids = sum(
        1 for _ in f
    )


duplicate_tx_ids = (
    tx_count - unique_tx_ids
)


print(
    f"Total transaction rows : "
    f"{tx_count:,}"
)

print(
    f"Unique transaction IDs  : "
    f"{unique_tx_ids:,}"
)

print(
    f"Duplicate transaction IDs: "
    f"{duplicate_tx_ids:,}"
)


os.remove(id_temp)
os.remove(id_sorted)


tx_id_gate = (
    tx_count == EXPECTED_TX
    and unique_tx_ids == EXPECTED_TX
    and blank_ids == 0
)


print(
    "✅ PASS"
    if tx_id_gate
    else "❌ FAIL"
)


# --------------------------------------------------------------------------------------------------
# 5. LEGITIMATE BANK-PAIR COVERAGE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. LEGITIMATE BANK-PAIR COVERAGE")
print("=" * 80)


legitimate_pairs = set()


with open(
    LEGIT_PAIRS_FILE,
    "r",
    newline="",
    encoding="utf-8"
) as f:

    reader = csv.DictReader(f)

    pair_header = reader.fieldnames

    fb_col = find_column(
        pair_header,
        [
            "From Bank",
            "from_bank",
            "source_bank",
            "Source Bank"
        ]
    )

    tb_col = find_column(
        pair_header,
        [
            "To Bank",
            "to_bank",
            "destination_bank",
            "Destination Bank"
        ]
    )


    if (
        fb_col is None
        or tb_col is None
    ):

        raise RuntimeError(
            "Could not identify bank columns "
            "in legitimate bank-pair file."
        )


    for row in reader:

        src = canonical_bank(
            row[fb_col]
        )

        dst = canonical_bank(
            row[tb_col]
        )

        if (
            src is not None
            and dst is not None
        ):

            legitimate_pairs.add(
                (src, dst)
            )


print(
    f"Legitimate bank pairs available : "
    f"{len(legitimate_pairs):,}"
)

print(
    f"Generated unique bank pairs      : "
    f"{len(bank_pairs):,}"
)


covered_pairs = sum(
    1
    for pair in bank_pairs
    if pair in legitimate_pairs
)


missing_pairs = (
    bank_pairs - legitimate_pairs
)


coverage_pct = (
    100.0 * covered_pairs / len(bank_pairs)
    if bank_pairs
    else 0.0
)


print(
    f"Observed in legitimate network  : "
    f"{covered_pairs:,}"
)

print(
    f"Unseen generated bank pairs      : "
    f"{len(missing_pairs):,}"
)

print(
    f"Coverage                         : "
    f"{coverage_pct:.6f}%"
)


if missing_pairs:

    print(
        "\nFirst 20 missing generated "
        "bank pairs:"
    )

    for pair in list(
        missing_pairs
    )[:20]:

        print(
            "  ",
            pair
        )


bank_pair_gate = (
    len(missing_pairs) == 0
)


print(
    "\n✅ PASS"
    if bank_pair_gate
    else "\n❌ FAIL"
)


# --------------------------------------------------------------------------------------------------
# 6. NEW ACCOUNT FILE AUDIT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. NEW ACCOUNT FILE AUDIT")
print("=" * 80)


account_count = 0

account_format_errors = 0
account_bank_missing = 0
account_entity_missing = 0
duplicate_account_keys = 0
duplicate_entity_ids = 0

account_key_seen = set()
entity_id_seen = set()


account_id_pattern = re.compile(
    r"^[A-Z0-9]{9}$"
)

entity_id_pattern = re.compile(
    r"^[A-Z0-9]{9}$"
)


with open(
    NEW_ACCOUNTS_FILE,
    "r",
    newline="",
    encoding="utf-8"
) as f:

    reader = csv.DictReader(f)

    account_header = reader.fieldnames


    bank_id_col = find_column(
        account_header,
        [
            "Bank ID",
            "Bank_ID",
            "bank_id",
            "bank"
        ]
    )

    account_num_col = find_column(
        account_header,
        [
            "Account Number",
            "Account",
            "account_number"
        ]
    )

    entity_id_col = find_column(
        account_header,
        [
            "Entity ID",
            "Entity_ID",
            "entity_id"
        ]
    )

    entity_name_col = find_column(
        account_header,
        [
            "Entity Name",
            "Entity_Name",
            "entity_name"
        ]
    )


    if (
        bank_id_col is None
        or account_num_col is None
        or entity_id_col is None
        or entity_name_col is None
    ):

        raise RuntimeError(
            "Could not identify all required "
            "new-account columns.\n"
            + str(account_header)
        )


    for row in reader:

        account_count += 1


        bank_id = canonical_bank(
            row[bank_id_col]
        )

        account_num = str(
            row[account_num_col]
        ).strip()

        entity_id = str(
            row[entity_id_col]
        ).strip()

        entity_name = str(
            row[entity_name_col]
        ).strip()


        if bank_id is None:
            account_bank_missing += 1


        if not account_id_pattern.match(
            account_num
        ):

            account_format_errors += 1


        if not entity_id_pattern.match(
            entity_id
        ):

            account_format_errors += 1


        if not entity_name:

            account_entity_missing += 1


        key = (
            bank_id,
            account_num
        )


        if key in account_key_seen:

            duplicate_account_keys += 1

        else:

            account_key_seen.add(
                key
            )


        if entity_id in entity_id_seen:

            duplicate_entity_ids += 1

        else:

            entity_id_seen.add(
                entity_id
            )


print(
    f"New account rows                 : "
    f"{account_count:,}"
)

print(
    f"Unique bank+account keys         : "
    f"{len(account_key_seen):,}"
)

print(
    f"Unique entity IDs                : "
    f"{len(entity_id_seen):,}"
)

print(
    f"Account format errors            : "
    f"{account_format_errors:,}"
)

print(
    f"Duplicate bank+account keys      : "
    f"{duplicate_account_keys:,}"
)

print(
    f"Duplicate entity IDs             : "
    f"{duplicate_entity_ids:,}"
)

print(
    f"Missing bank IDs                 : "
    f"{account_bank_missing:,}"
)

print(
    f"Missing entity names             : "
    f"{account_entity_missing:,}"
)


new_account_gate = (
    account_format_errors == 0
    and duplicate_account_keys == 0
    and account_bank_missing == 0
    and account_entity_missing == 0
)


print(
    "\n✅ PASS"
    if new_account_gate
    else "\n❌ FAIL"
)


# --------------------------------------------------------------------------------------------------
# 7. DISCOVER ORIGINAL HI-SMALL ACCOUNTS FILE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. ORIGINAL HI-SMALL ACCOUNTS FILE DISCOVERY")
print("=" * 80)


ORIGINAL_ACCOUNTS_FILE, accounts_candidates = (
    discover_original_accounts()
)


print(
    f"Candidates found: "
    f"{len(accounts_candidates)}"
)


for candidate in accounts_candidates:

    print(
        "  ",
        candidate
    )


if ORIGINAL_ACCOUNTS_FILE is None:

    print(
        "\n⚠️ ORIGINAL HI-SMALL ACCOUNTS FILE "
        "COULD NOT BE FOUND."
    )

    print(
        "The rest of the audit will continue, "
        "but the original-collision gate will "
        "be marked FAIL."
    )

    collision_gate = False

    original_collisions = None

    original_keys_count = None

else:

    print(
        "\nUsing original accounts file:"
    )

    print(
        ORIGINAL_ACCOUNTS_FILE
    )


    original_keys = set()


    with open(
        ORIGINAL_ACCOUNTS_FILE,
        "r",
        newline="",
        encoding="utf-8"
    ) as f:

        reader = csv.DictReader(f)

        original_header = reader.fieldnames


        original_bank_col = find_column(
            original_header,
            [
                "Bank ID",
                "Bank_ID",
                "bank_id",
                "bank"
            ]
        )

        original_account_col = find_column(
            original_header,
            [
                "Account Number",
                "Account",
                "account_number"
            ]
        )


        if (
            original_bank_col is None
            or original_account_col is None
        ):

            raise RuntimeError(
                "Could not identify original "
                "account columns:\n"
                + str(original_header)
            )


        for row in reader:

            original_keys.add(
                (
                    canonical_bank(
                        row[original_bank_col]
                    ),
                    str(
                        row[original_account_col]
                    ).strip()
                )
            )


    original_keys_count = (
        len(original_keys)
    )


    original_collisions = len(
        account_key_seen.intersection(
            original_keys
        )
    )


    print(
        f"Original bank+account keys      : "
        f"{original_keys_count:,}"
    )

    print(
        f"New/original collisions          : "
        f"{original_collisions:,}"
    )


    collision_gate = (
        original_collisions == 0
    )


    print(
        "\n✅ PASS"
        if collision_gate
        else "\n❌ FAIL"
    )


# --------------------------------------------------------------------------------------------------
# 8. PATTERNS FILE STREAMING AUDIT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. PATTERNS FILE VALIDATION")
print("=" * 80)


pattern_episode_ids = 0
pattern_begin = 0
pattern_end = 0
pattern_tx = 0

pattern_malformed_tx = 0
pattern_label_errors = 0

pattern_typology_episodes = Counter()
pattern_typology_tx = Counter()

current_typology = None
current_episode_tx = 0

pattern_id_values = set()


with open(
    PATTERNS_FILE,
    "r",
    encoding="utf-8"
) as f:

    for raw in f:

        line = raw.strip()


        if not line:
            continue


        if line.startswith(
            'id="'
        ):

            pattern_episode_ids += 1

            pattern_id_values.add(
                line
            )

            continue


        if line.startswith(
            "BEGIN LAUNDERING ATTEMPT"
        ):

            pattern_begin += 1


            m = re.search(
                r"BEGIN LAUNDERING ATTEMPT\s*-\s*([A-Z-]+)",
                line
            )


            current_typology = (
                m.group(1)
                if m
                else "UNKNOWN"
            )


            pattern_typology_episodes[
                current_typology
            ] += 1


            current_episode_tx = 0

            continue


        if line.startswith(
            "END LAUNDERING ATTEMPT"
        ):

            pattern_end += 1


            if current_typology is not None:

                pattern_typology_tx[
                    current_typology
                ] += current_episode_tx


            current_typology = None

            current_episode_tx = 0

            continue


        if current_typology is not None:

            parts = line.split(",")


            if len(parts) != 11:

                pattern_malformed_tx += 1

            else:

                try:

                    datetime.strptime(
                        parts[0].strip(),
                        "%Y/%m/%d %H:%M"
                    )

                except Exception:

                    pattern_malformed_tx += 1


                if parts[-1].strip() != "1":

                    pattern_label_errors += 1


            current_episode_tx += 1

            pattern_tx += 1


print(
    f"Pattern episode ID lines        : "
    f"{pattern_episode_ids:,}"
)

print(
    f"Unique pattern ID lines         : "
    f"{len(pattern_id_values):,}"
)

print(
    f"BEGIN episode markers           : "
    f"{pattern_begin:,}"
)

print(
    f"END episode markers             : "
    f"{pattern_end:,}"
)

print(
    f"Pattern transaction lines       : "
    f"{pattern_tx:,}"
)

print(
    f"Malformed pattern tx rows       : "
    f"{pattern_malformed_tx:,}"
)

print(
    f"Pattern label != 1              : "
    f"{pattern_label_errors:,}"
)


print("\nPattern transactions by typology:")


for typ in sorted(
    pattern_typology_tx
):

    actual = pattern_typology_tx[
        typ
    ]

    expected = EXPECTED_TYPOLOGY_TX.get(
        typ,
        "N/A"
    )

    print(
        f"  {typ:18s} "
        f"episodes="
        f"{pattern_typology_episodes[typ]:7,} "
        f"transactions="
        f"{actual:9,} "
        f"expected="
        f"{expected}"
    )


pattern_gate = (
    pattern_episode_ids == EXPECTED_EPISODES
    and len(pattern_id_values) == EXPECTED_EPISODES
    and pattern_begin == EXPECTED_EPISODES
    and pattern_end == EXPECTED_EPISODES
    and pattern_tx == EXPECTED_TX
    and pattern_malformed_tx == 0
    and pattern_label_errors == 0
    and all(
        pattern_typology_tx.get(
            typ,
            -1
        ) == expected
        for typ, expected
        in EXPECTED_TYPOLOGY_TX.items()
    )
)


print(
    "\n✅ PASS"
    if pattern_gate
    else "\n❌ FAIL"
)


# --------------------------------------------------------------------------------------------------
# 9. CSV ↔ PATTERNS TYPOLOGY CONSISTENCY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("9. CSV ↔ PATTERNS TYPOLOGY CONSISTENCY")
print("=" * 80)


csv_typology_gate = True


for typ, expected in EXPECTED_TYPOLOGY_TX.items():

    csv_count = typology_counts.get(
        typ,
        0
    )

    pattern_count = pattern_typology_tx.get(
        typ,
        0
    )


    passed = (
        csv_count == expected
        and pattern_count == expected
    )


    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} "
        f"| {typ:18s} "
        f"| CSV={csv_count:,} "
        f"| Patterns={pattern_count:,} "
        f"| Expected={expected:,}"
    )


    csv_typology_gate &= passed


print(
    "\n✅ PASS"
    if csv_typology_gate
    else "\n❌ FAIL"
)


# --------------------------------------------------------------------------------------------------
# 10. EPISODE AUDIT FILE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("10. EPISODE AUDIT FILE")
print("=" * 80)


episode_rows = 0
episode_tx_sum = 0

episode_typology_tx = Counter()
episode_typology_ep = Counter()


with open(
    EPISODE_AUDIT_FILE,
    "r",
    newline="",
    encoding="utf-8"
) as f:

    reader = csv.DictReader(f)

    eh = reader.fieldnames


    etyp = find_column(
        eh,
        [
            "typology",
            "Typology",
            "pattern_type",
            "pattern"
        ]
    )


    etx = find_column(
        eh,
        [
            "transaction_count",
            "tx_count",
            "transactions",
            "n_transactions"
        ]
    )


    if etx is None:

        raise RuntimeError(
            "Could not find transaction-count "
            "column in episode audit."
        )


    for row in reader:

        episode_rows += 1


        n = safe_int(
            row[etx]
        )


        if n is not None:

            episode_tx_sum += n


            if etyp is not None:

                typ = row[
                    etyp
                ].strip().upper()


                episode_typology_tx[
                    typ
                ] += n


                episode_typology_ep[
                    typ
                ] += 1


print(
    f"Episode-audit rows               : "
    f"{episode_rows:,}"
)

print(
    f"Expected episodes                : "
    f"{EXPECTED_EPISODES:,}"
)

print(
    f"Episode-audit transaction sum    : "
    f"{episode_tx_sum:,}"
)

print(
    f"Expected transactions            : "
    f"{EXPECTED_TX:,}"
)


if etyp is not None:

    print(
        "\nEpisode-audit typology totals:"
    )


    for typ in sorted(
        episode_typology_tx
    ):

        print(
            f"  {typ:18s} "
            f"episodes="
            f"{episode_typology_ep[typ]:7,} "
            f"transactions="
            f"{episode_typology_tx[typ]:9,}"
        )


episode_gate = (
    episode_rows == EXPECTED_EPISODES
    and episode_tx_sum == EXPECTED_TX
)


print(
    "\n✅ PASS"
    if episode_gate
    else "\n❌ FAIL"
)


# --------------------------------------------------------------------------------------------------
# 11. FINAL GLOBAL GATES
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("11. FINAL GLOBAL GATES")
print("=" * 80)


amount_gate = (
    min_amount_received > 0
    and math.isfinite(
        max_amount_received
    )
    and min_amount_paid > 0
    and math.isfinite(
        max_amount_paid
    )
)


episode_csv_gate = (
    idx_episode is not None
    and invalid_episode_ids == 0
    and len(unique_episode_ids) == EXPECTED_EPISODES
)


typology_gate = (
    idx_typology is not None
    and blank_typology == 0
    and csv_typology_gate
)


template_gate = (
    idx_template is not None
    and blank_template_ids == 0
)


gates = {

    "Exact transaction row count":
        tx_count == EXPECTED_TX,

    "Global transaction IDs unique":
        tx_id_gate,

    "No malformed transaction rows":
        malformed_rows == 0,

    "All laundering labels = 1":
        invalid_labels == 0,

    "All synthetic flags valid":
        (
            invalid_synthetic_flags == 0
            if synthetic_col is not None
            else True
        ),

    "Positive finite amounts":
        amount_gate,

    "Exact synthetic episode count":
        episode_csv_gate,

    "Typology integrity":
        typology_gate,

    "Template IDs present":
        template_gate,

    "Legitimate bank-pair coverage":
        bank_pair_gate,

    "New account integrity":
        new_account_gate,

    "No new/original account collisions":
        collision_gate,

    "Patterns file exact":
        pattern_gate,

    "CSV ↔ Patterns typology match":
        csv_typology_gate,

    "Episode audit exact":
        episode_gate,
}


all_pass = True


for name, passed in gates.items():

    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} "
        f"| {name}"
    )

    all_pass &= passed


# --------------------------------------------------------------------------------------------------
# 12. CREATE MANIFEST
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("12. CREATING MANIFEST")
print("=" * 80)


manifest = {

    "dataset":
        "IBM HI-Small",

    "generation_status":
        "COMPLETED",

    "audit_status":
        "PASS" if all_pass else "FAIL",


    "generation": {

        "synthetic_episodes":
            EXPECTED_EPISODES,

        "synthetic_transactions":
            EXPECTED_TX,

        "new_accounts_generated":
            account_count,
    },


    "expected_typology_transactions":
        EXPECTED_TYPOLOGY_TX,


    "resolved_source_files": {

        "original_accounts_file":
            ORIGINAL_ACCOUNTS_FILE,

        "legitimate_bank_pairs_file":
            LEGIT_PAIRS_FILE,
    },


    "outputs": {

        "transactions":
            os.path.basename(
                TRANSACTIONS_FILE
            ),

        "new_accounts":
            os.path.basename(
                NEW_ACCOUNTS_FILE
            ),

        "patterns":
            os.path.basename(
                PATTERNS_FILE
            ),

        "episode_audit":
            os.path.basename(
                EPISODE_AUDIT_FILE
            ),

        "manifest":
            os.path.basename(
                MANIFEST_FILE
            ),
    },


    "file_sizes_mb": {

        "transactions":
            round(
                file_size_mb(
                    TRANSACTIONS_FILE
                ),
                2
            ),

        "new_accounts":
            round(
                file_size_mb(
                    NEW_ACCOUNTS_FILE
                ),
                2
            ),

        "patterns":
            round(
                file_size_mb(
                    PATTERNS_FILE
                ),
                2
            ),

        "episode_audit":
            round(
                file_size_mb(
                    EPISODE_AUDIT_FILE
                ),
                2
            ),
    },


    "transaction_audit": {

        "rows":
            tx_count,

        "expected_rows":
            EXPECTED_TX,

        "unique_transaction_ids":
            unique_tx_ids,

        "duplicate_transaction_ids":
            duplicate_tx_ids,

        "malformed_rows":
            malformed_rows,

        "blank_transaction_ids":
            blank_ids,

        "invalid_laundering_labels":
            invalid_labels,

        "invalid_synthetic_flags":
            invalid_synthetic_flags,

        "unique_synthetic_episode_ids":
            len(unique_episode_ids),

        "unique_templates":
            len(template_counts),

        "min_timestamp":
            min_timestamp,

        "max_timestamp":
            max_timestamp,

        "min_amount_received":
            min_amount_received,

        "max_amount_received":
            max_amount_received,

        "min_amount_paid":
            min_amount_paid,

        "max_amount_paid":
            max_amount_paid,

        "unique_source_banks":
            len(source_banks),

        "unique_destination_banks":
            len(destination_banks),

        "unique_generated_bank_pairs":
            len(bank_pairs),

        "payment_format_counts":
            dict(format_counts),

        "payment_currency_counts":
            dict(payment_currency_counts),

        "receiving_currency_counts":
            dict(receiving_currency_counts),

        "typology_counts":
            dict(typology_counts),

        "template_counts":
            dict(template_counts),
    },


    "bank_network_audit": {

        "legitimate_pairs":
            len(legitimate_pairs),

        "generated_unique_pairs":
            len(bank_pairs),

        "covered_pairs":
            covered_pairs,

        "missing_pairs":
            len(missing_pairs),

        "coverage_percent":
            coverage_pct,
    },


    "new_account_audit": {

        "rows":
            account_count,

        "unique_bank_account_keys":
            len(account_key_seen),

        "unique_entity_ids":
            len(entity_id_seen),

        "format_errors":
            account_format_errors,

        "duplicate_bank_account_keys":
            duplicate_account_keys,

        "duplicate_entity_ids":
            duplicate_entity_ids,

        "missing_bank_ids":
            account_bank_missing,

        "missing_entity_names":
            account_entity_missing,

        "original_collisions":
            original_collisions,

        "original_account_keys":
            original_keys_count,
    },


    "patterns_audit": {

        "episode_id_lines":
            pattern_episode_ids,

        "unique_episode_id_lines":
            len(pattern_id_values),

        "begin_markers":
            pattern_begin,

        "end_markers":
            pattern_end,

        "transaction_lines":
            pattern_tx,

        "malformed_transaction_lines":
            pattern_malformed_tx,

        "label_errors":
            pattern_label_errors,

        "typology_episodes":
            dict(
                pattern_typology_episodes
            ),

        "typology_transactions":
            dict(
                pattern_typology_tx
            ),
    },


    "episode_audit": {

        "rows":
            episode_rows,

        "transaction_sum":
            episode_tx_sum,

        "typology_transactions":
            dict(
                episode_typology_tx
            ),

        "typology_episodes":
            dict(
                episode_typology_ep
            ),
    },


    "gates":
        gates,
}


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


print(
    "Manifest written to:"
)

print(
    MANIFEST_FILE
)

print(
    f"Manifest size: "
    f"{file_size_mb(MANIFEST_FILE):.4f} MB"
)


# --------------------------------------------------------------------------------------------------
# 13. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)


if all_pass:

    print(
        "✅✅✅ FINAL SYNTHETIC DATASET AUDIT PASSED ✅✅✅"
    )

    print(
        "=" * 100
    )

    print(
        f"Synthetic transactions : "
        f"{tx_count:,}"
    )

    print(
        f"Synthetic episodes     : "
        f"{len(unique_episode_ids):,}"
    )

    print(
        f"New accounts           : "
        f"{account_count:,}"
    )

    print(
        f"Patterns transactions  : "
        f"{pattern_tx:,}"
    )

    print(
        f"Generated bank pairs   : "
        f"{len(bank_pairs):,}"
    )

    print(
        f"Bank-pair coverage     : "
        f"{coverage_pct:.6f}%"
    )

    print(
        "Manifest               : CREATED"
    )

    print(
        "\n✅ Structural generation integrity is complete."
    )

else:

    print(
        "❌ FINAL SYNTHETIC DATASET AUDIT HAS FAILURES"
    )

    print(
        "=" * 100
    )

    print(
        "Do NOT proceed to GNN training yet."
    )

    print(
        "Inspect the failed gate(s) above."
    )


print("=" * 100)

CELL 21 — STREAMING FINAL SYNTHETIC DATA AUDIT

1. OUTPUT FILE CHECK
transactions       | exists=True  | size=844.22 MB | HI-Small_SYNTH_FULL_transactions.csv
new_accounts       | exists=True  | size=27.85 MB | HI-Small_SYNTH_FULL_new_accounts.csv
patterns           | exists=True  | size=570.76 MB | HI-Small_SYNTH_FULL_Patterns.txt
episode_audit      | exists=True  | size=43.50 MB | HI-Small_SYNTH_FULL_episode_audit.csv
legitimate_pairs   | exists=True  | size=5.26 MB | HI-Small_legitimate_bank_pairs.csv
manifest           | exists=False | HI-Small_SYNTH_FULL_manifest.json

2. TRANSACTION CSV STRUCTURE
Columns:
  [0] transaction_id
  [1] Timestamp
  [2] From Bank
  [3] Account
  [4] To Bank
  [5] Account.1
  [6] Amount Received
  [7] Receiving Currency
  [8] Amount Paid
  [9] Payment Currency
  [10] Payment Format
  [11] Is Laundering
  [12] is_synthetic
  [13] synthetic_episode_id
  [14] typology
  [15] source_template_id
  [16] template_source_order

✅ All required transaction column

In [32]:
# ==================================================================================================
# CELL 22 — SYNTHETIC vs ORIGINAL HI-SMALL BEHAVIORAL DNA AUDIT
#
# PURPOSE:
#   Compare the generated synthetic laundering data against ORIGINAL HI-Small behavior.
#
#   This cell:
#       - DOES NOT regenerate anything
#       - DOES NOT modify generated files
#       - DOES NOT train a model
#       - Creates an audit report only
#
# Main checks:
#   1. Timestamp/date coverage
#   2. Hour distribution
#   3. Day-of-week distribution
#   4. Amount distribution
#   5. Payment Format
#   6. Payment Currency
#   7. Receiving Currency
#   8. Source-bank concentration
#   9. Destination-bank concentration
#  10. Account activity/reuse
#  11. Cross-bank activity
# ==================================================================================================

import os
import csv
import json
import math
import glob
import random
from collections import Counter, defaultdict
from datetime import datetime


# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

BASE = "/kaggle/working/AML_GNN_ONLY/data"

SYN_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_transactions.csv"
)

OUT_JSON = os.path.join(
    BASE,
    "HI-Small_SYNTH_vs_ORIGINAL_behavioral_dna_audit.json"
)

OUT_CSV = os.path.join(
    BASE,
    "HI-Small_SYNTH_vs_ORIGINAL_behavioral_dna_summary.csv"
)


# --------------------------------------------------------------------------------------------------
# AUTO-DISCOVER ORIGINAL HI-SMALL TRANSACTION FILE
# --------------------------------------------------------------------------------------------------

def discover_original_transactions():

    candidates = []

    roots = [
        "/kaggle/input",
        "/kaggle/working"
    ]

    for root in roots:

        if not os.path.exists(root):
            continue

        pattern = os.path.join(
            root,
            "**",
            "HI-Small_Trans.csv"
        )

        candidates.extend(
            glob.glob(
                pattern,
                recursive=True
            )
        )

    candidates = sorted(
        set(
            os.path.abspath(x)
            for x in candidates
            if os.path.isfile(x)
        )
    )

    if len(candidates) == 0:
        return None

    # Prefer input dataset
    input_candidates = [
        x for x in candidates
        if x.startswith("/kaggle/input/")
    ]

    if input_candidates:
        return input_candidates[0]

    return candidates[0]


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def canonical_bank(x):

    if x is None:
        return None

    s = str(x).strip()

    if not s:
        return None

    if s.isdigit():
        return str(int(s))

    return s.upper()


def find_column(header, candidates):

    low = {
        str(x).strip().lower(): x
        for x in header
    }

    for c in candidates:

        if c.strip().lower() in low:
            return low[c.strip().lower()]

    for h in header:

        hl = str(h).strip().lower()

        for c in candidates:

            if c.strip().lower() in hl:
                return h

    return None


def pct(count, total):

    if total == 0:
        return 0.0

    return 100.0 * count / total


def quantiles_from_sample(values):

    if not values:
        return {}

    values = sorted(values)

    def q(p):

        idx = int(
            p * (len(values) - 1)
        )

        return values[idx]

    return {
        "p01": q(0.01),
        "p05": q(0.05),
        "p10": q(0.10),
        "p25": q(0.25),
        "p50": q(0.50),
        "p75": q(0.75),
        "p90": q(0.90),
        "p95": q(0.95),
        "p99": q(0.99),
        "min_sample": min(values),
        "max_sample": max(values),
    }


def distribution_pct(counter, total):

    return {
        k: {
            "count": v,
            "pct": pct(v, total)
        }
        for k, v in counter.items()
    }


# --------------------------------------------------------------------------------------------------
# START
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("CELL 22 — SYNTHETIC vs ORIGINAL HI-SMALL BEHAVIORAL DNA AUDIT")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# FILE DISCOVERY
# --------------------------------------------------------------------------------------------------

if not os.path.exists(SYN_FILE):

    raise FileNotFoundError(
        f"Missing synthetic transaction file:\n{SYN_FILE}"
    )


ORIGINAL_FILE = discover_original_transactions()


if ORIGINAL_FILE is None:

    raise FileNotFoundError(
        "Could not find original HI-Small_Trans.csv "
        "under /kaggle/input or /kaggle/working."
    )


print("\nSynthetic file:")
print(SYN_FILE)

print("\nOriginal file:")
print(ORIGINAL_FILE)


# --------------------------------------------------------------------------------------------------
# SAMPLE SETTINGS
# --------------------------------------------------------------------------------------------------

SAMPLE_SIZE = 200_000

random.seed(42)


# --------------------------------------------------------------------------------------------------
# GENERIC STATS CONTAINER
# --------------------------------------------------------------------------------------------------

def new_stats():

    return {
        "rows": 0,

        "min_timestamp": None,
        "max_timestamp": None,

        "hour": Counter(),
        "day_of_week": Counter(),
        "date": Counter(),

        "payment_format": Counter(),
        "payment_currency": Counter(),
        "receiving_currency": Counter(),

        "source_bank": Counter(),
        "destination_bank": Counter(),

        "source_account": Counter(),
        "destination_account": Counter(),

        "source_bank_account": Counter(),
        "destination_bank_account": Counter(),

        "amount_sample": [],
    }


# --------------------------------------------------------------------------------------------------
# STREAM FILE
# --------------------------------------------------------------------------------------------------

def stream_stats(
    path,
    stats_name,
    expected_columns=None
):

    stats = new_stats()

    with open(
        path,
        "r",
        newline="",
        encoding="utf-8"
    ) as f:

        reader = csv.DictReader(f)

        header = reader.fieldnames

        if header is None:
            raise RuntimeError(
                f"No header found in {path}"
            )

        timestamp_col = find_column(
            header,
            ["Timestamp"]
        )

        from_bank_col = find_column(
            header,
            ["From Bank", "from_bank"]
        )

        account_col = find_column(
            header,
            ["Account"]
        )

        to_bank_col = find_column(
            header,
            ["To Bank", "to_bank"]
        )

        account1_col = find_column(
            header,
            [
                "Account.1",
                "Account1",
                "To Account",
                "to_account"
            ]
        )

        amount_col = find_column(
            header,
            [
                "Amount Received",
                "amount_received"
            ]
        )

        pay_curr_col = find_column(
            header,
            [
                "Payment Currency",
                "payment_currency"
            ]
        )

        recv_curr_col = find_column(
            header,
            [
                "Receiving Currency",
                "receiving_currency"
            ]
        )

        format_col = find_column(
            header,
            [
                "Payment Format",
                "payment_format"
            ]
        )

        required = {
            "Timestamp": timestamp_col,
            "From Bank": from_bank_col,
            "Account": account_col,
            "To Bank": to_bank_col,
            "Account.1": account1_col,
            "Amount Received": amount_col,
            "Payment Currency": pay_curr_col,
            "Receiving Currency": recv_curr_col,
            "Payment Format": format_col,
        }

        missing = [
            k
            for k, v in required.items()
            if v is None
        ]

        if missing:

            raise RuntimeError(
                f"{stats_name}: missing columns: {missing}"
            )


        for row in reader:

            stats["rows"] += 1


            # --------------------------------------------------------------------------------------
            # TIMESTAMP
            # --------------------------------------------------------------------------------------

            ts = row[timestamp_col].strip()

            try:

                dt = datetime.strptime(
                    ts,
                    "%Y-%m-%d %H:%M:%S"
                )

            except Exception:

                try:

                    dt = datetime.strptime(
                        ts,
                        "%Y/%m/%d %H:%M"
                    )

                except Exception:

                    dt = None


            if dt is not None:

                if (
                    stats["min_timestamp"] is None
                    or dt < stats["min_timestamp"]
                ):
                    stats["min_timestamp"] = dt

                if (
                    stats["max_timestamp"] is None
                    or dt > stats["max_timestamp"]
                ):
                    stats["max_timestamp"] = dt

                stats["hour"][dt.hour] += 1

                stats["day_of_week"][
                    dt.strftime("%A")
                ] += 1

                stats["date"][
                    dt.strftime("%Y-%m-%d")
                ] += 1


            # --------------------------------------------------------------------------------------
            # CATEGORICAL
            # --------------------------------------------------------------------------------------

            stats["payment_format"][
                row[format_col].strip()
            ] += 1

            stats["payment_currency"][
                row[pay_curr_col].strip()
            ] += 1

            stats["receiving_currency"][
                row[recv_curr_col].strip()
            ] += 1


            # --------------------------------------------------------------------------------------
            # BANKS
            # --------------------------------------------------------------------------------------

            src = canonical_bank(
                row[from_bank_col]
            )

            dst = canonical_bank(
                row[to_bank_col]
            )

            acc_src = row[
                account_col
            ].strip()

            acc_dst = row[
                account1_col
            ].strip()


            stats["source_bank"][
                src
            ] += 1

            stats["destination_bank"][
                dst
            ] += 1


            stats["source_account"][
                acc_src
            ] += 1

            stats["destination_account"][
                acc_dst
            ] += 1


            stats["source_bank_account"][
                (src, acc_src)
            ] += 1

            stats["destination_bank_account"][
                (dst, acc_dst)
            ] += 1


            # --------------------------------------------------------------------------------------
            # AMOUNT RESERVOIR SAMPLE
            # --------------------------------------------------------------------------------------

            try:

                amount = float(
                    row[amount_col]
                )

                if (
                    math.isfinite(amount)
                    and amount > 0
                ):

                    n = len(
                        stats["amount_sample"]
                    )

                    if n < SAMPLE_SIZE:

                        stats["amount_sample"].append(
                            amount
                        )

                    else:

                        # Reservoir sampling
                        j = random.randint(
                            0,
                            stats["rows"] - 1
                        )

                        if j < SAMPLE_SIZE:

                            stats[
                                "amount_sample"
                            ][j] = amount

            except Exception:
                pass


    return stats


# --------------------------------------------------------------------------------------------------
# STREAM ORIGINAL
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. STREAMING ORIGINAL HI-SMALL")
print("=" * 80)

original = stream_stats(
    ORIGINAL_FILE,
    "ORIGINAL"
)

print(
    f"Original rows: "
    f"{original['rows']:,}"
)

print(
    f"Original min timestamp: "
    f"{original['min_timestamp']}"
)

print(
    f"Original max timestamp: "
    f"{original['max_timestamp']}"
)


# --------------------------------------------------------------------------------------------------
# STREAM SYNTHETIC
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. STREAMING SYNTHETIC LAUNDERING DATA")
print("=" * 80)

synthetic = stream_stats(
    SYN_FILE,
    "SYNTHETIC"
)

print(
    f"Synthetic rows: "
    f"{synthetic['rows']:,}"
)

print(
    f"Synthetic min timestamp: "
    f"{synthetic['min_timestamp']}"
)

print(
    f"Synthetic max timestamp: "
    f"{synthetic['max_timestamp']}"
)


# --------------------------------------------------------------------------------------------------
# 3. TIMESTAMP COVERAGE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. TIMESTAMP COVERAGE")
print("=" * 80)

print(
    f"Original : "
    f"{original['min_timestamp']} "
    f"→ "
    f"{original['max_timestamp']}"
)

print(
    f"Synthetic: "
    f"{synthetic['min_timestamp']} "
    f"→ "
    f"{synthetic['max_timestamp']}"
)


original_days = set(
    original["date"].keys()
)

synthetic_days = set(
    synthetic["date"].keys()
)

missing_synthetic_days = (
    original_days - synthetic_days
)


print(
    f"Original active dates : "
    f"{len(original_days)}"
)

print(
    f"Synthetic active dates: "
    f"{len(synthetic_days)}"
)

print(
    f"Dates missing in synthetic: "
    f"{len(missing_synthetic_days)}"
)


if missing_synthetic_days:

    print(
        "\n⚠️ Missing synthetic dates:"
    )

    for d in sorted(
        missing_synthetic_days
    ):

        print(
            "  ",
            d
        )


# --------------------------------------------------------------------------------------------------
# 4. HOUR DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. HOUR DISTRIBUTION")
print("=" * 80)

print(
    f"{'Hour':>6} "
    f"{'Original %':>14} "
    f"{'Synthetic %':>14} "
    f"{'Abs Diff':>14}"
)

hour_abs_diffs = []

for hour in range(24):

    op = pct(
        original["hour"].get(hour, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["hour"].get(hour, 0),
        synthetic["rows"]
    )

    diff = abs(
        op - sp
    )

    hour_abs_diffs.append(
        diff
    )

    print(
        f"{hour:6d} "
        f"{op:14.4f} "
        f"{sp:14.4f} "
        f"{diff:14.4f}"
    )

hour_mae = (
    sum(hour_abs_diffs)
    / len(hour_abs_diffs)
)


print(
    f"\nHour-distribution MAE: "
    f"{hour_mae:.6f} percentage points"
)


# --------------------------------------------------------------------------------------------------
# 5. DAY OF WEEK
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. DAY OF WEEK DISTRIBUTION")
print("=" * 80)

days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

dow_abs_diffs = []

for d in days:

    op = pct(
        original["day_of_week"].get(d, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["day_of_week"].get(d, 0),
        synthetic["rows"]
    )

    diff = abs(
        op - sp
    )

    dow_abs_diffs.append(
        diff
    )

    print(
        f"{d:12s} "
        f"| Original={op:8.4f}% "
        f"| Synthetic={sp:8.4f}% "
        f"| AbsDiff={diff:8.4f}"
    )


dow_mae = (
    sum(dow_abs_diffs)
    / len(dow_abs_diffs)
)


print(
    f"\nDay-of-week MAE: "
    f"{dow_mae:.6f} percentage points"
)


# --------------------------------------------------------------------------------------------------
# 6. PAYMENT FORMAT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. PAYMENT FORMAT")
print("=" * 80)

all_formats = sorted(
    set(
        original["payment_format"]
    )
    | set(
        synthetic["payment_format"]
    )
)

format_abs_diffs = []

for fmt in all_formats:

    op = pct(
        original["payment_format"].get(fmt, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["payment_format"].get(fmt, 0),
        synthetic["rows"]
    )

    diff = abs(
        op - sp
    )

    format_abs_diffs.append(
        diff
    )

    print(
        f"{fmt:20s} "
        f"| Original={op:8.4f}% "
        f"| Synthetic={sp:8.4f}% "
        f"| AbsDiff={diff:8.4f}"
    )


format_mae = (
    sum(format_abs_diffs)
    / len(format_abs_diffs)
)


# --------------------------------------------------------------------------------------------------
# 7. PAYMENT CURRENCY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. PAYMENT CURRENCY")
print("=" * 80)

all_pc = sorted(
    set(
        original["payment_currency"]
    )
    | set(
        synthetic["payment_currency"]
    )
)

currency_abs_diffs = []

for curr in all_pc:

    op = pct(
        original["payment_currency"].get(curr, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["payment_currency"].get(curr, 0),
        synthetic["rows"]
    )

    diff = abs(
        op - sp
    )

    currency_abs_diffs.append(
        diff
    )

    print(
        f"{curr:20s} "
        f"| Original={op:8.4f}% "
        f"| Synthetic={sp:8.4f}% "
        f"| AbsDiff={diff:8.4f}"
    )


currency_mae = (
    sum(currency_abs_diffs)
    / len(currency_abs_diffs)
)


# --------------------------------------------------------------------------------------------------
# 8. RECEIVING CURRENCY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. RECEIVING CURRENCY")
print("=" * 80)

all_rc = sorted(
    set(
        original["receiving_currency"]
    )
    | set(
        synthetic["receiving_currency"]
    )
)

recv_abs_diffs = []

for curr in all_rc:

    op = pct(
        original["receiving_currency"].get(curr, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["receiving_currency"].get(curr, 0),
        synthetic["rows"]
    )

    diff = abs(
        op - sp
    )

    recv_abs_diffs.append(
        diff
    )

    print(
        f"{curr:20s} "
        f"| Original={op:8.4f}% "
        f"| Synthetic={sp:8.4f}% "
        f"| AbsDiff={diff:8.4f}"
    )


recv_currency_mae = (
    sum(recv_abs_diffs)
    / len(recv_abs_diffs)
)


# --------------------------------------------------------------------------------------------------
# 9. AMOUNT DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("9. AMOUNT DISTRIBUTION")
print("=" * 80)

original_amount_q = quantiles_from_sample(
    original["amount_sample"]
)

synthetic_amount_q = quantiles_from_sample(
    synthetic["amount_sample"]
)


print(
    f"{'Quantile':>12} "
    f"{'Original':>20} "
    f"{'Synthetic':>20} "
    f"{'Ratio':>12}"
)


for key in [
    "p01",
    "p05",
    "p10",
    "p25",
    "p50",
    "p75",
    "p90",
    "p95",
    "p99"
]:

    ov = original_amount_q[key]
    sv = synthetic_amount_q[key]

    ratio = (
        sv / ov
        if ov != 0
        else None
    )

    print(
        f"{key:>12} "
        f"{ov:20.6f} "
        f"{sv:20.6f} "
        f"{ratio:12.4f}"
    )


# --------------------------------------------------------------------------------------------------
# 10. BANK COVERAGE + CONCENTRATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("10. BANK COVERAGE + CONCENTRATION")
print("=" * 80)


def top_share(counter, total, k):

    top = counter.most_common(k)

    return (
        sum(v for _, v in top) / total * 100
        if total
        else 0.0
    )


metrics = {

    "source_bank_unique": (
        len(original["source_bank"]),
        len(synthetic["source_bank"])
    ),

    "destination_bank_unique": (
        len(original["destination_bank"]),
        len(synthetic["destination_bank"])
    ),

    "source_account_unique": (
        len(original["source_account"]),
        len(synthetic["source_account"])
    ),

    "destination_account_unique": (
        len(original["destination_account"]),
        len(synthetic["destination_account"])
    ),

    "source_bank_account_unique": (
        len(original["source_bank_account"]),
        len(synthetic["source_bank_account"])
    ),

    "destination_bank_account_unique": (
        len(original["destination_bank_account"]),
        len(synthetic["destination_bank_account"])
    ),
}


for name, (ov, sv) in metrics.items():

    print(
        f"{name:30s} "
        f"| Original={ov:,} "
        f"| Synthetic={sv:,}"
    )


print("\nTop-bank transaction concentration:")


for k in [1, 5, 10, 25, 50, 100]:

    obs = top_share(
        original["source_bank"],
        original["rows"],
        k
    )

    syn = top_share(
        synthetic["source_bank"],
        synthetic["rows"],
        k
    )

    print(
        f"Top {k:3d} source banks "
        f"| Original={obs:9.4f}% "
        f"| Synthetic={syn:9.4f}%"
    )


print("\nTop destination-bank concentration:")


for k in [1, 5, 10, 25, 50, 100]:

    obs = top_share(
        original["destination_bank"],
        original["rows"],
        k
    )

    syn = top_share(
        synthetic["destination_bank"],
        synthetic["rows"],
        k
    )

    print(
        f"Top {k:3d} destination banks "
        f"| Original={obs:9.4f}% "
        f"| Synthetic={syn:9.4f}%"
    )


# --------------------------------------------------------------------------------------------------
# 11. ACCOUNT REUSE DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("11. ACCOUNT REUSE DISTRIBUTION")
print("=" * 80)


def activity_distribution(
    counter,
    total_accounts
):

    buckets = Counter()

    for _, n in counter.items():

        if n == 1:
            buckets["1"] += 1

        elif n == 2:
            buckets["2"] += 1

        elif 3 <= n <= 5:
            buckets["3-5"] += 1

        elif 6 <= n <= 10:
            buckets["6-10"] += 1

        elif 11 <= n <= 25:
            buckets["11-25"] += 1

        elif 26 <= n <= 50:
            buckets["26-50"] += 1

        elif 51 <= n <= 100:
            buckets["51-100"] += 1

        elif 101 <= n <= 250:
            buckets["101-250"] += 1

        elif 251 <= n <= 500:
            buckets["251-500"] += 1

        elif 501 <= n <= 1000:
            buckets["501-1000"] += 1

        else:
            buckets[">1000"] += 1


    return {
        k: {
            "count": v,
            "pct": pct(
                v,
                total_accounts
            )
        }
        for k, v in buckets.items()
    }


orig_src_activity = activity_distribution(
    original["source_account"],
    len(original["source_account"])
)

syn_src_activity = activity_distribution(
    synthetic["source_account"],
    len(synthetic["source_account"])
)

orig_dst_activity = activity_distribution(
    original["destination_account"],
    len(original["destination_account"])
)

syn_dst_activity = activity_distribution(
    synthetic["destination_account"],
    len(synthetic["destination_account"])
)


bucket_order = [
    "1",
    "2",
    "3-5",
    "6-10",
    "11-25",
    "26-50",
    "51-100",
    "101-250",
    "251-500",
    "501-1000",
    ">1000"
]


print("\nSOURCE ACCOUNT ACTIVITY")

for b in bucket_order:

    o = orig_src_activity.get(
        b,
        {"pct": 0.0}
    )["pct"]

    s = syn_src_activity.get(
        b,
        {"pct": 0.0}
    )["pct"]

    print(
        f"{b:10s} "
        f"| Original={o:9.4f}% "
        f"| Synthetic={s:9.4f}% "
        f"| AbsDiff={abs(o-s):8.4f}"
    )


print("\nDESTINATION ACCOUNT ACTIVITY")

for b in bucket_order:

    o = orig_dst_activity.get(
        b,
        {"pct": 0.0}
    )["pct"]

    s = syn_dst_activity.get(
        b,
        {"pct": 0.0}
    )["pct"]

    print(
        f"{b:10s} "
        f"| Original={o:9.4f}% "
        f"| Synthetic={s:9.4f}% "
        f"| AbsDiff={abs(o-s):8.4f}"
    )


# --------------------------------------------------------------------------------------------------
# 12. SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("12. BEHAVIORAL DNA SUMMARY")
print("=" * 80)


summary_rows = [

    (
        "Row count",
        original["rows"],
        synthetic["rows"],
        (
            synthetic["rows"]
            / original["rows"]
            if original["rows"]
            else None
        )
    ),

    (
        "Unique source banks",
        len(original["source_bank"]),
        len(synthetic["source_bank"]),
        None
    ),

    (
        "Unique destination banks",
        len(original["destination_bank"]),
        len(synthetic["destination_bank"]),
        None
    ),

    (
        "Unique source accounts",
        len(original["source_account"]),
        len(synthetic["source_account"]),
        None
    ),

    (
        "Unique destination accounts",
        len(original["destination_account"]),
        len(synthetic["destination_account"]),
        None
    ),

    (
        "Hour MAE (percentage points)",
        None,
        None,
        hour_mae
    ),

    (
        "Day-of-week MAE (percentage points)",
        None,
        None,
        dow_mae
    ),

    (
        "Payment-format MAE (percentage points)",
        None,
        None,
        format_mae
    ),

    (
        "Payment-currency MAE (percentage points)",
        None,
        None,
        currency_mae
    ),

    (
        "Receiving-currency MAE (percentage points)",
        None,
        None,
        recv_currency_mae
    ),

]


for name, ov, sv, metric in summary_rows:

    if (
        ov is not None
        and sv is not None
    ):

        print(
            f"{name:40s} "
            f"| Original={ov:,} "
            f"| Synthetic={sv:,}"
        )

    else:

        print(
            f"{name:40s} "
            f"| Metric={metric:.6f}"
        )


# --------------------------------------------------------------------------------------------------
# 13. SAVE AUDIT JSON
# --------------------------------------------------------------------------------------------------

audit = {

    "synthetic_file":
        SYN_FILE,

    "original_file":
        ORIGINAL_FILE,

    "row_counts": {
        "original":
            original["rows"],
        "synthetic":
            synthetic["rows"],
    },

    "timestamps": {

        "original_min":
            str(
                original["min_timestamp"]
            ),

        "original_max":
            str(
                original["max_timestamp"]
            ),

        "synthetic_min":
            str(
                synthetic["min_timestamp"]
            ),

        "synthetic_max":
            str(
                synthetic["max_timestamp"]
            ),

        "original_active_dates":
            sorted(original_days),

        "synthetic_active_dates":
            sorted(synthetic_days),

        "missing_synthetic_dates":
            sorted(
                missing_synthetic_days
            ),
    },

    "distribution_metrics": {

        "hour_mae_percentage_points":
            hour_mae,

        "day_of_week_mae_percentage_points":
            dow_mae,

        "payment_format_mae_percentage_points":
            format_mae,

        "payment_currency_mae_percentage_points":
            currency_mae,

        "receiving_currency_mae_percentage_points":
            recv_currency_mae,
    },

    "amount_quantiles_sample":
        {
            "original":
                original_amount_q,

            "synthetic":
                synthetic_amount_q,
        },

    "bank_coverage": {

        "original_unique_source_banks":
            len(
                original["source_bank"]
            ),

        "synthetic_unique_source_banks":
            len(
                synthetic["source_bank"]
            ),

        "original_unique_destination_banks":
            len(
                original["destination_bank"]
            ),

        "synthetic_unique_destination_banks":
            len(
                synthetic["destination_bank"]
            ),

        "original_top_source_bank_shares": {
            str(k):
                top_share(
                    original["source_bank"],
                    original["rows"],
                    k
                )
            for k in [1, 5, 10, 25, 50, 100]
        },

        "synthetic_top_source_bank_shares": {
            str(k):
                top_share(
                    synthetic["source_bank"],
                    synthetic["rows"],
                    k
                )
            for k in [1, 5, 10, 25, 50, 100]
        },

        "original_top_destination_bank_shares": {
            str(k):
                top_share(
                    original["destination_bank"],
                    original["rows"],
                    k
                )
            for k in [1, 5, 10, 25, 50, 100]
        },

        "synthetic_top_destination_bank_shares": {
            str(k):
                top_share(
                    synthetic["destination_bank"],
                    synthetic["rows"],
                    k
                )
            for k in [1, 5, 10, 25, 50, 100]
        },
    },

    "account_coverage": {

        "original_unique_source_accounts":
            len(
                original["source_account"]
            ),

        "synthetic_unique_source_accounts":
            len(
                synthetic["source_account"]
            ),

        "original_unique_destination_accounts":
            len(
                original["destination_account"]
            ),

        "synthetic_unique_destination_accounts":
            len(
                synthetic["destination_account"]
            ),
    },

    "payment_format": {

        "original":
            distribution_pct(
                original["payment_format"],
                original["rows"]
            ),

        "synthetic":
            distribution_pct(
                synthetic["payment_format"],
                synthetic["rows"]
            ),
    },

    "payment_currency": {

        "original":
            distribution_pct(
                original["payment_currency"],
                original["rows"]
            ),

        "synthetic":
            distribution_pct(
                synthetic["payment_currency"],
                synthetic["rows"]
            ),
    },

    "receiving_currency": {

        "original":
            distribution_pct(
                original["receiving_currency"],
                original["rows"]
            ),

        "synthetic":
            distribution_pct(
                synthetic["receiving_currency"],
                synthetic["rows"]
            ),
    },

    "hour_distribution": {

        "original":
            distribution_pct(
                original["hour"],
                original["rows"]
            ),

        "synthetic":
            distribution_pct(
                synthetic["hour"],
                synthetic["rows"]
            ),
    },

    "day_of_week_distribution": {

        "original":
            distribution_pct(
                original["day_of_week"],
                original["rows"]
            ),

        "synthetic":
            distribution_pct(
                synthetic["day_of_week"],
                synthetic["rows"]
            ),
    },
}


with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        indent=2
    )


# --------------------------------------------------------------------------------------------------
# SAVE COMPACT CSV
# --------------------------------------------------------------------------------------------------

with open(
    OUT_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "metric",
        "original",
        "synthetic",
        "metric_value"
    ])

    writer.writerow([
        "row_count",
        original["rows"],
        synthetic["rows"],
        ""
    ])

    writer.writerow([
        "unique_source_banks",
        len(original["source_bank"]),
        len(synthetic["source_bank"]),
        ""
    ])

    writer.writerow([
        "unique_destination_banks",
        len(original["destination_bank"]),
        len(synthetic["destination_bank"]),
        ""
    ])

    writer.writerow([
        "unique_source_accounts",
        len(original["source_account"]),
        len(synthetic["source_account"]),
        ""
    ])

    writer.writerow([
        "unique_destination_accounts",
        len(original["destination_account"]),
        len(synthetic["destination_account"]),
        ""
    ])

    writer.writerow([
        "hour_mae_percentage_points",
        "",
        "",
        hour_mae
    ])

    writer.writerow([
        "day_of_week_mae_percentage_points",
        "",
        "",
        dow_mae
    ])

    writer.writerow([
        "payment_format_mae_percentage_points",
        "",
        "",
        format_mae
    ])

    writer.writerow([
        "payment_currency_mae_percentage_points",
        "",
        "",
        currency_mae
    ])

    writer.writerow([
        "receiving_currency_mae_percentage_points",
        "",
        "",
        recv_currency_mae
    ])


# --------------------------------------------------------------------------------------------------
# FINAL
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("✅ CELL 22 COMPLETE")
print("=" * 100)

print(
    "Audit JSON:"
)

print(
    OUT_JSON
)

print(
    "\nAudit CSV:"
)

print(
    OUT_CSV
)

print(
    "\nIMPORTANT:"
)

print(
    "This is a behavioral comparison only."
)

print(
    "No synthetic data was regenerated or modified."
)

print("=" * 100)

CELL 22 — SYNTHETIC vs ORIGINAL HI-SMALL BEHAVIORAL DNA AUDIT

Synthetic file:
/kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_FULL_transactions.csv

Original file:
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv

1. STREAMING ORIGINAL HI-SMALL


RuntimeError: ORIGINAL: missing columns: ['Account.1']

In [34]:
# ==================================================================================================
# FINAL MERGE CELL — CORRECTED FOR IBM DUPLICATE "Account" COLUMNS
#
# Creates:
#   1) HI-Small_FINAL_transactions_10146336.csv
#   2) HI-Small_FINAL_accounts_1004953.csv
#   3) HI-Small_FINAL_Patterns.txt
#
# IMPORTANT:
#   - DOES NOT regenerate synthetic data
#   - Uses the already-completed synthetic files
#   - Original HI-Small transaction file is read POSITIONALLY because
#     it contains two columns named "Account"
# ==================================================================================================

import os
import csv
import shutil
from IPython.display import FileLink, display


# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

BASE = "/kaggle/working/AML_GNN_ONLY/data"

ORIGINAL_TRANS = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Trans.csv"
)

ORIGINAL_ACCOUNTS = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_accounts.csv"
)

ORIGINAL_PATTERNS = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Patterns.txt"
)

SYNTH_TRANS = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_transactions.csv"
)

SYNTH_ACCOUNTS = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_new_accounts.csv"
)

SYNTH_PATTERNS = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_Patterns.txt"
)


# --------------------------------------------------------------------------------------------------
# FINAL OUTPUT FILES
# --------------------------------------------------------------------------------------------------

FINAL_TRANS = os.path.join(
    BASE,
    "HI-Small_FINAL_transactions_10146336.csv"
)

FINAL_ACCOUNTS = os.path.join(
    BASE,
    "HI-Small_FINAL_accounts_1004953.csv"
)

FINAL_PATTERNS = os.path.join(
    BASE,
    "HI-Small_FINAL_Patterns.txt"
)


# --------------------------------------------------------------------------------------------------
# EXPECTED COUNTS
# --------------------------------------------------------------------------------------------------

EXPECTED_ORIGINAL_TRANS = 5_078_345
EXPECTED_SYNTH_TRANS = 5_067_991
EXPECTED_FINAL_TRANS = 10_146_336

EXPECTED_ORIGINAL_ACCOUNTS = 518_581
EXPECTED_SYNTH_ACCOUNTS = 486_372
EXPECTED_FINAL_ACCOUNTS = 1_004_953

EXPECTED_ORIGINAL_PATTERN_EPISODES = 370
EXPECTED_SYNTH_PATTERN_EPISODES = 584_706
EXPECTED_FINAL_PATTERN_EPISODES = 585_076

EXPECTED_ORIGINAL_PATTERN_TX = 3_209
EXPECTED_SYNTH_PATTERN_TX = 5_067_991
EXPECTED_FINAL_PATTERN_TX = 5_071_200


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def norm_col(x):
    return (
        str(x)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace(".", "")
    )


def find_column(header, names):

    normalized = {
        norm_col(h): h
        for h in header
    }

    for name in names:

        key = norm_col(name)

        if key in normalized:
            return normalized[key]

    for h in header:

        hl = norm_col(h)

        for name in names:

            if norm_col(name) in hl:
                return h

    return None


def file_mb(path):
    return os.path.getsize(path) / (1024 * 1024)


# --------------------------------------------------------------------------------------------------
# START
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("FINAL HI-SMALL MERGE — CORRECTED")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. INPUT FILE CHECK
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. INPUT FILE CHECK")
print("=" * 80)

inputs = {
    "Original transactions": ORIGINAL_TRANS,
    "Original accounts": ORIGINAL_ACCOUNTS,
    "Original patterns": ORIGINAL_PATTERNS,
    "Synthetic transactions": SYNTH_TRANS,
    "Synthetic accounts": SYNTH_ACCOUNTS,
    "Synthetic patterns": SYNTH_PATTERNS,
}

for name, path in inputs.items():

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing {name}:\n{path}"
        )

    print(
        f"✅ {name:25s} | "
        f"{file_mb(path):,.2f} MB"
    )


# --------------------------------------------------------------------------------------------------
# 2. FINAL TRANSACTION SCHEMA
# --------------------------------------------------------------------------------------------------

FINAL_TRANSACTION_COLUMNS = [
    "transaction_id",
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
    "is_synthetic",
    "synthetic_episode_id",
    "typology",
    "source_template_id",
    "template_source_order",
]


# --------------------------------------------------------------------------------------------------
# 3. INSPECT ORIGINAL TRANSACTION HEADER POSITIONALLY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. ORIGINAL TRANSACTION HEADER")
print("=" * 80)


with open(
    ORIGINAL_TRANS,
    "r",
    newline="",
    encoding="utf-8-sig"
) as f:

    reader = csv.reader(f)

    original_header = next(reader)


print(
    "Original header:"
)

for i, col in enumerate(original_header):

    print(
        f"  [{i}] {col}"
    )


# Expected IBM structure:
#
# 0 Timestamp
# 1 From Bank
# 2 Account            <- SOURCE ACCOUNT
# 3 To Bank
# 4 Account            <- DESTINATION ACCOUNT
# 5 Amount Received
# 6 Receiving Currency
# 7 Amount Paid
# 8 Payment Currency
# 9 Payment Format
# 10 Is Laundering


if len(original_header) != 11:

    raise RuntimeError(
        "Unexpected original HI-Small transaction "
        f"column count: {len(original_header)}"
    )


# Positional mapping — IMPORTANT
ORIG_TS = 0
ORIG_FROM_BANK = 1
ORIG_SOURCE_ACCOUNT = 2
ORIG_TO_BANK = 3
ORIG_DEST_ACCOUNT = 4
ORIG_AMOUNT_RECEIVED = 5
ORIG_RECEIVING_CURRENCY = 6
ORIG_AMOUNT_PAID = 7
ORIG_PAYMENT_CURRENCY = 8
ORIG_PAYMENT_FORMAT = 9
ORIG_IS_LAUNDERING = 10


# Verify expected names in the expected positions
expected_names = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]

if original_header != expected_names:

    raise RuntimeError(
        "Original HI-Small header differs from expected positional schema.\n"
        f"Actual  : {original_header}\n"
        f"Expected: {expected_names}"
    )


print(
    "\n✅ Original duplicate Account columns resolved positionally:"
)

print(
    "   original column [2] Account -> final 'Account'"
)

print(
    "   original column [4] Account -> final 'Account.1'"
)


# --------------------------------------------------------------------------------------------------
# 4. INSPECT SYNTHETIC TRANSACTION FILE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. SYNTHETIC TRANSACTION SCHEMA")
print("=" * 80)


with open(
    SYNTH_TRANS,
    "r",
    newline="",
    encoding="utf-8"
) as f:

    reader = csv.reader(f)

    synth_header = next(reader)


for i, col in enumerate(synth_header):

    print(
        f"  [{i}] {col}"
    )


missing_synthetic_columns = [
    col
    for col in FINAL_TRANSACTION_COLUMNS
    if col not in synth_header
]


if missing_synthetic_columns:

    raise RuntimeError(
        "Synthetic transaction file missing columns:\n"
        + str(missing_synthetic_columns)
    )


print(
    "\n✅ Synthetic transaction schema validated."
)


# --------------------------------------------------------------------------------------------------
# 5. MERGE TRANSACTIONS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. MERGING TRANSACTIONS")
print("=" * 80)


# Remove any partial output
if os.path.exists(FINAL_TRANS):

    os.remove(FINAL_TRANS)

    print(
        "Removed previous partial final transaction file."
    )


original_count = 0
synthetic_count = 0


with open(
    FINAL_TRANS,
    "w",
    newline="",
    encoding="utf-8"
) as out_f:

    writer = csv.DictWriter(
        out_f,
        fieldnames=FINAL_TRANSACTION_COLUMNS
    )

    writer.writeheader()


    # ==============================================================================================
    # ORIGINAL TRANSACTIONS — POSITIONAL READ
    # ==============================================================================================

    print(
        "\nWriting original transactions..."
    )

    with open(
        ORIGINAL_TRANS,
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        reader = csv.reader(f)

        next(reader)


        for row in reader:

            if len(row) != 11:

                raise RuntimeError(
                    "Unexpected original transaction row length "
                    f"at row {original_count + 2}: "
                    f"{len(row)}"
                )


            original_count += 1


            writer.writerow({

                "transaction_id":
                    f"ORIG_{original_count:010d}",

                "Timestamp":
                    row[ORIG_TS],

                "From Bank":
                    row[ORIG_FROM_BANK],

                # SOURCE ACCOUNT
                "Account":
                    row[ORIG_SOURCE_ACCOUNT],

                "To Bank":
                    row[ORIG_TO_BANK],

                # DESTINATION ACCOUNT
                "Account.1":
                    row[ORIG_DEST_ACCOUNT],

                "Amount Received":
                    row[ORIG_AMOUNT_RECEIVED],

                "Receiving Currency":
                    row[ORIG_RECEIVING_CURRENCY],

                "Amount Paid":
                    row[ORIG_AMOUNT_PAID],

                "Payment Currency":
                    row[ORIG_PAYMENT_CURRENCY],

                "Payment Format":
                    row[ORIG_PAYMENT_FORMAT],

                "Is Laundering":
                    row[ORIG_IS_LAUNDERING],

                "is_synthetic":
                    0,

                "synthetic_episode_id":
                    "",

                "typology":
                    "",

                "source_template_id":
                    "",

                "template_source_order":
                    "",
            })


            if (
                original_count % 1_000_000 == 0
            ):

                print(
                    f"  Original written: "
                    f"{original_count:,}"
                )


    print(
        f"✅ Original transactions written: "
        f"{original_count:,}"
    )


    # ==============================================================================================
    # SYNTHETIC TRANSACTIONS
    # ==============================================================================================

    print(
        "\nWriting synthetic transactions..."
    )

    with open(
        SYNTH_TRANS,
        "r",
        newline="",
        encoding="utf-8"
    ) as f:

        reader = csv.DictReader(f)


        for row in reader:

            synthetic_count += 1


            writer.writerow({

                col:
                    row[col]
                for col in FINAL_TRANSACTION_COLUMNS

            })


            if (
                synthetic_count % 1_000_000 == 0
            ):

                print(
                    f"  Synthetic written: "
                    f"{synthetic_count:,}"
                )


    print(
        f"✅ Synthetic transactions written: "
        f"{synthetic_count:,}"
    )


final_transaction_count = (
    original_count
    + synthetic_count
)


print(
    "\nTransaction merge result:"
)

print(
    f"  Original  : {original_count:,}"
)

print(
    f"  Synthetic : {synthetic_count:,}"
)

print(
    f"  FINAL     : {final_transaction_count:,}"
)

print(
    f"  EXPECTED  : {EXPECTED_FINAL_TRANS:,}"
)


if (
    original_count != EXPECTED_ORIGINAL_TRANS
    or synthetic_count != EXPECTED_SYNTH_TRANS
    or final_transaction_count != EXPECTED_FINAL_TRANS
):

    raise RuntimeError(
        "❌ FINAL TRANSACTION COUNT VALIDATION FAILED."
    )


print(
    "✅ Transaction merge PASSED."
)


# --------------------------------------------------------------------------------------------------
# 6. MERGE ACCOUNTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. MERGING ACCOUNTS")
print("=" * 80)


FINAL_ACCOUNT_COLUMNS = [
    "Bank Name",
    "Bank ID",
    "Account Number",
    "Entity ID",
    "Entity Name",
]


def resolve_account_columns(header):

    return {

        "Bank Name":
            find_column(
                header,
                [
                    "Bank Name",
                    "BankName"
                ]
            ),

        "Bank ID":
            find_column(
                header,
                [
                    "Bank ID",
                    "BankID",
                    "bank_id"
                ]
            ),

        "Account Number":
            find_column(
                header,
                [
                    "Account Number",
                    "AccountNumber",
                    "Account"
                ]
            ),

        "Entity ID":
            find_column(
                header,
                [
                    "Entity ID",
                    "EntityID",
                    "entity_id"
                ]
            ),

        "Entity Name":
            find_column(
                header,
                [
                    "Entity Name",
                    "EntityName",
                    "entity_name"
                ]
            ),
    }


# Read original account header
with open(
    ORIGINAL_ACCOUNTS,
    "r",
    newline="",
    encoding="utf-8-sig"
) as f:

    reader = csv.DictReader(f)

    original_account_header = (
        reader.fieldnames
    )


# Read synthetic account header
with open(
    SYNTH_ACCOUNTS,
    "r",
    newline="",
    encoding="utf-8-sig"
) as f:

    reader = csv.DictReader(f)

    synthetic_account_header = (
        reader.fieldnames
    )


print(
    "Original accounts:"
)

print(
    original_account_header
)


print(
    "\nSynthetic accounts:"
)

print(
    synthetic_account_header
)


original_account_cols = (
    resolve_account_columns(
        original_account_header
    )
)

synthetic_account_cols = (
    resolve_account_columns(
        synthetic_account_header
    )
)


missing_original_accounts = [
    k
    for k, v in original_account_cols.items()
    if v is None
]

missing_synthetic_accounts = [
    k
    for k, v in synthetic_account_cols.items()
    if v is None
]


if missing_original_accounts:

    raise RuntimeError(
        "Original accounts missing:\n"
        + str(missing_original_accounts)
    )


if missing_synthetic_accounts:

    raise RuntimeError(
        "Synthetic accounts missing:\n"
        + str(missing_synthetic_accounts)
    )


if os.path.exists(FINAL_ACCOUNTS):

    os.remove(FINAL_ACCOUNTS)


account_original_count = 0
account_synthetic_count = 0


with open(
    FINAL_ACCOUNTS,
    "w",
    newline="",
    encoding="utf-8"
) as out_f:

    writer = csv.DictWriter(
        out_f,
        fieldnames=FINAL_ACCOUNT_COLUMNS
    )

    writer.writeheader()


    # ==============================================================================================
    # ORIGINAL ACCOUNTS
    # ==============================================================================================

    with open(
        ORIGINAL_ACCOUNTS,
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:

            account_original_count += 1

            writer.writerow({

                col:
                    row[
                        original_account_cols[col]
                    ]

                for col
                in FINAL_ACCOUNT_COLUMNS

            })


    # ==============================================================================================
    # SYNTHETIC ACCOUNTS
    # ==============================================================================================

    with open(
        SYNTH_ACCOUNTS,
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:

            account_synthetic_count += 1

            writer.writerow({

                col:
                    row[
                        synthetic_account_cols[col]
                    ]

                for col
                in FINAL_ACCOUNT_COLUMNS

            })


final_account_count = (
    account_original_count
    + account_synthetic_count
)


print(
    f"Original accounts : "
    f"{account_original_count:,}"
)

print(
    f"Synthetic accounts: "
    f"{account_synthetic_count:,}"
)

print(
    f"FINAL accounts    : "
    f"{final_account_count:,}"
)

print(
    f"EXPECTED accounts : "
    f"{EXPECTED_FINAL_ACCOUNTS:,}"
)


if (
    account_original_count != EXPECTED_ORIGINAL_ACCOUNTS
    or account_synthetic_count != EXPECTED_SYNTH_ACCOUNTS
    or final_account_count != EXPECTED_FINAL_ACCOUNTS
):

    raise RuntimeError(
        "❌ FINAL ACCOUNT COUNT VALIDATION FAILED."
    )


print(
    "✅ Account merge PASSED."
)


# --------------------------------------------------------------------------------------------------
# 7. MERGE PATTERNS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. MERGING PATTERNS.TXT")
print("=" * 80)


if os.path.exists(FINAL_PATTERNS):

    os.remove(FINAL_PATTERNS)


# Copy original
with open(
    FINAL_PATTERNS,
    "wb"
) as out_f:

    with open(
        ORIGINAL_PATTERNS,
        "rb"
    ) as f:

        shutil.copyfileobj(
            f,
            out_f,
            length=8 * 1024 * 1024
        )


# Ensure newline between original and synthetic pattern files
with open(
    ORIGINAL_PATTERNS,
    "rb"
) as f:

    if os.path.getsize(ORIGINAL_PATTERNS) > 0:

        f.seek(-1, os.SEEK_END)

        last_byte = f.read(1)

        if last_byte != b"\n":

            with open(
                FINAL_PATTERNS,
                "ab"
            ) as out_f:

                out_f.write(
                    b"\n"
                )


# Append synthetic
with open(
    FINAL_PATTERNS,
    "ab"
) as out_f:

    with open(
        SYNTH_PATTERNS,
        "rb"
    ) as f:

        shutil.copyfileobj(
            f,
            out_f,
            length=8 * 1024 * 1024
        )


print(
    f"Original Patterns : "
    f"{file_mb(ORIGINAL_PATTERNS):,.2f} MB"
)

print(
    f"Synthetic Patterns: "
    f"{file_mb(SYNTH_PATTERNS):,.2f} MB"
)

print(
    f"FINAL Patterns    : "
    f"{file_mb(FINAL_PATTERNS):,.2f} MB"
)

print(
    "✅ Pattern merge completed."
)


# --------------------------------------------------------------------------------------------------
# 8. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. FINAL MERGED FILE VALIDATION")
print("=" * 80)


# --------------------------------------------------------------------------------
# Transactions
# --------------------------------------------------------------------------------

with open(
    FINAL_TRANS,
    "r",
    encoding="utf-8"
) as f:

    final_tx_lines = sum(
        1
        for _ in f
    ) - 1


# --------------------------------------------------------------------------------
# Accounts
# --------------------------------------------------------------------------------

with open(
    FINAL_ACCOUNTS,
    "r",
    encoding="utf-8"
) as f:

    final_account_lines = sum(
        1
        for _ in f
    ) - 1


# --------------------------------------------------------------------------------
# Patterns
# --------------------------------------------------------------------------------

final_pattern_ids = 0
final_pattern_begin = 0
final_pattern_end = 0
final_pattern_tx = 0

inside_episode = False


with open(
    FINAL_PATTERNS,
    "r",
    encoding="utf-8"
) as f:

    for raw in f:

        line = raw.strip()

        if not line:
            continue


        if line.startswith(
            'id="'
        ):

            final_pattern_ids += 1
            continue


        if line.startswith(
            "BEGIN LAUNDERING ATTEMPT"
        ):

            final_pattern_begin += 1
            inside_episode = True
            continue


        if line.startswith(
            "END LAUNDERING ATTEMPT"
        ):

            final_pattern_end += 1
            inside_episode = False
            continue


        if inside_episode:

            final_pattern_tx += 1


print(
    f"Final transactions    : "
    f"{final_tx_lines:,}"
)

print(
    f"Expected transactions : "
    f"{EXPECTED_FINAL_TRANS:,}"
)

print(
    f"Final accounts        : "
    f"{final_account_lines:,}"
)

print(
    f"Expected accounts     : "
    f"{EXPECTED_FINAL_ACCOUNTS:,}"
)

print(
    f"Final pattern IDs     : "
    f"{final_pattern_ids:,}"
)

print(
    f"Expected pattern IDs  : "
    f"{EXPECTED_FINAL_PATTERN_EPISODES:,}"
)

print(
    f"Final BEGIN markers   : "
    f"{final_pattern_begin:,}"
)

print(
    f"Final END markers     : "
    f"{final_pattern_end:,}"
)

print(
    f"Final pattern TX      : "
    f"{final_pattern_tx:,}"
)

print(
    f"Expected pattern TX   : "
    f"{EXPECTED_FINAL_PATTERN_TX:,}"
)


final_validation = {

    "transactions":
        final_tx_lines == EXPECTED_FINAL_TRANS,

    "accounts":
        final_account_lines == EXPECTED_FINAL_ACCOUNTS,

    "pattern_ids":
        final_pattern_ids == EXPECTED_FINAL_PATTERN_EPISODES,

    "pattern_begin":
        final_pattern_begin == EXPECTED_FINAL_PATTERN_EPISODES,

    "pattern_end":
        final_pattern_end == EXPECTED_FINAL_PATTERN_EPISODES,

    "pattern_transactions":
        final_pattern_tx == EXPECTED_FINAL_PATTERN_TX,
}


# --------------------------------------------------------------------------------
# Print gates
# --------------------------------------------------------------------------------

print("\nFinal validation gates:")

all_pass = True

for name, passed in final_validation.items():

    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} | {name}"
    )

    all_pass &= passed


if not all_pass:

    raise RuntimeError(
        "\n❌ FINAL MERGED DATASET VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 9. FINAL FILE INFORMATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. FINAL FILES")
print("=" * 80)


for path in [
    FINAL_TRANS,
    FINAL_ACCOUNTS,
    FINAL_PATTERNS,
]:

    print(
        f"{os.path.basename(path):50s} "
        f"{file_mb(path):12,.2f} MB"
    )


# --------------------------------------------------------------------------------------------------
# 10. KAGGLE DOWNLOAD LINKS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("9. DOWNLOAD LINKS")
print("=" * 80)


print(
    "\n⬇️ FINAL TRANSACTIONS — 10,146,336 rows"
)

display(
    FileLink(FINAL_TRANS)
)


print(
    "\n⬇️ FINAL ACCOUNTS — 1,004,953 rows"
)

display(
    FileLink(FINAL_ACCOUNTS)
)


print(
    "\n⬇️ FINAL PATTERNS.TXT — 585,076 episodes"
)

display(
    FileLink(FINAL_PATTERNS)
)


# --------------------------------------------------------------------------------------------------
# DONE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print(
    "✅✅✅ ALL THREE FINAL MERGED FILES ARE READY ✅✅✅"
)
print("=" * 100)

print(
    f"Transactions : {FINAL_TRANS}"
)

print(
    f"Accounts     : {FINAL_ACCOUNTS}"
)

print(
    f"Patterns     : {FINAL_PATTERNS}"
)

print("=" * 100)

FINAL HI-SMALL MERGE — CORRECTED

1. INPUT FILE CHECK
✅ Original transactions     | 453.63 MB
✅ Original accounts         | 32.48 MB
✅ Original patterns         | 0.31 MB
✅ Synthetic transactions    | 844.22 MB
✅ Synthetic accounts        | 27.85 MB
✅ Synthetic patterns        | 570.76 MB

2. ORIGINAL TRANSACTION HEADER
Original header:
  [0] Timestamp
  [1] From Bank
  [2] Account
  [3] To Bank
  [4] Account
  [5] Amount Received
  [6] Receiving Currency
  [7] Amount Paid
  [8] Payment Currency
  [9] Payment Format
  [10] Is Laundering

✅ Original duplicate Account columns resolved positionally:
   original column [2] Account -> final 'Account'
   original column [4] Account -> final 'Account.1'

3. SYNTHETIC TRANSACTION SCHEMA
  [0] transaction_id
  [1] Timestamp
  [2] From Bank
  [3] Account
  [4] To Bank
  [5] Account.1
  [6] Amount Received
  [7] Receiving Currency
  [8] Amount Paid
  [9] Payment Currency
  [10] Payment Format
  [11] Is Laundering
  [12] is_synthetic
  [13] synthe

RuntimeError: 
❌ FINAL MERGED DATASET VALIDATION FAILED.

In [35]:
# ==================================================================================================
# FINAL VALIDATION + DOWNLOAD
#
# IMPORTANT:
#   The previous merge is ALREADY CORRECT.
#
#   The failure happened because:
#       - Original IBM Patterns file has 370 episodes
#       - Original IBM Patterns file has 0 id="..." lines
#       - Synthetic Patterns file has 584,706 id="..." lines
#       - Final file therefore correctly has 584,706 ID lines
#       - BUT it has 585,076 BEGIN and END markers
#
#   This cell:
#       1. Validates the already-created final files
#       2. Confirms original Patterns ID count
#       3. Creates the final manifest
#       4. Shows download links
#
#   NO DATA REGENERATION.
# ==================================================================================================

import os
import csv
import json
from IPython.display import FileLink, display


# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

BASE = "/kaggle/working/AML_GNN_ONLY/data"

ORIGINAL_TRANS = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Trans.csv"
)

ORIGINAL_ACCOUNTS = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_accounts.csv"
)

ORIGINAL_PATTERNS = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Patterns.txt"
)

FINAL_TRANS = os.path.join(
    BASE,
    "HI-Small_FINAL_transactions_10146336.csv"
)

FINAL_ACCOUNTS = os.path.join(
    BASE,
    "HI-Small_FINAL_accounts_1004953.csv"
)

FINAL_PATTERNS = os.path.join(
    BASE,
    "HI-Small_FINAL_Patterns.txt"
)

MANIFEST = os.path.join(
    BASE,
    "HI-Small_FINAL_manifest.json"
)


# --------------------------------------------------------------------------------------------------
# EXPECTED VALUES
# --------------------------------------------------------------------------------------------------

EXPECTED_FINAL_TRANS = 10_146_336
EXPECTED_FINAL_ACCOUNTS = 1_004_953

ORIGINAL_PATTERN_EPISODES = 370
SYNTH_PATTERN_EPISODES = 584_706
EXPECTED_FINAL_PATTERN_EPISODES = (
    ORIGINAL_PATTERN_EPISODES
    + SYNTH_PATTERN_EPISODES
)

ORIGINAL_PATTERN_TX = 3_209
SYNTH_PATTERN_TX = 5_067_991
EXPECTED_FINAL_PATTERN_TX = (
    ORIGINAL_PATTERN_TX
    + SYNTH_PATTERN_TX
)


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def mb(path):
    return os.path.getsize(path) / (1024 * 1024)


def require(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)


# --------------------------------------------------------------------------------------------------
# START
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("FINAL HI-SMALL MERGED DATASET VALIDATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. FILE EXISTENCE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. FINAL FILE EXISTENCE")
print("=" * 80)

for name, path in {
    "FINAL TRANSACTIONS": FINAL_TRANS,
    "FINAL ACCOUNTS": FINAL_ACCOUNTS,
    "FINAL PATTERNS": FINAL_PATTERNS,
}.items():

    require(path)

    print(
        f"✅ {name:20s} | "
        f"{mb(path):,.2f} MB | "
        f"{os.path.basename(path)}"
    )


# --------------------------------------------------------------------------------------------------
# 2. COUNT ID LINES IN ORIGINAL PATTERNS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. ORIGINAL IBM PATTERNS STRUCTURE")
print("=" * 80)

original_pattern_ids = 0
original_begin = 0
original_end = 0
original_tx = 0

inside = False


with open(
    ORIGINAL_PATTERNS,
    "r",
    encoding="utf-8"
) as f:

    for raw in f:

        line = raw.strip()

        if not line:
            continue

        if line.startswith('id="'):

            original_pattern_ids += 1

        elif line.startswith(
            "BEGIN LAUNDERING ATTEMPT"
        ):

            original_begin += 1
            inside = True

        elif line.startswith(
            "END LAUNDERING ATTEMPT"
        ):

            original_end += 1
            inside = False

        elif inside:

            original_tx += 1


print(
    f"Original ID lines       : "
    f"{original_pattern_ids:,}"
)

print(
    f"Original BEGIN markers  : "
    f"{original_begin:,}"
)

print(
    f"Original END markers    : "
    f"{original_end:,}"
)

print(
    f"Original TX lines       : "
    f"{original_tx:,}"
)


# The observed original file structure is:
#   0 ID lines
#   370 BEGIN
#   370 END
#   3,209 TX

original_pattern_gate = (
    original_pattern_ids == 0
    and original_begin == ORIGINAL_PATTERN_EPISODES
    and original_end == ORIGINAL_PATTERN_EPISODES
    and original_tx == ORIGINAL_PATTERN_TX
)


print(
    "\n✅ Original IBM pattern structure PASS"
    if original_pattern_gate
    else "\n❌ Original IBM pattern structure FAIL"
)


# --------------------------------------------------------------------------------------------------
# 3. FINAL TRANSACTION COUNT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. FINAL TRANSACTION FILE")
print("=" * 80)


with open(
    FINAL_TRANS,
    "r",
    encoding="utf-8"
) as f:

    final_transaction_rows = (
        sum(1 for _ in f) - 1
    )


print(
    f"Final transaction rows : "
    f"{final_transaction_rows:,}"
)

print(
    f"Expected                : "
    f"{EXPECTED_FINAL_TRANS:,}"
)


transaction_gate = (
    final_transaction_rows
    == EXPECTED_FINAL_TRANS
)


print(
    "✅ PASS"
    if transaction_gate
    else "❌ FAIL"
)


# --------------------------------------------------------------------------------------------------
# 4. FINAL ACCOUNT COUNT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. FINAL ACCOUNT FILE")
print("=" * 80)


with open(
    FINAL_ACCOUNTS,
    "r",
    encoding="utf-8"
) as f:

    final_account_rows = (
        sum(1 for _ in f) - 1
    )


print(
    f"Final account rows : "
    f"{final_account_rows:,}"
)

print(
    f"Expected           : "
    f"{EXPECTED_FINAL_ACCOUNTS:,}"
)


account_gate = (
    final_account_rows
    == EXPECTED_FINAL_ACCOUNTS
)


print(
    "✅ PASS"
    if account_gate
    else "❌ FAIL"
)


# --------------------------------------------------------------------------------------------------
# 5. FINAL PATTERNS VALIDATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. FINAL PATTERNS FILE")
print("=" * 80)


final_pattern_ids = 0
final_pattern_begin = 0
final_pattern_end = 0
final_pattern_tx = 0

inside = False


with open(
    FINAL_PATTERNS,
    "r",
    encoding="utf-8"
) as f:

    for raw in f:

        line = raw.strip()

        if not line:
            continue


        if line.startswith('id="'):

            final_pattern_ids += 1


        elif line.startswith(
            "BEGIN LAUNDERING ATTEMPT"
        ):

            final_pattern_begin += 1
            inside = True


        elif line.startswith(
            "END LAUNDERING ATTEMPT"
        ):

            final_pattern_end += 1
            inside = False


        elif inside:

            final_pattern_tx += 1


print(
    f"Final ID lines       : "
    f"{final_pattern_ids:,}"
)

print(
    f"Final BEGIN markers  : "
    f"{final_pattern_begin:,}"
)

print(
    f"Final END markers    : "
    f"{final_pattern_end:,}"
)

print(
    f"Final TX lines       : "
    f"{final_pattern_tx:,}"
)


# IMPORTANT:
# ID lines are NOT expected to equal the number of episodes because
# the original IBM Patterns.txt contributes 0 id="..." lines.

pattern_id_expected = (
    original_pattern_ids
    + SYNTH_PATTERN_EPISODES
)


print(
    f"\nExpected ID lines    : "
    f"{pattern_id_expected:,}"
)

print(
    f"Expected BEGIN/END   : "
    f"{EXPECTED_FINAL_PATTERN_EPISODES:,}"
)

print(
    f"Expected TX lines    : "
    f"{EXPECTED_FINAL_PATTERN_TX:,}"
)


pattern_id_gate = (
    final_pattern_ids
    == pattern_id_expected
)

pattern_begin_gate = (
    final_pattern_begin
    == EXPECTED_FINAL_PATTERN_EPISODES
)

pattern_end_gate = (
    final_pattern_end
    == EXPECTED_FINAL_PATTERN_EPISODES
)

pattern_tx_gate = (
    final_pattern_tx
    == EXPECTED_FINAL_PATTERN_TX
)


print(
    f"\n{'✅ PASS' if pattern_id_gate else '❌ FAIL'} "
    "| Pattern ID lines"
)

print(
    f"{'✅ PASS' if pattern_begin_gate else '❌ FAIL'} "
    "| Pattern BEGIN markers"
)

print(
    f"{'✅ PASS' if pattern_end_gate else '❌ FAIL'} "
    "| Pattern END markers"
)

print(
    f"{'✅ PASS' if pattern_tx_gate else '❌ FAIL'} "
    "| Pattern transaction lines"
)


# --------------------------------------------------------------------------------------------------
# 6. FINAL GLOBAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. FINAL GLOBAL VALIDATION")
print("=" * 80)


gates = {

    "Final transactions exact":
        transaction_gate,

    "Final accounts exact":
        account_gate,

    "Original IBM Patterns structure":
        original_pattern_gate,

    "Final Pattern ID lines":
        pattern_id_gate,

    "Final Pattern BEGIN markers":
        pattern_begin_gate,

    "Final Pattern END markers":
        pattern_end_gate,

    "Final Pattern transaction lines":
        pattern_tx_gate,
}


all_pass = True


for name, passed in gates.items():

    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} "
        f"| {name}"
    )

    all_pass &= passed


# --------------------------------------------------------------------------------------------------
# 7. CREATE FINAL MANIFEST
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. FINAL MANIFEST")
print("=" * 80)


manifest_data = {

    "dataset":
        "IBM HI-Small + pattern-aware synthetic laundering data",

    "status":
        "VALIDATED" if all_pass else "VALIDATION_FAILED",

    "transactions": {

        "original":
            5_078_345,

        "synthetic":
            5_067_991,

        "final":
            final_transaction_rows,
    },

    "accounts": {

        "original":
            518_581,

        "synthetic":
            486_372,

        "final":
            final_account_rows,
    },

    "patterns": {

        "original": {

            "episodes":
                original_begin,

            "id_lines":
                original_pattern_ids,

            "transaction_lines":
                original_tx,
        },

        "synthetic": {

            "episodes":
                SYNTH_PATTERN_EPISODES,

            "id_lines":
                SYNTH_PATTERN_EPISODES,

            "transaction_lines":
                SYNTH_PATTERN_TX,
        },

        "final": {

            "episodes":
                final_pattern_begin,

            "id_lines":
                final_pattern_ids,

            "begin_markers":
                final_pattern_begin,

            "end_markers":
                final_pattern_end,

            "transaction_lines":
                final_pattern_tx,
        },
    },

    "important_note":
        (
            "The original IBM HI-Small Patterns.txt contains "
            "370 BEGIN/END laundering episodes but 0 id=\"...\" "
            "lines. Therefore final ID-line count is 584,706, "
            "while final episode count is 585,076."
        ),

    "files": {

        "transactions":
            FINAL_TRANS,

        "accounts":
            FINAL_ACCOUNTS,

        "patterns":
            FINAL_PATTERNS,
    },

    "file_sizes_mb": {

        "transactions":
            round(mb(FINAL_TRANS), 2),

        "accounts":
            round(mb(FINAL_ACCOUNTS), 2),

        "patterns":
            round(mb(FINAL_PATTERNS), 2),
    },

    "validation_gates":
        gates,
}


with open(
    MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest_data,
        f,
        indent=2
    )


print(
    f"Manifest created:\n{MANIFEST}"
)


# --------------------------------------------------------------------------------------------------
# 8. DOWNLOAD LINKS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. DOWNLOAD FINAL FILES")
print("=" * 80)


print(
    "\n⬇️ 1. FINAL TRANSACTIONS — 10,146,336 rows"
)

display(
    FileLink(FINAL_TRANS)
)


print(
    "\n⬇️ 2. FINAL ACCOUNTS — 1,004,953 rows"
)

display(
    FileLink(FINAL_ACCOUNTS)
)


print(
    "\n⬇️ 3. FINAL PATTERNS — 585,076 episodes"
)

display(
    FileLink(FINAL_PATTERNS)
)


print("\n" + "=" * 100)


if all_pass:

    print(
        "✅✅✅ FINAL DATASET PACKAGE VALIDATED ✅✅✅"
    )

    print("=" * 100)

    print(
        "Transactions : 10,146,336"
    )

    print(
        "Accounts     : 1,004,953"
    )

    print(
        "Pattern eps. : 585,076"
    )

    print(
        "Pattern IDs  : 584,706"
    )

    print(
        "Pattern TX   : 5,071,200"
    )

    print(
        "\nAll three final files are ready for download."
    )

else:

    print(
        "❌ FINAL VALIDATION FAILED"
    )

print("=" * 100)

FINAL HI-SMALL MERGED DATASET VALIDATION

1. FINAL FILE EXISTENCE
✅ FINAL TRANSACTIONS   | 1,409.24 MB | HI-Small_FINAL_transactions_10146336.csv
✅ FINAL ACCOUNTS       | 60.82 MB | HI-Small_FINAL_accounts_1004953.csv
✅ FINAL PATTERNS       | 571.07 MB | HI-Small_FINAL_Patterns.txt

2. ORIGINAL IBM PATTERNS STRUCTURE
Original ID lines       : 0
Original BEGIN markers  : 370
Original END markers    : 370
Original TX lines       : 3,209

✅ Original IBM pattern structure PASS

3. FINAL TRANSACTION FILE
Final transaction rows : 10,146,336
Expected                : 10,146,336
✅ PASS

4. FINAL ACCOUNT FILE
Final account rows : 1,004,953
Expected           : 1,004,953
✅ PASS

5. FINAL PATTERNS FILE
Final ID lines       : 584,706
Final BEGIN markers  : 585,076
Final END markers    : 585,076
Final TX lines       : 5,071,200

Expected ID lines    : 584,706
Expected BEGIN/END   : 585,076
Expected TX lines    : 5,071,200

✅ PASS | Pattern ID lines
✅ PASS | Pattern BEGIN markers
✅ PASS | Pattern EN

/kaggle/working/AML_GNN_ONLY/data/HI-Small_FINAL_transactions_10146336.csv


⬇️ 2. FINAL ACCOUNTS — 1,004,953 rows


/kaggle/working/AML_GNN_ONLY/data/HI-Small_FINAL_accounts_1004953.csv


⬇️ 3. FINAL PATTERNS — 585,076 episodes


/kaggle/working/AML_GNN_ONLY/data/HI-Small_FINAL_Patterns.txt


✅✅✅ FINAL DATASET PACKAGE VALIDATED ✅✅✅
Transactions : 10,146,336
Accounts     : 1,004,953
Pattern eps. : 585,076
Pattern IDs  : 584,706
Pattern TX   : 5,071,200

All three final files are ready for download.


In [37]:
# ==================================================================================================
# CELL 22 — CORRECTED BEHAVIORAL DNA AUDIT
# Synthetic laundering vs ORIGINAL HI-Small behavior
#
# DOES NOT modify or regenerate any dataset.
# ==================================================================================================

import os
import csv
import json
import math
import random
from collections import Counter
from datetime import datetime


# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

BASE = "/kaggle/working/AML_GNN_ONLY/data"

SYN_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_transactions.csv"
)

ORIGINAL_FILE = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Trans.csv"
)

OUT_JSON = os.path.join(
    BASE,
    "HI-Small_SYNTH_vs_ORIGINAL_behavioral_dna_audit.json"
)

OUT_CSV = os.path.join(
    BASE,
    "HI-Small_SYNTH_vs_ORIGINAL_behavioral_dna_summary.csv"
)

SAMPLE_SIZE = 200_000
random.seed(42)


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def canonical_bank(x):
    s = str(x).strip()
    if not s:
        return None
    if s.isdigit():
        return str(int(s))
    return s.upper()


def pct(x, total):
    return 100.0 * x / total if total else 0.0


def quantile(values, q):
    if not values:
        return None
    values = sorted(values)
    idx = int(q * (len(values) - 1))
    return values[idx]


def distribution(counter, total):
    return {
        str(k): {
            "count": int(v),
            "pct": pct(v, total)
        }
        for k, v in counter.items()
    }


def new_stats():
    return {
        "rows": 0,
        "min_timestamp": None,
        "max_timestamp": None,
        "hour": Counter(),
        "day_of_week": Counter(),
        "date": Counter(),
        "payment_format": Counter(),
        "payment_currency": Counter(),
        "receiving_currency": Counter(),
        "source_bank": Counter(),
        "destination_bank": Counter(),
        "source_account": Counter(),
        "destination_account": Counter(),
        "amount_sample": [],
    }


# --------------------------------------------------------------------------------------------------
# STREAM ORIGINAL — POSITIONAL BECAUSE IBM HAS TWO "Account" COLUMNS
# --------------------------------------------------------------------------------------------------

def stream_original(path):

    stats = new_stats()

    with open(
        path,
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        reader = csv.reader(f)
        header = next(reader)

        expected_header = [
            "Timestamp",
            "From Bank",
            "Account",
            "To Bank",
            "Account",
            "Amount Received",
            "Receiving Currency",
            "Amount Paid",
            "Payment Currency",
            "Payment Format",
            "Is Laundering"
        ]

        if header != expected_header:
            raise RuntimeError(
                "Unexpected original HI-Small header:\n"
                f"{header}"
            )

        for row in reader:

            if len(row) != 11:
                continue

            stats["rows"] += 1

            ts = row[0].strip()

            try:
                dt = datetime.strptime(
                    ts,
                    "%Y-%m-%d %H:%M:%S"
                )
            except Exception:
                try:
                    dt = datetime.strptime(
                        ts,
                        "%Y/%m/%d %H:%M"
                    )
                except Exception:
                    dt = None

            if dt is not None:

                if stats["min_timestamp"] is None or dt < stats["min_timestamp"]:
                    stats["min_timestamp"] = dt

                if stats["max_timestamp"] is None or dt > stats["max_timestamp"]:
                    stats["max_timestamp"] = dt

                stats["hour"][dt.hour] += 1
                stats["day_of_week"][dt.strftime("%A")] += 1
                stats["date"][dt.strftime("%Y-%m-%d")] += 1

            stats["payment_format"][row[9].strip()] += 1
            stats["payment_currency"][row[8].strip()] += 1
            stats["receiving_currency"][row[6].strip()] += 1

            src_bank = canonical_bank(row[1])
            dst_bank = canonical_bank(row[3])

            src_account = row[2].strip()
            dst_account = row[4].strip()

            stats["source_bank"][src_bank] += 1
            stats["destination_bank"][dst_bank] += 1

            stats["source_account"][src_account] += 1
            stats["destination_account"][dst_account] += 1

            try:
                amount = float(row[5])

                if math.isfinite(amount) and amount > 0:

                    n = len(stats["amount_sample"])

                    if n < SAMPLE_SIZE:

                        stats["amount_sample"].append(amount)

                    else:

                        j = random.randint(
                            0,
                            stats["rows"] - 1
                        )

                        if j < SAMPLE_SIZE:
                            stats["amount_sample"][j] = amount

            except Exception:
                pass

    return stats


# --------------------------------------------------------------------------------------------------
# STREAM SYNTHETIC
# --------------------------------------------------------------------------------------------------

def stream_synthetic(path):

    stats = new_stats()

    with open(
        path,
        "r",
        newline="",
        encoding="utf-8"
    ) as f:

        reader = csv.DictReader(f)

        required = [
            "Timestamp",
            "From Bank",
            "Account",
            "To Bank",
            "Account.1",
            "Amount Received",
            "Receiving Currency",
            "Payment Currency",
            "Payment Format",
        ]

        missing = [
            c for c in required
            if c not in reader.fieldnames
        ]

        if missing:
            raise RuntimeError(
                "Synthetic file missing columns:\n"
                + str(missing)
            )

        for row in reader:

            stats["rows"] += 1

            ts = row["Timestamp"].strip()

            try:
                dt = datetime.strptime(
                    ts,
                    "%Y-%m-%d %H:%M:%S"
                )
            except Exception:
                try:
                    dt = datetime.strptime(
                        ts,
                        "%Y/%m/%d %H:%M"
                    )
                except Exception:
                    dt = None

            if dt is not None:

                if stats["min_timestamp"] is None or dt < stats["min_timestamp"]:
                    stats["min_timestamp"] = dt

                if stats["max_timestamp"] is None or dt > stats["max_timestamp"]:
                    stats["max_timestamp"] = dt

                stats["hour"][dt.hour] += 1
                stats["day_of_week"][dt.strftime("%A")] += 1
                stats["date"][dt.strftime("%Y-%m-%d")] += 1

            stats["payment_format"][
                row["Payment Format"].strip()
            ] += 1

            stats["payment_currency"][
                row["Payment Currency"].strip()
            ] += 1

            stats["receiving_currency"][
                row["Receiving Currency"].strip()
            ] += 1

            src_bank = canonical_bank(
                row["From Bank"]
            )

            dst_bank = canonical_bank(
                row["To Bank"]
            )

            stats["source_bank"][src_bank] += 1
            stats["destination_bank"][dst_bank] += 1

            stats["source_account"][
                row["Account"].strip()
            ] += 1

            stats["destination_account"][
                row["Account.1"].strip()
            ] += 1

            try:
                amount = float(
                    row["Amount Received"]
                )

                if math.isfinite(amount) and amount > 0:

                    n = len(
                        stats["amount_sample"]
                    )

                    if n < SAMPLE_SIZE:

                        stats["amount_sample"].append(
                            amount
                        )

                    else:

                        j = random.randint(
                            0,
                            stats["rows"] - 1
                        )

                        if j < SAMPLE_SIZE:
                            stats["amount_sample"][j] = amount

            except Exception:
                pass

    return stats


# --------------------------------------------------------------------------------------------------
# RUN
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("CELL 22 — CORRECTED BEHAVIORAL DNA AUDIT")
print("=" * 100)

print("\n1. Streaming ORIGINAL HI-Small...")
original = stream_original(ORIGINAL_FILE)

print(
    f"Original rows     : {original['rows']:,}"
)

print(
    f"Original range    : "
    f"{original['min_timestamp']} -> "
    f"{original['max_timestamp']}"
)

print("\n2. Streaming SYNTHETIC laundering data...")
synthetic = stream_synthetic(SYN_FILE)

print(
    f"Synthetic rows    : {synthetic['rows']:,}"
)

print(
    f"Synthetic range   : "
    f"{synthetic['min_timestamp']} -> "
    f"{synthetic['max_timestamp']}"
)


# --------------------------------------------------------------------------------------------------
# TIMESTAMP
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. TIMESTAMP COVERAGE")
print("=" * 80)

original_dates = set(original["date"])
synthetic_dates = set(synthetic["date"])

missing_dates = original_dates - synthetic_dates

print(
    f"Original active dates : {len(original_dates)}"
)

print(
    f"Synthetic active dates: {len(synthetic_dates)}"
)

print(
    f"Missing synthetic dates: {len(missing_dates)}"
)

if missing_dates:
    print("\nMissing dates:")
    for d in sorted(missing_dates):
        print(" ", d)


# --------------------------------------------------------------------------------------------------
# HOUR
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. HOUR DISTRIBUTION")
print("=" * 80)

hour_diffs = []

for h in range(24):

    op = pct(
        original["hour"].get(h, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["hour"].get(h, 0),
        synthetic["rows"]
    )

    diff = abs(op - sp)
    hour_diffs.append(diff)

    print(
        f"{h:02d}:00 "
        f"| Original={op:8.4f}% "
        f"| Synthetic={sp:8.4f}% "
        f"| AbsDiff={diff:8.4f}"
    )

hour_mae = sum(hour_diffs) / len(hour_diffs)

print(
    f"\nHour MAE = {hour_mae:.6f} percentage points"
)


# --------------------------------------------------------------------------------------------------
# DAY OF WEEK
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. DAY-OF-WEEK DISTRIBUTION")
print("=" * 80)

days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

dow_diffs = []

for d in days:

    op = pct(
        original["day_of_week"].get(d, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["day_of_week"].get(d, 0),
        synthetic["rows"]
    )

    diff = abs(op - sp)
    dow_diffs.append(diff)

    print(
        f"{d:10s} "
        f"| Original={op:8.4f}% "
        f"| Synthetic={sp:8.4f}% "
        f"| AbsDiff={diff:8.4f}"
    )

dow_mae = sum(dow_diffs) / len(dow_diffs)

print(
    f"\nDay-of-week MAE = "
    f"{dow_mae:.6f} percentage points"
)


# --------------------------------------------------------------------------------------------------
# PAYMENT FORMAT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. PAYMENT FORMAT")
print("=" * 80)

formats = sorted(
    set(original["payment_format"])
    | set(synthetic["payment_format"])
)

format_diffs = []

for fmt in formats:

    op = pct(
        original["payment_format"].get(fmt, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["payment_format"].get(fmt, 0),
        synthetic["rows"]
    )

    diff = abs(op - sp)
    format_diffs.append(diff)

    print(
        f"{fmt:18s} "
        f"| Original={op:8.4f}% "
        f"| Synthetic={sp:8.4f}% "
        f"| AbsDiff={diff:8.4f}"
    )

format_mae = sum(format_diffs) / len(format_diffs)

print(
    f"\nPayment-format MAE = "
    f"{format_mae:.6f} percentage points"
)


# --------------------------------------------------------------------------------------------------
# PAYMENT CURRENCY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. PAYMENT CURRENCY")
print("=" * 80)

payment_currencies = sorted(
    set(original["payment_currency"])
    | set(synthetic["payment_currency"])
)

payment_currency_diffs = []

for curr in payment_currencies:

    op = pct(
        original["payment_currency"].get(curr, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["payment_currency"].get(curr, 0),
        synthetic["rows"]
    )

    diff = abs(op - sp)
    payment_currency_diffs.append(diff)

    print(
        f"{curr:20s} "
        f"| Original={op:8.4f}% "
        f"| Synthetic={sp:8.4f}% "
        f"| AbsDiff={diff:8.4f}"
    )

payment_currency_mae = (
    sum(payment_currency_diffs)
    / len(payment_currency_diffs)
)

print(
    f"\nPayment-currency MAE = "
    f"{payment_currency_mae:.6f} percentage points"
)


# --------------------------------------------------------------------------------------------------
# RECEIVING CURRENCY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. RECEIVING CURRENCY")
print("=" * 80)

receiving_currencies = sorted(
    set(original["receiving_currency"])
    | set(synthetic["receiving_currency"])
)

receiving_currency_diffs = []

for curr in receiving_currencies:

    op = pct(
        original["receiving_currency"].get(curr, 0),
        original["rows"]
    )

    sp = pct(
        synthetic["receiving_currency"].get(curr, 0),
        synthetic["rows"]
    )

    diff = abs(op - sp)
    receiving_currency_diffs.append(diff)

    print(
        f"{curr:20s} "
        f"| Original={op:8.4f}% "
        f"| Synthetic={sp:8.4f}% "
        f"| AbsDiff={diff:8.4f}"
    )

receiving_currency_mae = (
    sum(receiving_currency_diffs)
    / len(receiving_currency_diffs)
)

print(
    f"\nReceiving-currency MAE = "
    f"{receiving_currency_mae:.6f} percentage points"
)


# --------------------------------------------------------------------------------------------------
# AMOUNTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("9. AMOUNT DISTRIBUTION")
print("=" * 80)

quantile_points = [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
]

print(
    f"{'Quantile':>10} "
    f"{'Original':>20} "
    f"{'Synthetic':>20} "
    f"{'Ratio':>12}"
)

amount_ratios = []

for q in quantile_points:

    ov = quantile(
        original["amount_sample"],
        q
    )

    sv = quantile(
        synthetic["amount_sample"],
        q
    )

    ratio = (
        sv / ov
        if ov is not None
        and ov != 0
        else None
    )

    if ratio is not None:
        amount_ratios.append(ratio)

    print(
        f"p{int(q*100):02d} "
        f"{ov:20.6f} "
        f"{sv:20.6f} "
        f"{ratio:12.4f}"
    )


# --------------------------------------------------------------------------------------------------
# BANK COVERAGE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("10. BANK / ACCOUNT COVERAGE")
print("=" * 80)

original_source_banks = set(
    original["source_bank"]
)

synthetic_source_banks = set(
    synthetic["source_bank"]
)

original_dest_banks = set(
    original["destination_bank"]
)

synthetic_dest_banks = set(
    synthetic["destination_bank"]
)

print(
    f"Unique source banks:"
)

print(
    f"  Original : {len(original_source_banks):,}"
)

print(
    f"  Synthetic: {len(synthetic_source_banks):,}"
)

print(
    f"  Synthetic covers "
    f"{pct(len(synthetic_source_banks.intersection(original_source_banks)), len(original_source_banks)):.4f}% "
    f"of original source banks"
)


print(
    f"\nUnique destination banks:"
)

print(
    f"  Original : {len(original_dest_banks):,}"
)

print(
    f"  Synthetic: {len(synthetic_dest_banks):,}"
)

print(
    f"  Synthetic covers "
    f"{pct(len(synthetic_dest_banks.intersection(original_dest_banks)), len(original_dest_banks)):.4f}% "
    f"of original destination banks"
)


print(
    f"\nUnique source accounts:"
)

print(
    f"  Original : {len(original['source_account']):,}"
)

print(
    f"  Synthetic: {len(synthetic['source_account']):,}"
)


print(
    f"\nUnique destination accounts:"
)

print(
    f"  Original : {len(original['destination_account']):,}"
)

print(
    f"  Synthetic: {len(synthetic['destination_account']):,}"
)


# --------------------------------------------------------------------------------------------------
# BANK CONCENTRATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("11. BANK CONCENTRATION")
print("=" * 80)


def top_share(counter, total, k):
    return (
        100.0 * sum(
            v
            for _, v in counter.most_common(k)
        ) / total
        if total
        else 0.0
    )


print("\nSource-bank concentration:")

for k in [1, 5, 10, 25, 50, 100]:

    o = top_share(
        original["source_bank"],
        original["rows"],
        k
    )

    s = top_share(
        synthetic["source_bank"],
        synthetic["rows"],
        k
    )

    print(
        f"Top {k:3d} "
        f"| Original={o:10.4f}% "
        f"| Synthetic={s:10.4f}%"
    )


print("\nDestination-bank concentration:")

for k in [1, 5, 10, 25, 50, 100]:

    o = top_share(
        original["destination_bank"],
        original["rows"],
        k
    )

    s = top_share(
        synthetic["destination_bank"],
        synthetic["rows"],
        k
    )

    print(
        f"Top {k:3d} "
        f"| Original={o:10.4f}% "
        f"| Synthetic={s:10.4f}%"
    )


# --------------------------------------------------------------------------------------------------
# ACCOUNT REUSE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("12. ACCOUNT REUSE")
print("=" * 80)


def activity_stats(counter):

    values = list(
        counter.values()
    )

    if not values:
        return {}

    values.sort()

    n = len(values)

    def q(p):
        return values[
            int(
                p * (n - 1)
            )
        ]

    buckets = Counter()

    for v in values:

        if v == 1:
            buckets["1"] += 1

        elif v == 2:
            buckets["2"] += 1

        elif 3 <= v <= 5:
            buckets["3-5"] += 1

        elif 6 <= v <= 10:
            buckets["6-10"] += 1

        elif 11 <= v <= 25:
            buckets["11-25"] += 1

        elif 26 <= v <= 50:
            buckets["26-50"] += 1

        elif 51 <= v <= 100:
            buckets["51-100"] += 1

        elif 101 <= v <= 250:
            buckets["101-250"] += 1

        elif 251 <= v <= 500:
            buckets["251-500"] += 1

        elif 501 <= v <= 1000:
            buckets["501-1000"] += 1

        else:
            buckets[">1000"] += 1

    return {
        "min": min(values),
        "p25": q(0.25),
        "median": q(0.50),
        "p75": q(0.75),
        "p90": q(0.90),
        "p95": q(0.95),
        "p99": q(0.99),
        "max": max(values),
        "buckets": buckets,
    }


orig_src_activity = activity_stats(
    original["source_account"]
)

syn_src_activity = activity_stats(
    synthetic["source_account"]
)

orig_dst_activity = activity_stats(
    original["destination_account"]
)

syn_dst_activity = activity_stats(
    synthetic["destination_account"]
)


print("\nSOURCE ACCOUNT ACTIVITY")

for key in [
    "min",
    "p25",
    "median",
    "p75",
    "p90",
    "p95",
    "p99",
    "max"
]:

    print(
        f"{key:8s} "
        f"| Original={orig_src_activity[key]:12} "
        f"| Synthetic={syn_src_activity[key]:12}"
    )


print("\nDESTINATION ACCOUNT ACTIVITY")

for key in [
    "min",
    "p25",
    "median",
    "p75",
    "p90",
    "p95",
    "p99",
    "max"
]:

    print(
        f"{key:8s} "
        f"| Original={orig_dst_activity[key]:12} "
        f"| Synthetic={syn_dst_activity[key]:12}"
    )


# --------------------------------------------------------------------------------------------------
# SAVE AUDIT
# --------------------------------------------------------------------------------------------------

audit = {

    "original_rows":
        original["rows"],

    "synthetic_rows":
        synthetic["rows"],

    "timestamp": {

        "original_min":
            str(original["min_timestamp"]),

        "original_max":
            str(original["max_timestamp"]),

        "synthetic_min":
            str(synthetic["min_timestamp"]),

        "synthetic_max":
            str(synthetic["max_timestamp"]),

        "original_active_dates":
            sorted(original_dates),

        "synthetic_active_dates":
            sorted(synthetic_dates),

        "missing_synthetic_dates":
            sorted(missing_dates),
    },

    "distribution_mae": {

        "hour":
            hour_mae,

        "day_of_week":
            dow_mae,

        "payment_format":
            format_mae,

        "payment_currency":
            payment_currency_mae,

        "receiving_currency":
            receiving_currency_mae,
    },

    "amount_quantiles": {

        "original": {
            f"p{int(q*100):02d}":
                quantile(
                    original["amount_sample"],
                    q
                )
            for q in quantile_points
        },

        "synthetic": {
            f"p{int(q*100):02d}":
                quantile(
                    synthetic["amount_sample"],
                    q
                )
            for q in quantile_points
        },
    },

    "bank_coverage": {

        "original_source_banks":
            len(original_source_banks),

        "synthetic_source_banks":
            len(synthetic_source_banks),

        "original_destination_banks":
            len(original_dest_banks),

        "synthetic_destination_banks":
            len(synthetic_dest_banks),
    },

    "payment_format":

        {
            "original":
                distribution(
                    original["payment_format"],
                    original["rows"]
                ),

            "synthetic":
                distribution(
                    synthetic["payment_format"],
                    synthetic["rows"]
                ),
        },

    "payment_currency":

        {
            "original":
                distribution(
                    original["payment_currency"],
                    original["rows"]
                ),

            "synthetic":
                distribution(
                    synthetic["payment_currency"],
                    synthetic["rows"]
                ),
        },

    "receiving_currency":

        {
            "original":
                distribution(
                    original["receiving_currency"],
                    original["rows"]
                ),

            "synthetic":
                distribution(
                    synthetic["receiving_currency"],
                    synthetic["rows"]
                ),
        },

    "hour_distribution":

        {
            "original":
                distribution(
                    original["hour"],
                    original["rows"]
                ),

            "synthetic":
                distribution(
                    synthetic["hour"],
                    synthetic["rows"]
                ),
        },
}


with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        indent=2
    )


with open(
    OUT_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "metric",
        "value"
    ])

    writer.writerow([
        "original_rows",
        original["rows"]
    ])

    writer.writerow([
        "synthetic_rows",
        synthetic["rows"]
    ])

    writer.writerow([
        "hour_mae_percentage_points",
        hour_mae
    ])

    writer.writerow([
        "day_of_week_mae_percentage_points",
        dow_mae
    ])

    writer.writerow([
        "payment_format_mae_percentage_points",
        format_mae
    ])

    writer.writerow([
        "payment_currency_mae_percentage_points",
        payment_currency_mae
    ])

    writer.writerow([
        "receiving_currency_mae_percentage_points",
        receiving_currency_mae
    ])

    writer.writerow([
        "original_source_banks",
        len(original_source_banks)
    ])

    writer.writerow([
        "synthetic_source_banks",
        len(synthetic_source_banks)
    ])

    writer.writerow([
        "original_destination_banks",
        len(original_dest_banks)
    ])

    writer.writerow([
        "synthetic_destination_banks",
        len(synthetic_dest_banks)
    ])


# --------------------------------------------------------------------------------------------------
# DONE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("✅ CELL 22 COMPLETE")
print("=" * 100)

print(
    f"Audit JSON: {OUT_JSON}"
)

print(
    f"Audit CSV : {OUT_CSV}"
)

print("=" * 100)

CELL 22 — CORRECTED BEHAVIORAL DNA AUDIT

1. Streaming ORIGINAL HI-Small...
Original rows     : 5,078,345
Original range    : 2022-09-01 00:00:00 -> 2022-09-18 16:18:00

2. Streaming SYNTHETIC laundering data...
Synthetic rows    : 5,067,991
Synthetic range   : 2022-09-01 00:00:00 -> 2022-09-14 19:51:00

3. TIMESTAMP COVERAGE
Original active dates : 18
Synthetic active dates: 14
Missing synthetic dates: 4

Missing dates:
  2022-09-15
  2022-09-16
  2022-09-17
  2022-09-18

4. HOUR DISTRIBUTION
00:00 | Original= 12.4987% | Synthetic=  4.9561% | AbsDiff=  7.5426
01:00 | Original=  3.8148% | Synthetic=  4.0008% | AbsDiff=  0.1860
02:00 | Original=  3.8033% | Synthetic=  4.0594% | AbsDiff=  0.2562
03:00 | Original=  3.8054% | Synthetic=  4.0574% | AbsDiff=  0.2521
04:00 | Original=  3.8037% | Synthetic=  4.0374% | AbsDiff=  0.2337
05:00 | Original=  3.8182% | Synthetic=  4.0783% | AbsDiff=  0.2602
06:00 | Original=  3.8291% | Synthetic=  4.0222% | AbsDiff=  0.1931
07:00 | Original=  3.8071

In [39]:
# ==================================================================================================
# CELL 23 — FULL-SCALE PATTERN MORPHOLOGY AUDIT
#
# PURPOSE:
#   Verify that every synthetic laundering episode preserves the structural morphology
#   of its assigned IBM template.
#
# Checks:
#   1. Episode continuity
#   2. Typology consistency
#   3. Source-template consistency
#   4. Transaction count consistency
#   5. Unique node count
#   6. Unique edge count
#   7. Max in-degree / out-degree
#   8. Exact node degree signature
#   9. Template source-order preservation
#  10. Relative timestamp structure preservation
#
# IMPORTANT:
#   - DOES NOT modify any file
#   - DOES NOT regenerate data
#   - DOES NOT train the GNN
# ==================================================================================================

import os
import csv
import json
from collections import defaultdict, Counter


# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

BASE = "/kaggle/working/AML_GNN_ONLY/data"

SYN_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_transactions.csv"
)

EPISODE_AUDIT_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_episode_audit.csv"
)

TEMPLATE_FILE = os.path.join(
    BASE,
    "HI-Small_pattern_templates.json"
)

OUT_JSON = os.path.join(
    BASE,
    "HI-Small_SYNTH_full_morphology_audit.json"
)

OUT_CSV = os.path.join(
    BASE,
    "HI-Small_SYNTH_full_morphology_summary.csv"
)

EXPECTED_EPISODES = 584_706
EXPECTED_TX = 5_067_991


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def parse_timestamp(ts):
    ts = str(ts).strip()

    try:
        from datetime import datetime

        return datetime.strptime(
            ts,
            "%Y-%m-%d %H:%M:%S"
        )

    except Exception:

        from datetime import datetime

        return datetime.strptime(
            ts,
            "%Y/%m/%d %H:%M"
        )


def canonical_node(bank, account):
    return (
        str(bank).strip()
        + "|"
        + str(account).strip()
    )


def freeze_degree_signature(in_deg, out_deg):
    """
    Exact unlabeled directed degree signature.

    Each node represented as:
        (in_degree, out_degree)

    Sorted so node IDs do not affect the signature.
    """

    return tuple(
        sorted(
            (
                int(in_deg.get(node, 0)),
                int(out_deg.get(node, 0))
            )
            for node in set(in_deg) | set(out_deg)
        )
    )


def finish_episode(
    state,
    template_reference,
    results
):
    """
    Finalize one episode and compare with its template reference.
    """

    if state is None:
        return

    episode_id = state["episode_id"]
    template_id = state["template_id"]
    typology = state["typology"]

    tx_count = state["tx_count"]

    nodes = state["nodes"]

    unique_edges = state["edges"]

    in_deg = state["in_deg"]
    out_deg = state["out_deg"]

    degree_signature = freeze_degree_signature(
        in_deg,
        out_deg
    )

    max_in = (
        max(in_deg.values())
        if in_deg
        else 0
    )

    max_out = (
        max(out_deg.values())
        if out_deg
        else 0
    )

    source_nodes = sum(
        1
        for node in nodes
        if (
            in_deg.get(node, 0) == 0
            and out_deg.get(node, 0) > 0
        )
    )

    destination_nodes = sum(
        1
        for node in nodes
        if (
            out_deg.get(node, 0) == 0
            and in_deg.get(node, 0) > 0
        )
    )

    intermediate_nodes = sum(
        1
        for node in nodes
        if (
            in_deg.get(node, 0) > 0
            and out_deg.get(node, 0) > 0
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Relative timestamp sequence
    # ----------------------------------------------------------------------------------------------

    timestamps = state["timestamps"]

    if timestamps:

        t0 = min(timestamps)

        relative_minutes = tuple(
            int(
                (t - t0).total_seconds() // 60
            )
            for t in timestamps
        )

        relative_minutes_sorted = tuple(
            sorted(relative_minutes)
        )

    else:

        relative_minutes = tuple()
        relative_minutes_sorted = tuple()


    current_signature = {
        "typology": typology,
        "tx_count": tx_count,
        "node_count": len(nodes),
        "edge_count": len(unique_edges),
        "max_in_degree": max_in,
        "max_out_degree": max_out,
        "source_nodes": source_nodes,
        "destination_nodes": destination_nodes,
        "intermediate_nodes": intermediate_nodes,
        "degree_signature": degree_signature,
        "relative_minutes_sorted":
            relative_minutes_sorted,
    }


    results["episodes"] += 1
    results["transactions"] += tx_count

    results["typology_episodes"][
        typology
    ] += 1

    results["typology_transactions"][
        typology
    ] += tx_count


    # ----------------------------------------------------------------------------------------------
    # Reference template comparison
    # ----------------------------------------------------------------------------------------------

    if template_id not in template_reference:

        template_reference[
            template_id
        ] = {

            **current_signature,

            "_first_episode":
                episode_id
        }

        results[
            "new_template_ids"
        ].add(
            template_id
        )

        results[
            "template_typology"
        ][template_id] = typology

        results[
            "template_counts"
        ][template_id] += 1

        return


    ref = template_reference[
        template_id
    ]

    results[
        "template_counts"
    ][template_id] += 1


    # ----------------------------------------------------------------------------------------------
    # Typology consistency
    # ----------------------------------------------------------------------------------------------

    if typology != ref["typology"]:

        results[
            "typology_mismatches"
        ] += 1


    # ----------------------------------------------------------------------------------------------
    # Transaction count
    # ----------------------------------------------------------------------------------------------

    if tx_count != ref["tx_count"]:

        results[
            "transaction_count_mismatches"
        ] += 1


    # ----------------------------------------------------------------------------------------------
    # Node count
    # ----------------------------------------------------------------------------------------------

    if (
        len(nodes)
        != ref["node_count"]
    ):

        results[
            "node_count_mismatches"
        ] += 1


    # ----------------------------------------------------------------------------------------------
    # Edge count
    # ----------------------------------------------------------------------------------------------

    if (
        len(unique_edges)
        != ref["edge_count"]
    ):

        results[
            "edge_count_mismatches"
        ] += 1


    # ----------------------------------------------------------------------------------------------
    # Degree structure
    # ----------------------------------------------------------------------------------------------

    if (
        degree_signature
        != ref["degree_signature"]
    ):

        results[
            "degree_signature_mismatches"
        ] += 1


    # ----------------------------------------------------------------------------------------------
    # Max in/out degree
    # ----------------------------------------------------------------------------------------------

    if (
        max_in
        != ref["max_in_degree"]
    ):

        results[
            "max_in_degree_mismatches"
        ] += 1


    if (
        max_out
        != ref["max_out_degree"]
    ):

        results[
            "max_out_degree_mismatches"
        ] += 1


    # ----------------------------------------------------------------------------------------------
    # Role counts
    # ----------------------------------------------------------------------------------------------

    if (
        source_nodes
        != ref["source_nodes"]
    ):

        results[
            "source_node_mismatches"
        ] += 1


    if (
        destination_nodes
        != ref["destination_nodes"]
    ):

        results[
            "destination_node_mismatches"
        ] += 1


    if (
        intermediate_nodes
        != ref["intermediate_nodes"]
    ):

        results[
            "intermediate_node_mismatches"
        ] += 1


    # ----------------------------------------------------------------------------------------------
    # Relative timestamp structure
    # ----------------------------------------------------------------------------------------------

    if (
        relative_minutes_sorted
        != ref["relative_minutes_sorted"]
    ):

        results[
            "timestamp_structure_mismatches"
        ] += 1


    # ----------------------------------------------------------------------------------------------
    # Source-order consistency
    # ----------------------------------------------------------------------------------------------

    source_order = state[
        "source_orders"
    ]

    if source_order:

        expected_order = tuple(
            range(
                len(source_order)
            )
        )

        if source_order != expected_order:

            results[
                "source_order_mismatches"
            ] += 1


# --------------------------------------------------------------------------------------------------
# INITIALIZE
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("CELL 23 — FULL-SCALE PATTERN MORPHOLOGY AUDIT")
print("=" * 100)

if not os.path.exists(SYN_FILE):

    raise FileNotFoundError(
        SYN_FILE
    )


results = {

    "episodes": 0,

    "transactions": 0,

    "typology_episodes": Counter(),

    "typology_transactions": Counter(),

    "template_counts": Counter(),

    "template_typology": {},

    "new_template_ids": set(),

    "typology_mismatches": 0,

    "transaction_count_mismatches": 0,

    "node_count_mismatches": 0,

    "edge_count_mismatches": 0,

    "degree_signature_mismatches": 0,

    "max_in_degree_mismatches": 0,

    "max_out_degree_mismatches": 0,

    "source_node_mismatches": 0,

    "destination_node_mismatches": 0,

    "intermediate_node_mismatches": 0,

    "timestamp_structure_mismatches": 0,

    "source_order_mismatches": 0,
}


template_reference = {}


# Current episode
current_episode = None
closed_episode_ids = set()

rows_seen = 0


# --------------------------------------------------------------------------------------------------
# STREAM TRANSACTIONS
# --------------------------------------------------------------------------------------------------

print("\nStreaming synthetic transactions...")

with open(
    SYN_FILE,
    "r",
    newline="",
    encoding="utf-8"
) as f:

    reader = csv.DictReader(f)

    required = [
        "synthetic_episode_id",
        "typology",
        "source_template_id",
        "template_source_order",
        "Timestamp",
        "From Bank",
        "Account",
        "To Bank",
        "Account.1",
    ]

    missing = [
        c
        for c in required
        if c not in reader.fieldnames
    ]

    if missing:

        raise RuntimeError(
            "Missing required columns:\n"
            + str(missing)
        )


    for row in reader:

        rows_seen += 1

        episode_id = row[
            "synthetic_episode_id"
        ].strip()

        typology = row[
            "typology"
        ].strip().upper()

        template_id = row[
            "source_template_id"
        ].strip()

        template_order = int(
            row[
                "template_source_order"
            ]
        )

        # ------------------------------------------------------------------------------------------
        # New episode
        # ------------------------------------------------------------------------------------------

        if (
            current_episode is None
            or episode_id
            != current_episode["episode_id"]
        ):

            # Close previous
            if current_episode is not None:

                finish_episode(
                    current_episode,
                    template_reference,
                    results
                )

                closed_episode_ids.add(
                    current_episode[
                        "episode_id"
                    ]
                )


            # Detect a reappearing episode
            if episode_id in closed_episode_ids:

                raise RuntimeError(
                    "Episode IDs are not contiguous. "
                    f"Episode {episode_id} reappeared."
                )


            current_episode = {

                "episode_id":
                    episode_id,

                "typology":
                    typology,

                "template_id":
                    template_id,

                "tx_count":
                    0,

                "nodes":
                    set(),

                "edges":
                    set(),

                "in_deg":
                    defaultdict(int),

                "out_deg":
                    defaultdict(int),

                "timestamps":
                    [],

                "source_orders":
                    [],
            }


        # ------------------------------------------------------------------------------------------
        # Consistency within episode
        # ------------------------------------------------------------------------------------------

        if typology != current_episode["typology"]:

            results[
                "typology_mismatches"
            ] += 1


        if template_id != current_episode["template_id"]:

            results[
                "transaction_count_mismatches"
            ] += 1


        # ------------------------------------------------------------------------------------------
        # Graph nodes / edges
        # ------------------------------------------------------------------------------------------

        src = canonical_node(
            row["From Bank"],
            row["Account"]
        )

        dst = canonical_node(
            row["To Bank"],
            row["Account.1"]
        )

        edge = (
            src,
            dst
        )

        current_episode["nodes"].add(
            src
        )

        current_episode["nodes"].add(
            dst
        )

        current_episode["edges"].add(
            edge
        )

        current_episode["out_deg"][src] += 1
        current_episode["in_deg"][dst] += 1


        # ------------------------------------------------------------------------------------------
        # Time structure
        # ------------------------------------------------------------------------------------------

        try:

            current_episode[
                "timestamps"
            ].append(
                parse_timestamp(
                    row["Timestamp"]
                )
            )

        except Exception:

            pass


        # ------------------------------------------------------------------------------------------
        # Template source order
        # ------------------------------------------------------------------------------------------

        current_episode[
            "source_orders"
        ].append(
            template_order
        )

        current_episode[
            "tx_count"
        ] += 1


        # ------------------------------------------------------------------------------------------
        # Progress
        # ------------------------------------------------------------------------------------------

        if rows_seen % 1_000_000 == 0:

            print(
                f"  Rows processed: "
                f"{rows_seen:,}"
            )


# Finish final episode
if current_episode is not None:

    finish_episode(
        current_episode,
        template_reference,
        results
    )

    closed_episode_ids.add(
        current_episode[
            "episode_id"
        ]
    )


# --------------------------------------------------------------------------------------------------
# BASIC COUNTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. BASIC MORPHOLOGY COUNTS")
print("=" * 80)

print(
    f"Transactions processed : "
    f"{rows_seen:,}"
)

print(
    f"Expected transactions   : "
    f"{EXPECTED_TX:,}"
)

print(
    f"Episodes processed      : "
    f"{results['episodes']:,}"
)

print(
    f"Expected episodes       : "
    f"{EXPECTED_EPISODES:,}"
)

print(
    f"Unique templates seen   : "
    f"{len(template_reference):,}"
)


# --------------------------------------------------------------------------------------------------
# TYPOLOGY SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. TYPOLOGY MORPHOLOGY")
print("=" * 80)

for typ in sorted(
    results["typology_episodes"]
):

    print(
        f"{typ:18s} "
        f"| Episodes="
        f"{results['typology_episodes'][typ]:8,} "
        f"| Transactions="
        f"{results['typology_transactions'][typ]:10,}"
    )


# --------------------------------------------------------------------------------------------------
# MISMATCH SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. MORPHOLOGY MISMATCH SUMMARY")
print("=" * 80)

mismatch_metrics = {

    "Typology mismatches":
        results["typology_mismatches"],

    "Transaction-count mismatches":
        results["transaction_count_mismatches"],

    "Node-count mismatches":
        results["node_count_mismatches"],

    "Edge-count mismatches":
        results["edge_count_mismatches"],

    "Degree-signature mismatches":
        results["degree_signature_mismatches"],

    "Max in-degree mismatches":
        results["max_in_degree_mismatches"],

    "Max out-degree mismatches":
        results["max_out_degree_mismatches"],

    "Source-node mismatches":
        results["source_node_mismatches"],

    "Destination-node mismatches":
        results["destination_node_mismatches"],

    "Intermediate-node mismatches":
        results["intermediate_node_mismatches"],

    "Timestamp-structure mismatches":
        results["timestamp_structure_mismatches"],

    "Source-order mismatches":
        results["source_order_mismatches"],
}


all_morphology_zero = True

for name, value in mismatch_metrics.items():

    passed = (
        value == 0
    )

    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} "
        f"| {name:35s} | {value:,}"
    )

    all_morphology_zero &= passed


# --------------------------------------------------------------------------------------------------
# TEMPLATE USAGE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. TEMPLATE USAGE")
print("=" * 80)

template_usage_values = list(
    results["template_counts"].values()
)

if template_usage_values:

    print(
        f"Templates used : "
        f"{len(template_usage_values):,}"
    )

    print(
        f"Minimum episodes/template : "
        f"{min(template_usage_values):,}"
    )

    print(
        f"Maximum episodes/template : "
        f"{max(template_usage_values):,}"
    )

    print(
        f"Median episodes/template  : "
        f"{sorted(template_usage_values)[
            len(template_usage_values)//2
        ]:,}"
    )


# --------------------------------------------------------------------------------------------------
# SAVE JSON
# --------------------------------------------------------------------------------------------------

audit_json = {

    "transactions_processed":
        rows_seen,

    "expected_transactions":
        EXPECTED_TX,

    "episodes_processed":
        results["episodes"],

    "expected_episodes":
        EXPECTED_EPISODES,

    "unique_templates":
        len(template_reference),

    "typology_episodes":
        dict(
            results["typology_episodes"]
        ),

    "typology_transactions":
        dict(
            results["typology_transactions"]
        ),

    "template_counts":
        {
            str(k): int(v)
            for k, v
            in results["template_counts"].items()
        },

    "mismatch_metrics":
        mismatch_metrics,

    "morphology_pass":
        (
            rows_seen == EXPECTED_TX
            and results["episodes"]
                == EXPECTED_EPISODES
            and len(template_reference)
                == 370
            and all_morphology_zero
        ),
}


with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_json,
        f,
        indent=2
    )


# --------------------------------------------------------------------------------------------------
# SAVE SUMMARY CSV
# --------------------------------------------------------------------------------------------------

with open(
    OUT_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "metric",
        "value"
    ])

    writer.writerow([
        "transactions_processed",
        rows_seen
    ])

    writer.writerow([
        "expected_transactions",
        EXPECTED_TX
    ])

    writer.writerow([
        "episodes_processed",
        results["episodes"]
    ])

    writer.writerow([
        "expected_episodes",
        EXPECTED_EPISODES
    ])

    writer.writerow([
        "unique_templates",
        len(template_reference)
    ])

    for name, value in mismatch_metrics.items():

        writer.writerow([
            name,
            value
        ])


# --------------------------------------------------------------------------------------------------
# FINAL GATE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)

if audit_json["morphology_pass"]:

    print(
        "✅✅✅ FULL-SCALE PATTERN MORPHOLOGY PASSED ✅✅✅"
    )

else:

    print(
        "❌ FULL-SCALE PATTERN MORPHOLOGY HAS MISMATCHES"
    )

print("=" * 100)

print(
    f"Audit JSON: {OUT_JSON}"
)

print(
    f"Audit CSV : {OUT_CSV}"
)

print("=" * 100)

CELL 23 — FULL-SCALE PATTERN MORPHOLOGY AUDIT

Streaming synthetic transactions...
  Rows processed: 1,000,000
  Rows processed: 2,000,000
  Rows processed: 3,000,000
  Rows processed: 4,000,000
  Rows processed: 5,000,000

1. BASIC MORPHOLOGY COUNTS
Transactions processed : 5,067,991
Expected transactions   : 5,067,991
Episodes processed      : 584,706
Expected episodes       : 584,706
Unique templates seen   : 370

2. TYPOLOGY MORPHOLOGY
BIPARTITE          | Episodes=  77,451 | Transactions=   415,357
CYCLE              | Episodes=  85,293 | Transactions=   453,261
FAN-IN             | Episodes=  63,218 | Transactions=   502,219
FAN-OUT            | Episodes=  75,862 | Transactions=   540,123
GATHER-SCATTER     | Episodes=  80,605 | Transactions= 1,130,782
RANDOM             | Episodes=  64,782 | Transactions=   301,647
SCATTER-GATHER     | Episodes=  69,540 | Transactions=   988,646
STACK              | Episodes=  67,955 | Transactions=   735,956

3. MORPHOLOGY MISMATCH SUMMARY
✅ PA

In [40]:
# ==================================================================================================
# CELL 24 — ORIGINAL LAUNDERING vs SYNTHETIC LAUNDERING AUDIT
#
# IMPORTANT:
#   Compare ONLY:
#       Original laundering  = 5,177
#       Synthetic laundering = 5,067,991
#
#   DO NOT compare legitimate transactions against synthetic laundering here.
#
# Checks:
#   1. Amount mean / median / quantiles
#   2. Payment Format
#   3. Payment Currency
#   4. Receiving Currency
#   5. Hour distribution
#   6. Day-of-week distribution
#   7. Source-account activity
#   8. Destination-account activity
#   9. Source-bank concentration
#  10. Destination-bank concentration
#  11. Bank/account diversity
#
# This cell DOES NOT modify or regenerate any data.
# ==================================================================================================

import os
import csv
import math
import random
from collections import Counter
from datetime import datetime


# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

BASE = "/kaggle/working/AML_GNN_ONLY/data"

ORIGINAL_FILE = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Trans.csv"
)

SYNTHETIC_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_transactions.csv"
)

OUT_JSON = os.path.join(
    BASE,
    "HI-Small_original_laundering_vs_synthetic_laundering.json"
)


# --------------------------------------------------------------------------------------------------
# EXPECTED
# --------------------------------------------------------------------------------------------------

EXPECTED_ORIGINAL_LAUNDERING = 5_177
EXPECTED_SYNTHETIC_LAUNDERING = 5_067_991


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def canonical_bank(x):

    s = str(x).strip()

    if not s:
        return None

    if s.isdigit():
        return str(int(s))

    return s.upper()


def pct(x, total):

    return (
        100.0 * x / total
        if total
        else 0.0
    )


def qvalue(values, q):

    if not values:
        return None

    values = sorted(values)

    idx = int(
        q * (len(values) - 1)
    )

    return values[idx]


def distribution_pct(counter, total):

    return {
        str(k): {
            "count": int(v),
            "pct": pct(v, total)
        }
        for k, v in counter.items()
    }


def new_stats():

    return {

        "rows": 0,

        "amounts": [],

        "payment_format": Counter(),

        "payment_currency": Counter(),

        "receiving_currency": Counter(),

        "hour": Counter(),

        "day_of_week": Counter(),

        "date": Counter(),

        "source_account": Counter(),

        "destination_account": Counter(),

        "source_bank": Counter(),

        "destination_bank": Counter(),
    }


# --------------------------------------------------------------------------------------------------
# STREAM ORIGINAL LAUNDERING
# --------------------------------------------------------------------------------------------------

def stream_original_laundering(path):

    stats = new_stats()

    with open(
        path,
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        reader = csv.reader(f)

        header = next(reader)

        expected_header = [
            "Timestamp",
            "From Bank",
            "Account",
            "To Bank",
            "Account",
            "Amount Received",
            "Receiving Currency",
            "Amount Paid",
            "Payment Currency",
            "Payment Format",
            "Is Laundering"
        ]

        if header != expected_header:

            raise RuntimeError(
                "Unexpected original HI-Small header:\n"
                + str(header)
            )


        for row in reader:

            if len(row) != 11:
                continue


            # ONLY LAUNDERING
            label = row[10].strip()

            if label not in {"1", "1.0"}:
                continue


            stats["rows"] += 1


            # ------------------------------------------------------------------
            # AMOUNT
            # ------------------------------------------------------------------

            try:

                amount = float(
                    row[5]
                )

                if (
                    math.isfinite(amount)
                    and amount > 0
                ):

                    stats["amounts"].append(
                        amount
                    )

            except Exception:
                pass


            # ------------------------------------------------------------------
            # CATEGORICAL
            # ------------------------------------------------------------------

            stats["payment_format"][
                row[9].strip()
            ] += 1

            stats["payment_currency"][
                row[8].strip()
            ] += 1

            stats["receiving_currency"][
                row[6].strip()
            ] += 1


            # ------------------------------------------------------------------
            # TIME
            # ------------------------------------------------------------------

            try:

                dt = datetime.strptime(
                    row[0].strip(),
                    "%Y-%m-%d %H:%M:%S"
                )

            except Exception:

                try:

                    dt = datetime.strptime(
                        row[0].strip(),
                        "%Y/%m/%d %H:%M"
                    )

                except Exception:

                    dt = None


            if dt is not None:

                stats["hour"][
                    dt.hour
                ] += 1

                stats["day_of_week"][
                    dt.strftime("%A")
                ] += 1

                stats["date"][
                    dt.strftime("%Y-%m-%d")
                ] += 1


            # ------------------------------------------------------------------
            # BANKS / ACCOUNTS
            # ------------------------------------------------------------------

            src_bank = canonical_bank(
                row[1]
            )

            dst_bank = canonical_bank(
                row[3]
            )

            src_account = row[2].strip()
            dst_account = row[4].strip()


            stats["source_bank"][
                src_bank
            ] += 1

            stats["destination_bank"][
                dst_bank
            ] += 1

            stats["source_account"][
                src_account
            ] += 1

            stats["destination_account"][
                dst_account
            ] += 1


    return stats


# --------------------------------------------------------------------------------------------------
# STREAM SYNTHETIC LAUNDERING
# --------------------------------------------------------------------------------------------------

def stream_synthetic_laundering(path):

    stats = new_stats()

    with open(
        path,
        "r",
        newline="",
        encoding="utf-8"
    ) as f:

        reader = csv.DictReader(f)


        for row in reader:

            # Synthetic file contains only laundering rows,
            # but explicitly verify it.
            label = row[
                "Is Laundering"
            ].strip()

            if label not in {
                "1",
                "1.0"
            }:

                continue


            stats["rows"] += 1


            # ------------------------------------------------------------------
            # AMOUNT
            # ------------------------------------------------------------------

            try:

                amount = float(
                    row[
                        "Amount Received"
                    ]
                )

                if (
                    math.isfinite(amount)
                    and amount > 0
                ):

                    stats["amounts"].append(
                        amount
                    )

            except Exception:
                pass


            # ------------------------------------------------------------------
            # CATEGORICAL
            # ------------------------------------------------------------------

            stats["payment_format"][
                row[
                    "Payment Format"
                ].strip()
            ] += 1

            stats["payment_currency"][
                row[
                    "Payment Currency"
                ].strip()
            ] += 1

            stats["receiving_currency"][
                row[
                    "Receiving Currency"
                ].strip()
            ] += 1


            # ------------------------------------------------------------------
            # TIME
            # ------------------------------------------------------------------

            try:

                dt = datetime.strptime(
                    row["Timestamp"].strip(),
                    "%Y-%m-%d %H:%M:%S"
                )

            except Exception:

                try:

                    dt = datetime.strptime(
                        row["Timestamp"].strip(),
                        "%Y/%m/%d %H:%M"
                    )

                except Exception:

                    dt = None


            if dt is not None:

                stats["hour"][
                    dt.hour
                ] += 1

                stats["day_of_week"][
                    dt.strftime("%A")
                ] += 1

                stats["date"][
                    dt.strftime("%Y-%m-%d")
                ] += 1


            # ------------------------------------------------------------------
            # BANKS / ACCOUNTS
            # ------------------------------------------------------------------

            src_bank = canonical_bank(
                row["From Bank"]
            )

            dst_bank = canonical_bank(
                row["To Bank"]
            )

            src_account = row[
                "Account"
            ].strip()

            dst_account = row[
                "Account.1"
            ].strip()


            stats["source_bank"][
                src_bank
            ] += 1

            stats["destination_bank"][
                dst_bank
            ] += 1

            stats["source_account"][
                src_account
            ] += 1

            stats["destination_account"][
                dst_account
            ] += 1


    return stats


# --------------------------------------------------------------------------------------------------
# START
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("CELL 24 — ORIGINAL LAUNDERING vs SYNTHETIC LAUNDERING")
print("=" * 100)


print("\nStreaming ORIGINAL LAUNDERING...")

original = stream_original_laundering(
    ORIGINAL_FILE
)

print(
    f"Original laundering rows: "
    f"{original['rows']:,}"
)


print("\nStreaming SYNTHETIC LAUNDERING...")

synthetic = stream_synthetic_laundering(
    SYNTHETIC_FILE
)

print(
    f"Synthetic laundering rows: "
    f"{synthetic['rows']:,}"
)


# --------------------------------------------------------------------------------------------------
# COUNT CHECK
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. COUNT CHECK")
print("=" * 80)

print(
    f"Original laundering : "
    f"{original['rows']:,}"
)

print(
    f"Expected             : "
    f"{EXPECTED_ORIGINAL_LAUNDERING:,}"
)

print(
    f"Synthetic laundering: "
    f"{synthetic['rows']:,}"
)

print(
    f"Expected             : "
    f"{EXPECTED_SYNTHETIC_LAUNDERING:,}"
)


count_gate = (
    original["rows"]
    == EXPECTED_ORIGINAL_LAUNDERING
    and
    synthetic["rows"]
    == EXPECTED_SYNTHETIC_LAUNDERING
)


print(
    "\n✅ PASS"
    if count_gate
    else "\n❌ FAIL"
)


# --------------------------------------------------------------------------------------------------
# AMOUNT STATISTICS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. AMOUNT DISTRIBUTION")
print("=" * 80)


amount_quantiles = [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
]


def amount_stats(values):

    if not values:
        return {}

    return {

        "mean":
            sum(values) / len(values),

        "median":
            qvalue(values, 0.50),

        "p01":
            qvalue(values, 0.01),

        "p05":
            qvalue(values, 0.05),

        "p10":
            qvalue(values, 0.10),

        "p25":
            qvalue(values, 0.25),

        "p75":
            qvalue(values, 0.75),

        "p90":
            qvalue(values, 0.90),

        "p95":
            qvalue(values, 0.95),

        "p99":
            qvalue(values, 0.99),

        "min":
            min(values),

        "max":
            max(values),
    }


orig_amount = amount_stats(
    original["amounts"]
)

syn_amount = amount_stats(
    synthetic["amounts"]
)


print(
    f"{'Metric':12s} "
    f"{'Original laundering':>24s} "
    f"{'Synthetic laundering':>24s} "
    f"{'Ratio':>12s}"
)


for key in [
    "mean",
    "median",
    "p01",
    "p05",
    "p10",
    "p25",
    "p75",
    "p90",
    "p95",
    "p99",
    "min",
    "max",
]:

    ov = orig_amount[key]
    sv = syn_amount[key]

    ratio = (
        sv / ov
        if ov != 0
        else None
    )

    print(
        f"{key:12s} "
        f"{ov:24.6f} "
        f"{sv:24.6f} "
        f"{ratio:12.4f}"
    )


# --------------------------------------------------------------------------------------------------
# PAYMENT FORMAT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. PAYMENT FORMAT")
print("=" * 80)


formats = sorted(
    set(original["payment_format"])
    | set(synthetic["payment_format"])
)


for fmt in formats:

    op = pct(
        original["payment_format"].get(
            fmt,
            0
        ),
        original["rows"]
    )

    sp = pct(
        synthetic["payment_format"].get(
            fmt,
            0
        ),
        synthetic["rows"]
    )

    print(
        f"{fmt:18s} "
        f"| Original={op:9.4f}% "
        f"| Synthetic={sp:9.4f}% "
        f"| AbsDiff={abs(op-sp):9.4f}"
    )


# --------------------------------------------------------------------------------------------------
# PAYMENT CURRENCY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. PAYMENT CURRENCY")
print("=" * 80)


currencies = sorted(
    set(original["payment_currency"])
    | set(synthetic["payment_currency"])
)


for curr in currencies:

    op = pct(
        original["payment_currency"].get(
            curr,
            0
        ),
        original["rows"]
    )

    sp = pct(
        synthetic["payment_currency"].get(
            curr,
            0
        ),
        synthetic["rows"]
    )

    print(
        f"{curr:20s} "
        f"| Original={op:9.4f}% "
        f"| Synthetic={sp:9.4f}% "
        f"| AbsDiff={abs(op-sp):9.4f}"
    )


# --------------------------------------------------------------------------------------------------
# RECEIVING CURRENCY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. RECEIVING CURRENCY")
print("=" * 80)


currencies = sorted(
    set(original["receiving_currency"])
    | set(synthetic["receiving_currency"])
)


for curr in currencies:

    op = pct(
        original["receiving_currency"].get(
            curr,
            0
        ),
        original["rows"]
    )

    sp = pct(
        synthetic["receiving_currency"].get(
            curr,
            0
        ),
        synthetic["rows"]
    )

    print(
        f"{curr:20s} "
        f"| Original={op:9.4f}% "
        f"| Synthetic={sp:9.4f}% "
        f"| AbsDiff={abs(op-sp):9.4f}"
    )


# --------------------------------------------------------------------------------------------------
# HOUR DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. HOUR DISTRIBUTION")
print("=" * 80)


hour_diffs = []


for h in range(24):

    op = pct(
        original["hour"].get(
            h,
            0
        ),
        original["rows"]
    )

    sp = pct(
        synthetic["hour"].get(
            h,
            0
        ),
        synthetic["rows"]
    )

    diff = abs(
        op - sp
    )

    hour_diffs.append(
        diff
    )

    print(
        f"{h:02d}:00 "
        f"| Original={op:9.4f}% "
        f"| Synthetic={sp:9.4f}% "
        f"| AbsDiff={diff:9.4f}"
    )


hour_mae = (
    sum(hour_diffs)
    / len(hour_diffs)
)


print(
    f"\nHour MAE: "
    f"{hour_mae:.6f} percentage points"
)


# --------------------------------------------------------------------------------------------------
# DAY-OF-WEEK
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. DAY-OF-WEEK DISTRIBUTION")
print("=" * 80)


days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]


dow_diffs = []


for day in days:

    op = pct(
        original["day_of_week"].get(
            day,
            0
        ),
        original["rows"]
    )

    sp = pct(
        synthetic["day_of_week"].get(
            day,
            0
        ),
        synthetic["rows"]
    )

    diff = abs(
        op - sp
    )

    dow_diffs.append(
        diff
    )

    print(
        f"{day:10s} "
        f"| Original={op:9.4f}% "
        f"| Synthetic={sp:9.4f}% "
        f"| AbsDiff={diff:9.4f}"
    )


dow_mae = (
    sum(dow_diffs)
    / len(dow_diffs)
)


print(
    f"\nDay-of-week MAE: "
    f"{dow_mae:.6f} percentage points"
)


# --------------------------------------------------------------------------------------------------
# BANK / ACCOUNT DIVERSITY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. BANK / ACCOUNT DIVERSITY")
print("=" * 80)


print(
    f"{'Metric':30s} "
    f"{'Original':>15s} "
    f"{'Synthetic':>15s}"
)


metrics = [

    (
        "Unique source banks",
        len(original["source_bank"]),
        len(synthetic["source_bank"])
    ),

    (
        "Unique destination banks",
        len(original["destination_bank"]),
        len(synthetic["destination_bank"])
    ),

    (
        "Unique source accounts",
        len(original["source_account"]),
        len(synthetic["source_account"])
    ),

    (
        "Unique destination accounts",
        len(original["destination_account"]),
        len(synthetic["destination_account"])
    ),
]


for name, ov, sv in metrics:

    print(
        f"{name:30s} "
        f"{ov:15,} "
        f"{sv:15,}"
    )


# --------------------------------------------------------------------------------------------------
# ACCOUNT REUSE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("9. ACCOUNT REUSE")
print("=" * 80)


def reuse_stats(counter):

    values = sorted(
        counter.values()
    )

    if not values:
        return {}

    n = len(values)

    def q(p):

        return values[
            int(
                p * (n - 1)
            )
        ]

    return {

        "min":
            values[0],

        "p25":
            q(0.25),

        "median":
            q(0.50),

        "p75":
            q(0.75),

        "p90":
            q(0.90),

        "p95":
            q(0.95),

        "p99":
            q(0.99),

        "max":
            values[-1],
    }


orig_src_reuse = reuse_stats(
    original["source_account"]
)

syn_src_reuse = reuse_stats(
    synthetic["source_account"]
)

orig_dst_reuse = reuse_stats(
    original["destination_account"]
)

syn_dst_reuse = reuse_stats(
    synthetic["destination_account"]
)


print("\nSOURCE ACCOUNT REUSE")

for key in [
    "min",
    "p25",
    "median",
    "p75",
    "p90",
    "p95",
    "p99",
    "max"
]:

    print(
        f"{key:8s} "
        f"| Original={orig_src_reuse[key]:12,} "
        f"| Synthetic={syn_src_reuse[key]:12,}"
    )


print("\nDESTINATION ACCOUNT REUSE")

for key in [
    "min",
    "p25",
    "median",
    "p75",
    "p90",
    "p95",
    "p99",
    "max"
]:

    print(
        f"{key:8s} "
        f"| Original={orig_dst_reuse[key]:12,} "
        f"| Synthetic={syn_dst_reuse[key]:12,}"
    )


# --------------------------------------------------------------------------------------------------
# BANK CONCENTRATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("10. BANK CONCENTRATION")
print("=" * 80)


def top_share(counter, total, k):

    return (
        100.0
        * sum(
            v
            for _, v
            in counter.most_common(k)
        )
        / total
        if total
        else 0.0
    )


print("\nSOURCE BANKS")

for k in [
    1,
    5,
    10,
    25,
    50,
]:

    op = top_share(
        original["source_bank"],
        original["rows"],
        k
    )

    sp = top_share(
        synthetic["source_bank"],
        synthetic["rows"],
        k
    )

    print(
        f"Top {k:2d} "
        f"| Original={op:9.4f}% "
        f"| Synthetic={sp:9.4f}% "
        f"| AbsDiff={abs(op-sp):9.4f}"
    )


print("\nDESTINATION BANKS")

for k in [
    1,
    5,
    10,
    25,
    50,
]:

    op = top_share(
        original["destination_bank"],
        original["rows"],
        k
    )

    sp = top_share(
        synthetic["destination_bank"],
        synthetic["rows"],
        k
    )

    print(
        f"Top {k:2d} "
        f"| Original={op:9.4f}% "
        f"| Synthetic={sp:9.4f}% "
        f"| AbsDiff={abs(op-sp):9.4f}"
    )


# --------------------------------------------------------------------------------------------------
# SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("11. SUMMARY")
print("=" * 80)


print(
    f"Original laundering rows : "
    f"{original['rows']:,}"
)

print(
    f"Synthetic laundering rows: "
    f"{synthetic['rows']:,}"
)

print(
    f"Original median amount   : "
    f"{orig_amount['median']:,.6f}"
)

print(
    f"Synthetic median amount  : "
    f"{syn_amount['median']:,.6f}"
)

print(
    f"Original mean amount     : "
    f"{orig_amount['mean']:,.6f}"
)

print(
    f"Synthetic mean amount    : "
    f"{syn_amount['mean']:,.6f}"
)

print(
    f"Hour MAE                 : "
    f"{hour_mae:.6f} pp"
)

print(
    f"Day-of-week MAE          : "
    f"{dow_mae:.6f} pp"
)


# --------------------------------------------------------------------------------------------------
# SAVE JSON
# --------------------------------------------------------------------------------------------------

audit = {

    "comparison":
        "Original laundering vs Synthetic laundering",

    "rows": {

        "original_laundering":
            original["rows"],

        "synthetic_laundering":
            synthetic["rows"],
    },

    "amount": {

        "original":
            orig_amount,

        "synthetic":
            syn_amount,
    },

    "payment_format": {

        "original":
            distribution_pct(
                original["payment_format"],
                original["rows"]
            ),

        "synthetic":
            distribution_pct(
                synthetic["payment_format"],
                synthetic["rows"]
            ),
    },

    "payment_currency": {

        "original":
            distribution_pct(
                original["payment_currency"],
                original["rows"]
            ),

        "synthetic":
            distribution_pct(
                synthetic["payment_currency"],
                synthetic["rows"]
            ),
    },

    "receiving_currency": {

        "original":
            distribution_pct(
                original["receiving_currency"],
                original["rows"]
            ),

        "synthetic":
            distribution_pct(
                synthetic["receiving_currency"],
                synthetic["rows"]
            ),
    },

    "hour_mae_percentage_points":
        hour_mae,

    "day_of_week_mae_percentage_points":
        dow_mae,

    "original_unique_source_banks":
        len(original["source_bank"]),

    "synthetic_unique_source_banks":
        len(synthetic["source_bank"]),

    "original_unique_destination_banks":
        len(original["destination_bank"]),

    "synthetic_unique_destination_banks":
        len(synthetic["destination_bank"]),

    "original_unique_source_accounts":
        len(original["source_account"]),

    "synthetic_unique_source_accounts":
        len(synthetic["source_account"]),

    "original_unique_destination_accounts":
        len(original["destination_account"]),

    "synthetic_unique_destination_accounts":
        len(synthetic["destination_account"]),
}


with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    import json

    json.dump(
        audit,
        f,
        indent=2
    )


# --------------------------------------------------------------------------------------------------
# FINAL
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("✅ CELL 24 COMPLETE")
print("=" * 100)

print(
    f"Audit saved to:\n{OUT_JSON}"
)

print("=" * 100)

CELL 24 — ORIGINAL LAUNDERING vs SYNTHETIC LAUNDERING

Streaming ORIGINAL LAUNDERING...
Original laundering rows: 5,177

Streaming SYNTHETIC LAUNDERING...
Synthetic laundering rows: 5,067,991

1. COUNT CHECK
Original laundering : 5,177
Expected             : 5,177
Synthetic laundering: 5,067,991
Expected             : 5,067,991

✅ PASS

2. AMOUNT DISTRIBUTION
Metric            Original laundering     Synthetic laundering        Ratio
mean                  36135310.412813          21383007.836351       0.5917
median                    8667.210000              1465.920000       0.1691
p01                          0.995281                 0.014792       0.0149
p05                        213.850000                 3.413636       0.0160
p10                        672.380000                17.142333       0.0255
p25                       2634.970000               150.797968       0.0572
p75                      18832.270000             17584.697184       0.9338
p90                     170985

In [41]:
# ==================================================================================================
# CELL 25 — TYPOLOGY-WISE LAUNDERING DNA AUDIT
#
# Compare:
#   ORIGINAL IBM PATTERN TRANSACTIONS
#          vs
#   SYNTHETIC TRANSACTIONS
#
# Broken down by:
#   BIPARTITE
#   CYCLE
#   FAN-IN
#   FAN-OUT
#   GATHER-SCATTER
#   RANDOM
#   SCATTER-GATHER
#   STACK
#
# This tells us exactly how the generator differs from the ORIGINAL laundering patterns.
#
# DOES NOT modify or regenerate data.
# ==================================================================================================

import os
import csv
import json
import math
from collections import Counter, defaultdict
from datetime import datetime


# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

BASE = "/kaggle/working/AML_GNN_ONLY/data"

ORIGINAL_PATTERNS = (
    "/kaggle/input/datasets/"
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Patterns.txt"
)

SYNTHETIC_FILE = os.path.join(
    BASE,
    "HI-Small_SYNTH_FULL_transactions.csv"
)

OUT_JSON = os.path.join(
    BASE,
    "HI-Small_typology_wise_laundering_dna_audit.json"
)


TYPOLOGIES = [
    "BIPARTITE",
    "CYCLE",
    "FAN-IN",
    "FAN-OUT",
    "GATHER-SCATTER",
    "RANDOM",
    "SCATTER-GATHER",
    "STACK",
]


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def pct(x, total):

    return (
        100.0 * x / total
        if total
        else 0.0
    )


def parse_timestamp(x):

    x = x.strip()

    try:

        return datetime.strptime(
            x,
            "%Y/%m/%d %H:%M"
        )

    except Exception:

        return datetime.strptime(
            x,
            "%Y-%m-%d %H:%M:%S"
        )


def quantile(values, q):

    if not values:
        return None

    values = sorted(values)

    idx = int(
        q * (len(values) - 1)
    )

    return values[idx]


def new_typology_stats():

    return {
        "rows": 0,
        "amounts": [],
        "payment_format": Counter(),
        "payment_currency": Counter(),
        "receiving_currency": Counter(),
        "hour": Counter(),
        "day_of_week": Counter(),
        "source_banks": Counter(),
        "destination_banks": Counter(),
        "source_accounts": Counter(),
        "destination_accounts": Counter(),
    }


# --------------------------------------------------------------------------------------------------
# ORIGINAL IBM PATTERNS
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("CELL 25 — TYPOLOGY-WISE LAUNDERING DNA AUDIT")
print("=" * 100)

print("\n1. Reading ORIGINAL IBM Patterns.txt ...")


original = {
    typ: new_typology_stats()
    for typ in TYPOLOGIES
}


current_typology = None


with open(
    ORIGINAL_PATTERNS,
    "r",
    encoding="utf-8"
) as f:

    for raw in f:

        line = raw.strip()

        if not line:
            continue


        if line.startswith(
            "BEGIN LAUNDERING ATTEMPT"
        ):

            parts = line.split("-")

            # Robust extraction from:
            # BEGIN LAUNDERING ATTEMPT - FAN-OUT: ...
            typology_part = line.split(
                " - ",
                1
            )[1]

            current_typology = (
                typology_part
                .split(":")[0]
                .strip()
            )

            if current_typology not in original:

                current_typology = None

            continue


        if line.startswith(
            "END LAUNDERING ATTEMPT"
        ):

            current_typology = None
            continue


        # Ignore ID lines
        if line.startswith(
            'id="'
        ):

            continue


        if current_typology is None:
            continue


        parts = line.split(",")

        if len(parts) != 11:
            continue


        stats = original[
            current_typology
        ]


        stats["rows"] += 1


        # ----------------------------------------------------------------------
        # amount
        # ----------------------------------------------------------------------

        try:

            amount = float(
                parts[5]
            )

            if (
                math.isfinite(amount)
                and amount > 0
            ):

                stats[
                    "amounts"
                ].append(amount)

        except Exception:
            pass


        # ----------------------------------------------------------------------
        # categorical
        # ----------------------------------------------------------------------

        stats["payment_format"][
            parts[9].strip()
        ] += 1

        stats["payment_currency"][
            parts[8].strip()
        ] += 1

        stats["receiving_currency"][
            parts[6].strip()
        ] += 1


        # ----------------------------------------------------------------------
        # timestamp
        # ----------------------------------------------------------------------

        try:

            dt = parse_timestamp(
                parts[0]
            )

            stats["hour"][
                dt.hour
            ] += 1

            stats["day_of_week"][
                dt.strftime("%A")
            ] += 1

        except Exception:
            pass


        # ----------------------------------------------------------------------
        # bank/account
        # ----------------------------------------------------------------------

        src_bank = parts[1].strip()
        dst_bank = parts[3].strip()

        src_account = parts[2].strip()
        dst_account = parts[4].strip()


        stats["source_banks"][
            src_bank
        ] += 1

        stats["destination_banks"][
            dst_bank
        ] += 1

        stats["source_accounts"][
            src_account
        ] += 1

        stats["destination_accounts"][
            dst_account
        ] += 1


print("\nOriginal IBM pattern transaction counts:")

for typ in TYPOLOGIES:

    print(
        f"  {typ:18s} "
        f"{original[typ]['rows']:8,}"
    )


# --------------------------------------------------------------------------------------------------
# SYNTHETIC
# --------------------------------------------------------------------------------------------------

print("\n2. Streaming SYNTHETIC transactions ...")


synthetic = {
    typ: new_typology_stats()
    for typ in TYPOLOGIES
}


with open(
    SYNTHETIC_FILE,
    "r",
    newline="",
    encoding="utf-8"
) as f:

    reader = csv.DictReader(f)


    for row in reader:

        typology = row[
            "typology"
        ].strip().upper()


        if typology not in synthetic:
            continue


        stats = synthetic[
            typology
        ]


        stats["rows"] += 1


        # ----------------------------------------------------------------------
        # amount
        # ----------------------------------------------------------------------

        try:

            amount = float(
                row["Amount Received"]
            )

            if (
                math.isfinite(amount)
                and amount > 0
            ):

                stats[
                    "amounts"
                ].append(amount)

        except Exception:
            pass


        # ----------------------------------------------------------------------
        # categorical
        # ----------------------------------------------------------------------

        stats["payment_format"][
            row[
                "Payment Format"
            ].strip()
        ] += 1

        stats["payment_currency"][
            row[
                "Payment Currency"
            ].strip()
        ] += 1

        stats["receiving_currency"][
            row[
                "Receiving Currency"
            ].strip()
        ] += 1


        # ----------------------------------------------------------------------
        # timestamp
        # ----------------------------------------------------------------------

        try:

            dt = parse_timestamp(
                row["Timestamp"]
            )

            stats["hour"][
                dt.hour
            ] += 1

            stats["day_of_week"][
                dt.strftime("%A")
            ] += 1

        except Exception:
            pass


        # ----------------------------------------------------------------------
        # banks/accounts
        # ----------------------------------------------------------------------

        src_bank = row[
            "From Bank"
        ].strip()

        dst_bank = row[
            "To Bank"
        ].strip()

        src_account = row[
            "Account"
        ].strip()

        dst_account = row[
            "Account.1"
        ].strip()


        stats["source_banks"][
            src_bank
        ] += 1

        stats["destination_banks"][
            dst_bank
        ] += 1

        stats["source_accounts"][
            src_account
        ] += 1

        stats["destination_accounts"][
            dst_account
        ] += 1


# --------------------------------------------------------------------------------------------------
# BASIC COUNTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. TYPOLOGY COUNTS")
print("=" * 80)

for typ in TYPOLOGIES:

    print(
        f"{typ:18s} "
        f"| Original={original[typ]['rows']:9,} "
        f"| Synthetic={synthetic[typ]['rows']:10,}"
    )


# --------------------------------------------------------------------------------------------------
# AMOUNT COMPARISON
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. AMOUNT DISTRIBUTION BY TYPOLOGY")
print("=" * 80)


amount_quantiles = [
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
]


for typ in TYPOLOGIES:

    oa = original[typ]["amounts"]
    sa = synthetic[typ]["amounts"]


    if not oa or not sa:

        continue


    print(
        "\n"
        + typ
    )

    print(
        f"  {'Metric':10s} "
        f"{'Original':>18s} "
        f"{'Synthetic':>18s} "
        f"{'Ratio':>12s}"
    )


    for q in amount_quantiles:

        ov = quantile(
            oa,
            q
        )

        sv = quantile(
            sa,
            q
        )

        ratio = (
            sv / ov
            if ov != 0
            else None
        )


        print(
            f"  p{int(q*100):02d} "
            f"{ov:18.4f} "
            f"{sv:18.4f} "
            f"{ratio:12.4f}"
        )


# --------------------------------------------------------------------------------------------------
# PAYMENT FORMAT BY TYPOLOGY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. PAYMENT FORMAT BY TYPOLOGY")
print("=" * 80)


for typ in TYPOLOGIES:

    print(
        f"\n[{typ}]"
    )


    all_formats = sorted(
        set(
            original[typ]["payment_format"]
        )
        |
        set(
            synthetic[typ]["payment_format"]
        )
    )


    for fmt in all_formats:

        op = pct(
            original[typ]["payment_format"].get(
                fmt,
                0
            ),
            original[typ]["rows"]
        )

        sp = pct(
            synthetic[typ]["payment_format"].get(
                fmt,
                0
            ),
            synthetic[typ]["rows"]
        )


        print(
            f"  {fmt:18s} "
            f"| Original={op:9.3f}% "
            f"| Synthetic={sp:9.3f}% "
            f"| AbsDiff={abs(op-sp):9.3f}"
        )


# --------------------------------------------------------------------------------------------------
# PAYMENT CURRENCY BY TYPOLOGY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. PAYMENT CURRENCY BY TYPOLOGY")
print("=" * 80)


for typ in TYPOLOGIES:

    print(
        f"\n[{typ}]"
    )


    all_curr = sorted(
        set(
            original[typ]["payment_currency"]
        )
        |
        set(
            synthetic[typ]["payment_currency"]
        )
    )


    for curr in all_curr:

        op = pct(
            original[typ]["payment_currency"].get(
                curr,
                0
            ),
            original[typ]["rows"]
        )

        sp = pct(
            synthetic[typ]["payment_currency"].get(
                curr,
                0
            ),
            synthetic[typ]["rows"]
        )


        print(
            f"  {curr:20s} "
            f"| Original={op:9.3f}% "
            f"| Synthetic={sp:9.3f}% "
            f"| AbsDiff={abs(op-sp):9.3f}"
        )


# --------------------------------------------------------------------------------------------------
# RECEIVING CURRENCY BY TYPOLOGY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. RECEIVING CURRENCY BY TYPOLOGY")
print("=" * 80)


for typ in TYPOLOGIES:

    print(
        f"\n[{typ}]"
    )


    all_curr = sorted(
        set(
            original[typ]["receiving_currency"]
        )
        |
        set(
            synthetic[typ]["receiving_currency"]
        )
    )


    for curr in all_curr:

        op = pct(
            original[typ]["receiving_currency"].get(
                curr,
                0
            ),
            original[typ]["rows"]
        )

        sp = pct(
            synthetic[typ]["receiving_currency"].get(
                curr,
                0
            ),
            synthetic[typ]["rows"]
        )


        print(
            f"  {curr:20s} "
            f"| Original={op:9.3f}% "
            f"| Synthetic={sp:9.3f}% "
            f"| AbsDiff={abs(op-sp):9.3f}"
        )


# --------------------------------------------------------------------------------------------------
# BANK / ACCOUNT DIVERSITY BY TYPOLOGY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. BANK / ACCOUNT DIVERSITY BY TYPOLOGY")
print("=" * 80)


for typ in TYPOLOGIES:

    print(
        f"\n[{typ}]"
    )

    print(
        f"  Source banks      : "
        f"{len(original[typ]['source_banks']):8,} "
        f"-> "
        f"{len(synthetic[typ]['source_banks']):8,}"
    )

    print(
        f"  Destination banks : "
        f"{len(original[typ]['destination_banks']):8,} "
        f"-> "
        f"{len(synthetic[typ]['destination_banks']):8,}"
    )

    print(
        f"  Source accounts   : "
        f"{len(original[typ]['source_accounts']):8,} "
        f"-> "
        f"{len(synthetic[typ]['source_accounts']):8,}"
    )

    print(
        f"  Destination accounts:"
        f"{len(original[typ]['destination_accounts']):8,} "
        f"-> "
        f"{len(synthetic[typ]['destination_accounts']):8,}"
    )


# --------------------------------------------------------------------------------------------------
# DAY / HOUR BY TYPOLOGY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("9. TEMPORAL DNA BY TYPOLOGY")
print("=" * 80)


for typ in TYPOLOGIES:

    print(
        f"\n[{typ}]"
    )


    print(
        "  Hours:"
    )


    for h in range(24):

        ov = pct(
            original[typ]["hour"].get(
                h,
                0
            ),
            original[typ]["rows"]
        )

        sv = pct(
            synthetic[typ]["hour"].get(
                h,
                0
            ),
            synthetic[typ]["rows"]
        )


        if ov > 0 or sv > 0:

            print(
                f"    {h:02d}: "
                f"O={ov:7.3f}% "
                f"S={sv:7.3f}%"
            )


# --------------------------------------------------------------------------------------------------
# SAVE COMPACT JSON
# --------------------------------------------------------------------------------------------------

def compact_stats(stats):

    return {

        "rows":
            stats["rows"],

        "amount": {

            "mean":
                (
                    sum(stats["amounts"])
                    / len(stats["amounts"])
                    if stats["amounts"]
                    else None
                ),

            "median":
                quantile(
                    stats["amounts"],
                    0.50
                ),

            "p90":
                quantile(
                    stats["amounts"],
                    0.90
                ),

            "p95":
                quantile(
                    stats["amounts"],
                    0.95
                ),

            "p99":
                quantile(
                    stats["amounts"],
                    0.99
                ),
        },

        "payment_format":
            dict(
                stats["payment_format"]
            ),

        "payment_currency":
            dict(
                stats["payment_currency"]
            ),

        "receiving_currency":
            dict(
                stats["receiving_currency"]
            ),

        "unique_source_banks":
            len(stats["source_banks"]),

        "unique_destination_banks":
            len(stats["destination_banks"]),

        "unique_source_accounts":
            len(stats["source_accounts"]),

        "unique_destination_accounts":
            len(stats["destination_accounts"]),
    }


audit = {

    "original": {
        typ:
            compact_stats(
                original[typ]
            )
        for typ in TYPOLOGIES
    },

    "synthetic": {
        typ:
            compact_stats(
                synthetic[typ]
            )
        for typ in TYPOLOGIES
    },
}


with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        indent=2
    )


# --------------------------------------------------------------------------------------------------
# FINAL
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("✅ CELL 25 COMPLETE")
print("=" * 100)

print(
    f"Saved audit:\n{OUT_JSON}"
)

print("=" * 100)

CELL 25 — TYPOLOGY-WISE LAUNDERING DNA AUDIT

1. Reading ORIGINAL IBM Patterns.txt ...

Original IBM pattern transaction counts:
  BIPARTITE               263
  CYCLE                   287
  FAN-IN                  318
  FAN-OUT                 342
  GATHER-SCATTER          716
  RANDOM                  191
  SCATTER-GATHER          626
  STACK                   466

2. Streaming SYNTHETIC transactions ...

3. TYPOLOGY COUNTS
BIPARTITE          | Original=      263 | Synthetic=   415,357
CYCLE              | Original=      287 | Synthetic=   453,261
FAN-IN             | Original=      318 | Synthetic=   502,219
FAN-OUT            | Original=      342 | Synthetic=   540,123
GATHER-SCATTER     | Original=      716 | Synthetic= 1,130,782
RANDOM             | Original=      191 | Synthetic=   301,647
SCATTER-GATHER     | Original=      626 | Synthetic=   988,646
STACK              | Original=      466 | Synthetic=   735,956

4. AMOUNT DISTRIBUTION BY TYPOLOGY

BIPARTITE
  Metric           

In [43]:
# ============================================================
# CELL 26A — LOCATE FINAL ORIGINAL + FINAL BALANCED FILE
# ============================================================

import os

# ------------------------------------------------------------
# SEARCH ALL CSV FILES
# ------------------------------------------------------------

SEARCH_DIRS = [
    "/kaggle/working",
    "/kaggle/input"
]

matches = []

for base_dir in SEARCH_DIRS:
    for root, dirs, files in os.walk(base_dir):
        for fname in files:
            if fname.lower().endswith(".csv"):
                path = os.path.join(root, fname)

                if (
                    "hi-small" in fname.lower()
                    or "final" in fname.lower()
                    or "transactions" in fname.lower()
                ):
                    matches.append(path)

print("=" * 80)
print("POSSIBLE HI-SMALL / FINAL CSV FILES")
print("=" * 80)

for p in sorted(matches):
    size_mb = os.path.getsize(p) / (1024 ** 2)
    print(f"{size_mb:10.2f} MB | {p}")

# ------------------------------------------------------------
# IDENTIFY FILES
# ------------------------------------------------------------

ORIG_PATH = None
FINAL_TX_PATH = None

for p in matches:

    fname = os.path.basename(p).lower()

    # Original IBM HI-Small transactions
    if fname == "hi-small_trans.csv":
        ORIG_PATH = p

    # Our final 50:50 merged transactions
    if (
        "hi-small_final_transactions_10146336" in fname
        or (
            "hi-small_final" in fname
            and "transactions" in fname
        )
    ):
        FINAL_TX_PATH = p

print("\n" + "=" * 80)
print("SELECTED FILES")
print("=" * 80)

print("ORIGINAL:")
print(ORIG_PATH)

print("\nFINAL BALANCED:")
print(FINAL_TX_PATH)

# ------------------------------------------------------------
# VALIDATE PATHS
# ------------------------------------------------------------

if ORIG_PATH is None:
    raise FileNotFoundError(
        "Original HI-Small_Trans.csv was not found."
    )

if FINAL_TX_PATH is None:
    raise FileNotFoundError(
        "HI-Small_FINAL_transactions_10146336.csv was not found. "
        "Check whether the final merged file is attached to this Kaggle session."
    )

# ------------------------------------------------------------
# BASIC FILE INFO
# ------------------------------------------------------------

orig_size = os.path.getsize(ORIG_PATH) / (1024 ** 2)
final_size = os.path.getsize(FINAL_TX_PATH) / (1024 ** 2)

print("\n" + "=" * 80)
print("FILE VALIDATION")
print("=" * 80)

print(f"Original file size : {orig_size:,.2f} MB")
print(f"Final file size    : {final_size:,.2f} MB")

print("\n✅ Original file located")
print("✅ Final balanced transaction file located")
print("\nNow we can run the sample-size-adjusted amount audit.")

POSSIBLE HI-SMALL / FINAL CSV FILES
    453.63 MB | /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv
     32.48 MB | /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_accounts.csv
     60.82 MB | /kaggle/working/AML_GNN_ONLY/data/HI-Small_FINAL_accounts_1004953.csv
   1409.24 MB | /kaggle/working/AML_GNN_ONLY/data/HI-Small_FINAL_transactions_10146336.csv
     43.50 MB | /kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_FULL_episode_audit.csv
     27.85 MB | /kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_FULL_new_accounts.csv
    844.22 MB | /kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_FULL_transactions.csv
      0.33 MB | /kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_PILOT_V2_episode_audit.csv
      0.25 MB | /kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_PILOT_V2_new_accounts.csv
      7.25 MB | /kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_PILOT_V2_transactions.csv
      0.23 MB 

In [44]:
# ============================================================
# CELL 26B — SAMPLE-SIZE-ADJUSTED LAUNDERING AMOUNT AUDIT
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.stats import ks_2samp, wasserstein_distance

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

ORIG_PATH = (
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Trans.csv"
)

SYN_PATH = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_SYNTH_FULL_transactions.csv"
)

print("Original :", ORIG_PATH)
print("Synthetic:", SYN_PATH)

assert os.path.exists(ORIG_PATH), "Original file not found"
assert os.path.exists(SYN_PATH), "Synthetic file not found"


# ------------------------------------------------------------
# 1. LOAD ORIGINAL LAUNDERING AMOUNTS ONLY
# ------------------------------------------------------------

print("\nLoading original laundering amounts...")

orig = pd.read_csv(
    ORIG_PATH,
    usecols=[5, 10],
    header=0
)

orig.columns = ["Amount", "Is_Laundering"]

orig["Amount"] = pd.to_numeric(
    orig["Amount"],
    errors="coerce"
)

orig["Is_Laundering"] = pd.to_numeric(
    orig["Is_Laundering"],
    errors="coerce"
)

orig_laundering = (
    orig.loc[
        orig["Is_Laundering"] == 1,
        "Amount"
    ]
    .dropna()
    .to_numpy(dtype=np.float64)
)

orig_laundering = orig_laundering[
    orig_laundering > 0
]

del orig

print(
    f"Original laundering transactions: "
    f"{len(orig_laundering):,}"
)


# ------------------------------------------------------------
# 2. LOAD SYNTHETIC LAUNDERING AMOUNTS
# ------------------------------------------------------------

print("\nLoading synthetic laundering amounts...")

# Synthetic file is large, so read only Amount column.
syn_chunks = []

for chunk in pd.read_csv(
    SYN_PATH,
    usecols=[5],
    header=0,
    chunksize=500_000
):
    chunk.columns = ["Amount"]

    vals = pd.to_numeric(
        chunk["Amount"],
        errors="coerce"
    ).dropna().to_numpy(dtype=np.float64)

    vals = vals[vals > 0]

    syn_chunks.append(vals)

    print(
        f"  Loaded {sum(len(x) for x in syn_chunks):,} "
        f"synthetic amounts",
        end="\r"
    )

synthetic_laundering = np.concatenate(
    syn_chunks
)

del syn_chunks

print(
    f"\nSynthetic laundering transactions: "
    f"{len(synthetic_laundering):,}"
)


# ------------------------------------------------------------
# 3. SANITY CHECK
# ------------------------------------------------------------

EXPECTED_ORIGINAL = 5_177
EXPECTED_SYNTHETIC = 5_067_991

if len(orig_laundering) != EXPECTED_ORIGINAL:
    print(
        f"⚠️ Original count differs from expected "
        f"{EXPECTED_ORIGINAL:,}"
    )
else:
    print("✅ Original laundering count exact")

if len(synthetic_laundering) != EXPECTED_SYNTHETIC:
    print(
        f"⚠️ Synthetic count differs from expected "
        f"{EXPECTED_SYNTHETIC:,}"
    )
else:
    print("✅ Synthetic laundering count exact")


# ------------------------------------------------------------
# 4. RAW-SCALE QUANTILES
# ------------------------------------------------------------

q_names = [
    "P01", "P05", "P10", "P25",
    "P50", "P75", "P90", "P95", "P99"
]

q_values = [1, 5, 10, 25, 50, 75, 90, 95, 99]

orig_q = np.percentile(
    orig_laundering,
    q_values
)

syn_q = np.percentile(
    synthetic_laundering,
    q_values
)

raw_rows = []

for name, oq, sq in zip(q_names, orig_q, syn_q):

    raw_rows.append({
        "Quantile": name,
        "Original": oq,
        "Synthetic": sq,
        "Synthetic_to_Original_Ratio":
            sq / oq if oq != 0 else np.nan
    })

raw_qdf = pd.DataFrame(raw_rows)

print("\n" + "=" * 90)
print("RAW AMOUNT DISTRIBUTION")
print("=" * 90)

print(
    raw_qdf.to_string(
        index=False,
        formatters={
            "Original": "{:,.4f}".format,
            "Synthetic": "{:,.4f}".format,
            "Synthetic_to_Original_Ratio":
                "{:.4f}".format
        }
    )
)


# ------------------------------------------------------------
# 5. SUMMARY STATISTICS
# ------------------------------------------------------------

summary_stats = pd.DataFrame({
    "Metric": [
        "Count",
        "Mean",
        "Median",
        "Std",
        "Minimum",
        "Maximum"
    ],
    "Original": [
        len(orig_laundering),
        np.mean(orig_laundering),
        np.median(orig_laundering),
        np.std(orig_laundering),
        np.min(orig_laundering),
        np.max(orig_laundering)
    ],
    "Synthetic": [
        len(synthetic_laundering),
        np.mean(synthetic_laundering),
        np.median(synthetic_laundering),
        np.std(synthetic_laundering),
        np.min(synthetic_laundering),
        np.max(synthetic_laundering)
    ]
})

print("\n" + "=" * 90)
print("SUMMARY STATISTICS")
print("=" * 90)

print(
    summary_stats.to_string(
        index=False,
        formatters={
            "Original": "{:,.6f}".format,
            "Synthetic": "{:,.6f}".format
        }
    )
)


# ------------------------------------------------------------
# 6. LOG-SPACE DISTRIBUTIONS
# ------------------------------------------------------------

orig_log = np.log10(orig_laundering)
syn_log = np.log10(synthetic_laundering)

full_ks = ks_2samp(
    orig_log,
    syn_log,
    alternative="two-sided"
).statistic

full_wd = wasserstein_distance(
    orig_log,
    syn_log
)

print("\n" + "=" * 90)
print("FULL DISTRIBUTION DISTANCE")
print("=" * 90)

print(f"KS statistic                     : {full_ks:.6f}")
print(f"Wasserstein distance (log10)     : {full_wd:.6f}")


# ------------------------------------------------------------
# 7. SAME-SIZE BOOTSTRAP
# ------------------------------------------------------------

N = len(orig_log)
N_BOOT = 200

rng = np.random.default_rng(42)

ks_orig_syn = []
wd_orig_syn = []

ks_syn_syn = []
wd_syn_syn = []

bootstrap_quantiles = {
    q: []
    for q in q_names
}

print("\nRunning same-size bootstrap...")

for b in range(N_BOOT):

    # Same number of synthetic transactions as
    # the original laundering population.
    syn_idx_1 = rng.choice(
        len(synthetic_laundering),
        size=N,
        replace=False
    )

    syn_sample_1 = synthetic_laundering[
        syn_idx_1
    ]

    syn_log_1 = np.log10(syn_sample_1)

    # Independent synthetic sample
    syn_idx_2 = rng.choice(
        len(synthetic_laundering),
        size=N,
        replace=False
    )

    syn_sample_2 = synthetic_laundering[
        syn_idx_2
    ]

    syn_log_2 = np.log10(syn_sample_2)

    # --------------------------------------------
    # Original vs Synthetic
    # --------------------------------------------

    ks_orig_syn.append(
        ks_2samp(
            orig_log,
            syn_log_1
        ).statistic
    )

    wd_orig_syn.append(
        wasserstein_distance(
            orig_log,
            syn_log_1
        )
    )

    # --------------------------------------------
    # Synthetic vs Synthetic
    # --------------------------------------------

    ks_syn_syn.append(
        ks_2samp(
            syn_log_1,
            syn_log_2
        ).statistic
    )

    wd_syn_syn.append(
        wasserstein_distance(
            syn_log_1,
            syn_log_2
        )
    )

    # --------------------------------------------
    # Synthetic quantiles
    # --------------------------------------------

    sample_q = np.percentile(
        syn_sample_1,
        q_values
    )

    for name, value in zip(
        q_names,
        sample_q
    ):
        bootstrap_quantiles[name].append(value)

    if (b + 1) % 25 == 0:
        print(
            f"  Bootstrap {b + 1}/{N_BOOT}"
        )


# ------------------------------------------------------------
# 8. DISTANCE COMPARISON
# ------------------------------------------------------------

ks_orig_median = np.median(ks_orig_syn)
ks_orig_low = np.percentile(
    ks_orig_syn,
    2.5
)
ks_orig_high = np.percentile(
    ks_orig_syn,
    97.5
)

wd_orig_median = np.median(wd_orig_syn)
wd_orig_low = np.percentile(
    wd_orig_syn,
    2.5
)
wd_orig_high = np.percentile(
    wd_orig_syn,
    97.5
)

ks_syn_median = np.median(ks_syn_syn)
ks_syn_p95 = np.percentile(
    ks_syn_syn,
    95
)

wd_syn_median = np.median(wd_syn_syn)
wd_syn_p95 = np.percentile(
    wd_syn_syn,
    95
)

print("\n" + "=" * 90)
print("SAME-SIZE SAMPLE EFFECT")
print("=" * 90)

print(
    f"Synthetic vs Synthetic KS median : "
    f"{ks_syn_median:.6f}"
)

print(
    f"Synthetic vs Synthetic KS P95    : "
    f"{ks_syn_p95:.6f}"
)

print(
    f"Original vs Synthetic KS median  : "
    f"{ks_orig_median:.6f}"
)

print(
    f"Original vs Synthetic KS 95% CI  : "
    f"[{ks_orig_low:.6f}, {ks_orig_high:.6f}]"
)

print()

print(
    f"Synthetic vs Synthetic WD median : "
    f"{wd_syn_median:.6f}"
)

print(
    f"Synthetic vs Synthetic WD P95    : "
    f"{wd_syn_p95:.6f}"
)

print(
    f"Original vs Synthetic WD median  : "
    f"{wd_orig_median:.6f}"
)

print(
    f"Original vs Synthetic WD 95% CI  : "
    f"[{wd_orig_low:.6f}, {wd_orig_high:.6f}]"
)


# ------------------------------------------------------------
# 9. SAME-SIZE QUANTILE INTERVALS
# ------------------------------------------------------------

rows = []

for name, oq in zip(
    q_names,
    orig_q
):

    vals = np.array(
        bootstrap_quantiles[name]
    )

    median_val = np.median(vals)
    low_val = np.percentile(
        vals,
        2.5
    )
    high_val = np.percentile(
        vals,
        97.5
    )

    rows.append({
        "Quantile": name,
        "Original": oq,
        "Synthetic_Median": median_val,
        "Synthetic_2.5pct": low_val,
        "Synthetic_97.5pct": high_val,
        "Synthetic_to_Original":
            median_val / oq
            if oq != 0 else np.nan,
        "Original_inside_95pct":
            low_val <= oq <= high_val
    })

boot_qdf = pd.DataFrame(rows)

print("\n" + "=" * 90)
print("SAME-SIZE QUANTILE BOOTSTRAP")
print("=" * 90)

print(
    boot_qdf.to_string(
        index=False,
        formatters={
            "Original": "{:,.4f}".format,
            "Synthetic_Median": "{:,.4f}".format,
            "Synthetic_2.5pct": "{:,.4f}".format,
            "Synthetic_97.5pct": "{:,.4f}".format,
            "Synthetic_to_Original":
                "{:.4f}".format
        }
    )
)


# ------------------------------------------------------------
# 10. COUNT HOW MANY QUANTILES ARE OUTSIDE
# ------------------------------------------------------------

outside_count = (
    ~boot_qdf["Original_inside_95pct"]
).sum()

print("\n" + "=" * 90)
print("QUANTILE OVERLAP RESULT")
print("=" * 90)

print(
    f"Original quantiles outside synthetic "
    f"95% intervals: "
    f"{outside_count}/{len(boot_qdf)}"
)


# ------------------------------------------------------------
# 11. FINAL AUTOMATIC DIAGNOSIS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("AUTOMATIC DIAGNOSIS")
print("=" * 90)

distance_ratio = (
    ks_orig_median / ks_syn_p95
    if ks_syn_p95 > 0 else np.inf
)

if outside_count <= 2 and distance_ratio <= 1.5:

    print(
        "🟢 AMOUNT DISTRIBUTION IS REASONABLY CONSISTENT."
    )
    print(
        "Most quantile differences are compatible with "
        "sample-size variation."
    )

elif outside_count <= 4:

    print(
        "🟡 MODERATE AMOUNT-DISTRIBUTION MISMATCH."
    )
    print(
        "Some differences appear systematic, but the "
        "distributions are not completely disconnected."
    )

else:

    print(
        "🔴 CLEAR AMOUNT-DISTRIBUTION MISMATCH."
    )
    print(
        "The synthetic amount distribution differs from "
        "the original laundering population beyond "
        "ordinary same-size sampling variation."
    )

print("\n✅ CELL 26B COMPLETE")


# ------------------------------------------------------------
# 12. SAVE RESULTS
# ------------------------------------------------------------

summary_stats.to_csv(
    "/kaggle/working/amount_distribution_summary.csv",
    index=False
)

raw_qdf.to_csv(
    "/kaggle/working/amount_distribution_raw_quantiles.csv",
    index=False
)

boot_qdf.to_csv(
    "/kaggle/working/amount_distribution_bootstrap_quantiles.csv",
    index=False
)

pd.DataFrame({
    "Metric": [
        "Full_KS",
        "Full_Wasserstein_log10",
        "SameSize_SYN_SYN_KS_Median",
        "SameSize_SYN_SYN_KS_P95",
        "ORIG_SYN_KS_Median",
        "ORIG_SYN_KS_95pct_Low",
        "ORIG_SYN_KS_95pct_High",
        "SameSize_SYN_SYN_WD_Median",
        "SameSize_SYN_SYN_WD_P95",
        "ORIG_SYN_WD_Median",
        "Original_Quantiles_Outside_95pct"
    ],
    "Value": [
        full_ks,
        full_wd,
        ks_syn_median,
        ks_syn_p95,
        ks_orig_median,
        ks_orig_low,
        ks_orig_high,
        wd_syn_median,
        wd_syn_p95,
        wd_orig_median,
        outside_count
    ]
}).to_csv(
    "/kaggle/working/amount_distribution_distance_summary.csv",
    index=False
)

print("\nSaved audit files:")
print("  amount_distribution_summary.csv")
print("  amount_distribution_raw_quantiles.csv")
print("  amount_distribution_bootstrap_quantiles.csv")
print("  amount_distribution_distance_summary.csv")

Original : /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv
Synthetic: /kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_FULL_transactions.csv

Loading original laundering amounts...
Original laundering transactions: 5,177

Loading synthetic laundering amounts...
  Loaded 468,825 synthetic amounts
Synthetic laundering transactions: 468,825
✅ Original laundering count exact
⚠️ Synthetic count differs from expected 5,067,991

RAW AMOUNT DISTRIBUTION
Quantile        Original                                                                                                                                                                                                                                                                Synthetic                                                                                                                                                                       Synthetic_to_Original_Ratio
     P01     

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:188: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


  Bootstrap 25/200
  Bootstrap 50/200
  Bootstrap 75/200
  Bootstrap 100/200
  Bootstrap 125/200
  Bootstrap 150/200
  Bootstrap 175/200
  Bootstrap 200/200

SAME-SIZE SAMPLE EFFECT
Synthetic vs Synthetic KS median : 0.016805
Synthetic vs Synthetic KS P95    : 0.025546
Original vs Synthetic KS median  : 0.971122
Original vs Synthetic KS 95% CI  : [0.968515, 0.973735]

Synthetic vs Synthetic WD median : 0.555801
Synthetic vs Synthetic WD P95    : 1.073852
Original vs Synthetic WD median  : 12.463797
Original vs Synthetic WD 95% CI  : [11.713539, 13.420070]

SAME-SIZE QUANTILE BOOTSTRAP
Quantile        Original                                                                                                                                                                                                                                                         Synthetic_Median                                                                                                                       

In [46]:
# ============================================================
# CELL 26C — CORRECT AMOUNT DISTRIBUTION AUDIT
# Reads "Amount Received" by COLUMN NAME
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, wasserstein_distance

ORIG_PATH = (
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Trans.csv"
)

SYN_PATH = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_SYNTH_FULL_transactions.csv"
)

# ------------------------------------------------------------
# 1. ORIGINAL — LAUNDERING ONLY
# ------------------------------------------------------------

print("=" * 90)
print("CELL 26C — CORRECT AMOUNT AUDIT")
print("=" * 90)

print("\nLoading original laundering transactions...")

orig_parts = []

for chunk in pd.read_csv(
    ORIG_PATH,
    usecols=["Amount Received", "Is Laundering"],
    chunksize=500_000,
    low_memory=False
):

    chunk["Amount Received"] = pd.to_numeric(
        chunk["Amount Received"],
        errors="coerce"
    )

    chunk["Is Laundering"] = pd.to_numeric(
        chunk["Is Laundering"],
        errors="coerce"
    )

    vals = chunk.loc[
        chunk["Is Laundering"] == 1,
        "Amount Received"
    ].dropna()

    vals = vals[vals > 0]

    orig_parts.append(
        vals.to_numpy(dtype=np.float64)
    )

orig_amounts = np.concatenate(orig_parts)

del orig_parts

print(
    f"Original laundering amounts: "
    f"{len(orig_amounts):,}"
)


# ------------------------------------------------------------
# 2. SYNTHETIC — AMOUNT RECEIVED
# ------------------------------------------------------------

print("\nLoading synthetic laundering amounts...")

syn_parts = []

for chunk_no, chunk in enumerate(
    pd.read_csv(
        SYN_PATH,
        usecols=["Amount Received"],
        chunksize=500_000,
        low_memory=False
    ),
    start=1
):

    vals = pd.to_numeric(
        chunk["Amount Received"],
        errors="coerce"
    ).dropna()

    vals = vals[vals > 0]

    syn_parts.append(
        vals.to_numpy(dtype=np.float64)
    )

    loaded = sum(
        len(x) for x in syn_parts
    )

    print(
        f"  Chunk {chunk_no:02d} | "
        f"loaded {loaded:,}",
        end="\r"
    )

synthetic_amounts = np.concatenate(
    syn_parts
)

del syn_parts

print(
    f"\nSynthetic amounts: "
    f"{len(synthetic_amounts):,}"
)


# ------------------------------------------------------------
# 3. EXPECTED COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("COUNT CHECK")
print("=" * 90)

print(f"Original expected : 5,177")
print(f"Original actual   : {len(orig_amounts):,}")

print(f"Synthetic expected: 5,067,991")
print(f"Synthetic actual  : {len(synthetic_amounts):,}")

assert len(orig_amounts) == 5_177, (
    f"Original laundering count mismatch: "
    f"{len(orig_amounts):,}"
)

assert len(synthetic_amounts) == 5_067_991, (
    f"Synthetic amount count mismatch: "
    f"{len(synthetic_amounts):,}"
)

print("\n✅ Counts exactly match expected values")


# ------------------------------------------------------------
# 4. BASIC STATISTICS
# ------------------------------------------------------------

def stats_array(x):

    return {
        "Count": len(x),
        "Mean": np.mean(x),
        "Median": np.median(x),
        "Std": np.std(x),
        "Min": np.min(x),
        "Max": np.max(x),
        "P01": np.percentile(x, 1),
        "P05": np.percentile(x, 5),
        "P10": np.percentile(x, 10),
        "P25": np.percentile(x, 25),
        "P50": np.percentile(x, 50),
        "P75": np.percentile(x, 75),
        "P90": np.percentile(x, 90),
        "P95": np.percentile(x, 95),
        "P99": np.percentile(x, 99),
    }


orig_stats = stats_array(orig_amounts)
syn_stats = stats_array(synthetic_amounts)

stat_df = pd.DataFrame({
    "Metric": list(orig_stats.keys()),
    "Original_Laundering": list(orig_stats.values()),
    "Synthetic_Laundering": [
        syn_stats[k]
        for k in orig_stats.keys()
    ]
})

print("\n" + "=" * 90)
print("CORRECT AMOUNT STATISTICS")
print("=" * 90)

print(
    stat_df.to_string(
        index=False,
        formatters={
            "Original_Laundering": "{:,.6f}".format,
            "Synthetic_Laundering": "{:,.6f}".format
        }
    )
)


# ------------------------------------------------------------
# 5. QUANTILE RATIOS
# ------------------------------------------------------------

quantiles = [
    1, 5, 10, 25, 50,
    75, 90, 95, 99
]

rows = []

for q in quantiles:

    oq = np.percentile(
        orig_amounts,
        q
    )

    sq = np.percentile(
        synthetic_amounts,
        q
    )

    rows.append({
        "Quantile": f"P{q:02d}",
        "Original": oq,
        "Synthetic": sq,
        "Synthetic_to_Original_Ratio":
            sq / oq if oq != 0 else np.nan
    })

quantile_df = pd.DataFrame(rows)

print("\n" + "=" * 90)
print("CORRECT QUANTILE COMPARISON")
print("=" * 90)

print(
    quantile_df.to_string(
        index=False,
        formatters={
            "Original": "{:,.4f}".format,
            "Synthetic": "{:,.4f}".format,
            "Synthetic_to_Original_Ratio":
                "{:.4f}".format
        }
    )
)


# ------------------------------------------------------------
# 6. LOG-SPACE FULL DISTRIBUTION TEST
# ------------------------------------------------------------

orig_log = np.log10(orig_amounts)
syn_log = np.log10(synthetic_amounts)

ks_full = ks_2samp(
    orig_log,
    syn_log
).statistic

wd_full = wasserstein_distance(
    orig_log,
    syn_log
)

print("\n" + "=" * 90)
print("FULL DISTRIBUTION DISTANCE")
print("=" * 90)

print(
    f"KS statistic                  : "
    f"{ks_full:.6f}"
)

print(
    f"Wasserstein distance (log10) : "
    f"{wd_full:.6f}"
)


# ------------------------------------------------------------
# 7. SAME-SIZE SAMPLING TEST
# ------------------------------------------------------------

N = len(orig_amounts)
BOOT = 200

rng = np.random.default_rng(42)

ks_orig_syn = []
wd_orig_syn = []

ks_syn_syn = []
wd_syn_syn = []

for i in range(BOOT):

    a = rng.choice(
        synthetic_amounts,
        size=N,
        replace=False
    )

    b = rng.choice(
        synthetic_amounts,
        size=N,
        replace=False
    )

    a_log = np.log10(a)
    b_log = np.log10(b)

    ks_orig_syn.append(
        ks_2samp(
            orig_log,
            a_log
        ).statistic
    )

    wd_orig_syn.append(
        wasserstein_distance(
            orig_log,
            a_log
        )
    )

    ks_syn_syn.append(
        ks_2samp(
            a_log,
            b_log
        ).statistic
    )

    wd_syn_syn.append(
        wasserstein_distance(
            a_log,
            b_log
        )
    )

    if (i + 1) % 25 == 0:
        print(
            f"Bootstrap {i + 1}/{BOOT}"
        )


# ------------------------------------------------------------
# 8. DISTANCE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("SAME-SIZE SAMPLE EFFECT")
print("=" * 90)

print(
    f"Synthetic vs Synthetic KS median : "
    f"{np.median(ks_syn_syn):.6f}"
)

print(
    f"Synthetic vs Synthetic KS P95    : "
    f"{np.percentile(ks_syn_syn, 95):.6f}"
)

print(
    f"Original vs Synthetic KS median  : "
    f"{np.median(ks_orig_syn):.6f}"
)

print(
    f"Original vs Synthetic KS 95% CI  : "
    f"[{np.percentile(ks_orig_syn, 2.5):.6f}, "
    f"{np.percentile(ks_orig_syn, 97.5):.6f}]"
)

print()

print(
    f"Synthetic vs Synthetic WD median : "
    f"{np.median(wd_syn_syn):.6f}"
)

print(
    f"Synthetic vs Synthetic WD P95    : "
    f"{np.percentile(wd_syn_syn, 95):.6f}"
)

print(
    f"Original vs Synthetic WD median  : "
    f"{np.median(wd_orig_syn):.6f}"
)

print(
    f"Original vs Synthetic WD 95% CI  : "
    f"[{np.percentile(wd_orig_syn, 2.5):.6f}, "
    f"{np.percentile(wd_orig_syn, 97.5):.6f}]"
)


# ------------------------------------------------------------
# 9. DIRECT SAME-SIZE QUANTILE BOOTSTRAP
# ------------------------------------------------------------

boot_rows = []

for q in quantiles:

    original_q = np.percentile(
        orig_amounts,
        q
    )

    values = []

    for _ in range(BOOT):

        sample = rng.choice(
            synthetic_amounts,
            size=N,
            replace=False
        )

        values.append(
            np.percentile(
                sample,
                q
            )
        )

    values = np.asarray(values)

    low = np.percentile(
        values,
        2.5
    )

    med = np.median(values)

    high = np.percentile(
        values,
        97.5
    )

    boot_rows.append({
        "Quantile": f"P{q:02d}",
        "Original": original_q,
        "Synthetic_Median": med,
        "Synthetic_2.5pct": low,
        "Synthetic_97.5pct": high,
        "Ratio": med / original_q
            if original_q != 0 else np.nan,
        "Original_inside_95pct":
            low <= original_q <= high
    })

boot_df = pd.DataFrame(
    boot_rows
)

print("\n" + "=" * 90)
print("SAME-SIZE QUANTILE BOOTSTRAP")
print("=" * 90)

print(
    boot_df.to_string(
        index=False,
        formatters={
            "Original": "{:,.4f}".format,
            "Synthetic_Median": "{:,.4f}".format,
            "Synthetic_2.5pct": "{:,.4f}".format,
            "Synthetic_97.5pct": "{:,.4f}".format,
            "Ratio": "{:.4f}".format
        }
    )
)


# ------------------------------------------------------------
# 10. FINAL INTERPRETATION
# ------------------------------------------------------------

outside = (
    ~boot_df["Original_inside_95pct"]
).sum()

print("\n" + "=" * 90)
print("FINAL DIAGNOSIS")
print("=" * 90)

print(
    f"Original quantiles outside synthetic "
    f"95% intervals: {outside}/9"
)

if outside <= 2:
    print(
        "🟢 AMOUNT DISTRIBUTION IS REASONABLY CLOSE."
    )
    print(
        "The remaining differences are largely "
        "compatible with sampling variation."
    )

elif outside <= 4:
    print(
        "🟡 MODERATE AMOUNT DISTRIBUTION DIFFERENCE."
    )
    print(
        "Some amount characteristics may need tuning."
    )

else:
    print(
        "🔴 GENUINE AMOUNT DISTRIBUTION MISMATCH."
    )
    print(
        "The synthetic generator's amount behavior should "
        "be improved before final model training."
    )


# ------------------------------------------------------------
# 11. SAVE
# ------------------------------------------------------------

stat_df.to_csv(
    "/kaggle/working/amount_audit_corrected_statistics.csv",
    index=False
)

quantile_df.to_csv(
    "/kaggle/working/amount_audit_corrected_quantiles.csv",
    index=False
)

boot_df.to_csv(
    "/kaggle/working/amount_audit_corrected_bootstrap.csv",
    index=False
)

print("\n✅ CELL 26C COMPLETE")

CELL 26C — CORRECT AMOUNT AUDIT

Loading original laundering transactions...
Original laundering amounts: 5,177

Loading synthetic laundering amounts...
  Chunk 11 | loaded 5,067,991
Synthetic amounts: 5,067,991

COUNT CHECK
Original expected : 5,177
Original actual   : 5,177
Synthetic expected: 5,067,991
Synthetic actual  : 5,067,991

✅ Counts exactly match expected values

CORRECT AMOUNT STATISTICS
Metric   Original_Laundering  Synthetic_Laundering
 Count          5,177.000000      5,067,991.000000
  Mean     36,135,310.412813     21,383,007.836351
Median          8,667.210000          1,465.920000
   Std  1,527,771,094.710159    584,454,802.649593
   Min              0.003227              0.000001
   Max 84,853,144,179.580002 25,747,762,666.419998
   P01              1.000002              0.014793
   P05            213.970000              3.413659
   P10            673.148000             17.142333
   P25          2,634.970000            150.798097
   P50          8,667.210000       

In [47]:
# ============================================================
# CELL 28 — AMOUNT CALIBRATION OF SYNTHETIC LAUNDERING
# ============================================================
#
# Purpose:
#   Transform synthetic Amount Received/Amount Paid so that
#   their marginal monetary distribution follows the ORIGINAL
#   HI-Small laundering distribution.
#
# Important:
#   - NO graph changes
#   - NO account changes
#   - NO bank changes
#   - NO timestamp changes
#   - NO episode changes
#   - NO typology changes
#   - Only amount fields are calibrated
#
# Method:
#   Empirical quantile mapping in log10 space.
#
# ============================================================

import os
import csv
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

ORIG_PATH = (
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Trans.csv"
)

SYN_PATH = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_SYNTH_FULL_transactions.csv"
)

OUT_PATH = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_SYNTH_FULL_V2_transactions.csv"
)

print("=" * 90)
print("CELL 28 — AMOUNT CALIBRATION")
print("=" * 90)


# ------------------------------------------------------------
# 1. LOAD ORIGINAL LAUNDERING AMOUNTS
# ------------------------------------------------------------

print("\n[1/6] Loading original laundering amounts...")

orig_parts = []

for chunk in pd.read_csv(
    ORIG_PATH,
    usecols=[5, 10],
    chunksize=500_000,
    low_memory=False
):

    chunk_amount = pd.to_numeric(
        chunk.iloc[:, 0],
        errors="coerce"
    )

    laundering = pd.to_numeric(
        chunk.iloc[:, 1],
        errors="coerce"
    )

    vals = chunk_amount[
        laundering == 1
    ].dropna()

    vals = vals[
        vals > 0
    ]

    orig_parts.append(
        vals.to_numpy(dtype=np.float64)
    )

original_laundering = np.concatenate(
    orig_parts
)

del orig_parts

assert len(original_laundering) == 5_177

original_laundering.sort()

original_log = np.log10(
    original_laundering
)

print(
    f"Original laundering amounts: "
    f"{len(original_laundering):,}"
)


# ------------------------------------------------------------
# 2. LOAD ALL SYNTHETIC AMOUNT RECEIVED VALUES
# ------------------------------------------------------------

print("\n[2/6] Loading synthetic Amount Received...")

synthetic_amount_parts = []

for chunk_id, chunk in enumerate(
    pd.read_csv(
        SYN_PATH,
        usecols=["Amount Received"],
        chunksize=500_000,
        low_memory=False
    ),
    start=1
):

    vals = pd.to_numeric(
        chunk["Amount Received"],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    synthetic_amount_parts.append(vals)

    loaded = sum(
        len(x) for x in synthetic_amount_parts
    )

    print(
        f"  chunk {chunk_id:02d} | "
        f"{loaded:,} rows",
        end="\r"
    )

synthetic_amounts = np.concatenate(
    synthetic_amount_parts
)

del synthetic_amount_parts

assert len(synthetic_amounts) == 5_067_991

print(
    f"\nSynthetic amounts: "
    f"{len(synthetic_amounts):,}"
)


# ------------------------------------------------------------
# 3. CLEAN + SORT SYNTHETIC AMOUNTS
# ------------------------------------------------------------

print("\n[3/6] Preparing empirical quantile mapping...")

if not np.all(np.isfinite(synthetic_amounts)):
    raise ValueError(
        "Synthetic Amount Received contains non-finite values."
    )

if np.any(synthetic_amounts <= 0):
    raise ValueError(
        "Synthetic Amount Received contains zero/negative values."
    )

synthetic_log_sorted = np.sort(
    np.log10(synthetic_amounts)
)

N_SYN = len(synthetic_log_sorted)
N_ORIG = len(original_log)

# Empirical probabilities associated with
# the original laundering distribution.
orig_u = (
    np.arange(N_ORIG) + 0.5
) / N_ORIG

# ------------------------------------------------------------
# 4. REPORT CURRENT VS TARGET
# ------------------------------------------------------------

print("\nCurrent synthetic amount distribution:")
print(
    f"  P01  : {np.percentile(synthetic_amounts, 1):,.4f}"
)
print(
    f"  P10  : {np.percentile(synthetic_amounts, 10):,.4f}"
)
print(
    f"  P50  : {np.percentile(synthetic_amounts, 50):,.4f}"
)
print(
    f"  P90  : {np.percentile(synthetic_amounts, 90):,.4f}"
)
print(
    f"  P99  : {np.percentile(synthetic_amounts, 99):,.4f}"
)

print("\nTarget original laundering distribution:")
print(
    f"  P01  : {np.percentile(original_laundering, 1):,.4f}"
)
print(
    f"  P10  : {np.percentile(original_laundering, 10):,.4f}"
)
print(
    f"  P50  : {np.percentile(original_laundering, 50):,.4f}"
)
print(
    f"  P90  : {np.percentile(original_laundering, 90):,.4f}"
)
print(
    f"  P99  : {np.percentile(original_laundering, 99):,.4f}"
)


# ------------------------------------------------------------
# 5. MAP EACH SYNTHETIC AMOUNT TO TARGET DISTRIBUTION
# ------------------------------------------------------------

def calibrate_amounts(
    values,
    sorted_log_values,
    target_log_values,
    target_u
):

    values = np.asarray(
        values,
        dtype=np.float64
    )

    log_values = np.log10(
        values
    )

    # Find empirical CDF position.
    left = np.searchsorted(
        sorted_log_values,
        log_values,
        side="left"
    )

    right = np.searchsorted(
        sorted_log_values,
        log_values,
        side="right"
    )

    # Mid-rank handles duplicate amounts more fairly.
    rank = (
        (left.astype(np.float64) +
         right.astype(np.float64))
        / 2.0
    )

    # Convert rank to probability.
    u = (
        (rank + 0.5)
        / len(sorted_log_values)
    )

    # Keep strictly within interpolation range.
    u = np.clip(
        u,
        target_u[0],
        target_u[-1]
    )

    # Piecewise-linear quantile mapping
    # performed in log10 space.
    mapped_log = np.interp(
        u,
        target_u,
        target_log_values
    )

    mapped = np.power(
        10.0,
        mapped_log
    )

    mapped = np.maximum(
        mapped,
        1e-6
    )

    return mapped


# Test the transformation on a small sample first.

test_sample = synthetic_amounts[:100_000]

test_mapped = calibrate_amounts(
    test_sample,
    synthetic_log_sorted,
    original_log,
    orig_u
)

if not np.all(
    np.isfinite(test_mapped)
):
    raise ValueError(
        "Calibration produced non-finite amounts."
    )

if np.any(test_mapped <= 0):
    raise ValueError(
        "Calibration produced non-positive amounts."
    )

print("\n✅ Calibration function passed sanity check")


# ------------------------------------------------------------
# 6. STREAM-REWRITE COMPLETE SYNTHETIC FILE
# ------------------------------------------------------------

print("\n[4/6] Rewriting calibrated synthetic file...")
print(f"Output: {OUT_PATH}")

# Remove previous partial output if present.
if os.path.exists(OUT_PATH):
    os.remove(OUT_PATH)

total_rows = 0

with open(
    SYN_PATH,
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as fin, open(
    OUT_PATH,
    "w",
    encoding="utf-8",
    newline=""
) as fout:

    reader = csv.reader(fin)
    writer = csv.writer(fout)

    header = next(reader)

    # Locate fields by name.
    try:
        received_idx = header.index(
            "Amount Received"
        )
    except ValueError:
        raise ValueError(
            "Amount Received column not found."
        )

    try:
        paid_idx = header.index(
            "Amount Paid"
        )
    except ValueError:
        raise ValueError(
            "Amount Paid column not found."
        )

    writer.writerow(header)

    # Process rows in Python batches so that the
    # calibration itself stays vectorized.

    BATCH_SIZE = 100_000

    rows_buffer = []
    received_buffer = []
    paid_buffer = []

    def flush_batch():

        nonlocal_dummy = None

    # Manual batch loop.
    for row in reader:

        rows_buffer.append(row)

        try:
            r = float(
                row[received_idx]
            )
        except Exception:
            raise ValueError(
                f"Invalid Amount Received: "
                f"{row[received_idx]}"
            )

        try:
            p = float(
                row[paid_idx]
            )
        except Exception:
            raise ValueError(
                f"Invalid Amount Paid: "
                f"{row[paid_idx]}"
            )

        received_buffer.append(r)
        paid_buffer.append(p)

        if len(rows_buffer) >= BATCH_SIZE:

            old_received = np.asarray(
                received_buffer,
                dtype=np.float64
            )

            old_paid = np.asarray(
                paid_buffer,
                dtype=np.float64
            )

            new_received = calibrate_amounts(
                old_received,
                synthetic_log_sorted,
                original_log,
                orig_u
            )

            # Preserve the transaction's existing
            # paid/received relationship.
            ratio = np.divide(
                old_paid,
                old_received,
                out=np.ones_like(old_paid),
                where=old_received != 0
            )

            new_paid = (
                new_received * ratio
            )

            # Protect against numerical overflow.
            new_paid = np.clip(
                new_paid,
                1e-6,
                np.finfo(np.float64).max
            )

            for j, row in enumerate(
                rows_buffer
            ):

                row[received_idx] = (
                    f"{new_received[j]:.6f}"
                )

                row[paid_idx] = (
                    f"{new_paid[j]:.6f}"
                )

                writer.writerow(row)

            total_rows += len(
                rows_buffer
            )

            rows_buffer.clear()
            received_buffer.clear()
            paid_buffer.clear()

            print(
                f"  rewritten {total_rows:,}"
                f"/5,067,991",
                end="\r"
            )

    # Flush final partial batch.
    if rows_buffer:

        old_received = np.asarray(
            received_buffer,
            dtype=np.float64
        )

        old_paid = np.asarray(
            paid_buffer,
            dtype=np.float64
        )

        new_received = calibrate_amounts(
            old_received,
            synthetic_log_sorted,
            original_log,
            orig_u
        )

        ratio = np.divide(
            old_paid,
            old_received,
            out=np.ones_like(old_paid),
            where=old_received != 0
        )

        new_paid = (
            new_received * ratio
        )

        new_paid = np.clip(
            new_paid,
            1e-6,
            np.finfo(np.float64).max
        )

        for j, row in enumerate(
            rows_buffer
        ):

            row[received_idx] = (
                f"{new_received[j]:.6f}"
            )

            row[paid_idx] = (
                f"{new_paid[j]:.6f}"
            )

            writer.writerow(row)

        total_rows += len(
            rows_buffer
        )

print(
    f"\n\nRows written: {total_rows:,}"
)

assert total_rows == 5_067_991

print("\n✅ Full calibrated synthetic file written")


# ------------------------------------------------------------
# 7. FILE SIZE
# ------------------------------------------------------------

size_gb = (
    os.path.getsize(OUT_PATH)
    / (1024 ** 3)
)

print(
    f"Output file size: "
    f"{size_gb:.2f} GB"
)

print("\n" + "=" * 90)
print("CELL 28 COMPLETE")
print("=" * 90)

print(
    "\nCreated:"
)
print(OUT_PATH)

print(
    "\nNext step will be a complete validation of:"
)
print(
    "  1. amount distribution"
)
print(
    "  2. row count"
)
print(
    "  3. graph/episode integrity"
)
print(
    "  4. timestamp preservation"
)
print(
    "  5. typology preservation"
)

CELL 28 — AMOUNT CALIBRATION

[1/6] Loading original laundering amounts...
Original laundering amounts: 5,177

[2/6] Loading synthetic Amount Received...
  chunk 11 | 5,067,991 rows
Synthetic amounts: 5,067,991

[3/6] Preparing empirical quantile mapping...

Current synthetic amount distribution:
  P01  : 0.0148
  P10  : 17.1423
  P50  : 1,465.9200
  P90  : 276,577.6935
  P99  : 59,921,433.3667

Target original laundering distribution:
  P01  : 1.0000
  P10  : 673.1480
  P50  : 8,667.2100
  P90  : 171,438.2920
  P99  : 23,420,046.8380

✅ Calibration function passed sanity check

[4/6] Rewriting calibrated synthetic file...
Output: /kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_FULL_V2_transactions.csv
  rewritten 5,000,000/5,067,991

Rows written: 5,067,991

✅ Full calibrated synthetic file written
Output file size: 0.83 GB

CELL 28 COMPLETE

Created:
/kaggle/working/AML_GNN_ONLY/data/HI-Small_SYNTH_FULL_V2_transactions.csv

Next step will be a complete validation of:
  1. amount dis

In [45]:
# ============================================================
# CELL 27 — DIAGNOSE SYNTHETIC FILE CORRUPTION
# ============================================================

import os
import csv
import numpy as np
import pandas as pd

SYN_PATH = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_SYNTH_FULL_transactions.csv"
)

print("=" * 90)
print("SYNTHETIC FILE DIAGNOSTIC")
print("=" * 90)

# ------------------------------------------------------------
# 1. RAW PHYSICAL LINE COUNT
# ------------------------------------------------------------

with open(
    SYN_PATH,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    raw_line_count = sum(1 for _ in f)

print(f"\nRaw physical lines : {raw_line_count:,}")
print("Expected data rows : 5,067,991")


# ------------------------------------------------------------
# 2. HEADER + FIRST 10 RAW ROWS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("HEADER + FIRST 10 ROWS")
print("=" * 90)

with open(
    SYN_PATH,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for i in range(11):
        line = f.readline()

        if not line:
            break

        print(f"{i:02d}: {line.rstrip()}")


# ------------------------------------------------------------
# 3. CSV STRUCTURE CHECK
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CSV STRUCTURE CHECK")
print("=" * 90)

with open(
    SYN_PATH,
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as f:

    reader = csv.reader(f)

    header = next(reader)

    print("Number of header fields:", len(header))

    for i, col in enumerate(header):
        print(f"[{i}] {col!r}")

    good = 0
    bad = 0
    first_bad = None

    for line_no, row in enumerate(reader, start=2):

        if len(row) == len(header):
            good += 1
        else:
            bad += 1

            if first_bad is None:
                first_bad = (line_no, len(row), row)

        # Enough to diagnose structural corruption
        if line_no >= 100_002:
            break

print("\nRows checked :", good + bad)
print("Good rows    :", good)
print("Bad rows     :", bad)

if first_bad is not None:
    print("\nFIRST BAD ROW:")
    print("Line number :", first_bad[0])
    print("Field count :", first_bad[1])
    print("Row        :", first_bad[2])


# ------------------------------------------------------------
# 4. READ ONLY AMOUNT COLUMN
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("AMOUNT COLUMN DIAGNOSTIC")
print("=" * 90)

amount_parts = []

for chunk in pd.read_csv(
    SYN_PATH,
    usecols=[5],
    chunksize=100_000,
    low_memory=False
):

    vals = pd.to_numeric(
        chunk.iloc[:, 0],
        errors="coerce"
    )

    amount_parts.append(vals)

amounts = pd.concat(
    amount_parts,
    ignore_index=True
)

a = amounts.dropna().to_numpy(
    dtype=np.float64
)

print(f"Rows readable by pandas : {len(amounts):,}")
print(f"NaN amounts             : {amounts.isna().sum():,}")
print(f"Finite amounts          : {np.isfinite(a).sum():,}")

finite = a[np.isfinite(a)]

if len(finite) > 0:

    print("\nBasic statistics")
    print("----------------")
    print(f"Min    : {np.min(finite):,.10g}")
    print(f"Median : {np.median(finite):,.10g}")
    print(f"Mean   : {np.mean(finite):,.10g}")
    print(f"Max    : {np.max(finite):,.10g}")

    print("\nQuantiles")
    print("---------")

    for q in [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9, 100]:

        value = np.percentile(
            finite,
            q
        )

        print(
            f"P{q:<5} : {value:,.10g}"
        )


# ------------------------------------------------------------
# 5. EXTREME VALUE COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("EXTREME AMOUNT COUNTS")
print("=" * 90)

thresholds = [
    1,
    10,
    100,
    1_000,
    10_000,
    100_000,
    1_000_000,
    10_000_000,
    100_000_000,
    1_000_000_000,
    1e12,
    1e15,
    1e20,
    1e50
]

for t in thresholds:

    count = np.sum(
        finite > t
    )

    print(
        f"> {t:>14,.0f} : {count:,}"
    )


# ------------------------------------------------------------
# 6. MOST FREQUENT AMOUNTS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("MOST COMMON EXACT AMOUNTS")
print("=" * 90)

rounded = pd.Series(
    finite
).round(8)

common = rounded.value_counts().head(20)

for value, count in common.items():

    print(
        f"{value:>30,.8f} : {count:,}"
    )


# ------------------------------------------------------------
# 7. CHECK WHETHER VALUES LOOK POWER/SCALING GENERATED
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("REPRESENTATIVE LARGE VALUES")
print("=" * 90)

large = np.sort(
    finite[
        finite > 1_000_000
    ]
)

if len(large) > 0:

    sample_idx = np.linspace(
        0,
        len(large) - 1,
        min(20, len(large)),
        dtype=int
    )

    for x in large[sample_idx]:

        print(
            f"{x:.15e}"
        )

else:
    print("No amounts > 1,000,000")

print("\n✅ CELL 27 COMPLETE")

SYNTHETIC FILE DIAGNOSTIC

Raw physical lines : 5,067,992
Expected data rows : 5,067,991

HEADER + FIRST 10 ROWS
00: transaction_id,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,is_synthetic,synthetic_episode_id,typology,source_template_id,template_source_order
01: BSX045L9N8UA3V3I,2022-09-10 23:26:00,001,800453EF0,018549,805E94300,134.510000,Rupee,134.510000,Rupee,Cash,1,1,SYN_BIPARTITE_0000000,BIPARTITE,BIPARTITE_0101,0
02: 9VSDBFX8R7GLGPK1,2022-09-02 15:14:00,001,8146E6E70,011405,807FCC2F0,470.645953,US Dollar,470.645953,US Dollar,Cheque,1,1,SYN_BIPARTITE_0000001,BIPARTITE,BIPARTITE_0233,0
03: D6L43N7BK70DUS7B,2022-09-02 00:35:00,01665,800710650,010,8001DDC50,216.758051,US Dollar,216.758051,US Dollar,ACH,1,1,SYN_BIPARTITE_0000001,BIPARTITE,BIPARTITE_0233,1
04: ELH5L6ICM5W8GDT3,2022-09-01 04:06:00,014650,807BAEB10,0113935,80576E930,9181.159823,Euro,9181.159823,Euro,Bitcoin,1,1,SYN_BIPARTITE_0

/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:1986: RuntimeWarning: overflow encountered in multiply
  values = self.values.round(decimals)  # type: ignore[union-attr]


In [48]:
# ============================================================
# CELL 29 — VALIDATE CALIBRATED SYNTHETIC V2
# ============================================================
#
# Checks:
#   1. Exact row count
#   2. Amount distribution vs original laundering
#   3. Same-size KS/Wasserstein
#   4. Quantile matching
#   5. Non-amount columns preserved
#   6. Timestamp preserved
#   7. Episode IDs preserved
#   8. Typologies preserved
#   9. Synthetic transaction IDs preserved
#  10. Graph endpoints preserved
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.stats import ks_2samp, wasserstein_distance

ORIG_PATH = (
    "/kaggle/input/datasets/ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml/"
    "HI-Small_Trans.csv"
)

V1_PATH = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_SYNTH_FULL_transactions.csv"
)

V2_PATH = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_SYNTH_FULL_V2_transactions.csv"
)

EXPECTED_SYNTH = 5_067_991
EXPECTED_ORIG_LAUNDERING = 5_177

print("=" * 100)
print("CELL 29 — CALIBRATED SYNTHETIC VALIDATION")
print("=" * 100)


# ============================================================
# 1. ORIGINAL LAUNDERING AMOUNTS
# ============================================================

print("\n[1/8] Loading original laundering amounts...")

orig_parts = []

for chunk in pd.read_csv(
    ORIG_PATH,
    usecols=["Amount Received", "Is Laundering"],
    chunksize=500_000,
    low_memory=False
):

    chunk["Amount Received"] = pd.to_numeric(
        chunk["Amount Received"],
        errors="coerce"
    )

    chunk["Is Laundering"] = pd.to_numeric(
        chunk["Is Laundering"],
        errors="coerce"
    )

    vals = chunk.loc[
        chunk["Is Laundering"] == 1,
        "Amount Received"
    ].dropna()

    vals = vals[vals > 0]

    orig_parts.append(
        vals.to_numpy(dtype=np.float64)
    )

original_amounts = np.concatenate(orig_parts)

del orig_parts

assert len(original_amounts) == EXPECTED_ORIG_LAUNDERING

print(
    f"Original laundering amounts: "
    f"{len(original_amounts):,}"
)


# ============================================================
# 2. LOAD V1 + V2 AMOUNTS
# ============================================================

print("\n[2/8] Loading V1 and V2 amounts...")

v1_amounts = []
v2_amounts = []

for v1_chunk, v2_chunk in zip(
    pd.read_csv(
        V1_PATH,
        usecols=["Amount Received"],
        chunksize=500_000,
        low_memory=False
    ),
    pd.read_csv(
        V2_PATH,
        usecols=["Amount Received"],
        chunksize=500_000,
        low_memory=False
    )
):

    v1_amounts.append(
        pd.to_numeric(
            v1_chunk["Amount Received"],
            errors="coerce"
        ).to_numpy(dtype=np.float64)
    )

    v2_amounts.append(
        pd.to_numeric(
            v2_chunk["Amount Received"],
            errors="coerce"
        ).to_numpy(dtype=np.float64)
    )

v1_amounts = np.concatenate(v1_amounts)
v2_amounts = np.concatenate(v2_amounts)

assert len(v1_amounts) == EXPECTED_SYNTH
assert len(v2_amounts) == EXPECTED_SYNTH

print(
    f"V1 synthetic amounts: "
    f"{len(v1_amounts):,}"
)

print(
    f"V2 synthetic amounts: "
    f"{len(v2_amounts):,}"
)


# ============================================================
# 3. BASIC AMOUNT STATISTICS
# ============================================================

print("\n[3/8] Amount statistics...")

def make_stats(x):

    return {
        "Mean": np.mean(x),
        "Median": np.median(x),
        "P01": np.percentile(x, 1),
        "P05": np.percentile(x, 5),
        "P10": np.percentile(x, 10),
        "P25": np.percentile(x, 25),
        "P50": np.percentile(x, 50),
        "P75": np.percentile(x, 75),
        "P90": np.percentile(x, 90),
        "P95": np.percentile(x, 95),
        "P99": np.percentile(x, 99)
    }


orig_stats = make_stats(original_amounts)
v1_stats = make_stats(v1_amounts)
v2_stats = make_stats(v2_amounts)

stats_df = pd.DataFrame({
    "Metric": list(orig_stats.keys()),
    "Original_Laundering": list(orig_stats.values()),
    "V1_Synthetic": [
        v1_stats[k] for k in orig_stats.keys()
    ],
    "V2_Calibrated": [
        v2_stats[k] for k in orig_stats.keys()
    ]
})

print(
    stats_df.to_string(
        index=False,
        formatters={
            "Original_Laundering":
                "{:,.4f}".format,
            "V1_Synthetic":
                "{:,.4f}".format,
            "V2_Calibrated":
                "{:,.4f}".format
        }
    )
)


# ============================================================
# 4. QUANTILE RATIOS
# ============================================================

print("\n" + "=" * 100)
print("QUANTILE COMPARISON")
print("=" * 100)

quantiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

q_rows = []

for q in quantiles:

    oq = np.percentile(
        original_amounts,
        q
    )

    v1q = np.percentile(
        v1_amounts,
        q
    )

    v2q = np.percentile(
        v2_amounts,
        q
    )

    q_rows.append({
        "Quantile": f"P{q:02d}",
        "Original": oq,
        "V1": v1q,
        "V2": v2q,
        "V1_to_Original":
            v1q / oq if oq != 0 else np.nan,
        "V2_to_Original":
            v2q / oq if oq != 0 else np.nan
    })

qdf = pd.DataFrame(q_rows)

print(
    qdf.to_string(
        index=False,
        formatters={
            "Original":
                "{:,.4f}".format,
            "V1":
                "{:,.4f}".format,
            "V2":
                "{:,.4f}".format,
            "V1_to_Original":
                "{:.4f}".format,
            "V2_to_Original":
                "{:.4f}".format
        }
    )
)


# ============================================================
# 5. FULL DISTRIBUTION DISTANCES
# ============================================================

print("\n" + "=" * 100)
print("DISTRIBUTION DISTANCE")
print("=" * 100)

orig_log = np.log10(
    original_amounts
)

v1_log = np.log10(
    v1_amounts
)

v2_log = np.log10(
    v2_amounts
)

v1_ks = ks_2samp(
    orig_log,
    v1_log
).statistic

v2_ks = ks_2samp(
    orig_log,
    v2_log
).statistic

v1_wd = wasserstein_distance(
    orig_log,
    v1_log
)

v2_wd = wasserstein_distance(
    orig_log,
    v2_log
)

print(
    f"Original vs V1 KS               : "
    f"{v1_ks:.6f}"
)

print(
    f"Original vs V2 KS               : "
    f"{v2_ks:.6f}"
)

print(
    f"Original vs V1 Wasserstein      : "
    f"{v1_wd:.6f}"
)

print(
    f"Original vs V2 Wasserstein      : "
    f"{v2_wd:.6f}"
)


# ============================================================
# 6. CHECK NON-AMOUNT DATA PRESERVATION
# ============================================================

print("\n" + "=" * 100)
print("STRUCTURAL PRESERVATION CHECK")
print("=" * 100)

# Columns whose values MUST NOT change.
protected_columns = [
    "transaction_id",
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Receiving Currency",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
    "is_synthetic",
    "synthetic_episode_id",
    "typology",
    "source_template_id",
    "template_source_order"
]

comparison_columns = [
    "transaction_id",
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
    "is_synthetic",
    "synthetic_episode_id",
    "typology",
    "source_template_id",
    "template_source_order"
]

diff_counts = {
    col: 0
    for col in protected_columns
}

rows_checked = 0

for v1_chunk, v2_chunk in zip(
    pd.read_csv(
        V1_PATH,
        usecols=comparison_columns,
        chunksize=100_000,
        low_memory=False
    ),
    pd.read_csv(
        V2_PATH,
        usecols=comparison_columns,
        chunksize=100_000,
        low_memory=False
    )
):

    # Make sure rows remain aligned.
    assert len(v1_chunk) == len(v2_chunk)

    for col in protected_columns:

        a = v1_chunk[col].fillna("__NA__").astype(str)
        b = v2_chunk[col].fillna("__NA__").astype(str)

        diff_counts[col] += int(
            (a != b).sum()
        )

    rows_checked += len(v1_chunk)

    print(
        f"  Checked {rows_checked:,} rows",
        end="\r"
    )

print(
    f"\nRows checked: {rows_checked:,}"
)

print("\nProtected-column differences:")

for col, count in diff_counts.items():

    status = "✅" if count == 0 else "❌"

    print(
        f"{status} {col:24s}: "
        f"{count:,}"
    )


# ============================================================
# 7. EPISODE / TYPOLOGY / ENDPOINT COUNTS
# ============================================================

print("\n" + "=" * 100)
print("EPISODE / TYPOLOGY / GRAPH PRESERVATION")
print("=" * 100)

# Use only structural columns for this comparison.
struct_cols = [
    "synthetic_episode_id",
    "typology",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Timestamp"
]

v1_struct_parts = []
v2_struct_parts = []

for v1_chunk, v2_chunk in zip(
    pd.read_csv(
        V1_PATH,
        usecols=struct_cols,
        chunksize=500_000,
        low_memory=False
    ),
    pd.read_csv(
        V2_PATH,
        usecols=struct_cols,
        chunksize=500_000,
        low_memory=False
    )
):

    v1_struct_parts.append(v1_chunk)
    v2_struct_parts.append(v2_chunk)

v1_struct = pd.concat(
    v1_struct_parts,
    ignore_index=True
)

v2_struct = pd.concat(
    v2_struct_parts,
    ignore_index=True
)

del v1_struct_parts, v2_struct_parts


# Episode count
v1_episode_count = (
    v1_struct["synthetic_episode_id"]
    .nunique()
)

v2_episode_count = (
    v2_struct["synthetic_episode_id"]
    .nunique()
)

print(
    f"V1 episodes: {v1_episode_count:,}"
)

print(
    f"V2 episodes: {v2_episode_count:,}"
)

print(
    "✅ Episode count preserved"
    if v1_episode_count == v2_episode_count
    else "❌ Episode count changed"
)


# Typology counts
v1_typ = (
    v1_struct["typology"]
    .value_counts()
    .sort_index()
)

v2_typ = (
    v2_struct["typology"]
    .value_counts()
    .sort_index()
)

typology_compare = pd.DataFrame({
    "V1": v1_typ,
    "V2": v2_typ
}).fillna(0).astype(int)

typology_compare["Difference"] = (
    typology_compare["V2"]
    - typology_compare["V1"]
)

print("\nTypology transaction counts:")
print(typology_compare.to_string())

assert np.all(
    typology_compare["Difference"] == 0
)

print(
    "\n✅ Typology transaction counts preserved"
)


# Endpoint preservation
v1_endpoints = v1_struct[
    [
        "From Bank",
        "Account",
        "To Bank",
        "Account.1"
    ]
].astype(str)

v2_endpoints = v2_struct[
    [
        "From Bank",
        "Account",
        "To Bank",
        "Account.1"
    ]
].astype(str)

endpoint_diff = (
    v1_endpoints
    != v2_endpoints
).any(axis=1).sum()

print(
    f"\nEndpoint row differences: "
    f"{endpoint_diff:,}"
)

if endpoint_diff == 0:
    print(
        "✅ All graph endpoints preserved"
    )
else:
    print(
        "❌ Graph endpoints changed"
    )


# Timestamp preservation
v1_ts = (
    pd.to_datetime(
        v1_struct["Timestamp"]
    )
)

v2_ts = (
    pd.to_datetime(
        v2_struct["Timestamp"]
    )
)

timestamp_diff = (
    v1_ts != v2_ts
).sum()

print(
    f"Timestamp differences: "
    f"{timestamp_diff:,}"
)

if timestamp_diff == 0:
    print(
        "✅ All timestamps preserved"
    )
else:
    print(
        "❌ Timestamps changed"
    )


# ============================================================
# 8. FINAL DECISION
# ============================================================

print("\n" + "=" * 100)
print("FINAL V2 DECISION")
print("=" * 100)

q_v2_ratios = qdf[
    "V2_to_Original"
].to_numpy()

# Percentage closeness in log space.
log_ratio_error = np.mean(
    np.abs(
        np.log10(
            np.maximum(
                q_v2_ratios,
                1e-12
            )
        )
    )
)

protected_changes = sum(
    diff_counts.values()
)

structure_ok = (
    endpoint_diff == 0
    and timestamp_diff == 0
    and v1_episode_count == v2_episode_count
    and np.all(
        typology_compare["Difference"] == 0
    )
)

amount_improved = (
    v2_ks < v1_ks
    and v2_wd < v1_wd
)

print(
    f"V1 KS : {v1_ks:.6f}"
)

print(
    f"V2 KS : {v2_ks:.6f}"
)

print(
    f"V1 WD : {v1_wd:.6f}"
)

print(
    f"V2 WD : {v2_wd:.6f}"
)

print(
    f"Protected-column changes: "
    f"{protected_changes:,}"
)

print(
    f"Graph/episode structure OK: "
    f"{structure_ok}"
)

print(
    f"Amount distribution improved: "
    f"{amount_improved}"
)

if (
    amount_improved
    and protected_changes == 0
    and structure_ok
):

    print(
        "\n🟢 V2 PASSES CALIBRATION VALIDATION"
    )

    print(
        "The monetary distribution improved while "
        "the AML graph structure remained unchanged."
    )

elif (
    protected_changes == 0
    and structure_ok
):

    print(
        "\n🟡 STRUCTURE IS SAFE, BUT AMOUNT IMPROVEMENT "
        "NEEDS REVIEW"
    )

else:

    print(
        "\n🔴 V2 FAILED STRUCTURAL PRESERVATION"
    )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

stats_df.to_csv(
    "/kaggle/working/"
    "amount_validation_v2_statistics.csv",
    index=False
)

qdf.to_csv(
    "/kaggle/working/"
    "amount_validation_v2_quantiles.csv",
    index=False
)

typology_compare.to_csv(
    "/kaggle/working/"
    "amount_validation_v2_typology.csv"
)

print(
    "\n✅ CELL 29 COMPLETE"
)

CELL 29 — CALIBRATED SYNTHETIC VALIDATION

[1/8] Loading original laundering amounts...
Original laundering amounts: 5,177

[2/8] Loading V1 and V2 amounts...
V1 synthetic amounts: 5,067,991
V2 synthetic amounts: 5,067,991

[3/8] Amount statistics...
Metric Original_Laundering    V1_Synthetic   V2_Calibrated
  Mean     36,135,310.4128 21,383,007.8364 35,540,072.6022
Median          8,667.2100      1,465.9200      8,667.2212
   P01              1.0000          0.0148          0.9970
   P05            213.9700          3.4137        213.9026
   P10            673.1480         17.1423        672.6370
   P25          2,634.9700        150.7981      2,634.6883
   P50          8,667.2100      1,465.9200      8,667.2212
   P75         18,832.2700     17,584.7302     18,834.5069
   P90        171,438.2920    276,577.6935    171,890.8613
   P95        943,718.0220  1,575,623.5649    953,049.3934
   P99     23,420,046.8380 59,921,433.3667 23,867,222.9808

QUANTILE COMPARISON
Quantile        Orig

In [49]:
# ============================================================
# CELL 30 — BUILD FINAL V2 BALANCED TRAINING DATASET
# ============================================================
#
# Existing final file:
#   Original HI-Small + V1 synthetic
#
# New final file:
#   Original HI-Small + V2 calibrated synthetic
#
# ONLY synthetic amount columns are replaced.
#
# Everything else is preserved:
#   transaction_id
#   Timestamp
#   From Bank
#   Account
#   To Bank
#   Account.1
#   currencies
#   payment format
#   Is Laundering
#   is_synthetic
#   synthetic_episode_id
#   typology
#   source_template_id
#   template_source_order
#
# ============================================================

import os
import csv
import shutil

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

OLD_FINAL = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_FINAL_transactions_10146336.csv"
)

V1_PATH = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_SYNTH_FULL_transactions.csv"
)

V2_PATH = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_SYNTH_FULL_V2_transactions.csv"
)

FINAL_V2 = (
    "/kaggle/working/AML_GNN_ONLY/data/"
    "HI-Small_FINAL_V2_transactions_10146336.csv"
)


EXPECTED_ORIGINAL = 5_078_345
EXPECTED_SYNTH = 5_067_991
EXPECTED_FINAL = 10_146_336


print("=" * 100)
print("CELL 30 — FINAL V2 TRAINING DATASET")
print("=" * 100)


# ------------------------------------------------------------
# 1. VERIFY FILES
# ------------------------------------------------------------

for path in [
    OLD_FINAL,
    V1_PATH,
    V2_PATH
]:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )

    size_gb = (
        os.path.getsize(path)
        / (1024 ** 3)
    )

    print(
        f"✅ {os.path.basename(path)} "
        f"({size_gb:.2f} GB)"
    )


# ------------------------------------------------------------
# 2. VALIDATE V1/V2 ALIGNMENT
# ------------------------------------------------------------

print("\n[1/5] Validating V1 ↔ V2 transaction alignment...")

checked = 0

v1_reader = pd.read_csv(
    V1_PATH,
    usecols=[
        "transaction_id",
        "Timestamp",
        "From Bank",
        "Account",
        "To Bank",
        "Account.1",
        "synthetic_episode_id",
        "typology",
        "source_template_id",
        "template_source_order"
    ],
    chunksize=200_000,
    low_memory=False
)

v2_reader = pd.read_csv(
    V2_PATH,
    usecols=[
        "transaction_id",
        "Timestamp",
        "From Bank",
        "Account",
        "To Bank",
        "Account.1",
        "synthetic_episode_id",
        "typology",
        "source_template_id",
        "template_source_order"
    ],
    chunksize=200_000,
    low_memory=False
)

for chunk_no, (v1, v2) in enumerate(
    zip(v1_reader, v2_reader),
    start=1
):

    if len(v1) != len(v2):
        raise ValueError(
            f"V1/V2 chunk length mismatch "
            f"at chunk {chunk_no}"
        )

    for col in v1.columns:

        a = (
            v1[col]
            .fillna("__NA__")
            .astype(str)
        )

        b = (
            v2[col]
            .fillna("__NA__")
            .astype(str)
        )

        mismatches = (
            a != b
        ).sum()

        if mismatches:
            raise ValueError(
                f"V1/V2 mismatch in "
                f"{col}: {mismatches:,}"
            )

    checked += len(v1)

    print(
        f"  Checked {checked:,} V1/V2 rows",
        end="\r"
    )

print(
    f"\n✅ V1/V2 alignment verified: "
    f"{checked:,} rows"
)


# ------------------------------------------------------------
# 3. REWRITE OLD FINAL FILE
# ------------------------------------------------------------

print("\n[2/5] Creating final V2 merged file...")

if os.path.exists(FINAL_V2):
    os.remove(FINAL_V2)


# We need to consume V2 only as synthetic rows
v2_stream = pd.read_csv(
    V2_PATH,
    chunksize=100_000,
    low_memory=False
)

v2_buffer = pd.DataFrame()

v2_buffer_pos = 0

synthetic_rows_replaced = 0
total_rows_written = 0
original_rows_seen = 0


def get_next_v2_rows(n):

    global v2_buffer
    global v2_buffer_pos

    parts = []

    remaining = n

    while remaining > 0:

        available = (
            len(v2_buffer)
            - v2_buffer_pos
        )

        if available == 0:

            try:
                v2_buffer = next(
                    v2_stream
                )

                v2_buffer_pos = 0

                available = len(
                    v2_buffer
                )

            except StopIteration:

                raise ValueError(
                    "V2 file ended before "
                    "all synthetic rows were consumed."
                )

        take = min(
            remaining,
            available
        )

        parts.append(
            v2_buffer.iloc[
                v2_buffer_pos:
                v2_buffer_pos + take
            ].copy()
        )

        v2_buffer_pos += take
        remaining -= take

    return pd.concat(
        parts,
        ignore_index=True
    )


# ------------------------------------------------------------
# STREAM OLD FINAL
# ------------------------------------------------------------

first_write = True

for chunk_no, final_chunk in enumerate(
    pd.read_csv(
        OLD_FINAL,
        chunksize=100_000,
        low_memory=False
    ),
    start=1
):

    if "is_synthetic" not in final_chunk.columns:
        raise ValueError(
            "Old final file does not contain "
            "'is_synthetic' column."
        )

    synthetic_mask = (
        pd.to_numeric(
            final_chunk["is_synthetic"],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
        == 1
    )

    n_synthetic = int(
        synthetic_mask.sum()
    )

    n_original = (
        len(final_chunk)
        - n_synthetic
    )

    original_rows_seen += n_original

    # --------------------------------------------------------
    # Replace only synthetic rows
    # --------------------------------------------------------

    if n_synthetic > 0:

        v2_rows = get_next_v2_rows(
            n_synthetic
        )

        # Transaction IDs must align.
        final_ids = (
            final_chunk.loc[
                synthetic_mask,
                "transaction_id"
            ]
            .astype(str)
            .to_numpy()
        )

        v2_ids = (
            v2_rows[
                "transaction_id"
            ]
            .astype(str)
            .to_numpy()
        )

        if not np.array_equal(
            final_ids,
            v2_ids
        ):
            raise ValueError(
                "Synthetic transaction IDs "
                "do not align between old "
                "final file and V2."
            )

        # Replace calibrated amounts.
        final_chunk.loc[
            synthetic_mask,
            "Amount Received"
        ] = (
            v2_rows[
                "Amount Received"
            ]
            .to_numpy()
        )

        final_chunk.loc[
            synthetic_mask,
            "Amount Paid"
        ] = (
            v2_rows[
                "Amount Paid"
            ]
            .to_numpy()
        )

        synthetic_rows_replaced += (
            n_synthetic
        )

    # --------------------------------------------------------
    # Write chunk
    # --------------------------------------------------------

    final_chunk.to_csv(
        FINAL_V2,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )

    first_write = False

    total_rows_written += len(
        final_chunk
    )

    print(
        f"  Chunk {chunk_no:03d} | "
        f"written {total_rows_written:,} | "
        f"synthetic replaced "
        f"{synthetic_rows_replaced:,}",
        end="\r"
    )


# ------------------------------------------------------------
# 4. FINAL COUNTS
# ------------------------------------------------------------

print("\n\n[3/5] Final row-count validation...")

print(
    f"Original rows retained  : "
    f"{original_rows_seen:,}"
)

print(
    f"Synthetic rows replaced : "
    f"{synthetic_rows_replaced:,}"
)

print(
    f"Total rows written      : "
    f"{total_rows_written:,}"
)

if original_rows_seen != EXPECTED_ORIGINAL:
    raise ValueError(
        f"Original row count incorrect: "
        f"{original_rows_seen:,}"
    )

if synthetic_rows_replaced != EXPECTED_SYNTH:
    raise ValueError(
        f"Synthetic row count incorrect: "
        f"{synthetic_rows_replaced:,}"
    )

if total_rows_written != EXPECTED_FINAL:
    raise ValueError(
        f"Final row count incorrect: "
        f"{total_rows_written:,}"
    )

print("✅ Final row count exact")


# ------------------------------------------------------------
# 5. CHECK FINAL FILE METADATA + SIZE
# ------------------------------------------------------------

print("\n[4/5] Checking final file...")

with open(
    FINAL_V2,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    header = f.readline().strip()

print("\nFinal header:")
print(header)

size_gb = (
    os.path.getsize(FINAL_V2)
    / (1024 ** 3)
)

print(
    f"\nFinal V2 file size: "
    f"{size_gb:.2f} GB"
)


# ------------------------------------------------------------
# 6. QUICK FINAL DISTRIBUTION CHECK
# ------------------------------------------------------------

print("\n[5/5] Quick final laundering balance check...")

counts = {
    "legitimate": 0,
    "laundering": 0,
    "synthetic": 0
}

for chunk in pd.read_csv(
    FINAL_V2,
    usecols=[
        "Is Laundering",
        "is_synthetic"
    ],
    chunksize=500_000,
    low_memory=False
):

    laundering = pd.to_numeric(
        chunk["Is Laundering"],
        errors="coerce"
    ).fillna(0)

    synthetic = pd.to_numeric(
        chunk["is_synthetic"],
        errors="coerce"
    ).fillna(0)

    counts["laundering"] += int(
        (laundering == 1).sum()
    )

    counts["legitimate"] += int(
        (laundering == 0).sum()
    )

    counts["synthetic"] += int(
        (synthetic == 1).sum()
    )


print(
    f"Legitimate : {counts['legitimate']:,}"
)

print(
    f"Laundering : {counts['laundering']:,}"
)

print(
    f"Synthetic   : {counts['synthetic']:,}"
)

expected_balance = (
    counts["legitimate"]
    == counts["laundering"]
)

if not expected_balance:
    raise ValueError(
        "Final dataset is not exactly 50:50."
    )

if counts["synthetic"] != EXPECTED_SYNTH:
    raise ValueError(
        "Synthetic count in final file is incorrect."
    )

print(
    "\n✅ FINAL DATASET IS EXACTLY 50:50"
)

print("\n" + "=" * 100)
print("CELL 30 COMPLETE")
print("=" * 100)

print(
    "\nFINAL TRAINING FILE:"
)

print(FINAL_V2)

print(
    "\nNext stage:"
)

print(
    "FINAL V2 → feature engineering → "
    "Graph Transformer training → LI-Small external evaluation"
)

CELL 30 — FINAL V2 TRAINING DATASET
✅ HI-Small_FINAL_transactions_10146336.csv (1.38 GB)
✅ HI-Small_SYNTH_FULL_transactions.csv (0.82 GB)
✅ HI-Small_SYNTH_FULL_V2_transactions.csv (0.83 GB)

[1/5] Validating V1 ↔ V2 transaction alignment...
  Checked 5,067,991 V1/V2 rows
✅ V1/V2 alignment verified: 5,067,991 rows

[2/5] Creating final V2 merged file...
  Chunk 102 | written 10,146,336 | synthetic replaced 5,067,991

[3/5] Final row-count validation...
Original rows retained  : 5,078,345
Synthetic rows replaced : 5,067,991
Total rows written      : 10,146,336
✅ Final row count exact

[4/5] Checking final file...

Final header:
transaction_id,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,is_synthetic,synthetic_episode_id,typology,source_template_id,template_source_order

Final V2 file size: 1.35 GB

[5/5] Quick final laundering balance check...
Legitimate : 5,073,168
Laundering : 5,073,168
Synthe

In [51]:
# ============================================================
# CELL 32 — CREATE FINAL V2 PATTERNS.TXT
# ============================================================
#
# Original IBM pattern episodes:
#   unchanged
#
# Synthetic pattern episodes:
#   Amount Received -> V2 calibrated amount
#   Amount Paid     -> V2 calibrated amount
#
# Everything else remains unchanged.
#
# Output:
#   HI-Small_FINAL_V2_Patterns.txt
#
# ============================================================

import os
import csv
import re
import numpy as np
import pandas as pd


# ============================================================
# PATHS
# ============================================================

DATA_DIR = "/kaggle/working/AML_GNN_ONLY/data"

PATTERN_PATH = os.path.join(
    DATA_DIR,
    "HI-Small_FINAL_Patterns.txt"
)

V1_PATH = os.path.join(
    DATA_DIR,
    "HI-Small_SYNTH_FULL_transactions.csv"
)

V2_PATH = os.path.join(
    DATA_DIR,
    "HI-Small_SYNTH_FULL_V2_transactions.csv"
)

OUTPUT_PATH = os.path.join(
    DATA_DIR,
    "HI-Small_FINAL_V2_Patterns.txt"
)

EXPECTED_ORIGINAL_PATTERN_TX = 3_209
EXPECTED_SYNTH_PATTERN_TX = 5_067_991
EXPECTED_TOTAL_PATTERN_TX = (
    EXPECTED_ORIGINAL_PATTERN_TX
    + EXPECTED_SYNTH_PATTERN_TX
)

EXPECTED_BEGIN = 585_076
EXPECTED_END = 585_076
EXPECTED_ID_LINES = 584_706


print("=" * 100)
print("CELL 32 — FINAL V2 PATTERNS.TXT")
print("=" * 100)


# ============================================================
# 1. VERIFY INPUTS
# ============================================================

for path in [
    PATTERN_PATH,
    V1_PATH,
    V2_PATH
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"\nMissing file:\n{path}"
        )

    print(
        "✅",
        os.path.basename(path)
    )


# ============================================================
# 2. LOAD V1 SAMPLE FOR FIELD IDENTIFICATION
# ============================================================

print(
    "\n[1/8] Loading first 500 V1 synthetic amounts..."
)

V1_SAMPLE_N = 500

v1_sample_df = pd.read_csv(
    V1_PATH,
    usecols=[
        "Amount Received",
        "Amount Paid"
    ],
    nrows=V1_SAMPLE_N,
    low_memory=False
)

v1_received_sample = (
    v1_sample_df[
        "Amount Received"
    ]
    .to_numpy(dtype=np.float64)
)

v1_paid_sample = (
    v1_sample_df[
        "Amount Paid"
    ]
    .to_numpy(dtype=np.float64)
)

print(
    f"V1 sample rows: "
    f"{len(v1_received_sample):,}"
)


# ============================================================
# 3. LOAD ALL V2 AMOUNTS
# ============================================================

print(
    "\n[2/8] Loading all V2 calibrated amounts..."
)

v2_received_parts = []
v2_paid_parts = []

for chunk_no, chunk in enumerate(
    pd.read_csv(
        V2_PATH,
        usecols=[
            "Amount Received",
            "Amount Paid"
        ],
        chunksize=500_000,
        low_memory=False
    ),
    start=1
):

    v2_received_parts.append(
        chunk[
            "Amount Received"
        ].to_numpy(dtype=np.float64)
    )

    v2_paid_parts.append(
        chunk[
            "Amount Paid"
        ].to_numpy(dtype=np.float64)
    )

    loaded = sum(
        len(x)
        for x in v2_received_parts
    )

    print(
        f"  Chunk {chunk_no:02d} | "
        f"{loaded:,} rows",
        end="\r"
    )

v2_received = np.concatenate(
    v2_received_parts
)

v2_paid = np.concatenate(
    v2_paid_parts
)

del v2_received_parts
del v2_paid_parts

assert len(v2_received) == EXPECTED_SYNTH_PATTERN_TX
assert len(v2_paid) == EXPECTED_SYNTH_PATTERN_TX

print(
    f"\nV2 received amounts: "
    f"{len(v2_received):,}"
)

print(
    f"V2 paid amounts: "
    f"{len(v2_paid):,}"
)


# ============================================================
# 4. PATTERN LINE CLASSIFICATION
# ============================================================

print(
    "\n[3/8] Inspecting Patterns.txt..."
)


def is_control_line(line):

    s = line.strip()

    if not s:
        return True

    upper = s.upper()

    if (
        upper.startswith("BEGIN")
        or upper.startswith("END")
    ):
        return True

    if (
        s.startswith("id=")
        or s.startswith("ID=")
    ):
        return True

    return False


def detect_delimiter(line):

    candidates = [
        ",",
        "\t",
        "|",
        ";"
    ]

    scores = {
        d: line.count(d)
        for d in candidates
    }

    best = max(
        scores,
        key=scores.get
    )

    if scores[best] > 0:
        return best

    return None


def tokenize(line, delimiter):

    body = line.rstrip(
        "\r\n"
    )

    if delimiter is None:

        return re.split(
            r"\s+",
            body.strip()
        )

    return body.split(
        delimiter
    )


def join_tokens(tokens, delimiter):

    if delimiter is None:
        return " ".join(tokens)

    return delimiter.join(tokens)


# ------------------------------------------------------------
# First pass:
# Find the first 500 synthetic transaction lines.
# ------------------------------------------------------------

sample_pattern_lines = []

pattern_tx_count = 0
original_tx_count = 0

delimiter = None

with open(
    PATTERN_PATH,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for line in f:

        if is_control_line(line):
            continue

        pattern_tx_count += 1

        # First non-control line determines delimiter
        if delimiter is None:
            delimiter = detect_delimiter(line)

        # First 3,209 transaction lines are original IBM
        if pattern_tx_count <= EXPECTED_ORIGINAL_PATTERN_TX:

            original_tx_count += 1

            continue

        # Synthetic transaction lines
        synthetic_position = (
            pattern_tx_count
            - EXPECTED_ORIGINAL_PATTERN_TX
            - 1
        )

        if synthetic_position < V1_SAMPLE_N:

            sample_pattern_lines.append(
                line.rstrip("\r\n")
            )

        if len(sample_pattern_lines) >= V1_SAMPLE_N:

            break


print(
    f"Detected delimiter: "
    f"{delimiter!r}"
)

print(
    f"Original pattern tx encountered: "
    f"{original_tx_count:,}"
)

print(
    f"Synthetic sample lines: "
    f"{len(sample_pattern_lines):,}"
)

if len(sample_pattern_lines) != V1_SAMPLE_N:

    raise ValueError(
        "Could not collect enough synthetic "
        "pattern transaction lines."
    )


# ============================================================
# 5. INFER AMOUNT POSITIONS
# ============================================================

print(
    "\n[4/8] Identifying amount fields..."
)

tokenized_samples = [
    tokenize(
        line,
        delimiter
    )
    for line in sample_pattern_lines
]

max_tokens = max(
    len(tokens)
    for tokens in tokenized_samples
)

received_scores = {}
paid_scores = {}

for token_idx in range(max_tokens):

    received_matches = 0
    paid_matches = 0

    for i, tokens in enumerate(
        tokenized_samples
    ):

        if token_idx >= len(tokens):
            continue

        token = (
            tokens[token_idx]
            .strip()
            .strip('"')
            .strip("'")
        )

        try:
            value = float(token)

        except Exception:
            continue

        if np.isclose(
            value,
            v1_received_sample[i],
            rtol=1e-6,
            atol=1e-6
        ):
            received_matches += 1

        if np.isclose(
            value,
            v1_paid_sample[i],
            rtol=1e-6,
            atol=1e-6
        ):
            paid_matches += 1

    received_scores[
        token_idx
    ] = received_matches

    paid_scores[
        token_idx
    ] = paid_matches


print("\nAmount Received candidate positions:")

for idx, score in sorted(
    received_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]:

    print(
        f"  token {idx}: "
        f"{score}/{V1_SAMPLE_N}"
    )


print("\nAmount Paid candidate positions:")

for idx, score in sorted(
    paid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]:

    print(
        f"  token {idx}: "
        f"{score}/{V1_SAMPLE_N}"
    )


# ------------------------------------------------------------
# Choose fields
# ------------------------------------------------------------

received_idx = max(
    received_scores,
    key=received_scores.get
)

received_score = received_scores[
    received_idx
]

paid_idx = max(
    paid_scores,
    key=paid_scores.get
)

paid_score = paid_scores[
    paid_idx
]

MIN_MATCH = int(
    V1_SAMPLE_N * 0.90
)

if received_score < MIN_MATCH:

    raise ValueError(
        f"Could not identify Amount Received. "
        f"Best score = "
        f"{received_score}/{V1_SAMPLE_N}"
    )


if paid_score < MIN_MATCH:

    paid_idx = None

    print(
        "\n⚠️ Could not confidently identify "
        "a separate Amount Paid field."
    )

else:

    print(
        f"\n✅ Amount Received token position: "
        f"{received_idx}"
    )

    print(
        f"   Match score: "
        f"{received_score}/{V1_SAMPLE_N}"
    )

    print(
        f"✅ Amount Paid token position: "
        f"{paid_idx}"
    )

    print(
        f"   Match score: "
        f"{paid_score}/{V1_SAMPLE_N}"
    )


# ============================================================
# 6. SHOW FIRST 5 BEFORE/AFTER VALUES
# ============================================================

print(
    "\n[5/8] Checking first five synthetic mappings..."
)

for i in range(5):

    tokens = tokenize(
        sample_pattern_lines[i],
        delimiter
    )

    old_received = float(
        tokens[received_idx]
        .strip()
        .strip('"')
        .strip("'")
    )

    print(
        f"\nSynthetic pattern {i}:"
    )

    print(
        f"  V1 pattern received : "
        f"{old_received:,.6f}"
    )

    print(
        f"  V1 CSV received     : "
        f"{v1_received_sample[i]:,.6f}"
    )

    print(
        f"  V2 CSV received     : "
        f"{v2_received[i]:,.6f}"
    )

    if paid_idx is not None:

        old_paid = float(
            tokens[paid_idx]
            .strip()
            .strip('"')
            .strip("'")
        )

        print(
            f"  V1 pattern paid     : "
            f"{old_paid:,.6f}"
        )

        print(
            f"  V1 CSV paid         : "
            f"{v1_paid_sample[i]:,.6f}"
        )

        print(
            f"  V2 CSV paid         : "
            f"{v2_paid[i]:,.6f}"
        )


# ============================================================
# 7. WRITE FINAL PATTERNS STREAMING
# ============================================================

print(
    "\n[6/8] Writing final V2 Patterns.txt..."
)

if os.path.exists(
    OUTPUT_PATH
):

    os.remove(
        OUTPUT_PATH
    )


tx_index = 0
synthetic_index = 0

begin_count = 0
end_count = 0
id_count = 0

original_tx_written = 0
synthetic_tx_written = 0

with open(
    PATTERN_PATH,
    "r",
    encoding="utf-8",
    errors="replace"
) as fin, open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8",
    newline=""
) as fout:

    for line in fin:

        stripped = line.strip()

        # ----------------------------------------------------
        # CONTROL LINE
        # ----------------------------------------------------

        if is_control_line(line):

            fout.write(
                line
            )

            upper = stripped.upper()

            if upper.startswith("BEGIN"):
                begin_count += 1

            elif upper.startswith("END"):
                end_count += 1

            elif (
                stripped.startswith("id=")
                or stripped.startswith("ID=")
            ):
                id_count += 1

            continue

        # ----------------------------------------------------
        # TRANSACTION LINE
        # ----------------------------------------------------

        tx_index += 1

        # Original IBM pattern transaction
        if tx_index <= EXPECTED_ORIGINAL_PATTERN_TX:

            fout.write(
                line
            )

            original_tx_written += 1

            continue

        # Synthetic pattern transaction
        synthetic_index = (
            tx_index
            - EXPECTED_ORIGINAL_PATTERN_TX
            - 1
        )

        if synthetic_index >= EXPECTED_SYNTH_PATTERN_TX:

            raise ValueError(
                "More synthetic pattern transaction "
                "lines than expected."
            )

        tokens = tokenize(
            line,
            delimiter
        )

        # Replace Amount Received
        tokens[
            received_idx
        ] = f"{v2_received[synthetic_index]:.6f}"

        # Replace Amount Paid
        if paid_idx is not None:

            tokens[
                paid_idx
            ] = f"{v2_paid[synthetic_index]:.6f}"

        # Preserve newline style
        newline = (
            "\r\n"
            if line.endswith("\r\n")
            else "\n"
            if line.endswith("\n")
            else ""
        )

        fout.write(
            join_tokens(
                tokens,
                delimiter
            )
            + newline
        )

        synthetic_tx_written += 1

        if (
            synthetic_tx_written % 500_000
            == 0
        ):

            print(
                f"  Updated "
                f"{synthetic_tx_written:,}/"
                f"{EXPECTED_SYNTH_PATTERN_TX:,} "
                f"synthetic pattern transactions"
            )


# ============================================================
# 8. FINAL VALIDATION
# ============================================================

print(
    "\n[7/8] Final validation..."
)

total_tx_written = (
    original_tx_written
    + synthetic_tx_written
)

print(
    f"Original pattern tx : "
    f"{original_tx_written:,}"
)

print(
    f"Synthetic pattern tx: "
    f"{synthetic_tx_written:,}"
)

print(
    f"Total pattern tx    : "
    f"{total_tx_written:,}"
)

print(
    f"BEGIN lines         : "
    f"{begin_count:,}"
)

print(
    f"END lines           : "
    f"{end_count:,}"
)

print(
    f"ID lines            : "
    f"{id_count:,}"
)


assert (
    original_tx_written
    == EXPECTED_ORIGINAL_PATTERN_TX
)

assert (
    synthetic_tx_written
    == EXPECTED_SYNTH_PATTERN_TX
)

assert (
    total_tx_written
    == EXPECTED_TOTAL_PATTERN_TX
)

assert begin_count == EXPECTED_BEGIN
assert end_count == EXPECTED_END
assert id_count == EXPECTED_ID_LINES

print(
    "\n✅ Transaction counts correct"
)

print(
    "✅ BEGIN/END counts correct"
)

print(
    "✅ Synthetic ID-line count correct"
)


# ------------------------------------------------------------
# Verify file can be reopened
# ------------------------------------------------------------

print(
    "\nChecking output file readability..."
)

output_size_mb = (
    os.path.getsize(
        OUTPUT_PATH
    ) / (1024 ** 2)
)

print(
    f"Output size: "
    f"{output_size_mb:.2f} MB"
)


# ------------------------------------------------------------
# Verify synthetic amount samples
# ------------------------------------------------------------

print(
    "\nVerifying 10,000 synthetic V2 amounts..."
)

VERIFY_N = 10_000

verify_positions = np.linspace(
    0,
    EXPECTED_SYNTH_PATTERN_TX - 1,
    VERIFY_N,
    dtype=np.int64
)

verify_set = set(
    verify_positions.tolist()
)

synthetic_counter = 0
verified = 0

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for line in f:

        if is_control_line(line):
            continue

        if tx_index == 0:
            pass

        # We need synthetic transaction position
        # based on transaction-line order.
        #
        # This counter starts at zero for all pattern
        # transactions and skips original transactions.

        if synthetic_counter >= EXPECTED_SYNTH_PATTERN_TX:
            break

        # This pass tracks synthetic position differently.
        # Actual synthetic lines begin after 3,209 originals.
        break


# More robust targeted verification:
#
# Re-read transaction lines and verify selected positions.

tx_counter = 0
checks_done = 0

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for line in f:

        if is_control_line(line):
            continue

        if tx_counter < EXPECTED_ORIGINAL_PATTERN_TX:

            tx_counter += 1
            continue

        syn_pos = (
            tx_counter
            - EXPECTED_ORIGINAL_PATTERN_TX
        )

        if syn_pos in verify_set:

            tokens = tokenize(
                line,
                delimiter
            )

            actual_received = float(
                tokens[received_idx]
                .strip()
                .strip('"')
                .strip("'")
            )

            expected_received = (
                v2_received[syn_pos]
            )

            if not np.isclose(
                actual_received,
                expected_received,
                rtol=1e-6,
                atol=1e-5
            ):

                raise ValueError(
                    f"Amount Received mismatch "
                    f"at synthetic position "
                    f"{syn_pos}"
                )

            if paid_idx is not None:

                actual_paid = float(
                    tokens[paid_idx]
                    .strip()
                    .strip('"')
                    .strip("'")
                )

                expected_paid = (
                    v2_paid[syn_pos]
                )

                if not np.isclose(
                    actual_paid,
                    expected_paid,
                    rtol=1e-6,
                    atol=1e-5
                ):

                    raise ValueError(
                        f"Amount Paid mismatch "
                        f"at synthetic position "
                        f"{syn_pos}"
                    )

            checks_done += 1

        tx_counter += 1

        if (
            tx_counter
            >= EXPECTED_TOTAL_PATTERN_TX
        ):
            break


print(
    f"✅ Verified "
    f"{checks_done:,} synthetic V2 pattern amounts"
)


# ============================================================
# FINAL RESULT
# ============================================================

print("\n" + "=" * 100)
print("✅ CELL 32 COMPLETE")
print("=" * 100)

print(
    "\nFINAL PATTERNS FILE:"
)

print(
    OUTPUT_PATH
)

print(
    f"\nFile size: "
    f"{output_size_mb:.2f} MB"
)

print(
    "\nContents:"
)

print(
    f"  Original IBM pattern transactions : "
    f"{EXPECTED_ORIGINAL_PATTERN_TX:,}"
)

print(
    f"  Synthetic V2 pattern transactions: "
    f"{EXPECTED_SYNTH_PATTERN_TX:,}"
)

print(
    f"  Total pattern transactions        : "
    f"{EXPECTED_TOTAL_PATTERN_TX:,}"
)

print(
    "\n✅ Original pattern data preserved"
)

print(
    "✅ Synthetic pattern structure preserved"
)

print(
    "✅ Synthetic V2 amounts inserted"
)

print(
    "✅ Final Patterns.txt is ready"
)

CELL 32 — FINAL V2 PATTERNS.TXT
✅ HI-Small_FINAL_Patterns.txt
✅ HI-Small_SYNTH_FULL_transactions.csv
✅ HI-Small_SYNTH_FULL_V2_transactions.csv

[1/8] Loading first 500 V1 synthetic amounts...
V1 sample rows: 500

[2/8] Loading all V2 calibrated amounts...
  Chunk 11 | 5,067,991 rows
V2 received amounts: 5,067,991
V2 paid amounts: 5,067,991

[3/8] Inspecting Patterns.txt...
Detected delimiter: ','
Original pattern tx encountered: 3,209
Synthetic sample lines: 500

[4/8] Identifying amount fields...

Amount Received candidate positions:
  token 5: 500/500
  token 7: 500/500
  token 0: 0/500
  token 1: 0/500
  token 2: 0/500
  token 3: 0/500
  token 4: 0/500
  token 6: 0/500
  token 8: 0/500
  token 9: 0/500

Amount Paid candidate positions:
  token 5: 500/500
  token 7: 500/500
  token 0: 0/500
  token 1: 0/500
  token 2: 0/500
  token 3: 0/500
  token 4: 0/500
  token 6: 0/500
  token 8: 0/500
  token 9: 0/500

✅ Amount Received token position: 5
   Match score: 500/500
✅ Amount Paid to